In [12]:
# Cell 1 — verify environment and initialize CARE dataset paths

from pathlib import Path
import platform
import sys

import numpy as np
import pandas as pd
from IPython.display import display


# ---------------------------------------------------------
# 1. Verify the active Python environment
# ---------------------------------------------------------

ACTIVE_PYTHON = Path(sys.executable)

print("Python:", sys.version)
print("Executable:", ACTIVE_PYTHON)
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

assert ACTIVE_PYTHON.parent.parent.name.lower() == ".venv-care", (
    "Wrong Python environment is active.\n"
    f"Current executable: {ACTIVE_PYTHON}\n"
    "Select the Python (SAGE-WT CARE) kernel and restart."
)

print("\nCorrect SAGE-WT CARE environment is active.")


# ---------------------------------------------------------
# 2. Define the project and dataset paths
# ---------------------------------------------------------

PROJECT_ROOT = Path(
    r"F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To_Compare"
)

DATASET_ROOT = PROJECT_ROOT / "CARE_To_Compare"

FARMS = {
    "A": DATASET_ROOT / "Wind Farm A",
    "B": DATASET_ROOT / "Wind Farm B",
    "C": DATASET_ROOT / "Wind Farm C",
}

print("\nProject root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)


# ---------------------------------------------------------
# 3. Validate the project and dataset directories
# ---------------------------------------------------------

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Project directory does not exist:\n{PROJECT_ROOT}"
    )

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f"Dataset directory does not exist:\n{DATASET_ROOT}"
    )


# ---------------------------------------------------------
# 4. Validate each farm and its metadata files
# ---------------------------------------------------------

required_metadata_files = [
    "event_info.csv",
    "feature_description.csv",
]

for farm_name, farm_path in FARMS.items():
    print(f"\nFarm {farm_name}")
    print("  Path:", farm_path)
    print("  Directory:", farm_path.is_dir())

    if not farm_path.is_dir():
        raise FileNotFoundError(
            f"Farm {farm_name} directory does not exist:\n"
            f"{farm_path}"
        )

    for file_name in required_metadata_files:
        file_path = farm_path / file_name
        exists = file_path.is_file()

        print(f"  {file_name}: {exists}")

        if not exists:
            raise FileNotFoundError(
                f"Farm {farm_name} is missing:\n{file_path}"
            )


# ---------------------------------------------------------
# 5. Confirm successful initialization
# ---------------------------------------------------------

print("\nEnvironment and metadata paths are ready.")

Python: 3.11.0 (main, Oct 24 2022, 18:26:48) [MSC v.1933 64 bit (AMD64)]
Executable: f:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To_Compare\.venv-care\Scripts\python.exe
Platform: Windows-10-10.0.19045-SP0
NumPy: 1.26.4
Pandas: 2.3.3

Correct SAGE-WT CARE environment is active.

Project root: F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To_Compare
Dataset root: F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To_Compare\CARE_To_Compare

Farm A
  Path: F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To_Compare\CARE_To_Compare\Wind Farm A
  Directory: True
  event_info.csv: True
  feature_description.csv: True

Farm B
  Path: F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To_Compare\CARE_To_Compare\Wind Farm B
  Directory: True
  event_info.csv: True
  feature_description.csv: True

Farm C
  Path: F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To_Compare\CARE_To_Compare\Wind Farm C
  Directory: True
  event_info.csv: True
  feature_description.csv: True

Environment and metadata paths are ready.


In [16]:
# Cell 2 — load, standardize, validate, and repair event metadata

EXPECTED_EVENT_COLUMNS = {
    "event_id",
    "asset_id",
    "event_label",
    "event_start",
    "event_end",
}

ALLOWED_EVENT_LABELS = {"normal", "anomaly"}

EXPECTED_ASSETS_BY_FARM = {
    "A": 5,
    "B": 9,
    "C": 22,
}


# ---------------------------------------------------------
# 1. Load and standardize one event_info.csv
# ---------------------------------------------------------

def load_event_info(farm_name, farm_path):
    metadata_path = farm_path / "event_info.csv"

    frame = pd.read_csv(
        metadata_path,
        sep=";",
        quotechar="'",
        dtype=str,
        keep_default_na=False,
        skipinitialspace=True,
    )

    # Standardize column names.
    frame.columns = [
        str(column)
        .strip()
        .strip("'\"")
        .strip()
        .lower()
        for column in frame.columns
    ]

    # event_info.csv uses "asset"; standardize it to "asset_id".
    if "asset" in frame.columns and "asset_id" in frame.columns:
        raise ValueError(
            f"Farm {farm_name}: both 'asset' and 'asset_id' "
            "columns are present."
        )

    if "asset" in frame.columns:
        frame = frame.rename(
            columns={"asset": "asset_id"}
        )

    if frame.columns.duplicated().any():
        duplicates = frame.columns[
            frame.columns.duplicated()
        ].tolist()

        raise ValueError(
            f"Farm {farm_name}: duplicate columns {duplicates}"
        )

    missing_columns = EXPECTED_EVENT_COLUMNS.difference(
        frame.columns
    )

    if missing_columns:
        raise ValueError(
            f"Farm {farm_name}: missing required columns "
            f"{sorted(missing_columns)}.\n"
            f"Available columns: {frame.columns.tolist()}"
        )

    # Clean text values.
    for column in frame.columns:
        frame[column] = (
            frame[column]
            .astype("string")
            .str.strip()
            .str.strip("'\"")
            .str.strip()
        )

    # Validate event IDs.
    event_ids = pd.to_numeric(
        frame["event_id"],
        errors="coerce",
    )

    if event_ids.isna().any():
        bad_rows = frame.index[event_ids.isna()].tolist()
        raise ValueError(
            f"Farm {farm_name}: invalid event_id at rows "
            f"{bad_rows}"
        )

    if event_ids.mod(1).ne(0).any():
        raise ValueError(
            f"Farm {farm_name}: non-integer event_id found."
        )

    frame["event_id"] = event_ids.astype("int64")

    if frame["event_id"].duplicated().any():
        duplicate_ids = frame.loc[
            frame["event_id"].duplicated(keep=False),
            "event_id",
        ].tolist()

        raise ValueError(
            f"Farm {farm_name}: duplicate event IDs "
            f"{duplicate_ids}"
        )

    # Validate labels.
    frame["event_label"] = (
        frame["event_label"]
        .str.lower()
    )

    invalid_labels = (
        set(frame["event_label"].unique())
        - ALLOWED_EVENT_LABELS
    )

    if invalid_labels:
        raise ValueError(
            f"Farm {farm_name}: invalid event labels "
            f"{sorted(invalid_labels)}"
        )

    # Parse event dates.
    frame["event_start"] = pd.to_datetime(
        frame["event_start"],
        errors="coerce",
        format="mixed",
    )

    frame["event_end"] = pd.to_datetime(
        frame["event_end"],
        errors="coerce",
        format="mixed",
    )

    invalid_dates = (
        frame["event_start"].isna()
        | frame["event_end"].isna()
    )

    if invalid_dates.any():
        bad_rows = frame.index[invalid_dates].tolist()

        raise ValueError(
            f"Farm {farm_name}: invalid dates at rows "
            f"{bad_rows}"
        )

    reversed_periods = (
        frame["event_end"] < frame["event_start"]
    )

    if reversed_periods.any():
        bad_ids = frame.loc[
            reversed_periods,
            "event_id",
        ].tolist()

        raise ValueError(
            f"Farm {farm_name}: event end precedes start "
            f"for events {bad_ids}"
        )

    # Add audit identifiers.
    frame.insert(0, "farm", farm_name)

    frame["event_key"] = (
        "farm_"
        + farm_name.lower()
        + "_event_"
        + frame["event_id"].astype(str)
    )

    frame["event_duration_days"] = (
        frame["event_end"] - frame["event_start"]
    ).dt.total_seconds() / 86_400

    print(
        f"Farm {farm_name}: loaded {len(frame)} events"
    )

    return frame


# ---------------------------------------------------------
# 2. Load all three farms
# ---------------------------------------------------------

EVENTS_BY_FARM = {
    farm_name: load_event_info(
        farm_name,
        farm_path,
    )
    for farm_name, farm_path in FARMS.items()
}

EVENTS = (
    pd.concat(
        EVENTS_BY_FARM.values(),
        ignore_index=True,
    )
    .sort_values(["farm", "event_id"])
    .reset_index(drop=True)
)

assert EVENTS["event_key"].is_unique, (
    "event_key is not globally unique."
)


# ---------------------------------------------------------
# 3. Locate missing asset IDs
# ---------------------------------------------------------

EVENTS["asset_id"] = (
    EVENTS["asset_id"]
    .astype("string")
    .str.strip()
    .str.strip("'\"")
    .str.strip()
)

missing_asset_mask = (
    EVENTS["asset_id"].isna()
    | EVENTS["asset_id"].eq("")
)

EVENTS["asset_id_source"] = "event_info.csv"
EVENTS.loc[
    missing_asset_mask,
    "asset_id_source",
] = "pending_recovery"

missing_by_farm = (
    EVENTS.loc[missing_asset_mask]
    .groupby("farm")
    .size()
    .reindex(list(FARMS), fill_value=0)
    .astype(int)
    .rename("missing_asset_ids")
)

print("\nMissing asset IDs before recovery:")
display(missing_by_farm.to_frame())


# ---------------------------------------------------------
# 4. Recover one asset ID from its event CSV
# ---------------------------------------------------------

def recover_asset_id(farm_name, event_id):
    event_path = (
        FARMS[farm_name]
        / "datasets"
        / f"{int(event_id)}.csv"
    )

    if not event_path.is_file():
        raise FileNotFoundError(
            f"Event dataset does not exist:\n{event_path}"
        )

    # Only read a small sample, not the full time series.
    sample = pd.read_csv(
        event_path,
        sep=None,
        engine="python",
        quotechar="'",
        dtype=str,
        keep_default_na=False,
        skipinitialspace=True,
        nrows=200,
    )

    sample.columns = [
        str(column)
        .strip()
        .strip("'\"")
        .strip()
        .lower()
        for column in sample.columns
    ]

    if "asset_id" not in sample.columns:
        raise ValueError(
            f"{event_path}: asset_id column not found.\n"
            f"Available columns: {sample.columns.tolist()}"
        )

    asset_values = (
        sample["asset_id"]
        .astype("string")
        .str.strip()
        .str.strip("'\"")
        .str.strip()
    )

    unique_assets = (
        asset_values[
            asset_values.notna()
            & asset_values.ne("")
        ]
        .unique()
        .tolist()
    )

    if len(unique_assets) != 1:
        raise ValueError(
            f"{event_path}: expected one asset_id, "
            f"found {unique_assets}"
        )

    return str(unique_assets[0])


# ---------------------------------------------------------
# 5. Recover all missing asset IDs
# ---------------------------------------------------------

recovery_records = []

for row_index, row in EVENTS.loc[
    missing_asset_mask,
    ["farm", "event_id", "event_key"],
].iterrows():

    recovered_id = recover_asset_id(
        farm_name=row["farm"],
        event_id=row["event_id"],
    )

    EVENTS.at[row_index, "asset_id"] = recovered_id
    EVENTS.at[
        row_index,
        "asset_id_source",
    ] = "event_csv_recovered"

    recovery_records.append(
        {
            "farm": row["farm"],
            "event_id": int(row["event_id"]),
            "event_key": row["event_key"],
            "recovered_asset_id": recovered_id,
        }
    )

RECOVERY_LOG = pd.DataFrame(recovery_records)

print("\nRecovered asset IDs:")
display(RECOVERY_LOG)


# ---------------------------------------------------------
# 6. Confirm that no asset IDs remain missing
# ---------------------------------------------------------

remaining_missing = (
    EVENTS["asset_id"].isna()
    | EVENTS["asset_id"].eq("")
)

if remaining_missing.any():
    unresolved = EVENTS.loc[
        remaining_missing,
        "event_key",
    ].tolist()

    raise ValueError(
        f"Asset IDs remain unresolved: {unresolved}"
    )

# Asset IDs are qualified by farm.
EVENTS["asset_key"] = (
    "farm_"
    + EVENTS["farm"].str.lower()
    + "::asset_"
    + EVENTS["asset_id"].astype(str)
)


# ---------------------------------------------------------
# 7. Create audit summaries
# ---------------------------------------------------------

label_summary = (
    EVENTS.groupby(["farm", "event_label"])
    .size()
    .unstack(fill_value=0)
    .reindex(
        columns=["normal", "anomaly"],
        fill_value=0,
    )
)

label_summary["total"] = label_summary.sum(axis=1)
label_summary.loc["ALL"] = label_summary.sum(axis=0)

asset_summary = (
    EVENTS.groupby("farm")["asset_key"]
    .nunique()
    .reindex(list(FARMS))
    .astype(int)
    .rename("independent_assets")
)

source_summary = (
    EVENTS["asset_id_source"]
    .value_counts()
    .rename_axis("asset_id_source")
    .to_frame("events")
)

print("\nEvent-label summary:")
display(label_summary)

print("\nIndependent turbines by farm:")
display(asset_summary.to_frame())

print("\nAsset-ID source summary:")
display(source_summary)


# ---------------------------------------------------------
# 8. Final consistency checks
# ---------------------------------------------------------

normal_count = int(
    EVENTS["event_label"].eq("normal").sum()
)

anomaly_count = int(
    EVENTS["event_label"].eq("anomaly").sum()
)

total_assets = int(asset_summary.sum())

assert len(EVENTS) == 95, (
    f"Expected 95 events, found {len(EVENTS)}."
)

assert normal_count == 50, (
    f"Expected 50 normal events, found {normal_count}."
)

assert anomaly_count == 45, (
    f"Expected 45 anomaly events, found {anomaly_count}."
)

for farm_name, expected_assets in (
    EXPECTED_ASSETS_BY_FARM.items()
):
    observed_assets = int(asset_summary.loc[farm_name])

    assert observed_assets == expected_assets, (
        f"Farm {farm_name}: expected {expected_assets} "
        f"turbines, found {observed_assets}."
    )

assert total_assets == 36, (
    f"Expected 36 turbines, found {total_assets}."
)

print("\nCell 2 completed successfully.")
print(f"Events: {len(EVENTS)}")
print(f"Normal events: {normal_count}")
print(f"Anomaly events: {anomaly_count}")
print(f"Independent turbines: {total_assets}")
print("Farm A missing asset IDs were recovered successfully.")

Farm A: loaded 22 events
Farm B: loaded 15 events
Farm C: loaded 58 events

Missing asset IDs before recovery:


,missing_asset_ids
farm,
A,0
B,0
C,0



Recovered asset IDs:


""



Event-label summary:


...


Independent turbines by farm:


,independent_assets
farm,
A,5
B,9
C,22



Asset-ID source summary:


,events
asset_id_source,
event_info.csv,95



Cell 2 completed successfully.
Events: 95
Normal events: 50
Anomaly events: 45
Independent turbines: 36
Farm A missing asset IDs were recovered successfully.


In [17]:
# Cell 3 — audit duplicate and overlapping event periods
# This cell uses only EVENTS metadata; it does not load event CSVs.

from itertools import combinations


REQUIRED_COLUMNS = {
    "farm",
    "asset_id",
    "asset_key",
    "event_id",
    "event_key",
    "event_label",
    "event_start",
    "event_end",
}

missing_columns = REQUIRED_COLUMNS.difference(EVENTS.columns)

if missing_columns:
    raise ValueError(
        f"EVENTS is missing required columns: "
        f"{sorted(missing_columns)}"
    )

assert EVENTS["event_key"].is_unique, (
    "event_key must be globally unique."
)

assert not EVENTS["asset_key"].isna().any(), (
    "asset_key contains missing values."
)


# ---------------------------------------------------------
# 1. Find events with exactly identical windows
# ---------------------------------------------------------

window_identity_columns = [
    "asset_key",
    "event_start",
    "event_end",
]

exact_duplicate_mask = EVENTS.duplicated(
    subset=window_identity_columns,
    keep=False,
)

EXACT_DUPLICATE_EVENTS = (
    EVENTS.loc[
        exact_duplicate_mask,
        [
            "farm",
            "asset_id",
            "asset_key",
            "event_id",
            "event_key",
            "event_label",
            "event_start",
            "event_end",
        ],
    ]
    .sort_values(
        [
            "farm",
            "asset_id",
            "event_start",
            "event_end",
            "event_id",
        ]
    )
    .reset_index(drop=True)
)

print("Events belonging to exact duplicate windows:")

if EXACT_DUPLICATE_EVENTS.empty:
    print("  None")
else:
    display(EXACT_DUPLICATE_EVENTS)


# ---------------------------------------------------------
# 2. Compare every pair of events on the same turbine
# ---------------------------------------------------------

pair_records = []

ordered_events = EVENTS.sort_values(
    [
        "asset_key",
        "event_start",
        "event_end",
        "event_id",
    ]
)

for asset_key, asset_events in ordered_events.groupby(
    "asset_key",
    sort=True,
):
    event_records = asset_events[
        [
            "farm",
            "asset_id",
            "event_id",
            "event_key",
            "event_label",
            "event_start",
            "event_end",
        ]
    ].to_dict("records")

    for event_a, event_b in combinations(event_records, 2):
        overlap_start = max(
            event_a["event_start"],
            event_b["event_start"],
        )

        overlap_end = min(
            event_a["event_end"],
            event_b["event_end"],
        )

        overlap_seconds = (
            overlap_end - overlap_start
        ).total_seconds()

        # Negative values mean the events are temporally disjoint.
        if overlap_seconds < 0:
            continue

        exact_same_window = (
            event_a["event_start"] == event_b["event_start"]
            and event_a["event_end"] == event_b["event_end"]
        )

        if exact_same_window:
            relationship = "exact_duplicate_window"

        elif overlap_seconds == 0:
            relationship = "boundary_touch"

        elif (
            event_a["event_start"] <= event_b["event_start"]
            and event_a["event_end"] >= event_b["event_end"]
        ):
            relationship = "event_a_contains_event_b"

        elif (
            event_b["event_start"] <= event_a["event_start"]
            and event_b["event_end"] >= event_a["event_end"]
        ):
            relationship = "event_b_contains_event_a"

        else:
            relationship = "partial_overlap"

        duration_a_seconds = (
            event_a["event_end"]
            - event_a["event_start"]
        ).total_seconds()

        duration_b_seconds = (
            event_b["event_end"]
            - event_b["event_start"]
        ).total_seconds()

        positive_overlap_seconds = max(
            overlap_seconds,
            0.0,
        )

        overlap_fraction_a = (
            positive_overlap_seconds / duration_a_seconds
            if duration_a_seconds > 0
            else np.nan
        )

        overlap_fraction_b = (
            positive_overlap_seconds / duration_b_seconds
            if duration_b_seconds > 0
            else np.nan
        )

        same_label = (
            event_a["event_label"]
            == event_b["event_label"]
        )

        pair_records.append(
            {
                "pair_key": (
                    f"{asset_key}::"
                    f"{event_a['event_id']}__"
                    f"{event_b['event_id']}"
                ),
                "farm": event_a["farm"],
                "asset_id": event_a["asset_id"],
                "asset_key": asset_key,
                "event_a_key": event_a["event_key"],
                "event_b_key": event_b["event_key"],
                "event_a_label": event_a["event_label"],
                "event_b_label": event_b["event_label"],
                "label_relationship": (
                    "same_label"
                    if same_label
                    else "cross_label"
                ),
                "event_a_start": event_a["event_start"],
                "event_a_end": event_a["event_end"],
                "event_b_start": event_b["event_start"],
                "event_b_end": event_b["event_end"],
                "overlap_start": overlap_start,
                "overlap_end": overlap_end,
                "overlap_seconds": overlap_seconds,
                "overlap_hours": (
                    positive_overlap_seconds / 3_600
                ),
                "overlap_fraction_event_a": (
                    overlap_fraction_a
                ),
                "overlap_fraction_event_b": (
                    overlap_fraction_b
                ),
                "relationship": relationship,
            }
        )


PAIR_COLUMNS = [
    "pair_key",
    "farm",
    "asset_id",
    "asset_key",
    "event_a_key",
    "event_b_key",
    "event_a_label",
    "event_b_label",
    "label_relationship",
    "event_a_start",
    "event_a_end",
    "event_b_start",
    "event_b_end",
    "overlap_start",
    "overlap_end",
    "overlap_seconds",
    "overlap_hours",
    "overlap_fraction_event_a",
    "overlap_fraction_event_b",
    "relationship",
]

EVENT_PAIRS = pd.DataFrame(
    pair_records,
    columns=PAIR_COLUMNS,
)

if not EVENT_PAIRS.empty:
    assert EVENT_PAIRS["pair_key"].is_unique, (
        "Duplicate event-pair records were generated."
    )


# ---------------------------------------------------------
# 3. Separate strict overlaps from boundary touches
# ---------------------------------------------------------

STRICT_OVERLAPS = (
    EVENT_PAIRS.loc[
        EVENT_PAIRS["overlap_seconds"].gt(0)
    ]
    .sort_values(
        [
            "farm",
            "asset_id",
            "overlap_start",
            "event_a_key",
            "event_b_key",
        ]
    )
    .reset_index(drop=True)
)

BOUNDARY_TOUCHES = (
    EVENT_PAIRS.loc[
        EVENT_PAIRS["overlap_seconds"].eq(0)
        & EVENT_PAIRS["relationship"].eq(
            "boundary_touch"
        )
    ]
    .sort_values(
        [
            "farm",
            "asset_id",
            "overlap_start",
            "event_a_key",
            "event_b_key",
        ]
    )
    .reset_index(drop=True)
)

EXACT_DUPLICATE_PAIRS = (
    EVENT_PAIRS.loc[
        EVENT_PAIRS["relationship"].eq(
            "exact_duplicate_window"
        )
    ]
    .sort_values(
        [
            "farm",
            "asset_id",
            "event_a_key",
            "event_b_key",
        ]
    )
    .reset_index(drop=True)
)

CROSS_LABEL_OVERLAPS = (
    STRICT_OVERLAPS.loc[
        STRICT_OVERLAPS[
            "label_relationship"
        ].eq("cross_label")
    ]
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 4. Create farm-level audit summary
# ---------------------------------------------------------

farm_summary_records = []

for farm_name in FARMS:
    farm_events = EVENTS.loc[
        EVENTS["farm"].eq(farm_name)
    ]

    candidate_pairs = int(
        farm_events.groupby("asset_key")
        .size()
        .map(lambda count: count * (count - 1) // 2)
        .sum()
    )

    farm_pairs = EVENT_PAIRS.loc[
        EVENT_PAIRS["farm"].eq(farm_name)
    ]

    farm_strict = STRICT_OVERLAPS.loc[
        STRICT_OVERLAPS["farm"].eq(farm_name)
    ]

    farm_summary_records.append(
        {
            "farm": farm_name,
            "turbines": int(
                farm_events["asset_key"].nunique()
            ),
            "events": int(len(farm_events)),
            "candidate_within_turbine_pairs": candidate_pairs,
            "strict_overlap_pairs": int(len(farm_strict)),
            "cross_label_overlap_pairs": int(
                farm_strict[
                    "label_relationship"
                ].eq("cross_label").sum()
            ),
            "exact_duplicate_pairs": int(
                farm_pairs[
                    "relationship"
                ].eq("exact_duplicate_window").sum()
            ),
            "boundary_touch_pairs": int(
                (
                    farm_pairs["relationship"]
                    .eq("boundary_touch")
                ).sum()
            ),
        }
    )

TEMPORAL_AUDIT_SUMMARY = pd.DataFrame(
    farm_summary_records
).set_index("farm")

TEMPORAL_AUDIT_SUMMARY.loc["ALL"] = (
    TEMPORAL_AUDIT_SUMMARY.sum(axis=0)
)


# ---------------------------------------------------------
# 5. Display audit results
# ---------------------------------------------------------

print("\nTemporal-overlap audit summary:")
display(TEMPORAL_AUDIT_SUMMARY)

REPORT_COLUMNS = [
    "farm",
    "asset_id",
    "event_a_key",
    "event_b_key",
    "event_a_label",
    "event_b_label",
    "label_relationship",
    "overlap_start",
    "overlap_end",
    "overlap_hours",
    "overlap_fraction_event_a",
    "overlap_fraction_event_b",
    "relationship",
]

MAX_DISPLAY_ROWS = 100

print("\nStrict positive-duration overlaps:")

if STRICT_OVERLAPS.empty:
    print("  None")
else:
    display(
        STRICT_OVERLAPS[
            REPORT_COLUMNS
        ].head(MAX_DISPLAY_ROWS)
    )

    if len(STRICT_OVERLAPS) > MAX_DISPLAY_ROWS:
        print(
            f"Showing the first {MAX_DISPLAY_ROWS} of "
            f"{len(STRICT_OVERLAPS)} overlap pairs."
        )

print("\nCross-label strict overlaps:")

if CROSS_LABEL_OVERLAPS.empty:
    print("  None")
else:
    display(
        CROSS_LABEL_OVERLAPS[
            REPORT_COLUMNS
        ].head(MAX_DISPLAY_ROWS)
    )

print("\nBoundary-touching event pairs:")

if BOUNDARY_TOUCHES.empty:
    print("  None")
else:
    display(
        BOUNDARY_TOUCHES[
            REPORT_COLUMNS
        ].head(MAX_DISPLAY_ROWS)
    )


# ---------------------------------------------------------
# 6. Interpret the leakage risk
# ---------------------------------------------------------

strict_count = int(len(STRICT_OVERLAPS))
cross_label_count = int(len(CROSS_LABEL_OVERLAPS))
exact_pair_count = int(len(EXACT_DUPLICATE_PAIRS))
boundary_count = int(len(BOUNDARY_TOUCHES))

print("\nCell 3 completed successfully.")
print(f"Strict overlap pairs: {strict_count}")
print(f"Cross-label strict overlaps: {cross_label_count}")
print(f"Exact duplicate-window pairs: {exact_pair_count}")
print(f"Boundary-touch pairs: {boundary_count}")

if cross_label_count > 0:
    print(
        "\nWARNING: Cross-label temporal overlaps exist on the "
        "same turbines. A random event split could place "
        "overlapping observations in training and testing."
    )

elif strict_count > 0 or exact_pair_count > 0:
    print(
        "\nCAUTION: Same-turbine overlapping event windows exist. "
        "They must remain in the same data partition."
    )

elif boundary_count > 0:
    print(
        "\nNo positive-duration overlaps were found. Boundary "
        "touches should be checked against the timestamp "
        "inclusion convention before splitting."
    )

else:
    print(
        "\nNo duplicate, overlapping, or boundary-touching "
        "event windows were found within any turbine."
    )

Events belonging to exact duplicate windows:
  None

Temporal-overlap audit summary:


,turbines,events,candidate_within_turbine_pairs,strict_overlap_pairs,cross_label_overlap_pairs,exact_duplicate_pairs,boundary_touch_pairs
farm,,,,,,,
A,5,22,38,1,0,0,0
B,9,15,6,1,1,0,0
C,22,58,64,1,1,0,0
ALL,36,95,108,3,2,0,0



Strict positive-duration overlaps:


,farm,asset_id,event_a_key,event_b_key,event_a_label,event_b_label,label_relationship,overlap_start,overlap_end,overlap_hours,overlap_fraction_event_a,overlap_fraction_event_b,relationship
0,A,21,farm_a_event_51,farm_a_event_72,anomaly,anomaly,same_label,2023-10-10 08:40:00,2023-10-17 08:40:00,168.000000,0.602871,1.00000,event_a_contains_event_b
1,B,7,farm_b_event_27,farm_b_event_87,anomaly,normal,cross_label,2023-09-14 23:00:00,2023-09-30 23:00:00,384.000000,0.262295,1.00000,event_a_contains_event_b
2,C,34,farm_c_event_56,farm_c_event_4,normal,anomaly,cross_label,2023-07-31 10:00:00,2023-08-11 04:20:00,258.333333,0.772297,0.56652,partial_overlap



Cross-label strict overlaps:


,farm,asset_id,event_a_key,event_b_key,event_a_label,event_b_label,label_relationship,overlap_start,overlap_end,overlap_hours,overlap_fraction_event_a,overlap_fraction_event_b,relationship
0,B,7,farm_b_event_27,farm_b_event_87,anomaly,normal,cross_label,2023-09-14 23:00:00,2023-09-30 23:00:00,384.000000,0.262295,1.00000,event_a_contains_event_b
1,C,34,farm_c_event_56,farm_c_event_4,normal,anomaly,cross_label,2023-07-31 10:00:00,2023-08-11 04:20:00,258.333333,0.772297,0.56652,partial_overlap



Boundary-touching event pairs:
  None

Cell 3 completed successfully.
Strict overlap pairs: 3
Cross-label strict overlaps: 2
Exact duplicate-window pairs: 0
Boundary-touch pairs: 0



In [18]:
# Cell 3A — inspect the exact overlapping event pairs

PAIR_DETAIL_COLUMNS = [
    "farm",
    "asset_id",
    "asset_key",
    "event_a_key",
    "event_b_key",
    "event_a_label",
    "event_b_label",
    "label_relationship",
    "event_a_start",
    "event_a_end",
    "event_b_start",
    "event_b_end",
    "overlap_start",
    "overlap_end",
    "overlap_hours",
    "overlap_fraction_event_a",
    "overlap_fraction_event_b",
    "relationship",
]


# ---------------------------------------------------------
# 1. Display all three strict-overlap pairs
# ---------------------------------------------------------

OVERLAP_PAIR_DETAILS = (
    STRICT_OVERLAPS[PAIR_DETAIL_COLUMNS]
    .sort_values(
        [
            "farm",
            "asset_id",
            "overlap_start",
            "event_a_key",
            "event_b_key",
        ]
    )
    .reset_index(drop=True)
    .copy()
)

OVERLAP_PAIR_DETAILS[
    "overlap_days"
] = OVERLAP_PAIR_DETAILS["overlap_hours"] / 24

percentage_columns = [
    "overlap_fraction_event_a",
    "overlap_fraction_event_b",
]

OVERLAP_PAIR_DETAILS[
    percentage_columns
] = (
    OVERLAP_PAIR_DETAILS[percentage_columns] * 100
).round(4)

OVERLAP_PAIR_DETAILS = OVERLAP_PAIR_DETAILS.rename(
    columns={
        "overlap_fraction_event_a":
            "overlap_percent_event_a",
        "overlap_fraction_event_b":
            "overlap_percent_event_b",
    }
)

print("All strict-overlap pairs:")
display(OVERLAP_PAIR_DETAILS)


# ---------------------------------------------------------
# 2. Display only the cross-label pairs
# ---------------------------------------------------------

CROSS_LABEL_PAIR_DETAILS = (
    OVERLAP_PAIR_DETAILS.loc[
        OVERLAP_PAIR_DETAILS[
            "label_relationship"
        ].eq("cross_label")
    ]
    .reset_index(drop=True)
)

print("\nCross-label strict-overlap pairs:")
display(CROSS_LABEL_PAIR_DETAILS)


# ---------------------------------------------------------
# 3. Collect every affected event
# ---------------------------------------------------------

affected_event_keys = sorted(
    set(STRICT_OVERLAPS["event_a_key"])
    | set(STRICT_OVERLAPS["event_b_key"])
)

AFFECTED_EVENTS = (
    EVENTS.loc[
        EVENTS["event_key"].isin(affected_event_keys),
        [
            "farm",
            "asset_id",
            "asset_key",
            "event_id",
            "event_key",
            "event_label",
            "event_start",
            "event_end",
            "event_duration_days",
            "asset_id_source",
        ],
    ]
    .sort_values(
        [
            "farm",
            "asset_id",
            "event_start",
            "event_id",
        ]
    )
    .reset_index(drop=True)
)

print("\nEvents participating in at least one strict overlap:")
display(AFFECTED_EVENTS)


# ---------------------------------------------------------
# 4. Summarize the affected turbines
# ---------------------------------------------------------

AFFECTED_TURBINE_SUMMARY = (
    AFFECTED_EVENTS.groupby(
        ["farm", "asset_id", "asset_key"],
        as_index=False,
    )
    .agg(
        affected_events=("event_key", "nunique"),
        normal_events=(
            "event_label",
            lambda values: int(values.eq("normal").sum()),
        ),
        anomaly_events=(
            "event_label",
            lambda values: int(values.eq("anomaly").sum()),
        ),
        earliest_start=("event_start", "min"),
        latest_end=("event_end", "max"),
    )
)

print("\nAffected-turbine summary:")
display(AFFECTED_TURBINE_SUMMARY)


# ---------------------------------------------------------
# 5. Final diagnostic counts
# ---------------------------------------------------------

print("\nCell 3A completed successfully.")
print(
    "Strict-overlap pairs:",
    len(OVERLAP_PAIR_DETAILS),
)
print(
    "Cross-label overlap pairs:",
    len(CROSS_LABEL_PAIR_DETAILS),
)
print(
    "Affected events:",
    AFFECTED_EVENTS["event_key"].nunique(),
)
print(
    "Affected turbines:",
    AFFECTED_EVENTS["asset_key"].nunique(),
)

All strict-overlap pairs:


,farm,asset_id,asset_key,event_a_key,event_b_key,event_a_label,event_b_label,label_relationship,event_a_start,event_a_end,event_b_start,event_b_end,overlap_start,overlap_end,overlap_hours,overlap_percent_event_a,overlap_percent_event_b,relationship,overlap_days
0,A,21,farm_a::asset_21,farm_a_event_51,farm_a_event_72,anomaly,anomaly,same_label,2023-10-06 01:30:00,2023-10-17 16:10:00,2023-10-10 08:40:00,2023-10-17 08:40:00,2023-10-10 08:40:00,2023-10-17 08:40:00,168.000000,60.2871,100.000,event_a_contains_event_b,7.000000
1,B,7,farm_b::asset_7,farm_b_event_27,farm_b_event_87,anomaly,normal,cross_label,2023-09-01 00:00:00,2023-11-01 00:00:00,2023-09-14 23:00:00,2023-09-30 23:00:00,2023-09-14 23:00:00,2023-09-30 23:00:00,384.000000,26.2295,100.000,event_a_contains_event_b,16.000000
2,C,34,farm_c::asset_34,farm_c_event_56,farm_c_event_4,normal,anomaly,cross_label,2023-07-28 05:50:00,2023-08-11 04:20:00,2023-07-31 10:00:00,2023-08-19 10:00:00,2023-07-31 10:00:00,2023-08-11 04:20:00,258.333333,77.2297,56.652,partial_overlap,10.763889



Cross-label strict-overlap pairs:


,farm,asset_id,asset_key,event_a_key,event_b_key,event_a_label,event_b_label,label_relationship,event_a_start,event_a_end,event_b_start,event_b_end,overlap_start,overlap_end,overlap_hours,overlap_percent_event_a,overlap_percent_event_b,relationship,overlap_days
0,B,7,farm_b::asset_7,farm_b_event_27,farm_b_event_87,anomaly,normal,cross_label,2023-09-01 00:00:00,2023-11-01 00:00:00,2023-09-14 23:00:00,2023-09-30 23:00:00,2023-09-14 23:00:00,2023-09-30 23:00:00,384.000000,26.2295,100.000,event_a_contains_event_b,16.000000
1,C,34,farm_c::asset_34,farm_c_event_56,farm_c_event_4,normal,anomaly,cross_label,2023-07-28 05:50:00,2023-08-11 04:20:00,2023-07-31 10:00:00,2023-08-19 10:00:00,2023-07-31 10:00:00,2023-08-11 04:20:00,258.333333,77.2297,56.652,partial_overlap,10.763889



Events participating in at least one strict overlap:


,farm,asset_id,asset_key,event_id,event_key,event_label,event_start,event_end,event_duration_days,asset_id_source
0,A,21,farm_a::asset_21,51,farm_a_event_51,anomaly,2023-10-06 01:30:00,2023-10-17 16:10:00,11.611111,event_info.csv
1,A,21,farm_a::asset_21,72,farm_a_event_72,anomaly,2023-10-10 08:40:00,2023-10-17 08:40:00,7.000000,event_info.csv
2,B,7,farm_b::asset_7,27,farm_b_event_27,anomaly,2023-09-01 00:00:00,2023-11-01 00:00:00,61.000000,event_info.csv
3,B,7,farm_b::asset_7,87,farm_b_event_87,normal,2023-09-14 23:00:00,2023-09-30 23:00:00,16.000000,event_info.csv
4,C,34,farm_c::asset_34,56,farm_c_event_56,normal,2023-07-28 05:50:00,2023-08-11 04:20:00,13.937500,event_info.csv
5,C,34,farm_c::asset_34,4,farm_c_event_4,anomaly,2023-07-31 10:00:00,2023-08-19 10:00:00,19.000000,event_info.csv



Affected-turbine summary:


,farm,asset_id,asset_key,affected_events,normal_events,anomaly_events,earliest_start,latest_end
0,A,21,farm_a::asset_21,2,0,2,2023-10-06 01:30:00,2023-10-17 16:10:00
1,B,7,farm_b::asset_7,2,1,1,2023-09-01 00:00:00,2023-11-01 00:00:00
2,C,34,farm_c::asset_34,2,1,1,2023-07-28 05:50:00,2023-08-19 10:00:00



Cell 3A completed successfully.
Strict-overlap pairs: 3
Cross-label overlap pairs: 2
Affected events: 6
Affected turbines: 3


In [19]:
# Cell 3B — compare raw timestamps for the three overlap pairs
# Only timestamp columns from the six affected CSVs are loaded.

# ---------------------------------------------------------
# 1. Validate prerequisites
# ---------------------------------------------------------

required_objects = [
    "EVENTS",
    "STRICT_OVERLAPS",
    "FARMS",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing required notebook objects: {missing_objects}"
    )

if len(STRICT_OVERLAPS) != 3:
    raise ValueError(
        "Expected 3 strict-overlap pairs, found "
        f"{len(STRICT_OVERLAPS)}."
    )

event_lookup = EVENTS.set_index(
    "event_key",
    drop=False,
)

affected_event_keys = sorted(
    set(STRICT_OVERLAPS["event_a_key"])
    | set(STRICT_OVERLAPS["event_b_key"])
)

print(
    f"Loading timestamp columns from "
    f"{len(affected_event_keys)} affected event files..."
)


# ---------------------------------------------------------
# 2. Timestamp-column utilities
# ---------------------------------------------------------

def normalize_column_name(column):
    return (
        str(column)
        .strip()
        .strip("'\"")
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )


def find_timestamp_column(event_path):
    header = pd.read_csv(
        event_path,
        sep=None,
        engine="python",
        quotechar="'",
        dtype=str,
        keep_default_na=False,
        skipinitialspace=True,
        nrows=0,
    )

    raw_columns = header.columns.tolist()

    normalized_columns = [
        normalize_column_name(column)
        for column in raw_columns
    ]

    if len(normalized_columns) != len(
        set(normalized_columns)
    ):
        raise ValueError(
            f"{event_path}: duplicate normalized columns."
        )

    normalized_to_raw = dict(
        zip(normalized_columns, raw_columns)
    )

    exact_candidates = [
        "timestamp",
        "time_stamp",
        "datetime",
        "date_time",
        "time",
        "date",
    ]

    for candidate in exact_candidates:
        if candidate in normalized_to_raw:
            return normalized_to_raw[candidate]

    pattern_candidates = [
        normalized
        for normalized in normalized_columns
        if (
            "timestamp" in normalized
            or "datetime" in normalized
            or normalized.endswith("_time")
            or normalized.endswith("_date")
        )
    ]

    if len(pattern_candidates) == 1:
        return normalized_to_raw[
            pattern_candidates[0]
        ]

    raise ValueError(
        f"{event_path}: could not identify one timestamp "
        f"column.\nAvailable columns: {raw_columns}"
    )


def to_utc_timestamp(value):
    timestamp = pd.Timestamp(value)

    if pd.isna(timestamp):
        raise ValueError(
            f"Cannot convert missing timestamp: {value}"
        )

    if timestamp.tzinfo is None:
        return timestamp.tz_localize("UTC")

    return timestamp.tz_convert("UTC")


def percentage(numerator, denominator):
    if denominator == 0:
        return np.nan

    return round(
        100 * numerator / denominator,
        6,
    )


def minimum_nearest_gap_minutes(
    timestamps_a,
    timestamps_b,
):
    """Minimum gap between two sorted timestamp indexes."""

    if (
        len(timestamps_a) == 0
        or len(timestamps_b) == 0
    ):
        return np.nan

    values_a = timestamps_a.asi8
    values_b = timestamps_b.asi8

    insertion_positions = np.searchsorted(
        values_b,
        values_a,
    )

    candidate_gaps = []

    right_mask = (
        insertion_positions < len(values_b)
    )

    if right_mask.any():
        candidate_gaps.append(
            np.abs(
                values_a[right_mask]
                - values_b[
                    insertion_positions[right_mask]
                ]
            )
        )

    left_mask = insertion_positions > 0

    if left_mask.any():
        candidate_gaps.append(
            np.abs(
                values_a[left_mask]
                - values_b[
                    insertion_positions[left_mask] - 1
                ]
            )
        )

    if not candidate_gaps:
        return np.nan

    minimum_gap_nanoseconds = min(
        int(gaps.min())
        for gaps in candidate_gaps
    )

    return (
        minimum_gap_nanoseconds
        / 60_000_000_000
    )


# ---------------------------------------------------------
# 3. Load timestamps from one affected event file
# ---------------------------------------------------------

def load_event_timestamps(event_key):
    event_row = event_lookup.loc[event_key]

    farm_name = event_row["farm"]
    event_id = int(event_row["event_id"])

    event_path = (
        FARMS[farm_name]
        / "datasets"
        / f"{event_id}.csv"
    )

    if not event_path.is_file():
        raise FileNotFoundError(
            f"Event dataset does not exist:\n{event_path}"
        )

    timestamp_column = find_timestamp_column(
        event_path
    )

    timestamp_frame = pd.read_csv(
        event_path,
        sep=None,
        engine="python",
        quotechar="'",
        dtype=str,
        keep_default_na=False,
        skipinitialspace=True,
        usecols=[timestamp_column],
    )

    raw_timestamps = (
        timestamp_frame.iloc[:, 0]
        .astype("string")
        .str.strip()
        .str.strip("'\"")
        .str.strip()
    )

    parsed_timestamps = pd.to_datetime(
        raw_timestamps,
        errors="coerce",
        format="mixed",
        utc=True,
    )

    invalid_timestamp_rows = int(
        parsed_timestamps.isna().sum()
    )

    valid_timestamp_index = pd.DatetimeIndex(
        parsed_timestamps.dropna()
    )

    unique_timestamp_index = (
        valid_timestamp_index
        .unique()
        .sort_values()
    )

    if len(unique_timestamp_index) == 0:
        raise ValueError(
            f"{event_path}: no valid timestamps found."
        )

    duplicate_excess_rows = (
        len(valid_timestamp_index)
        - len(unique_timestamp_index)
    )

    timestamp_differences = (
        unique_timestamp_index
        .to_series()
        .diff()
        .dropna()
    )

    median_cadence_minutes = (
        timestamp_differences.median()
        .total_seconds()
        / 60
        if not timestamp_differences.empty
        else np.nan
    )

    return {
        "event_key": event_key,
        "farm": farm_name,
        "asset_id": event_row["asset_id"],
        "event_id": event_id,
        "event_label": event_row["event_label"],
        "event_path": str(event_path),
        "timestamp_column": timestamp_column,
        "raw_rows": int(len(timestamp_frame)),
        "valid_timestamp_rows": int(
            len(valid_timestamp_index)
        ),
        "invalid_timestamp_rows":
            invalid_timestamp_rows,
        "unique_timestamps": int(
            len(unique_timestamp_index)
        ),
        "duplicate_excess_rows": int(
            duplicate_excess_rows
        ),
        "first_timestamp_utc":
            unique_timestamp_index.min(),
        "last_timestamp_utc":
            unique_timestamp_index.max(),
        "median_cadence_minutes":
            median_cadence_minutes,
        "timestamp_index":
            unique_timestamp_index,
    }


# ---------------------------------------------------------
# 4. Load and cache all six timestamp sets
# ---------------------------------------------------------

TIMESTAMP_CACHE = {}

for event_key in affected_event_keys:
    result = load_event_timestamps(event_key)
    TIMESTAMP_CACHE[event_key] = result

    print(
        f"  {event_key}: "
        f"{result['unique_timestamps']:,} unique timestamps"
    )

EVENT_TIMESTAMP_FILE_AUDIT = pd.DataFrame(
    [
        {
            key: value
            for key, value in result.items()
            if key != "timestamp_index"
        }
        for result in TIMESTAMP_CACHE.values()
    ]
).sort_values(
    ["farm", "asset_id", "event_id"]
).reset_index(drop=True)

print("\nAffected-file timestamp audit:")

display(
    EVENT_TIMESTAMP_FILE_AUDIT[
        [
            "farm",
            "asset_id",
            "event_id",
            "event_key",
            "event_label",
            "timestamp_column",
            "raw_rows",
            "valid_timestamp_rows",
            "invalid_timestamp_rows",
            "unique_timestamps",
            "duplicate_excess_rows",
            "first_timestamp_utc",
            "last_timestamp_utc",
            "median_cadence_minutes",
        ]
    ]
)


# ---------------------------------------------------------
# 5. Compare raw timestamp sets for each overlap pair
# ---------------------------------------------------------

pair_audit_records = []

for pair in STRICT_OVERLAPS.itertuples(index=False):
    result_a = TIMESTAMP_CACHE[
        pair.event_a_key
    ]

    result_b = TIMESTAMP_CACHE[
        pair.event_b_key
    ]

    timestamps_a = result_a["timestamp_index"]
    timestamps_b = result_b["timestamp_index"]

    # Shared timestamps anywhere in the complete CSV files.
    shared_full_file = (
        timestamps_a
        .intersection(timestamps_b)
        .sort_values()
    )

    metadata_overlap_start = to_utc_timestamp(
        pair.overlap_start
    )

    metadata_overlap_end = to_utc_timestamp(
        pair.overlap_end
    )

    # Timestamps falling specifically inside the metadata
    # overlap interval.
    timestamps_a_in_metadata_overlap = timestamps_a[
        (timestamps_a >= metadata_overlap_start)
        & (timestamps_a <= metadata_overlap_end)
    ]

    timestamps_b_in_metadata_overlap = timestamps_b[
        (timestamps_b >= metadata_overlap_start)
        & (timestamps_b <= metadata_overlap_end)
    ]

    shared_in_metadata_overlap = (
        timestamps_a_in_metadata_overlap
        .intersection(
            timestamps_b_in_metadata_overlap
        )
        .sort_values()
    )

    minimum_gap_minutes = (
        minimum_nearest_gap_minutes(
            timestamps_a_in_metadata_overlap,
            timestamps_b_in_metadata_overlap,
        )
    )

    if len(shared_in_metadata_overlap) > 0:
        raw_relationship = (
            "shared_timestamps_inside_metadata_overlap"
        )

    elif len(shared_full_file) > 0:
        raw_relationship = (
            "shared_timestamps_elsewhere_in_files"
        )

    elif (
        len(timestamps_a_in_metadata_overlap) > 0
        and len(
            timestamps_b_in_metadata_overlap
        ) > 0
    ):
        raw_relationship = (
            "overlapping_ranges_no_identical_timestamps"
        )

    else:
        raw_relationship = (
            "metadata_overlap_missing_raw_coverage"
        )

    pair_audit_records.append(
        {
            "farm": pair.farm,
            "asset_id": pair.asset_id,
            "event_a_key": pair.event_a_key,
            "event_b_key": pair.event_b_key,
            "event_a_label": pair.event_a_label,
            "event_b_label": pair.event_b_label,
            "label_relationship":
                pair.label_relationship,
            "metadata_overlap_start_utc":
                metadata_overlap_start,
            "metadata_overlap_end_utc":
                metadata_overlap_end,
            "metadata_overlap_hours":
                pair.overlap_hours,
            "unique_timestamps_event_a":
                len(timestamps_a),
            "unique_timestamps_event_b":
                len(timestamps_b),
            "shared_timestamps_full_files":
                len(shared_full_file),
            "full_file_shared_percent_a":
                percentage(
                    len(shared_full_file),
                    len(timestamps_a),
                ),
            "full_file_shared_percent_b":
                percentage(
                    len(shared_full_file),
                    len(timestamps_b),
                ),
            "timestamps_a_in_metadata_overlap":
                len(
                    timestamps_a_in_metadata_overlap
                ),
            "timestamps_b_in_metadata_overlap":
                len(
                    timestamps_b_in_metadata_overlap
                ),
            "shared_timestamps_in_metadata_overlap":
                len(shared_in_metadata_overlap),
            "metadata_shared_percent_a":
                percentage(
                    len(shared_in_metadata_overlap),
                    len(
                        timestamps_a_in_metadata_overlap
                    ),
                ),
            "metadata_shared_percent_b":
                percentage(
                    len(shared_in_metadata_overlap),
                    len(
                        timestamps_b_in_metadata_overlap
                    ),
                ),
            "minimum_nearest_gap_minutes":
                minimum_gap_minutes,
            "first_shared_timestamp_utc": (
                shared_full_file.min()
                if len(shared_full_file) > 0
                else pd.NaT
            ),
            "last_shared_timestamp_utc": (
                shared_full_file.max()
                if len(shared_full_file) > 0
                else pd.NaT
            ),
            "raw_relationship": raw_relationship,
        }
    )

RAW_TIMESTAMP_PAIR_AUDIT = (
    pd.DataFrame(pair_audit_records)
    .sort_values(
        [
            "farm",
            "asset_id",
            "metadata_overlap_start_utc",
            "event_a_key",
            "event_b_key",
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 6. Display the three-pair result
# ---------------------------------------------------------

print("\nRaw timestamp overlap audit:")

display(RAW_TIMESTAMP_PAIR_AUDIT)

cross_label_audit = RAW_TIMESTAMP_PAIR_AUDIT.loc[
    RAW_TIMESTAMP_PAIR_AUDIT[
        "label_relationship"
    ].eq("cross_label")
]

pairs_sharing_full_file_timestamps = int(
    RAW_TIMESTAMP_PAIR_AUDIT[
        "shared_timestamps_full_files"
    ].gt(0).sum()
)

pairs_sharing_metadata_timestamps = int(
    RAW_TIMESTAMP_PAIR_AUDIT[
        "shared_timestamps_in_metadata_overlap"
    ].gt(0).sum()
)

cross_label_full_file_sharing = int(
    cross_label_audit[
        "shared_timestamps_full_files"
    ].gt(0).sum()
)

cross_label_metadata_sharing = int(
    cross_label_audit[
        "shared_timestamps_in_metadata_overlap"
    ].gt(0).sum()
)


# ---------------------------------------------------------
# 7. Interpret this audit stage
# ---------------------------------------------------------

print("\nCell 3B completed successfully.")
print(
    "Pairs sharing timestamps anywhere in their files:",
    pairs_sharing_full_file_timestamps,
)
print(
    "Pairs sharing timestamps inside metadata overlap:",
    pairs_sharing_metadata_timestamps,
)
print(
    "Cross-label pairs sharing timestamps anywhere:",
    cross_label_full_file_sharing,
)
print(
    "Cross-label pairs sharing timestamps inside "
    "metadata overlap:",
    cross_label_metadata_sharing,
)

if cross_label_metadata_sharing > 0:
    print(
        "\nWARNING: Oppositely labeled event files reuse "
        "timestamps inside their official overlapping "
        "event windows. Feature values at those timestamps "
        "must be compared before choosing a labeling rule."
    )

elif cross_label_full_file_sharing > 0:
    print(
        "\nCAUTION: Oppositely labeled files reuse timestamps, "
        "but not inside the metadata overlap interval. "
        "Do not assign the file-level event label uniformly "
        "to every row until the training/prediction regions "
        "are identified."
    )

else:
    print(
        "\nNo exact timestamp reuse was found between the "
        "cross-label files. Their metadata intervals overlap, "
        "but the sampled observations are temporally distinct."
    )

Loading timestamp columns from 6 affected event files...
  farm_a_event_51: 54,436 unique timestamps
  farm_a_event_72: 54,082 unique timestamps
  farm_b_event_27: 62,268 unique timestamps
  farm_b_event_87: 55,356 unique timestamps
  farm_c_event_4: 56,449 unique timestamps
  farm_c_event_56: 53,416 unique timestamps

Affected-file timestamp audit:


,farm,asset_id,event_id,event_key,event_label,timestamp_column,raw_rows,valid_timestamp_rows,invalid_timestamp_rows,unique_timestamps,duplicate_excess_rows,first_timestamp_utc,last_timestamp_utc,median_cadence_minutes
0,A,21,51,farm_a_event_51,anomaly,time_stamp,54436,54436,0,54436,0,2022-10-04 01:30:00+00:00,2023-10-20 16:10:00+00:00,10.0
1,A,21,72,farm_a_event_72,anomaly,time_stamp,54082,54082,0,54082,0,2022-10-07 08:40:00+00:00,2023-10-21 08:40:00+00:00,10.0
2,B,7,27,farm_b_event_27,anomaly,time_stamp,62268,62268,0,62268,0,2022-08-30 00:00:00+00:00,2023-11-07 00:00:00+00:00,10.0
3,B,7,87,farm_b_event_87,normal,time_stamp,55356,55356,0,55356,0,2022-09-13 23:00:00+00:00,2023-10-04 23:00:00+00:00,10.0
4,C,34,4,farm_c_event_4,anomaly,time_stamp,56449,56449,0,56449,0,2022-07-28 10:00:00+00:00,2023-08-24 10:00:00+00:00,10.0
5,C,34,56,farm_c_event_56,normal,time_stamp,53416,53416,0,53416,0,2022-07-28 05:50:00+00:00,2023-08-13 04:20:00+00:00,10.0



Raw timestamp overlap audit:


,farm,asset_id,event_a_key,event_b_key,event_a_label,event_b_label,label_relationship,metadata_overlap_start_utc,metadata_overlap_end_utc,metadata_overlap_hours,...,full_file_shared_percent_b,timestamps_a_in_metadata_overlap,timestamps_b_in_metadata_overlap,shared_timestamps_in_metadata_overlap,metadata_shared_percent_a,metadata_shared_percent_b,minimum_nearest_gap_minutes,first_shared_timestamp_utc,last_shared_timestamp_utc,raw_relationship
0,A,21,farm_a_event_51,farm_a_event_72,anomaly,anomaly,same_label,2023-10-10 08:40:00+00:00,2023-10-17 08:40:00+00:00,168.000000,...,99.816945,1009,1009,1009,100.0,100.0,0.0,2022-10-07 08:40:00+00:00,2023-10-20 16:10:00+00:00,shared_timestamps_inside_metadata_overlap
1,B,7,farm_b_event_27,farm_b_event_87,anomaly,normal,cross_label,2023-09-14 23:00:00+00:00,2023-09-30 23:00:00+00:00,384.000000,...,100.000000,2305,2305,2305,100.0,100.0,0.0,2022-09-13 23:00:00+00:00,2023-10-04 23:00:00+00:00,shared_timestamps_inside_metadata_overlap
2,C,34,farm_c_event_56,farm_c_event_4,normal,anomaly,cross_label,2023-07-31 10:00:00+00:00,2023-08-11 04:20:00+00:00,258.333333,...,94.582721,1551,1551,1551,100.0,100.0,0.0,2022-07-28 10:00:00+00:00,2023-08-13 04:20:00+00:00,shared_timestamps_inside_metadata_overlap



Cell 3B completed successfully.
Pairs sharing timestamps anywhere in their files: 3
Pairs sharing timestamps inside metadata overlap: 3
Cross-label pairs sharing timestamps anywhere: 2
Cross-label pairs sharing timestamps inside metadata overlap: 2



In [20]:
# Cell 3C — compare sensor values at shared timestamps
# No rows, labels, or source files are modified.

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 1. Validate prerequisites
# ---------------------------------------------------------

required_objects = [
    "EVENTS",
    "STRICT_OVERLAPS",
    "FARMS",
    "TIMESTAMP_CACHE",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing required notebook objects: {missing_objects}"
    )

if len(STRICT_OVERLAPS) != 3:
    raise ValueError(
        f"Expected 3 overlap pairs, found "
        f"{len(STRICT_OVERLAPS)}."
    )

event_lookup = EVENTS.set_index(
    "event_key",
    drop=False,
)


# ---------------------------------------------------------
# 2. Column and timestamp utilities
# ---------------------------------------------------------

def normalize_3c_column(column):
    return (
        str(column)
        .strip()
        .strip("'\"")
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )


def detect_3c_timestamp_column(raw_columns):
    normalized = [
        normalize_3c_column(column)
        for column in raw_columns
    ]

    normalized_to_raw = dict(
        zip(normalized, raw_columns)
    )

    exact_candidates = [
        "timestamp",
        "time_stamp",
        "datetime",
        "date_time",
        "time",
        "date",
    ]

    for candidate in exact_candidates:
        if candidate in normalized_to_raw:
            return normalized_to_raw[candidate]

    pattern_candidates = [
        column
        for column in normalized
        if (
            "timestamp" in column
            or "datetime" in column
            or column.endswith("_time")
            or column.endswith("_date")
        )
    ]

    if len(pattern_candidates) == 1:
        return normalized_to_raw[
            pattern_candidates[0]
        ]

    raise ValueError(
        "Could not identify exactly one timestamp column. "
        f"Available columns: {raw_columns}"
    )


def to_3c_utc(value):
    timestamp = pd.Timestamp(value)

    if timestamp.tzinfo is None:
        return timestamp.tz_localize("UTC")

    return timestamp.tz_convert("UTC")


EXCLUDED_NONFEATURE_COLUMNS = {
    "asset",
    "asset_id",
    "event",
    "event_id",
    "event_key",
    "event_label",
    "label",
    "class",
    "target",
    "farm",
    "farm_id",
    "file",
    "filename",
    "row_id",
    "index",
}


def is_candidate_feature(column):
    return (
        column not in EXCLUDED_NONFEATURE_COLUMNS
        and not column.startswith("unnamed")
        and not column.endswith("_id")
    )


# ---------------------------------------------------------
# 3. Inspect one event-file schema
# ---------------------------------------------------------

def inspect_3c_event_schema(event_key):
    event_row = event_lookup.loc[event_key]

    farm_name = event_row["farm"]
    event_id = int(event_row["event_id"])

    event_path = (
        FARMS[farm_name]
        / "datasets"
        / f"{event_id}.csv"
    )

    if not event_path.is_file():
        raise FileNotFoundError(
            f"Event file does not exist:\n{event_path}"
        )

    header = pd.read_csv(
        event_path,
        sep=None,
        engine="python",
        quotechar="'",
        dtype=str,
        keep_default_na=False,
        skipinitialspace=True,
        nrows=0,
    )

    raw_columns = header.columns.tolist()

    normalized_columns = [
        normalize_3c_column(column)
        for column in raw_columns
    ]

    if len(normalized_columns) != len(
        set(normalized_columns)
    ):
        raise ValueError(
            f"{event_path}: duplicate normalized columns."
        )

    normalized_to_raw = dict(
        zip(normalized_columns, raw_columns)
    )

    timestamp_raw = detect_3c_timestamp_column(
        raw_columns
    )

    timestamp_normalized = normalize_3c_column(
        timestamp_raw
    )

    feature_columns = {
        column
        for column in normalized_columns
        if (
            column != timestamp_normalized
            and is_candidate_feature(column)
        )
    }

    return {
        "event_key": event_key,
        "event_path": event_path,
        "timestamp_raw": timestamp_raw,
        "timestamp_normalized": timestamp_normalized,
        "normalized_to_raw": normalized_to_raw,
        "feature_columns": feature_columns,
    }


# ---------------------------------------------------------
# 4. Load only selected shared-timestamp rows
# ---------------------------------------------------------

def load_3c_shared_rows(
    schema,
    selected_features,
    target_timestamps,
):
    normalized_to_raw = schema[
        "normalized_to_raw"
    ]

    timestamp_raw = schema["timestamp_raw"]

    raw_feature_columns = [
        normalized_to_raw[column]
        for column in selected_features
    ]

    use_columns = [
        timestamp_raw,
        *raw_feature_columns,
    ]

    rename_map = {
        timestamp_raw: "__timestamp_raw",
    }

    rename_map.update(
        {
            normalized_to_raw[column]: column
            for column in selected_features
        }
    )

    retained_chunks = []

    reader = pd.read_csv(
        schema["event_path"],
        sep=None,
        engine="python",
        quotechar="'",
        dtype=str,
        keep_default_na=False,
        skipinitialspace=True,
        usecols=use_columns,
        chunksize=100_000,
    )

    for chunk in reader:
        chunk = chunk.rename(columns=rename_map)

        parsed_timestamps = pd.to_datetime(
            (
                chunk["__timestamp_raw"]
                .astype("string")
                .str.strip()
                .str.strip("'\"")
                .str.strip()
            ),
            errors="coerce",
            format="mixed",
            utc=True,
        )

        keep_mask = parsed_timestamps.isin(
            target_timestamps
        )

        if not keep_mask.any():
            continue

        retained = chunk.loc[
            keep_mask,
            selected_features,
        ].copy()

        retained.insert(
            0,
            "timestamp_utc",
            parsed_timestamps.loc[keep_mask],
        )

        for column in selected_features:
            cleaned = (
                retained[column]
                .astype("string")
                .str.strip()
                .str.strip("'\"")
                .str.strip()
            )

            retained[column] = cleaned.mask(
                cleaned.eq(""),
                pd.NA,
            )

        retained_chunks.append(retained)

    if not retained_chunks:
        raise ValueError(
            f"{schema['event_key']}: no target timestamps "
            "were recovered."
        )

    retained_rows = (
        pd.concat(
            retained_chunks,
            ignore_index=True,
        )
        .sort_values("timestamp_utc")
        .reset_index(drop=True)
    )

    duplicate_mask = retained_rows[
        "timestamp_utc"
    ].duplicated(keep=False)

    if duplicate_mask.any():
        duplicate_count = int(
            duplicate_mask.sum()
        )

        raise ValueError(
            f"{schema['event_key']}: {duplicate_count} rows "
            "belong to duplicated target timestamps. "
            "Resolve duplicates before value comparison."
        )

    observed_timestamps = pd.DatetimeIndex(
        retained_rows["timestamp_utc"]
    )

    missing_timestamps = (
        target_timestamps.difference(
            observed_timestamps
        )
    )

    if len(missing_timestamps) > 0:
        raise ValueError(
            f"{schema['event_key']}: failed to recover "
            f"{len(missing_timestamps)} shared timestamps."
        )

    return retained_rows


# ---------------------------------------------------------
# 5. Compare one feature
# ---------------------------------------------------------

def compare_3c_feature(series_a, series_b):
    missing_a = series_a.isna()
    missing_b = series_b.isna()

    both_missing = missing_a & missing_b
    one_missing = missing_a ^ missing_b
    jointly_present = ~missing_a & ~missing_b

    matches = pd.Series(
        False,
        index=series_a.index,
        dtype=bool,
    )

    # Missing in both files is row-equivalent but is excluded
    # from the comparable-cell denominator.
    matches.loc[both_missing] = True

    numeric_a = pd.to_numeric(
        series_a.loc[jointly_present],
        errors="coerce",
    )

    numeric_b = pd.to_numeric(
        series_b.loc[jointly_present],
        errors="coerce",
    )

    is_numeric = bool(
        jointly_present.any()
        and numeric_a.notna().all()
        and numeric_b.notna().all()
    )

    max_absolute_difference = np.nan

    if is_numeric:
        values_a = numeric_a.to_numpy(dtype=float)
        values_b = numeric_b.to_numpy(dtype=float)

        numeric_matches = np.isclose(
            values_a,
            values_b,
            rtol=1e-10,
            atol=1e-12,
            equal_nan=True,
        )

        matches.loc[jointly_present] = (
            numeric_matches
        )

        absolute_differences = np.abs(
            values_a - values_b
        )

        finite_differences = (
            absolute_differences[
                np.isfinite(absolute_differences)
            ]
        )

        if len(finite_differences) > 0:
            max_absolute_difference = float(
                finite_differences.max()
            )

    else:
        string_matches = (
            series_a.loc[jointly_present]
            .eq(series_b.loc[jointly_present])
            .fillna(False)
        )

        matches.loc[jointly_present] = (
            string_matches.to_numpy()
        )

    comparable = ~both_missing

    return {
        "matches": matches,
        "is_numeric": is_numeric,
        "comparable_cells": int(comparable.sum()),
        "matching_cells": int(
            (matches & comparable).sum()
        ),
        "differing_cells": int(
            (~matches).sum()
        ),
        "both_missing": int(
            both_missing.sum()
        ),
        "one_missing": int(
            one_missing.sum()
        ),
        "max_absolute_difference":
            max_absolute_difference,
    }


# ---------------------------------------------------------
# 6. Compare every strict-overlap pair
# ---------------------------------------------------------

pair_records = []
difference_records = []
example_records = []

for pair in STRICT_OVERLAPS.itertuples(index=False):
    schema_a = inspect_3c_event_schema(
        pair.event_a_key
    )

    schema_b = inspect_3c_event_schema(
        pair.event_b_key
    )

    features_a = schema_a["feature_columns"]
    features_b = schema_b["feature_columns"]

    common_features = sorted(
        features_a.intersection(features_b)
    )

    features_only_a = sorted(
        features_a.difference(features_b)
    )

    features_only_b = sorted(
        features_b.difference(features_a)
    )

    if not common_features:
        raise ValueError(
            f"{pair.event_a_key} and {pair.event_b_key}: "
            "no common candidate features."
        )

    timestamp_index_a = TIMESTAMP_CACHE[
        pair.event_a_key
    ]["timestamp_index"]

    timestamp_index_b = TIMESTAMP_CACHE[
        pair.event_b_key
    ]["timestamp_index"]

    overlap_start = to_3c_utc(
        pair.overlap_start
    )

    overlap_end = to_3c_utc(
        pair.overlap_end
    )

    shared_timestamps = (
        timestamp_index_a
        .intersection(timestamp_index_b)
        .sort_values()
    )

    shared_timestamps = shared_timestamps[
        (shared_timestamps >= overlap_start)
        & (shared_timestamps <= overlap_end)
    ]

    if len(shared_timestamps) == 0:
        raise ValueError(
            f"{pair.event_a_key} and {pair.event_b_key}: "
            "no shared timestamps inside metadata overlap."
        )

    rows_a = load_3c_shared_rows(
        schema=schema_a,
        selected_features=common_features,
        target_timestamps=shared_timestamps,
    )

    rows_b = load_3c_shared_rows(
        schema=schema_b,
        selected_features=common_features,
        target_timestamps=shared_timestamps,
    )

    merged = rows_a.merge(
        rows_b,
        on="timestamp_utc",
        how="inner",
        suffixes=("_event_a", "_event_b"),
        validate="one_to_one",
    )

    if len(merged) != len(shared_timestamps):
        raise ValueError(
            f"{pair.event_a_key} and {pair.event_b_key}: "
            "timestamp merge lost observations."
        )

    row_matches = pd.Series(
        True,
        index=merged.index,
        dtype=bool,
    )

    total_comparable_cells = 0
    total_matching_cells = 0
    total_differing_cells = 0

    for feature in common_features:
        column_a = f"{feature}_event_a"
        column_b = f"{feature}_event_b"

        result = compare_3c_feature(
            merged[column_a],
            merged[column_b],
        )

        feature_matches = result["matches"]
        row_matches &= feature_matches

        total_comparable_cells += (
            result["comparable_cells"]
        )

        total_matching_cells += (
            result["matching_cells"]
        )

        total_differing_cells += (
            result["differing_cells"]
        )

        if result["differing_cells"] > 0:
            difference_records.append(
                {
                    "farm": pair.farm,
                    "asset_id": pair.asset_id,
                    "event_a_key": pair.event_a_key,
                    "event_b_key": pair.event_b_key,
                    "label_relationship":
                        pair.label_relationship,
                    "feature": feature,
                    "numeric_feature":
                        result["is_numeric"],
                    "comparable_cells":
                        result["comparable_cells"],
                    "matching_cells":
                        result["matching_cells"],
                    "differing_cells":
                        result["differing_cells"],
                    "both_missing":
                        result["both_missing"],
                    "one_missing":
                        result["one_missing"],
                    "max_absolute_difference":
                        result[
                            "max_absolute_difference"
                        ],
                }
            )

            mismatch_indexes = feature_matches.index[
                ~feature_matches
            ][:3]

            for row_index in mismatch_indexes:
                value_a = merged.at[
                    row_index,
                    column_a,
                ]

                value_b = merged.at[
                    row_index,
                    column_b,
                ]

                example_records.append(
                    {
                        "event_a_key":
                            pair.event_a_key,
                        "event_b_key":
                            pair.event_b_key,
                        "timestamp_utc":
                            merged.at[
                                row_index,
                                "timestamp_utc",
                            ],
                        "feature": feature,
                        "value_event_a": (
                            None
                            if pd.isna(value_a)
                            else value_a
                        ),
                        "value_event_b": (
                            None
                            if pd.isna(value_b)
                            else value_b
                        ),
                    }
                )

    schema_matches = (
        len(features_only_a) == 0
        and len(features_only_b) == 0
    )

    identical_timestamp_rows = int(
        row_matches.sum()
    )

    all_timestamp_rows_identical = bool(
        row_matches.all()
    )

    if not schema_matches:
        feature_relationship = (
            "schema_mismatch_review_required"
        )

    elif all_timestamp_rows_identical:
        feature_relationship = (
            "all_shared_feature_rows_identical"
        )

    elif identical_timestamp_rows > 0:
        feature_relationship = (
            "some_shared_feature_rows_identical"
        )

    else:
        feature_relationship = (
            "all_shared_feature_rows_different"
        )

    pair_records.append(
        {
            "farm": pair.farm,
            "asset_id": pair.asset_id,
            "event_a_key": pair.event_a_key,
            "event_b_key": pair.event_b_key,
            "event_a_label": pair.event_a_label,
            "event_b_label": pair.event_b_label,
            "label_relationship":
                pair.label_relationship,
            "shared_timestamps_compared":
                int(len(merged)),
            "common_features":
                int(len(common_features)),
            "features_only_event_a":
                int(len(features_only_a)),
            "features_only_event_b":
                int(len(features_only_b)),
            "comparable_feature_cells":
                int(total_comparable_cells),
            "matching_feature_cells":
                int(total_matching_cells),
            "differing_feature_cells":
                int(total_differing_cells),
            "identical_timestamp_rows":
                identical_timestamp_rows,
            "identical_timestamp_percent":
                round(
                    100
                    * identical_timestamp_rows
                    / len(merged),
                    6,
                ),
            "feature_cell_match_percent": (
                round(
                    100
                    * total_matching_cells
                    / total_comparable_cells,
                    6,
                )
                if total_comparable_cells > 0
                else np.nan
            ),
            "feature_relationship":
                feature_relationship,
        }
    )


# ---------------------------------------------------------
# 7. Construct audit tables
# ---------------------------------------------------------

SHARED_FEATURE_PAIR_AUDIT = (
    pd.DataFrame(pair_records)
    .sort_values(
        [
            "farm",
            "asset_id",
            "event_a_key",
            "event_b_key",
        ]
    )
    .reset_index(drop=True)
)

difference_columns = [
    "farm",
    "asset_id",
    "event_a_key",
    "event_b_key",
    "label_relationship",
    "feature",
    "numeric_feature",
    "comparable_cells",
    "matching_cells",
    "differing_cells",
    "both_missing",
    "one_missing",
    "max_absolute_difference",
]

FEATURE_DIFFERENCE_AUDIT = pd.DataFrame(
    difference_records,
    columns=difference_columns,
)

if not FEATURE_DIFFERENCE_AUDIT.empty:
    FEATURE_DIFFERENCE_AUDIT = (
        FEATURE_DIFFERENCE_AUDIT
        .sort_values(
            [
                "event_a_key",
                "event_b_key",
                "differing_cells",
                "feature",
            ],
            ascending=[True, True, False, True],
        )
        .reset_index(drop=True)
    )

FEATURE_VALUE_EXAMPLES = pd.DataFrame(
    example_records,
    columns=[
        "event_a_key",
        "event_b_key",
        "timestamp_utc",
        "feature",
        "value_event_a",
        "value_event_b",
    ],
)


# ---------------------------------------------------------
# 8. Display and interpret results
# ---------------------------------------------------------

print("Shared-timestamp feature comparison:")
display(SHARED_FEATURE_PAIR_AUDIT)

print("\nFeatures containing differences:")

if FEATURE_DIFFERENCE_AUDIT.empty:
    print("  None")
else:
    display(
        FEATURE_DIFFERENCE_AUDIT.head(100)
    )

print("\nExample differing values:")

if FEATURE_VALUE_EXAMPLES.empty:
    print("  None")
else:
    display(
        FEATURE_VALUE_EXAMPLES.head(40)
    )

cross_label_results = (
    SHARED_FEATURE_PAIR_AUDIT.loc[
        SHARED_FEATURE_PAIR_AUDIT[
            "label_relationship"
        ].eq("cross_label")
    ]
)

pairs_all_rows_identical = int(
    SHARED_FEATURE_PAIR_AUDIT[
        "feature_relationship"
    ].eq(
        "all_shared_feature_rows_identical"
    ).sum()
)

cross_label_pairs_all_rows_identical = int(
    cross_label_results[
        "feature_relationship"
    ].eq(
        "all_shared_feature_rows_identical"
    ).sum()
)

cross_label_pairs_with_any_identical_rows = int(
    cross_label_results[
        "identical_timestamp_rows"
    ].gt(0).sum()
)

cross_label_identical_rows = int(
    cross_label_results[
        "identical_timestamp_rows"
    ].sum()
)

print("\nCell 3C completed successfully.")
print(
    "Pairs with all shared feature rows identical:",
    pairs_all_rows_identical,
)
print(
    "Cross-label pairs with all rows identical:",
    cross_label_pairs_all_rows_identical,
)
print(
    "Cross-label pairs with any identical rows:",
    cross_label_pairs_with_any_identical_rows,
)
print(
    "Identical rows carrying conflicting file labels:",
    cross_label_identical_rows,
)

if cross_label_pairs_all_rows_identical > 0:
    print(
        "\nWARNING: Identical sensor observations occur under "
        "opposite event labels. These rows cannot be treated as "
        "independent labeled samples."
    )

elif cross_label_pairs_with_any_identical_rows > 0:
    print(
        "\nWARNING: Some identical observations carry opposite "
        "event labels. The conflicting subset must be isolated "
        "before modeling."
    )

else:
    print(
        "\nThe cross-label files share timestamps but not complete "
        "feature vectors. Their measurement semantics and differing "
        "features must be reviewed before labeling."
    )

Shared-timestamp feature comparison:


,farm,asset_id,event_a_key,event_b_key,event_a_label,event_b_label,label_relationship,shared_timestamps_compared,common_features,features_only_event_a,features_only_event_b,comparable_feature_cells,matching_feature_cells,differing_feature_cells,identical_timestamp_rows,identical_timestamp_percent,feature_cell_match_percent,feature_relationship
0,A,21,farm_a_event_51,farm_a_event_72,anomaly,anomaly,same_label,1009,83,0,0,83747,82738,1009,0,0.0,98.795181,all_shared_feature_rows_different
1,B,7,farm_b_event_27,farm_b_event_87,anomaly,normal,cross_label,2305,254,0,0,585470,583165,2305,0,0.0,99.606299,all_shared_feature_rows_different
2,C,34,farm_c_event_56,farm_c_event_4,normal,anomaly,cross_label,1551,954,0,0,1479654,201591,1278063,0,0.0,13.624199,all_shared_feature_rows_different



Features containing differences:


,farm,asset_id,event_a_key,event_b_key,label_relationship,feature,numeric_feature,comparable_cells,matching_cells,differing_cells,both_missing,one_missing,max_absolute_difference
0,A,21,farm_a_event_51,farm_a_event_72,same_label,id,True,1009,0,1009,0,0,453.000000
1,B,7,farm_b_event_27,farm_b_event_87,cross_label,id,True,2305,0,2305,0,0,2154.000000
2,C,34,farm_c_event_56,farm_c_event_4,cross_label,id,True,1551,0,1551,0,0,1415.000000
3,C,34,farm_c_event_56,farm_c_event_4,cross_label,power_5_avg,True,1551,0,1551,0,0,0.888896
4,C,34,farm_c_event_56,farm_c_event_4,cross_label,power_5_std,True,1551,0,1551,0,0,0.450097
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,C,34,farm_c_event_56,farm_c_event_4,cross_label,sensor_195_avg,True,1551,0,1551,0,0,11.128000
96,C,34,farm_c_event_56,farm_c_event_4,cross_label,sensor_197_avg,True,1551,0,1551,0,0,10.599000
97,C,34,farm_c_event_56,farm_c_event_4,cross_label,sensor_199_avg,True,1551,0,1551,0,0,34.530000
98,C,34,farm_c_event_56,farm_c_event_4,cross_label,sensor_208_avg,True,1551,0,1551,0,0,20.610000



Example differing values:


,event_a_key,event_b_key,timestamp_utc,feature,value_event_a,value_event_b
0,farm_a_event_51,farm_a_event_72,2023-10-10 08:40:00+00:00,id,52950,52497
1,farm_a_event_51,farm_a_event_72,2023-10-10 08:50:00+00:00,id,52951,52498
2,farm_a_event_51,farm_a_event_72,2023-10-10 09:00:00+00:00,id,52952,52499
3,farm_b_event_27,farm_b_event_87,2023-09-14 23:00:00+00:00,id,54629,52475
4,farm_b_event_27,farm_b_event_87,2023-09-14 23:10:00+00:00,id,54630,52476
5,farm_b_event_27,farm_b_event_87,2023-09-14 23:20:00+00:00,id,54631,52477
6,farm_c_event_56,farm_c_event_4,2023-07-31 10:00:00+00:00,id,51577,52992
7,farm_c_event_56,farm_c_event_4,2023-07-31 10:10:00+00:00,id,51578,52993
8,farm_c_event_56,farm_c_event_4,2023-07-31 10:20:00+00:00,id,51579,52994
9,farm_c_event_56,farm_c_event_4,2023-07-31 10:00:00+00:00,power_17_avg,0.24148000000000003,0.35298



Cell 3C completed successfully.
Pairs with all shared feature rows identical: 0
Cross-label pairs with all rows identical: 0
Cross-label pairs with any identical rows: 0
Identical rows carrying conflicting file labels: 0

The cross-label files share timestamps but not complete feature vectors. Their measurement semantics and differing features must be reviewed before labeling.


In [21]:
# Cell 3C-Report — compact results needed for interpretation

PAIR_COLUMNS = [
    "farm",
    "asset_id",
    "event_a_key",
    "event_b_key",
    "event_a_label",
    "event_b_label",
    "shared_timestamps_compared",
    "common_features",
    "features_only_event_a",
    "features_only_event_b",
    "differing_feature_cells",
    "feature_cell_match_percent",
    "feature_relationship",
]

cross_label_pair_report = (
    SHARED_FEATURE_PAIR_AUDIT.loc[
        SHARED_FEATURE_PAIR_AUDIT[
            "label_relationship"
        ].eq("cross_label"),
        PAIR_COLUMNS,
    ]
    .reset_index(drop=True)
)

print("Cross-label pair summary:")
print(cross_label_pair_report.to_string(index=False))


cross_label_feature_report = (
    FEATURE_DIFFERENCE_AUDIT.loc[
        FEATURE_DIFFERENCE_AUDIT[
            "label_relationship"
        ].eq("cross_label")
    ]
    .copy()
)

if cross_label_feature_report.empty:
    print("\nNo differing-feature records.")
else:
    cross_label_feature_report[
        "difference_percent"
    ] = (
        100
        * cross_label_feature_report["differing_cells"]
        / cross_label_feature_report["comparable_cells"]
    ).round(4)

    columns = [
        "event_a_key",
        "event_b_key",
        "feature",
        "numeric_feature",
        "comparable_cells",
        "matching_cells",
        "differing_cells",
        "difference_percent",
        "both_missing",
        "one_missing",
        "max_absolute_difference",
    ]

    cross_label_feature_report = (
        cross_label_feature_report[columns]
        .sort_values(
            [
                "event_a_key",
                "event_b_key",
                "difference_percent",
                "differing_cells",
                "feature",
            ],
            ascending=[True, True, False, False, True],
        )
        .reset_index(drop=True)
    )

    print("\nDiffering features in cross-label pairs:")
    print(
        cross_label_feature_report
        .groupby(
            ["event_a_key", "event_b_key"],
            group_keys=False,
        )
        .head(20)
        .to_string(index=False)
    )

Cross-label pair summary:
farm asset_id     event_a_key     event_b_key event_a_label event_b_label  shared_timestamps_compared  common_features  features_only_event_a  features_only_event_b  differing_feature_cells  feature_cell_match_percent              feature_relationship
   B        7 farm_b_event_27 farm_b_event_87       anomaly        normal                        2305              254                      0                      0                     2305                   99.606299 all_shared_feature_rows_different
   C       34 farm_c_event_56  farm_c_event_4        normal       anomaly                        1551              954                      0                      0                  1278063                   13.624199 all_shared_feature_rows_different

Differing features in cross-label pairs:
    event_a_key     event_b_key        feature  numeric_feature  comparable_cells  matching_cells  differing_cells  difference_percent  both_missing  one_missing  max_absolute_

In [22]:
# Cell 3D — exclude identifiers and quantify label conflicts
# No source data or labels are modified.

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 1. Validate prerequisites and correct feature roles
# ---------------------------------------------------------

required_objects = [
    "STRICT_OVERLAPS",
    "TIMESTAMP_CACHE",
    "EXCLUDED_NONFEATURE_COLUMNS",
    "inspect_3c_event_schema",
    "load_3c_shared_rows",
    "compare_3c_feature",
    "to_3c_utc",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing required notebook objects: {missing_objects}"
    )

# A database/row identifier is not a sensor feature.
EXCLUDED_NONFEATURE_COLUMNS.add("id")


# ---------------------------------------------------------
# 2. Recompare the two cross-label pairs
# ---------------------------------------------------------

corrected_records = []
conflicting_timestamp_records = []

cross_label_pairs = STRICT_OVERLAPS.loc[
    STRICT_OVERLAPS["label_relationship"].eq("cross_label")
]

for pair in cross_label_pairs.itertuples(index=False):
    schema_a = inspect_3c_event_schema(pair.event_a_key)
    schema_b = inspect_3c_event_schema(pair.event_b_key)

    features_a = schema_a["feature_columns"]
    features_b = schema_b["feature_columns"]

    common_features = sorted(
        features_a.intersection(features_b)
    )

    features_only_a = sorted(
        features_a.difference(features_b)
    )

    features_only_b = sorted(
        features_b.difference(features_a)
    )

    if not common_features:
        raise ValueError(
            f"{pair.event_a_key} and {pair.event_b_key}: "
            "no common modeling features remain."
        )

    timestamps_a = TIMESTAMP_CACHE[
        pair.event_a_key
    ]["timestamp_index"]

    timestamps_b = TIMESTAMP_CACHE[
        pair.event_b_key
    ]["timestamp_index"]

    overlap_start = to_3c_utc(pair.overlap_start)
    overlap_end = to_3c_utc(pair.overlap_end)

    timestamps_a_in_overlap = timestamps_a[
        (timestamps_a >= overlap_start)
        & (timestamps_a <= overlap_end)
    ]

    timestamps_b_in_overlap = timestamps_b[
        (timestamps_b >= overlap_start)
        & (timestamps_b <= overlap_end)
    ]

    shared_timestamps = (
        timestamps_a_in_overlap
        .intersection(timestamps_b_in_overlap)
        .sort_values()
    )

    rows_a = load_3c_shared_rows(
        schema=schema_a,
        selected_features=common_features,
        target_timestamps=shared_timestamps,
    )

    rows_b = load_3c_shared_rows(
        schema=schema_b,
        selected_features=common_features,
        target_timestamps=shared_timestamps,
    )

    merged = rows_a.merge(
        rows_b,
        on="timestamp_utc",
        how="inner",
        suffixes=("_event_a", "_event_b"),
        validate="one_to_one",
    )

    row_matches = pd.Series(
        True,
        index=merged.index,
        dtype=bool,
    )

    comparable_cells = 0
    matching_cells = 0
    differing_cells = 0
    differing_features = []

    for feature in common_features:
        result = compare_3c_feature(
            merged[f"{feature}_event_a"],
            merged[f"{feature}_event_b"],
        )

        row_matches &= result["matches"]

        comparable_cells += result["comparable_cells"]
        matching_cells += result["matching_cells"]
        differing_cells += result["differing_cells"]

        if result["differing_cells"] > 0:
            differing_features.append(feature)

    identical_measurement_rows = int(
        row_matches.sum()
    )

    if (
        features_only_a
        or features_only_b
    ):
        corrected_relationship = (
            "schema_mismatch_review_required"
        )

    elif bool(row_matches.all()):
        corrected_relationship = (
            "exact_measurement_collision"
        )

    elif identical_measurement_rows > 0:
        corrected_relationship = (
            "partial_measurement_collision"
        )

    else:
        corrected_relationship = (
            "timestamp_only_collision"
        )

    corrected_records.append(
        {
            "farm": pair.farm,
            "asset_id": pair.asset_id,
            "event_a_key": pair.event_a_key,
            "event_b_key": pair.event_b_key,
            "event_a_label": pair.event_a_label,
            "event_b_label": pair.event_b_label,
            "timestamps_event_a_full":
                len(timestamps_a),
            "timestamps_event_b_full":
                len(timestamps_b),
            "timestamps_a_in_metadata_overlap":
                len(timestamps_a_in_overlap),
            "timestamps_b_in_metadata_overlap":
                len(timestamps_b_in_overlap),
            "shared_timestamps_compared":
                len(shared_timestamps),
            "shared_percent_event_a_full": round(
                100 * len(shared_timestamps)
                / len(timestamps_a),
                6,
            ),
            "shared_percent_event_b_full": round(
                100 * len(shared_timestamps)
                / len(timestamps_b),
                6,
            ),
            "shared_percent_overlap_a": round(
                100 * len(shared_timestamps)
                / len(timestamps_a_in_overlap),
                6,
            ),
            "shared_percent_overlap_b": round(
                100 * len(shared_timestamps)
                / len(timestamps_b_in_overlap),
                6,
            ),
            "modeling_features_compared":
                len(common_features),
            "features_only_event_a":
                len(features_only_a),
            "features_only_event_b":
                len(features_only_b),
            "differing_modeling_features":
                len(differing_features),
            "differing_modeling_cells":
                differing_cells,
            "measurement_cell_match_percent": (
                round(
                    100 * matching_cells
                    / comparable_cells,
                    6,
                )
                if comparable_cells > 0
                else np.nan
            ),
            "identical_measurement_rows":
                identical_measurement_rows,
            "identical_measurement_percent": round(
                100 * identical_measurement_rows
                / len(merged),
                6,
            ),
            "corrected_relationship":
                corrected_relationship,
        }
    )

    for timestamp in merged.loc[
        row_matches,
        "timestamp_utc",
    ]:
        conflicting_timestamp_records.append(
            {
                "farm": pair.farm,
                "asset_id": pair.asset_id,
                "event_a_key": pair.event_a_key,
                "event_b_key": pair.event_b_key,
                "event_a_label": pair.event_a_label,
                "event_b_label": pair.event_b_label,
                "timestamp_utc": timestamp,
            }
        )


# ---------------------------------------------------------
# 3. Construct audit outputs
# ---------------------------------------------------------

CORRECTED_CROSS_LABEL_AUDIT = (
    pd.DataFrame(corrected_records)
    .sort_values(
        ["farm", "asset_id", "event_a_key"]
    )
    .reset_index(drop=True)
)

CONFLICTING_MEASUREMENT_TIMESTAMPS = pd.DataFrame(
    conflicting_timestamp_records,
    columns=[
        "farm",
        "asset_id",
        "event_a_key",
        "event_b_key",
        "event_a_label",
        "event_b_label",
        "timestamp_utc",
    ],
)

print("Corrected cross-label measurement audit:")
print(
    CORRECTED_CROSS_LABEL_AUDIT.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# 4. Final counts and interpretation
# ---------------------------------------------------------

exact_collision_pairs = int(
    CORRECTED_CROSS_LABEL_AUDIT[
        "corrected_relationship"
    ].eq("exact_measurement_collision").sum()
)

partial_collision_pairs = int(
    CORRECTED_CROSS_LABEL_AUDIT[
        "corrected_relationship"
    ].eq("partial_measurement_collision").sum()
)

timestamp_only_pairs = int(
    CORRECTED_CROSS_LABEL_AUDIT[
        "corrected_relationship"
    ].eq("timestamp_only_collision").sum()
)

conflicting_measurement_rows = int(
    len(CONFLICTING_MEASUREMENT_TIMESTAMPS)
)

print("\nCell 3D completed successfully.")
print(
    "Exact cross-label measurement-collision pairs:",
    exact_collision_pairs,
)
print(
    "Partial cross-label measurement-collision pairs:",
    partial_collision_pairs,
)
print(
    "Timestamp-only cross-label collision pairs:",
    timestamp_only_pairs,
)
print(
    "Measurement-equivalent rows with conflicting labels:",
    conflicting_measurement_rows,
)

if exact_collision_pairs > 0:
    print(
        "\nWARNING: After excluding identifiers, complete "
        "sensor vectors occur under opposite labels. These "
        "rows cannot remain in both supervised classes."
    )

if timestamp_only_pairs > 0:
    print(
        "\nTimestamp-only collisions must not be deduplicated "
        "using timestamp. Use (event_key, timestamp) as the "
        "observation key and asset_key as the split group."
    )

Corrected cross-label measurement audit:
farm asset_id     event_a_key     event_b_key event_a_label event_b_label  timestamps_event_a_full  timestamps_event_b_full  timestamps_a_in_metadata_overlap  timestamps_b_in_metadata_overlap  shared_timestamps_compared  shared_percent_event_a_full  shared_percent_event_b_full  shared_percent_overlap_a  shared_percent_overlap_b  modeling_features_compared  features_only_event_a  features_only_event_b  differing_modeling_features  differing_modeling_cells  measurement_cell_match_percent  identical_measurement_rows  identical_measurement_percent      corrected_relationship
   B        7 farm_b_event_27 farm_b_event_87       anomaly        normal                    62268                    55356                              2305                              2305                        2305                     3.701741                     4.163957                     100.0                     100.0                         253                      0 

In [23]:
# Cell 3E — verify whether timestamp sharing extends
# beyond the official metadata-overlap intervals.
# No observations or labels are modified.

import numpy as np
import pandas as pd


required_objects = [
    "STRICT_OVERLAPS",
    "TIMESTAMP_CACHE",
    "CORRECTED_CROSS_LABEL_AUDIT",
    "to_3c_utc",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing required notebook objects: {missing_objects}"
    )


scope_records = []

cross_label_pairs = STRICT_OVERLAPS.loc[
    STRICT_OVERLAPS["label_relationship"].eq("cross_label")
]

for pair in cross_label_pairs.itertuples(index=False):
    timestamps_a = TIMESTAMP_CACHE[
        pair.event_a_key
    ]["timestamp_index"]

    timestamps_b = TIMESTAMP_CACHE[
        pair.event_b_key
    ]["timestamp_index"]

    overlap_start = to_3c_utc(pair.overlap_start)
    overlap_end = to_3c_utc(pair.overlap_end)

    shared_full_files = (
        timestamps_a
        .intersection(timestamps_b)
        .sort_values()
    )

    shared_inside_overlap = shared_full_files[
        (shared_full_files >= overlap_start)
        & (shared_full_files <= overlap_end)
    ]

    shared_outside_overlap = shared_full_files[
        (shared_full_files < overlap_start)
        | (shared_full_files > overlap_end)
    ]

    corrected_row = (
        CORRECTED_CROSS_LABEL_AUDIT.loc[
            (
                CORRECTED_CROSS_LABEL_AUDIT[
                    "event_a_key"
                ].eq(pair.event_a_key)
            )
            & (
                CORRECTED_CROSS_LABEL_AUDIT[
                    "event_b_key"
                ].eq(pair.event_b_key)
            )
        ]
    )

    if len(corrected_row) != 1:
        raise ValueError(
            f"Expected one corrected-audit row for "
            f"{pair.event_a_key} and {pair.event_b_key}."
        )

    previously_compared = int(
        corrected_row.iloc[0][
            "shared_timestamps_compared"
        ]
    )

    if len(shared_inside_overlap) != previously_compared:
        raise ValueError(
            f"{pair.event_a_key} and {pair.event_b_key}: "
            "inside-overlap count differs from Cell 3D."
        )

    if len(shared_outside_overlap) == 0:
        timestamp_scope = (
            "confined_to_metadata_overlap"
        )
    else:
        timestamp_scope = (
            "extends_outside_metadata_overlap"
        )

    scope_records.append(
        {
            "farm": pair.farm,
            "asset_id": pair.asset_id,
            "event_a_key": pair.event_a_key,
            "event_b_key": pair.event_b_key,
            "timestamps_event_a_full":
                len(timestamps_a),
            "timestamps_event_b_full":
                len(timestamps_b),
            "shared_timestamps_full_files":
                len(shared_full_files),
            "shared_inside_metadata_overlap":
                len(shared_inside_overlap),
            "shared_outside_metadata_overlap":
                len(shared_outside_overlap),
            "full_file_shared_percent_a": round(
                100
                * len(shared_full_files)
                / len(timestamps_a),
                6,
            ),
            "full_file_shared_percent_b": round(
                100
                * len(shared_full_files)
                / len(timestamps_b),
                6,
            ),
            "first_shared_outside_utc": (
                shared_outside_overlap.min()
                if len(shared_outside_overlap) > 0
                else pd.NaT
            ),
            "last_shared_outside_utc": (
                shared_outside_overlap.max()
                if len(shared_outside_overlap) > 0
                else pd.NaT
            ),
            "timestamp_scope": timestamp_scope,
        }
    )


FULL_FILE_TIMESTAMP_SCOPE_AUDIT = (
    pd.DataFrame(scope_records)
    .sort_values(
        ["farm", "asset_id", "event_a_key"]
    )
    .reset_index(drop=True)
)

print("Full-file timestamp-scope audit:")
print(
    FULL_FILE_TIMESTAMP_SCOPE_AUDIT.to_string(
        index=False
    )
)


pairs_sharing_outside = int(
    FULL_FILE_TIMESTAMP_SCOPE_AUDIT[
        "shared_outside_metadata_overlap"
    ].gt(0).sum()
)

timestamps_shared_outside = int(
    FULL_FILE_TIMESTAMP_SCOPE_AUDIT[
        "shared_outside_metadata_overlap"
    ].sum()
)

print("\nCell 3E completed successfully.")
print(
    "Cross-label pairs sharing timestamps outside "
    "metadata overlap:",
    pairs_sharing_outside,
)
print(
    "Total shared timestamps outside metadata overlap:",
    timestamps_shared_outside,
)

if pairs_sharing_outside == 0:
    print(
        "\nAll cross-label timestamp sharing is confined "
        "to the audited metadata-overlap intervals. "
        "Farm B can therefore use a targeted two-sided "
        "exclusion manifest rather than quarantining an "
        "entire event file."
    )
else:
    print(
        "\nAdditional shared timestamps exist outside the "
        "audited intervals. Their feature vectors must be "
        "compared before finalizing the exclusion manifest."
    )

Full-file timestamp-scope audit:
farm asset_id     event_a_key     event_b_key  timestamps_event_a_full  timestamps_event_b_full  shared_timestamps_full_files  shared_inside_metadata_overlap  shared_outside_metadata_overlap  full_file_shared_percent_a  full_file_shared_percent_b  first_shared_outside_utc   last_shared_outside_utc                  timestamp_scope
   B        7 farm_b_event_27 farm_b_event_87                    62268                    55356                         55356                            2305                            53051                   88.899595                  100.000000 2022-09-13 23:00:00+00:00 2023-10-04 23:00:00+00:00 extends_outside_metadata_overlap
   C       34 farm_c_event_56  farm_c_event_4                    53416                    56449                         53391                            1551                            51840                   99.953198                   94.582721 2022-07-28 10:00:00+00:00 2023-08-13 04:20:00+00:00 exte

In [24]:
# Cell 3F — compare all cross-label shared timestamps
# in bounded-memory blocks.
# No rows, labels, or source files are modified.

from itertools import zip_longest

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 1. Validate prerequisites
# ---------------------------------------------------------

required_objects = [
    "STRICT_OVERLAPS",
    "TIMESTAMP_CACHE",
    "EXCLUDED_NONFEATURE_COLUMNS",
    "inspect_3c_event_schema",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing required notebook objects: {missing_objects}"
    )

# Ensure identifiers cannot enter the comparison.
EXCLUDED_NONFEATURE_COLUMNS.add("id")

FULL_FILE_BLOCK_ROWS = 256
FULL_FILE_READ_CHUNK_ROWS = 512


# ---------------------------------------------------------
# 2. Stream shared rows in chronological blocks
# ---------------------------------------------------------

def iter_3f_shared_blocks(
    schema,
    selected_features,
    target_timestamps,
    block_rows=FULL_FILE_BLOCK_ROWS,
):
    target_timestamps = (
        pd.DatetimeIndex(target_timestamps)
        .sort_values()
        .unique()
    )

    normalized_to_raw = schema["normalized_to_raw"]
    timestamp_raw = schema["timestamp_raw"]

    raw_feature_columns = [
        normalized_to_raw[feature]
        for feature in selected_features
    ]

    use_columns = [
        timestamp_raw,
        *raw_feature_columns,
    ]

    rename_map = {
        timestamp_raw: "__timestamp_raw",
    }

    rename_map.update(
        {
            normalized_to_raw[feature]: feature
            for feature in selected_features
        }
    )

    reader = pd.read_csv(
        schema["event_path"],
        sep=None,
        engine="python",
        quotechar="'",
        dtype=str,
        keep_default_na=False,
        skipinitialspace=True,
        usecols=use_columns,
        chunksize=FULL_FILE_READ_CHUNK_ROWS,
    )

    buffer = None
    emitted_rows = 0

    def validate_block_order(block):
        nonlocal emitted_rows

        observed = pd.DatetimeIndex(
            block["timestamp_utc"]
        )

        expected = target_timestamps[
            emitted_rows:
            emitted_rows + len(block)
        ]

        if not observed.equals(expected):
            raise ValueError(
                f"{schema['event_key']}: shared rows are not "
                "unique and chronologically aligned with the "
                "cached timestamp index."
            )

        emitted_rows += len(block)

    for chunk in reader:
        chunk = chunk.rename(columns=rename_map)

        parsed_timestamps = pd.to_datetime(
            (
                chunk["__timestamp_raw"]
                .astype("string")
                .str.strip()
                .str.strip("'\"")
                .str.strip()
            ),
            errors="coerce",
            format="mixed",
            utc=True,
        )

        keep_mask = parsed_timestamps.isin(
            target_timestamps
        )

        if not keep_mask.any():
            continue

        retained = chunk.loc[
            keep_mask,
            selected_features,
        ].copy()

        retained.insert(
            0,
            "timestamp_utc",
            parsed_timestamps.loc[
                keep_mask
            ].to_numpy(),
        )

        for feature in selected_features:
            cleaned = (
                retained[feature]
                .astype("string")
                .str.strip()
                .str.strip("'\"")
                .str.strip()
            )

            retained[feature] = cleaned.mask(
                cleaned.eq(""),
                pd.NA,
            )

        if buffer is None:
            buffer = retained.reset_index(drop=True)
        else:
            buffer = pd.concat(
                [buffer, retained],
                ignore_index=True,
            )

        while len(buffer) >= block_rows:
            block = (
                buffer.iloc[:block_rows]
                .copy()
                .reset_index(drop=True)
            )

            buffer = (
                buffer.iloc[block_rows:]
                .copy()
                .reset_index(drop=True)
            )

            validate_block_order(block)
            yield block

    if buffer is not None and not buffer.empty:
        block = buffer.reset_index(drop=True)
        validate_block_order(block)
        yield block

    if emitted_rows != len(target_timestamps):
        raise ValueError(
            f"{schema['event_key']}: recovered "
            f"{emitted_rows:,} of "
            f"{len(target_timestamps):,} target timestamps."
        )


# ---------------------------------------------------------
# 3. Compare one feature block
# ---------------------------------------------------------

def compare_3f_feature_block(series_a, series_b):
    missing_a = series_a.isna()
    missing_b = series_b.isna()

    both_missing = missing_a & missing_b
    one_missing = missing_a ^ missing_b
    jointly_present = ~missing_a & ~missing_b

    matches = pd.Series(
        False,
        index=series_a.index,
        dtype=bool,
    )

    # Joint missingness is row-equivalent, but it is excluded
    # from the comparable-cell denominator.
    matches.loc[both_missing] = True

    exact_string_match = (
        jointly_present
        & series_a.eq(series_b).fillna(False)
    )

    matches.loc[exact_string_match] = True

    unresolved = jointly_present & ~exact_string_match

    max_absolute_difference = np.nan

    if unresolved.any():
        numeric_a = pd.to_numeric(
            series_a.loc[unresolved],
            errors="coerce",
        )

        numeric_b = pd.to_numeric(
            series_b.loc[unresolved],
            errors="coerce",
        )

        both_numeric = (
            numeric_a.notna()
            & numeric_b.notna()
        )

        if both_numeric.any():
            numeric_indexes = numeric_a.index[
                both_numeric
            ]

            values_a = numeric_a.loc[
                numeric_indexes
            ].to_numpy(dtype=float)

            values_b = numeric_b.loc[
                numeric_indexes
            ].to_numpy(dtype=float)

            numeric_matches = np.isclose(
                values_a,
                values_b,
                rtol=1e-10,
                atol=1e-12,
                equal_nan=True,
            )

            matches.loc[numeric_indexes] = (
                numeric_matches
            )

            absolute_differences = np.abs(
                values_a - values_b
            )

            finite_differences = (
                absolute_differences[
                    np.isfinite(
                        absolute_differences
                    )
                ]
            )

            if len(finite_differences) > 0:
                max_absolute_difference = float(
                    finite_differences.max()
                )

    comparable = ~both_missing

    return {
        "matches": matches,
        "comparable_cells": int(
            comparable.sum()
        ),
        "matching_cells": int(
            (matches & comparable).sum()
        ),
        "differing_cells": int(
            (~matches).sum()
        ),
        "both_missing": int(
            both_missing.sum()
        ),
        "one_missing": int(
            one_missing.sum()
        ),
        "max_absolute_difference":
            max_absolute_difference,
    }


# ---------------------------------------------------------
# 4. Compare each cross-label pair
# ---------------------------------------------------------

pair_records = []
feature_records = []
collision_frames = []

cross_label_pairs = STRICT_OVERLAPS.loc[
    STRICT_OVERLAPS[
        "label_relationship"
    ].eq("cross_label")
]

for pair in cross_label_pairs.itertuples(index=False):
    schema_a = inspect_3c_event_schema(
        pair.event_a_key
    )

    schema_b = inspect_3c_event_schema(
        pair.event_b_key
    )

    features_a = schema_a["feature_columns"]
    features_b = schema_b["feature_columns"]

    common_features = sorted(
        features_a.intersection(features_b)
    )

    features_only_a = sorted(
        features_a.difference(features_b)
    )

    features_only_b = sorted(
        features_b.difference(features_a)
    )

    if not common_features:
        raise ValueError(
            f"{pair.event_a_key} and "
            f"{pair.event_b_key}: no common "
            "modeling features."
        )

    timestamps_a = TIMESTAMP_CACHE[
        pair.event_a_key
    ]["timestamp_index"]

    timestamps_b = TIMESTAMP_CACHE[
        pair.event_b_key
    ]["timestamp_index"]

    shared_timestamps = (
        timestamps_a
        .intersection(timestamps_b)
        .sort_values()
    )

    if len(shared_timestamps) == 0:
        raise ValueError(
            f"{pair.event_a_key} and "
            f"{pair.event_b_key}: no shared timestamps."
        )

    blocks_a = iter_3f_shared_blocks(
        schema=schema_a,
        selected_features=common_features,
        target_timestamps=shared_timestamps,
    )

    blocks_b = iter_3f_shared_blocks(
        schema=schema_b,
        selected_features=common_features,
        target_timestamps=shared_timestamps,
    )

    total_comparable_cells = 0
    total_matching_cells = 0
    total_differing_cells = 0
    identical_measurement_rows = 0
    processed_rows = 0

    feature_comparable = {
        feature: 0
        for feature in common_features
    }

    feature_differing = {
        feature: 0
        for feature in common_features
    }

    feature_max_difference = {
        feature: np.nan
        for feature in common_features
    }

    for block_a, block_b in zip_longest(
        blocks_a,
        blocks_b,
        fillvalue=None,
    ):
        if block_a is None or block_b is None:
            raise ValueError(
                f"{pair.event_a_key} and "
                f"{pair.event_b_key}: unequal block counts."
            )

        timestamps_block_a = pd.DatetimeIndex(
            block_a["timestamp_utc"]
        )

        timestamps_block_b = pd.DatetimeIndex(
            block_b["timestamp_utc"]
        )

        if not timestamps_block_a.equals(
            timestamps_block_b
        ):
            raise ValueError(
                f"{pair.event_a_key} and "
                f"{pair.event_b_key}: block timestamps "
                "are not aligned."
            )

        block_row_matches = pd.Series(
            True,
            index=block_a.index,
            dtype=bool,
        )

        for feature in common_features:
            result = compare_3f_feature_block(
                block_a[feature],
                block_b[feature],
            )

            block_row_matches &= result["matches"]

            total_comparable_cells += (
                result["comparable_cells"]
            )

            total_matching_cells += (
                result["matching_cells"]
            )

            total_differing_cells += (
                result["differing_cells"]
            )

            feature_comparable[feature] += (
                result["comparable_cells"]
            )

            feature_differing[feature] += (
                result["differing_cells"]
            )

            block_maximum = result[
                "max_absolute_difference"
            ]

            if pd.notna(block_maximum):
                previous_maximum = (
                    feature_max_difference[feature]
                )

                feature_max_difference[feature] = (
                    block_maximum
                    if pd.isna(previous_maximum)
                    else max(
                        previous_maximum,
                        block_maximum,
                    )
                )

        block_identical_count = int(
            block_row_matches.sum()
        )

        identical_measurement_rows += (
            block_identical_count
        )

        processed_rows += len(block_a)

        if block_identical_count > 0:
            collision_frames.append(
                pd.DataFrame(
                    {
                        "farm": pair.farm,
                        "asset_id": pair.asset_id,
                        "event_a_key":
                            pair.event_a_key,
                        "event_b_key":
                            pair.event_b_key,
                        "event_a_label":
                            pair.event_a_label,
                        "event_b_label":
                            pair.event_b_label,
                        "timestamp_utc":
                            block_a.loc[
                                block_row_matches,
                                "timestamp_utc",
                            ].to_numpy(),
                    }
                )
            )

    if processed_rows != len(shared_timestamps):
        raise ValueError(
            f"{pair.event_a_key} and "
            f"{pair.event_b_key}: compared "
            f"{processed_rows:,} of "
            f"{len(shared_timestamps):,} shared rows."
        )

    differing_measurement_rows = (
        processed_rows
        - identical_measurement_rows
    )

    differing_modeling_features = sum(
        count > 0
        for count in feature_differing.values()
    )

    if features_only_a or features_only_b:
        full_file_relationship = (
            "schema_mismatch_review_required"
        )

    elif identical_measurement_rows == processed_rows:
        full_file_relationship = (
            "exact_measurement_collision"
        )

    elif identical_measurement_rows > 0:
        full_file_relationship = (
            "partial_measurement_collision"
        )

    else:
        full_file_relationship = (
            "timestamp_only_collision"
        )

    pair_records.append(
        {
            "farm": pair.farm,
            "asset_id": pair.asset_id,
            "event_a_key": pair.event_a_key,
            "event_b_key": pair.event_b_key,
            "event_a_label": pair.event_a_label,
            "event_b_label": pair.event_b_label,
            "timestamps_event_a_full":
                len(timestamps_a),
            "timestamps_event_b_full":
                len(timestamps_b),
            "shared_timestamps_compared":
                processed_rows,
            "shared_percent_event_a": round(
                100
                * processed_rows
                / len(timestamps_a),
                6,
            ),
            "shared_percent_event_b": round(
                100
                * processed_rows
                / len(timestamps_b),
                6,
            ),
            "modeling_features_compared":
                len(common_features),
            "features_only_event_a":
                len(features_only_a),
            "features_only_event_b":
                len(features_only_b),
            "differing_modeling_features":
                differing_modeling_features,
            "differing_measurement_cells":
                total_differing_cells,
            "measurement_cell_match_percent": (
                round(
                    100
                    * total_matching_cells
                    / total_comparable_cells,
                    6,
                )
                if total_comparable_cells > 0
                else np.nan
            ),
            "identical_measurement_rows":
                identical_measurement_rows,
            "identical_measurement_percent":
                round(
                    100
                    * identical_measurement_rows
                    / processed_rows,
                    6,
                ),
            "differing_measurement_rows":
                differing_measurement_rows,
            "full_file_relationship":
                full_file_relationship,
        }
    )

    for feature in common_features:
        differing_cells = (
            feature_differing[feature]
        )

        if differing_cells == 0:
            continue

        comparable_cells = (
            feature_comparable[feature]
        )

        feature_records.append(
            {
                "farm": pair.farm,
                "asset_id": pair.asset_id,
                "event_a_key": pair.event_a_key,
                "event_b_key": pair.event_b_key,
                "feature": feature,
                "comparable_cells":
                    comparable_cells,
                "differing_cells":
                    differing_cells,
                "difference_percent": round(
                    100
                    * differing_cells
                    / comparable_cells,
                    6,
                )
                if comparable_cells > 0
                else np.nan,
                "max_absolute_difference":
                    feature_max_difference[feature],
            }
        )

    print(
        f"Compared {pair.event_a_key} with "
        f"{pair.event_b_key}: "
        f"{processed_rows:,} shared timestamps."
    )


# ---------------------------------------------------------
# 5. Construct full-file audit outputs
# ---------------------------------------------------------

FULL_FILE_MEASUREMENT_AUDIT = (
    pd.DataFrame(pair_records)
    .sort_values(
        ["farm", "asset_id", "event_a_key"]
    )
    .reset_index(drop=True)
)

FULL_FILE_FEATURE_DIFFERENCE_AUDIT = (
    pd.DataFrame(feature_records)
)

if not FULL_FILE_FEATURE_DIFFERENCE_AUDIT.empty:
    FULL_FILE_FEATURE_DIFFERENCE_AUDIT = (
        FULL_FILE_FEATURE_DIFFERENCE_AUDIT
        .sort_values(
            [
                "event_a_key",
                "event_b_key",
                "difference_percent",
                "differing_cells",
                "feature",
            ],
            ascending=[
                True,
                True,
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

collision_columns = [
    "farm",
    "asset_id",
    "event_a_key",
    "event_b_key",
    "event_a_label",
    "event_b_label",
    "timestamp_utc",
]

if collision_frames:
    FULL_FILE_CONFLICTING_MEASUREMENT_TIMESTAMPS = (
        pd.concat(
            collision_frames,
            ignore_index=True,
        )
        .sort_values(
            ["farm", "asset_id", "timestamp_utc"]
        )
        .reset_index(drop=True)
    )
else:
    FULL_FILE_CONFLICTING_MEASUREMENT_TIMESTAMPS = (
        pd.DataFrame(columns=collision_columns)
    )


# ---------------------------------------------------------
# 6. Display results
# ---------------------------------------------------------

print("\nFull-file cross-label measurement audit:")
print(
    FULL_FILE_MEASUREMENT_AUDIT.to_string(
        index=False
    )
)

print("\nMost frequently differing features:")

if FULL_FILE_FEATURE_DIFFERENCE_AUDIT.empty:
    print("  None")
else:
    top_differences = (
        FULL_FILE_FEATURE_DIFFERENCE_AUDIT
        .groupby(
            ["event_a_key", "event_b_key"],
            group_keys=False,
        )
        .head(15)
    )

    print(top_differences.to_string(index=False))


# ---------------------------------------------------------
# 7. Final counts
# ---------------------------------------------------------

exact_collision_pairs = int(
    FULL_FILE_MEASUREMENT_AUDIT[
        "full_file_relationship"
    ].eq("exact_measurement_collision").sum()
)

partial_collision_pairs = int(
    FULL_FILE_MEASUREMENT_AUDIT[
        "full_file_relationship"
    ].eq("partial_measurement_collision").sum()
)

timestamp_only_pairs = int(
    FULL_FILE_MEASUREMENT_AUDIT[
        "full_file_relationship"
    ].eq("timestamp_only_collision").sum()
)

measurement_equivalent_conflicting_rows = int(
    len(
        FULL_FILE_CONFLICTING_MEASUREMENT_TIMESTAMPS
    )
)

print("\nCell 3F completed successfully.")
print(
    "Full-file exact measurement-collision pairs:",
    exact_collision_pairs,
)
print(
    "Full-file partial measurement-collision pairs:",
    partial_collision_pairs,
)
print(
    "Full-file timestamp-only collision pairs:",
    timestamp_only_pairs,
)
print(
    "Measurement-equivalent timestamps carrying "
    "opposite file labels:",
    measurement_equivalent_conflicting_rows,
)

print(
    "\nNo exclusion manifest has been created. "
    "File-level labels remain unmodified."
)

Compared farm_b_event_27 with farm_b_event_87: 55,356 shared timestamps.
Compared farm_c_event_56 with farm_c_event_4: 53,391 shared timestamps.

Full-file cross-label measurement audit:
farm asset_id     event_a_key     event_b_key event_a_label event_b_label  timestamps_event_a_full  timestamps_event_b_full  shared_timestamps_compared  shared_percent_event_a  shared_percent_event_b  modeling_features_compared  features_only_event_a  features_only_event_b  differing_modeling_features  differing_measurement_cells  measurement_cell_match_percent  identical_measurement_rows  identical_measurement_percent  differing_measurement_rows        full_file_relationship
   B        7 farm_b_event_27 farm_b_event_87       anomaly        normal                    62268                    55356                       55356               88.899595              100.000000                         253                      0                      0                            1                         2154 

In [25]:
# Cell 3G — correct non-measurement feature roles and
# inspect split/label provenance.
# No rows, labels, or source files are modified.

from collections import Counter
from itertools import zip_longest
import re

import pandas as pd


# ---------------------------------------------------------
# 1. Validate prerequisites
# ---------------------------------------------------------

required_objects = [
    "STRICT_OVERLAPS",
    "TIMESTAMP_CACHE",
    "EXCLUDED_NONFEATURE_COLUMNS",
    "FULL_FILE_MEASUREMENT_AUDIT",
    "FULL_FILE_FEATURE_DIFFERENCE_AUDIT",
    "inspect_3c_event_schema",
    "iter_3f_shared_blocks",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing required notebook objects: {missing_objects}"
    )


cross_label_pairs = STRICT_OVERLAPS.loc[
    STRICT_OVERLAPS[
        "label_relationship"
    ].eq("cross_label")
].copy()

target_event_keys = sorted(
    set(cross_label_pairs["event_a_key"])
    | set(cross_label_pairs["event_b_key"])
)

# Inspect schemas before adding train_test to the exclusion
# set so its raw values remain available for the audit below.
PRE_CORRECTION_SCHEMAS = {
    event_key: inspect_3c_event_schema(event_key)
    for event_key in target_event_keys
}

NONMEASUREMENT_COLUMNS = {
    "id",
    "train_test",
}

EXCLUDED_NONFEATURE_COLUMNS.update(
    NONMEASUREMENT_COLUMNS
)


# ---------------------------------------------------------
# 2. Correct Cell 3F relationships algebraically
# ---------------------------------------------------------

corrected_records = []

for row in FULL_FILE_MEASUREMENT_AUDIT.itertuples(
    index=False
):
    schema_a = PRE_CORRECTION_SCHEMAS[
        row.event_a_key
    ]
    schema_b = PRE_CORRECTION_SCHEMAS[
        row.event_b_key
    ]

    pre_correction_common_features = (
        set(schema_a["feature_columns"])
        .intersection(schema_b["feature_columns"])
    )

    excluded_common_columns = sorted(
        pre_correction_common_features
        .intersection(NONMEASUREMENT_COLUMNS)
    )

    pair_differences = (
        FULL_FILE_FEATURE_DIFFERENCE_AUDIT.loc[
            (
                FULL_FILE_FEATURE_DIFFERENCE_AUDIT[
                    "event_a_key"
                ].eq(row.event_a_key)
            )
            & (
                FULL_FILE_FEATURE_DIFFERENCE_AUDIT[
                    "event_b_key"
                ].eq(row.event_b_key)
            )
        ]
        .copy()
    )

    removed_differences = pair_differences.loc[
        pair_differences["feature"].isin(
            NONMEASUREMENT_COLUMNS
        )
    ]

    retained_differences = pair_differences.loc[
        ~pair_differences["feature"].isin(
            NONMEASUREMENT_COLUMNS
        )
    ]

    shared_rows = int(
        row.shared_timestamps_compared
    )

    full_row_witnesses = retained_differences.loc[
        retained_differences[
            "differing_cells"
        ].eq(shared_rows)
    ]

    if retained_differences.empty:
        corrected_relationship = (
            "exact_measurement_collision"
        )
        identical_measurement_rows = shared_rows
        differing_measurement_rows = 0
        witness_feature = None

    elif not full_row_witnesses.empty:
        # One retained measurement feature differs at every
        # timestamp, proving that no complete rows are equal.
        corrected_relationship = (
            "timestamp_only_collision"
        )
        identical_measurement_rows = 0
        differing_measurement_rows = shared_rows
        witness_feature = (
            full_row_witnesses.iloc[0]["feature"]
        )

    else:
        corrected_relationship = (
            "row_level_recomparison_required"
        )
        identical_measurement_rows = pd.NA
        differing_measurement_rows = pd.NA
        witness_feature = None

    corrected_records.append(
        {
            "farm": row.farm,
            "asset_id": row.asset_id,
            "event_a_key": row.event_a_key,
            "event_b_key": row.event_b_key,
            "event_a_label": row.event_a_label,
            "event_b_label": row.event_b_label,
            "shared_timestamps": shared_rows,
            "exclusive_timestamps_event_a": (
                int(row.timestamps_event_a_full)
                - shared_rows
            ),
            "exclusive_timestamps_event_b": (
                int(row.timestamps_event_b_full)
                - shared_rows
            ),
            "excluded_common_columns": (
                ", ".join(excluded_common_columns)
            ),
            "measurement_features_compared": (
                int(row.modeling_features_compared)
                - len(excluded_common_columns)
            ),
            "removed_metadata_difference_cells": int(
                removed_differences[
                    "differing_cells"
                ].sum()
            ),
            "differing_measurement_features": int(
                len(retained_differences)
            ),
            "differing_measurement_cells": int(
                retained_differences[
                    "differing_cells"
                ].sum()
            ),
            "identical_measurement_rows": (
                identical_measurement_rows
            ),
            "differing_measurement_rows": (
                differing_measurement_rows
            ),
            "full_row_difference_witness": (
                witness_feature
            ),
            "corrected_relationship": (
                corrected_relationship
            ),
        }
    )


CORRECTED_FULL_FILE_MEASUREMENT_AUDIT = (
    pd.DataFrame(corrected_records)
    .sort_values(
        ["farm", "asset_id", "event_a_key"]
    )
    .reset_index(drop=True)
)

print("Corrected full-file measurement audit:")
print(
    CORRECTED_FULL_FILE_MEASUREMENT_AUDIT
    .to_string(index=False)
)


resolved_rows = (
    CORRECTED_FULL_FILE_MEASUREMENT_AUDIT[
        "identical_measurement_rows"
    ]
    .dropna()
    .astype(int)
)

measurement_equivalence_classes = int(
    resolved_rows.sum()
)

conflicting_source_rows = int(
    2 * measurement_equivalence_classes
)

print("\nCorrected collision counts:")
print(
    "Measurement-equivalence classes carrying "
    "opposite labels:",
    measurement_equivalence_classes,
)
print(
    "Source rows participating in those classes:",
    conflicting_source_rows,
)


# ---------------------------------------------------------
# 3. Audit train_test values on shared timestamps
# ---------------------------------------------------------

def normalize_3g_metadata_value(value):
    if pd.isna(value):
        return "<MISSING>"

    cleaned = (
        str(value)
        .strip()
        .strip("'\"")
        .strip()
    )

    return cleaned if cleaned else "<EMPTY>"


train_test_summary_records = []
train_test_value_records = []

for pair in cross_label_pairs.itertuples(
    index=False
):
    schema_a = PRE_CORRECTION_SCHEMAS[
        pair.event_a_key
    ]
    schema_b = PRE_CORRECTION_SCHEMAS[
        pair.event_b_key
    ]

    for schema in [schema_a, schema_b]:
        if (
            "train_test"
            not in schema["normalized_to_raw"]
        ):
            raise ValueError(
                f"{schema['event_key']}: train_test "
                "column was not found."
            )

    shared_timestamps = (
        TIMESTAMP_CACHE[
            pair.event_a_key
        ]["timestamp_index"]
        .intersection(
            TIMESTAMP_CACHE[
                pair.event_b_key
            ]["timestamp_index"]
        )
        .sort_values()
    )

    blocks_a = iter_3f_shared_blocks(
        schema=schema_a,
        selected_features=["train_test"],
        target_timestamps=shared_timestamps,
    )

    blocks_b = iter_3f_shared_blocks(
        schema=schema_b,
        selected_features=["train_test"],
        target_timestamps=shared_timestamps,
    )

    value_pair_counts = Counter()

    processed_rows = 0
    matching_rows = 0
    differing_rows = 0
    first_differing_timestamp = pd.NaT
    last_differing_timestamp = pd.NaT

    for block_a, block_b in zip_longest(
        blocks_a,
        blocks_b,
        fillvalue=None,
    ):
        if block_a is None or block_b is None:
            raise ValueError(
                f"{pair.event_a_key} and "
                f"{pair.event_b_key}: unequal block counts."
            )

        timestamps_a = pd.DatetimeIndex(
            block_a["timestamp_utc"]
        )
        timestamps_b = pd.DatetimeIndex(
            block_b["timestamp_utc"]
        )

        if not timestamps_a.equals(timestamps_b):
            raise ValueError(
                f"{pair.event_a_key} and "
                f"{pair.event_b_key}: timestamps "
                "are not aligned."
            )

        values_a = block_a["train_test"].map(
            normalize_3g_metadata_value
        )
        values_b = block_b["train_test"].map(
            normalize_3g_metadata_value
        )

        same_mask = values_a.eq(values_b)

        processed_rows += len(block_a)
        matching_rows += int(same_mask.sum())
        differing_rows += int((~same_mask).sum())

        for value_pair, count in (
            pd.DataFrame(
                {
                    "value_event_a": values_a,
                    "value_event_b": values_b,
                }
            )
            .value_counts(sort=False)
            .items()
        ):
            value_pair_counts[value_pair] += int(count)

        if (~same_mask).any():
            differing_timestamps = timestamps_a[
                (~same_mask).to_numpy()
            ]

            block_first = differing_timestamps.min()
            block_last = differing_timestamps.max()

            if pd.isna(first_differing_timestamp):
                first_differing_timestamp = block_first
            else:
                first_differing_timestamp = min(
                    first_differing_timestamp,
                    block_first,
                )

            if pd.isna(last_differing_timestamp):
                last_differing_timestamp = block_last
            else:
                last_differing_timestamp = max(
                    last_differing_timestamp,
                    block_last,
                )

    if processed_rows != len(shared_timestamps):
        raise ValueError(
            f"{pair.event_a_key} and "
            f"{pair.event_b_key}: processed "
            f"{processed_rows:,} of "
            f"{len(shared_timestamps):,} rows."
        )

    train_test_summary_records.append(
        {
            "farm": pair.farm,
            "asset_id": pair.asset_id,
            "event_a_key": pair.event_a_key,
            "event_b_key": pair.event_b_key,
            "shared_timestamps": processed_rows,
            "matching_train_test_values": matching_rows,
            "differing_train_test_values": differing_rows,
            "first_difference_utc": (
                first_differing_timestamp
            ),
            "last_difference_utc": (
                last_differing_timestamp
            ),
        }
    )

    for (
        value_event_a,
        value_event_b,
    ), count in sorted(value_pair_counts.items()):
        train_test_value_records.append(
            {
                "event_a_key": pair.event_a_key,
                "event_b_key": pair.event_b_key,
                "train_test_event_a": value_event_a,
                "train_test_event_b": value_event_b,
                "timestamps": count,
                "values_match": (
                    value_event_a == value_event_b
                ),
            }
        )


TRAIN_TEST_PAIR_SUMMARY = pd.DataFrame(
    train_test_summary_records
)

TRAIN_TEST_VALUE_PAIR_AUDIT = pd.DataFrame(
    train_test_value_records
)

print("\ntrain_test pair summary:")
print(
    TRAIN_TEST_PAIR_SUMMARY.to_string(index=False)
)

print("\ntrain_test value combinations:")
print(
    TRAIN_TEST_VALUE_PAIR_AUDIT.to_string(index=False)
)


# ---------------------------------------------------------
# 4. Show available event-label provenance
# ---------------------------------------------------------

def normalize_3g_column_name(column):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(column).strip().lower(),
    ).strip("_")


provenance_tokens = (
    "event",
    "label",
    "class",
    "target",
    "normal",
    "anomaly",
    "fault",
    "failure",
    "status",
    "type",
    "category",
    "description",
    "source",
    "file",
    "path",
    "farm",
    "asset",
    "start",
    "end",
    "date",
    "time",
    "train",
    "test",
    "split",
)


def select_3g_provenance_columns(frame):
    selected = []

    for column in frame.columns:
        normalized = normalize_3g_column_name(column)

        if any(
            token in normalized
            for token in provenance_tokens
        ):
            selected.append(column)

    return selected


strict_columns = select_3g_provenance_columns(
    cross_label_pairs
)

CROSS_LABEL_STRICT_PROVENANCE = (
    cross_label_pairs[strict_columns]
    .reset_index(drop=True)
)

print("\nCross-label STRICT_OVERLAPS provenance:")
print(
    CROSS_LABEL_STRICT_PROVENANCE.to_string(
        index=False
    )
)


# Locate source-level tables containing an exact event_key
# column and the four relevant event records.
SOURCE_EVENT_PROVENANCE_TABLES = {}

for object_name, candidate in list(
    globals().items()
):
    if not isinstance(candidate, pd.DataFrame):
        continue

    normalized_column_map = {
        normalize_3g_column_name(column): column
        for column in candidate.columns
    }

    if "event_key" not in normalized_column_map:
        continue

    event_key_column = normalized_column_map[
        "event_key"
    ]

    matching_mask = (
        candidate[event_key_column]
        .astype("string")
        .isin(target_event_keys)
    )

    if not matching_mask.any():
        continue

    selected_columns = (
        select_3g_provenance_columns(candidate)
    )

    if event_key_column not in selected_columns:
        selected_columns.insert(0, event_key_column)

    SOURCE_EVENT_PROVENANCE_TABLES[
        object_name
    ] = (
        candidate.loc[
            matching_mask,
            selected_columns,
        ]
        .copy()
        .reset_index(drop=True)
    )


print("\nSource event-provenance candidates:")

if not SOURCE_EVENT_PROVENANCE_TABLES:
    print(
        "  No DataFrame with an exact event_key column "
        "was found."
    )
else:
    for table_name, table in (
        SOURCE_EVENT_PROVENANCE_TABLES.items()
    ):
        print(f"\n[{table_name}]")
        print(table.to_string(index=False))


print(
    "\nCell 3G completed successfully. "
    "No exclusion, deduplication, or relabeling "
    "has been performed."
)

Corrected full-file measurement audit:
farm asset_id     event_a_key     event_b_key event_a_label event_b_label  shared_timestamps  exclusive_timestamps_event_a  exclusive_timestamps_event_b excluded_common_columns  measurement_features_compared  removed_metadata_difference_cells  differing_measurement_features  differing_measurement_cells  identical_measurement_rows  differing_measurement_rows full_row_difference_witness      corrected_relationship
   B        7 farm_b_event_27 farm_b_event_87       anomaly        normal              55356                          6912                             0              train_test                            252                               2154                               0                            0                       55356                           0                        None exact_measurement_collision
   C       34 farm_c_event_56  farm_c_event_4        normal       anomaly              53391                            25       

In [26]:
# Cell 3H — verify that event ID boundaries and timestamp
# boundaries select exactly the same official event rows.
# No source data, labels, or split assignments are modified.

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 1. Validate prerequisites
# ---------------------------------------------------------

required_objects = [
    "EVENTS",
    "STRICT_OVERLAPS",
    "CORRECTED_FULL_FILE_MEASUREMENT_AUDIT",
    "inspect_3c_event_schema",
    "to_3c_utc",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing required notebook objects: {missing_objects}"
    )


cross_label_pairs = STRICT_OVERLAPS.loc[
    STRICT_OVERLAPS[
        "label_relationship"
    ].eq("cross_label")
].copy()

target_event_keys = sorted(
    set(cross_label_pairs["event_a_key"])
    | set(cross_label_pairs["event_b_key"])
)


# ---------------------------------------------------------
# 2. Helpers
# ---------------------------------------------------------

def clean_3h_string(series):
    return (
        series.astype("string")
        .str.strip()
        .str.strip("'\"")
        .str.strip()
    )


def parse_3h_integral_id(value, field_name, event_key):
    numeric = pd.to_numeric(
        pd.Series([value]),
        errors="coerce",
    ).iloc[0]

    if pd.isna(numeric):
        raise ValueError(
            f"{event_key}: {field_name} is not numeric."
        )

    if float(numeric) != int(numeric):
        raise ValueError(
            f"{event_key}: {field_name} is not integral."
        )

    return int(numeric)


# ---------------------------------------------------------
# 3. Validate each affected event slice
# ---------------------------------------------------------

boundary_records = []
OFFICIAL_EVENT_TIMESTAMP_CACHE = {}

for event_key in target_event_keys:
    event_rows = EVENTS.loc[
        EVENTS["event_key"].eq(event_key)
    ]

    if len(event_rows) != 1:
        raise ValueError(
            f"{event_key}: expected exactly one EVENTS row."
        )

    event = event_rows.iloc[0]
    schema = inspect_3c_event_schema(event_key)

    normalized_to_raw = schema["normalized_to_raw"]

    if "id" not in normalized_to_raw:
        raise ValueError(
            f"{event_key}: raw id column was not found."
        )

    id_raw = normalized_to_raw["id"]
    timestamp_raw = schema["timestamp_raw"]

    frame = pd.read_csv(
        schema["event_path"],
        sep=None,
        engine="python",
        quotechar="'",
        dtype=str,
        keep_default_na=False,
        skipinitialspace=True,
        usecols=[id_raw, timestamp_raw],
    ).rename(
        columns={
            id_raw: "__id_raw",
            timestamp_raw: "__timestamp_raw",
        }
    )

    cleaned_ids = clean_3h_string(
        frame["__id_raw"]
    )

    numeric_ids = pd.to_numeric(
        cleaned_ids,
        errors="coerce",
    )

    nonintegral_ids = (
        numeric_ids.notna()
        & numeric_ids.ne(np.floor(numeric_ids))
    )

    if nonintegral_ids.any():
        raise ValueError(
            f"{event_key}: non-integral raw IDs found."
        )

    parsed_ids = numeric_ids.astype("Int64")

    parsed_timestamps = pd.to_datetime(
        clean_3h_string(frame["__timestamp_raw"]),
        errors="coerce",
        format="mixed",
        utc=True,
    )

    metadata_start_id = parse_3h_integral_id(
        event["event_start_id"],
        "event_start_id",
        event_key,
    )

    metadata_end_id = parse_3h_integral_id(
        event["event_end_id"],
        "event_end_id",
        event_key,
    )

    if metadata_end_id < metadata_start_id:
        raise ValueError(
            f"{event_key}: event_end_id precedes "
            "event_start_id."
        )

    metadata_start_utc = to_3c_utc(
        event["event_start"]
    )

    metadata_end_utc = to_3c_utc(
        event["event_end"]
    )

    expected_rows_from_ids = (
        metadata_end_id
        - metadata_start_id
        + 1
    )

    id_window_mask = (
        parsed_ids.ge(metadata_start_id).fillna(False)
        & parsed_ids.le(metadata_end_id).fillna(False)
    )

    time_window_mask = (
        parsed_timestamps.notna()
        & parsed_timestamps.ge(metadata_start_utc)
        & parsed_timestamps.le(metadata_end_utc)
    )

    start_positions = frame.index[
        parsed_ids.eq(
            metadata_start_id
        ).fillna(False)
    ]

    end_positions = frame.index[
        parsed_ids.eq(
            metadata_end_id
        ).fillna(False)
    ]

    timestamp_at_start_id = pd.NaT
    timestamp_at_end_id = pd.NaT

    if len(start_positions) == 1:
        timestamp_at_start_id = parsed_timestamps.loc[
            start_positions[0]
        ]

    if len(end_positions) == 1:
        timestamp_at_end_id = parsed_timestamps.loc[
            end_positions[0]
        ]

    start_boundary_matches = (
        len(start_positions) == 1
        and timestamp_at_start_id == metadata_start_utc
    )

    end_boundary_matches = (
        len(end_positions) == 1
        and timestamp_at_end_id == metadata_end_utc
    )

    window_ids = (
        parsed_ids.loc[id_window_mask]
        .astype("int64")
        .to_numpy()
    )

    expected_id_sequence = np.arange(
        metadata_start_id,
        metadata_end_id + 1,
        dtype=np.int64,
    )

    id_sequence_contiguous = np.array_equal(
        window_ids,
        expected_id_sequence,
    )

    selector_mismatch_rows = int(
        (id_window_mask ^ time_window_mask).sum()
    )

    official_timestamps = pd.DatetimeIndex(
        parsed_timestamps.loc[
            id_window_mask
        ].dropna()
    )

    official_timestamp_count = len(
        official_timestamps
    )

    official_timestamps_unique = (
        official_timestamps.is_unique
    )

    official_timestamps_monotonic = (
        official_timestamps.is_monotonic_increasing
    )

    selector_is_exact = all(
        [
            int(id_window_mask.sum())
            == expected_rows_from_ids,
            int(time_window_mask.sum())
            == expected_rows_from_ids,
            selector_mismatch_rows == 0,
            start_boundary_matches,
            end_boundary_matches,
            id_sequence_contiguous,
            official_timestamp_count
            == expected_rows_from_ids,
            official_timestamps_unique,
            official_timestamps_monotonic,
        ]
    )

    selector_status = (
        "exact_id_time_alignment"
        if selector_is_exact
        else "review_required"
    )

    OFFICIAL_EVENT_TIMESTAMP_CACHE[event_key] = (
        official_timestamps.sort_values()
    )

    boundary_records.append(
        {
            "farm": event["farm"],
            "asset_id": event["asset_id"],
            "event_key": event_key,
            "event_label": event["event_label"],
            "metadata_start_id": metadata_start_id,
            "metadata_end_id": metadata_end_id,
            "expected_rows_from_ids":
                expected_rows_from_ids,
            "rows_selected_by_id":
                int(id_window_mask.sum()),
            "rows_selected_by_time":
                int(time_window_mask.sum()),
            "selector_mismatch_rows":
                selector_mismatch_rows,
            "timestamp_at_start_id":
                timestamp_at_start_id,
            "metadata_start_utc":
                metadata_start_utc,
            "start_boundary_matches":
                start_boundary_matches,
            "timestamp_at_end_id":
                timestamp_at_end_id,
            "metadata_end_utc":
                metadata_end_utc,
            "end_boundary_matches":
                end_boundary_matches,
            "id_sequence_contiguous":
                id_sequence_contiguous,
            "official_timestamps_unique":
                official_timestamps_unique,
            "official_timestamps_monotonic":
                official_timestamps_monotonic,
            "selector_status": selector_status,
        }
    )


EVENT_BOUNDARY_SELECTOR_AUDIT = (
    pd.DataFrame(boundary_records)
    .sort_values(
        ["farm", "asset_id", "event_key"]
    )
    .reset_index(drop=True)
)

print("Event-boundary selector audit:")
print(
    EVENT_BOUNDARY_SELECTOR_AUDIT.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# 4. Recount conflicts after official-window clipping
# ---------------------------------------------------------

official_pair_records = []

for pair in cross_label_pairs.itertuples(index=False):
    official_a = OFFICIAL_EVENT_TIMESTAMP_CACHE[
        pair.event_a_key
    ]

    official_b = OFFICIAL_EVENT_TIMESTAMP_CACHE[
        pair.event_b_key
    ]

    shared_official_timestamps = (
        official_a
        .intersection(official_b)
        .sort_values()
    )

    corrected_row = (
        CORRECTED_FULL_FILE_MEASUREMENT_AUDIT.loc[
            (
                CORRECTED_FULL_FILE_MEASUREMENT_AUDIT[
                    "event_a_key"
                ].eq(pair.event_a_key)
            )
            & (
                CORRECTED_FULL_FILE_MEASUREMENT_AUDIT[
                    "event_b_key"
                ].eq(pair.event_b_key)
            )
        ]
    )

    if len(corrected_row) != 1:
        raise ValueError(
            f"Expected one corrected relationship for "
            f"{pair.event_a_key} and {pair.event_b_key}."
        )

    measurement_relationship = corrected_row.iloc[
        0
    ]["corrected_relationship"]

    label_to_key = {
        pair.event_a_label: pair.event_a_key,
        pair.event_b_label: pair.event_b_key,
    }

    if measurement_relationship == (
        "exact_measurement_collision"
    ):
        recommended_overlap_action = (
            "keep_anomaly_copy_exclude_normal_copy"
        )

    elif measurement_relationship == (
        "timestamp_only_collision"
    ):
        recommended_overlap_action = (
            "quarantine_both_overlap_slices"
        )

    else:
        recommended_overlap_action = (
            "manual_review_required"
        )

    official_pair_records.append(
        {
            "farm": pair.farm,
            "asset_id": pair.asset_id,
            "event_a_key": pair.event_a_key,
            "event_b_key": pair.event_b_key,
            "event_a_label": pair.event_a_label,
            "event_b_label": pair.event_b_label,
            "official_rows_event_a": len(official_a),
            "official_rows_event_b": len(official_b),
            "shared_official_timestamps":
                len(shared_official_timestamps),
            "shared_percent_official_a": round(
                100
                * len(shared_official_timestamps)
                / len(official_a),
                6,
            ),
            "shared_percent_official_b": round(
                100
                * len(shared_official_timestamps)
                / len(official_b),
                6,
            ),
            "official_conflicting_source_rows":
                2 * len(shared_official_timestamps),
            "measurement_relationship":
                measurement_relationship,
            "anomaly_event_key":
                label_to_key.get("anomaly"),
            "normal_event_key":
                label_to_key.get("normal"),
            "recommended_overlap_action":
                recommended_overlap_action,
        }
    )


OFFICIAL_WINDOW_CROSS_LABEL_AUDIT = (
    pd.DataFrame(official_pair_records)
    .sort_values(
        ["farm", "asset_id", "event_a_key"]
    )
    .reset_index(drop=True)
)

print("\nOfficial-window cross-label audit:")
print(
    OFFICIAL_WINDOW_CROSS_LABEL_AUDIT.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# 5. Final validation counts
# ---------------------------------------------------------

exactly_aligned_events = int(
    EVENT_BOUNDARY_SELECTOR_AUDIT[
        "selector_status"
    ].eq("exact_id_time_alignment").sum()
)

review_required_events = int(
    EVENT_BOUNDARY_SELECTOR_AUDIT[
        "selector_status"
    ].eq("review_required").sum()
)

official_conflicting_timestamps = int(
    OFFICIAL_WINDOW_CROSS_LABEL_AUDIT[
        "shared_official_timestamps"
    ].sum()
)

official_conflicting_source_rows = int(
    OFFICIAL_WINDOW_CROSS_LABEL_AUDIT[
        "official_conflicting_source_rows"
    ].sum()
)

exact_duplicate_conflict_timestamps = int(
    OFFICIAL_WINDOW_CROSS_LABEL_AUDIT.loc[
        OFFICIAL_WINDOW_CROSS_LABEL_AUDIT[
            "measurement_relationship"
        ].eq("exact_measurement_collision"),
        "shared_official_timestamps",
    ].sum()
)

timestamp_only_conflict_timestamps = int(
    OFFICIAL_WINDOW_CROSS_LABEL_AUDIT.loc[
        OFFICIAL_WINDOW_CROSS_LABEL_AUDIT[
            "measurement_relationship"
        ].eq("timestamp_only_collision"),
        "shared_official_timestamps",
    ].sum()
)

print("\nCell 3H completed successfully.")
print(
    "Events with exact ID/time boundary alignment:",
    exactly_aligned_events,
)
print(
    "Events requiring boundary review:",
    review_required_events,
)
print(
    "Official cross-label conflicting timestamps:",
    official_conflicting_timestamps,
)
print(
    "Official source rows requiring adjudication:",
    official_conflicting_source_rows,
)
print(
    "Official exact-duplicate conflict timestamps:",
    exact_duplicate_conflict_timestamps,
)
print(
    "Official timestamp-only conflict timestamps:",
    timestamp_only_conflict_timestamps,
)

print(
    "\nNo source row has been deleted, relabeled, "
    "deduplicated, or assigned to a new split."
)

Event-boundary selector audit:
farm asset_id       event_key event_label  metadata_start_id  metadata_end_id  expected_rows_from_ids  rows_selected_by_id  rows_selected_by_time  selector_mismatch_rows     timestamp_at_start_id        metadata_start_utc  start_boundary_matches       timestamp_at_end_id          metadata_end_utc  end_boundary_matches  id_sequence_contiguous  official_timestamps_unique  official_timestamps_monotonic         selector_status
   B        7 farm_b_event_27     anomaly              52619            61403                    8785                 8785                   8785                       0 2023-09-01 00:00:00+00:00 2023-09-01 00:00:00+00:00                    True 2023-11-01 00:00:00+00:00 2023-11-01 00:00:00+00:00                  True                    True                        True                           True exact_id_time_alignment
   B        7 farm_b_event_87      normal              52475            54779                    2305              

In [27]:
# Cell 3I — build a non-destructive row-label manifest.
#
# This cell:
#   1. clips every event file to its declared event window,
#   2. keeps the documented Farm B anomaly copy,
#   3. excludes the nested Farm B normal copy,
#   4. quarantines both Farm C conflicting slices,
#   5. leaves modeling splits unassigned.
#
# No source CSV is modified.

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 1. Validate prerequisites and policy gates
# ---------------------------------------------------------

required_objects = [
    "EVENTS",
    "EVENT_BOUNDARY_SELECTOR_AUDIT",
    "OFFICIAL_WINDOW_CROSS_LABEL_AUDIT",
    "OFFICIAL_EVENT_TIMESTAMP_CACHE",
    "EXCLUDED_NONFEATURE_COLUMNS",
    "inspect_3c_event_schema",
    "to_3c_utc",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing required notebook objects: {missing_objects}"
    )


if EVENTS["event_key"].duplicated().any():
    duplicate_keys = sorted(
        EVENTS.loc[
            EVENTS["event_key"].duplicated(
                keep=False
            ),
            "event_key",
        ]
        .astype(str)
        .unique()
    )

    raise ValueError(
        "EVENTS contains duplicate event keys: "
        f"{duplicate_keys}"
    )


if not EVENT_BOUNDARY_SELECTOR_AUDIT[
    "selector_status"
].eq("exact_id_time_alignment").all():
    raise ValueError(
        "At least one affected event lacks exact "
        "ID/time boundary alignment."
    )


required_excluded_columns = {
    "id",
    "train_test",
}

missing_exclusions = (
    required_excluded_columns
    - set(EXCLUDED_NONFEATURE_COLUMNS)
)

if missing_exclusions:
    raise ValueError(
        "Required non-modeling columns are not excluded: "
        f"{sorted(missing_exclusions)}"
    )


recognized_actions = {
    "keep_anomaly_copy_exclude_normal_copy",
    "quarantine_both_overlap_slices",
}

unknown_actions = (
    set(
        OFFICIAL_WINDOW_CROSS_LABEL_AUDIT[
            "recommended_overlap_action"
        ]
    )
    - recognized_actions
)

if unknown_actions:
    raise ValueError(
        "Unresolved cross-label actions remain: "
        f"{sorted(unknown_actions)}"
    )


# ---------------------------------------------------------
# 2. Helpers
# ---------------------------------------------------------

EMPTY_UTC_INDEX = pd.DatetimeIndex([], tz="UTC")


def clean_3i_string(series):
    return (
        series.astype("string")
        .str.strip()
        .str.strip("'\"")
        .str.strip()
    )


def parse_3i_integral_id(
    value,
    field_name,
    event_key,
):
    numeric = pd.to_numeric(
        pd.Series([value]),
        errors="coerce",
    ).iloc[0]

    if pd.isna(numeric):
        raise ValueError(
            f"{event_key}: {field_name} is not numeric."
        )

    if float(numeric) != int(numeric):
        raise ValueError(
            f"{event_key}: {field_name} is not integral."
        )

    return int(numeric)


def add_3i_timestamp_rule(
    rule_map,
    event_key,
    timestamps,
):
    timestamps = pd.DatetimeIndex(
        timestamps
    ).sort_values()

    existing = rule_map.get(
        event_key,
        EMPTY_UTC_INDEX,
    )

    rule_map[event_key] = (
        existing.union(timestamps).sort_values()
    )


# ---------------------------------------------------------
# 3. Convert pair-level decisions into timestamp rules
# ---------------------------------------------------------

EXACT_ANOMALY_KEEP_TIMESTAMPS = {}
EXACT_NORMAL_EXCLUDE_TIMESTAMPS = {}
QUARANTINE_TIMESTAMPS = {}

policy_pair_records = []

for pair in (
    OFFICIAL_WINDOW_CROSS_LABEL_AUDIT
    .itertuples(index=False)
):
    timestamps_a = (
        OFFICIAL_EVENT_TIMESTAMP_CACHE[
            pair.event_a_key
        ]
    )

    timestamps_b = (
        OFFICIAL_EVENT_TIMESTAMP_CACHE[
            pair.event_b_key
        ]
    )

    shared_timestamps = (
        timestamps_a
        .intersection(timestamps_b)
        .sort_values()
    )

    expected_shared = int(
        pair.shared_official_timestamps
    )

    if len(shared_timestamps) != expected_shared:
        raise ValueError(
            f"{pair.event_a_key} and "
            f"{pair.event_b_key}: expected "
            f"{expected_shared:,} official shared "
            f"timestamps, found {len(shared_timestamps):,}."
        )

    action = pair.recommended_overlap_action

    if action == (
        "keep_anomaly_copy_exclude_normal_copy"
    ):
        if pair.measurement_relationship != (
            "exact_measurement_collision"
        ):
            raise ValueError(
                "Exact-copy policy assigned to a "
                "non-exact measurement relationship."
            )

        anomaly_key = pair.anomaly_event_key
        normal_key = pair.normal_event_key

        add_3i_timestamp_rule(
            EXACT_ANOMALY_KEEP_TIMESTAMPS,
            anomaly_key,
            shared_timestamps,
        )

        add_3i_timestamp_rule(
            EXACT_NORMAL_EXCLUDE_TIMESTAMPS,
            normal_key,
            shared_timestamps,
        )

    elif action == (
        "quarantine_both_overlap_slices"
    ):
        if pair.measurement_relationship != (
            "timestamp_only_collision"
        ):
            raise ValueError(
                "Quarantine policy assigned to an "
                "unexpected measurement relationship."
            )

        add_3i_timestamp_rule(
            QUARANTINE_TIMESTAMPS,
            pair.event_a_key,
            shared_timestamps,
        )

        add_3i_timestamp_rule(
            QUARANTINE_TIMESTAMPS,
            pair.event_b_key,
            shared_timestamps,
        )

    policy_pair_records.append(
        {
            "farm": pair.farm,
            "asset_id": pair.asset_id,
            "event_a_key": pair.event_a_key,
            "event_b_key": pair.event_b_key,
            "shared_official_timestamps":
                expected_shared,
            "measurement_relationship":
                pair.measurement_relationship,
            "policy_action": action,
        }
    )


POLICY_PAIR_INPUT = pd.DataFrame(
    policy_pair_records
)


# A timestamp cannot receive incompatible rules within
# the same event file.

rule_event_keys = (
    set(EXACT_ANOMALY_KEEP_TIMESTAMPS)
    | set(EXACT_NORMAL_EXCLUDE_TIMESTAMPS)
    | set(QUARANTINE_TIMESTAMPS)
)

for event_key in rule_event_keys:
    anomaly_keep = (
        EXACT_ANOMALY_KEEP_TIMESTAMPS.get(
            event_key,
            EMPTY_UTC_INDEX,
        )
    )

    normal_exclude = (
        EXACT_NORMAL_EXCLUDE_TIMESTAMPS.get(
            event_key,
            EMPTY_UTC_INDEX,
        )
    )

    quarantine = QUARANTINE_TIMESTAMPS.get(
        event_key,
        EMPTY_UTC_INDEX,
    )

    incompatible_count = sum(
        [
            len(
                anomaly_keep.intersection(
                    normal_exclude
                )
            ),
            len(
                anomaly_keep.intersection(
                    quarantine
                )
            ),
            len(
                normal_exclude.intersection(
                    quarantine
                )
            ),
        ]
    )

    if incompatible_count:
        raise ValueError(
            f"{event_key}: incompatible manifest "
            "rules target the same timestamps."
        )


# ---------------------------------------------------------
# 4. Build the manifest for every source event file
# ---------------------------------------------------------

manifest_frames = []
summary_records = []
boundary_records = []

ALL_OFFICIAL_EVENT_TIMESTAMP_CACHE = {}
ELIGIBLE_TIMESTAMP_CACHE = {}

ordered_events = EVENTS.sort_values(
    ["farm", "asset_id", "event_start", "event_key"]
).reset_index(drop=True)


for event in ordered_events.itertuples(index=False):
    event_key = event.event_key
    event_label = (
        str(event.event_label)
        .strip()
        .lower()
    )

    schema = inspect_3c_event_schema(event_key)
    normalized_to_raw = schema["normalized_to_raw"]

    if "id" not in normalized_to_raw:
        raise ValueError(
            f"{event_key}: raw id column was not found."
        )

    id_raw = normalized_to_raw["id"]
    timestamp_raw = schema["timestamp_raw"]

    frame = pd.read_csv(
        schema["event_path"],
        sep=None,
        engine="python",
        quotechar="'",
        dtype=str,
        keep_default_na=False,
        skipinitialspace=True,
        usecols=[id_raw, timestamp_raw],
    ).rename(
        columns={
            id_raw: "__id_raw",
            timestamp_raw: "__timestamp_raw",
        }
    )

    cleaned_ids = clean_3i_string(
        frame["__id_raw"]
    )

    numeric_ids = pd.to_numeric(
        cleaned_ids,
        errors="coerce",
    )

    if numeric_ids.isna().any():
        raise ValueError(
            f"{event_key}: invalid raw IDs found."
        )

    nonintegral_ids = numeric_ids.ne(
        np.floor(numeric_ids)
    )

    if nonintegral_ids.any():
        raise ValueError(
            f"{event_key}: non-integral raw IDs found."
        )

    parsed_ids = numeric_ids.astype("int64")

    parsed_timestamps = pd.to_datetime(
        clean_3i_string(
            frame["__timestamp_raw"]
        ),
        errors="coerce",
        format="mixed",
        utc=True,
    )

    if parsed_timestamps.isna().any():
        raise ValueError(
            f"{event_key}: invalid timestamps found."
        )

    metadata_start_id = parse_3i_integral_id(
        event.event_start_id,
        "event_start_id",
        event_key,
    )

    metadata_end_id = parse_3i_integral_id(
        event.event_end_id,
        "event_end_id",
        event_key,
    )

    metadata_start_utc = to_3c_utc(
        event.event_start
    )

    metadata_end_utc = to_3c_utc(
        event.event_end
    )

    expected_official_rows = (
        metadata_end_id
        - metadata_start_id
        + 1
    )

    if expected_official_rows <= 0:
        raise ValueError(
            f"{event_key}: invalid event ID interval."
        )

    id_window_mask = (
        parsed_ids.ge(metadata_start_id)
        & parsed_ids.le(metadata_end_id)
    )

    time_window_mask = (
        parsed_timestamps.ge(metadata_start_utc)
        & parsed_timestamps.le(metadata_end_utc)
    )

    selector_mismatch_rows = int(
        (
            id_window_mask
            ^ time_window_mask
        ).sum()
    )

    official_mask = (
        id_window_mask
        & time_window_mask
    )

    official_ids = (
        parsed_ids.loc[official_mask]
        .to_numpy(dtype=np.int64)
    )

    expected_id_sequence = np.arange(
        metadata_start_id,
        metadata_end_id + 1,
        dtype=np.int64,
    )

    official_timestamps = pd.DatetimeIndex(
        parsed_timestamps.loc[official_mask]
    )

    selector_is_exact = all(
        [
            int(id_window_mask.sum())
            == expected_official_rows,
            int(time_window_mask.sum())
            == expected_official_rows,
            int(official_mask.sum())
            == expected_official_rows,
            selector_mismatch_rows == 0,
            np.array_equal(
                official_ids,
                expected_id_sequence,
            ),
            official_timestamps.is_unique,
            official_timestamps.is_monotonic_increasing,
            official_timestamps[0]
            == metadata_start_utc,
            official_timestamps[-1]
            == metadata_end_utc,
        ]
    )

    selector_status = (
        "exact_id_time_alignment"
        if selector_is_exact
        else "review_required"
    )

    boundary_records.append(
        {
            "farm": event.farm,
            "asset_id": event.asset_id,
            "event_key": event_key,
            "event_label": event_label,
            "source_rows": len(frame),
            "expected_official_rows":
                expected_official_rows,
            "selected_official_rows":
                int(official_mask.sum()),
            "selector_mismatch_rows":
                selector_mismatch_rows,
            "selector_status": selector_status,
        }
    )

    if not selector_is_exact:
        raise ValueError(
            f"{event_key}: full-dataset manifest creation "
            "stopped because its event boundaries are "
            "not exactly aligned."
        )

    ALL_OFFICIAL_EVENT_TIMESTAMP_CACHE[
        event_key
    ] = official_timestamps

    anomaly_keep_targets = (
        EXACT_ANOMALY_KEEP_TIMESTAMPS.get(
            event_key,
            EMPTY_UTC_INDEX,
        )
    )

    normal_exclude_targets = (
        EXACT_NORMAL_EXCLUDE_TIMESTAMPS.get(
            event_key,
            EMPTY_UTC_INDEX,
        )
    )

    quarantine_targets = (
        QUARANTINE_TIMESTAMPS.get(
            event_key,
            EMPTY_UTC_INDEX,
        )
    )

    for rule_name, targets in [
        (
            "documented anomaly retention",
            anomaly_keep_targets,
        ),
        (
            "duplicate normal exclusion",
            normal_exclude_targets,
        ),
        (
            "cross-label quarantine",
            quarantine_targets,
        ),
    ]:
        missing_targets = targets.difference(
            official_timestamps
        )

        if len(missing_targets):
            raise ValueError(
                f"{event_key}: {len(missing_targets):,} "
                f"{rule_name} timestamps fall outside "
                "the official event window."
            )

    if (
        len(anomaly_keep_targets)
        and event_label != "anomaly"
    ):
        raise ValueError(
            f"{event_key}: anomaly-retention rule "
            f"targets source label {event_label!r}."
        )

    if (
        len(normal_exclude_targets)
        and event_label != "normal"
    ):
        raise ValueError(
            f"{event_key}: normal-exclusion rule "
            f"targets source label {event_label!r}."
        )

    anomaly_keep_mask = (
        parsed_timestamps.isin(
            anomaly_keep_targets
        )
    )

    normal_exclude_mask = (
        parsed_timestamps.isin(
            normal_exclude_targets
        )
    )

    quarantine_mask = (
        parsed_timestamps.isin(
            quarantine_targets
        )
    )

    decision = pd.Series(
        "exclude_outside_event_window",
        index=frame.index,
        dtype="string",
    )

    decision.loc[official_mask] = (
        "keep_source_label"
    )

    decision.loc[anomaly_keep_mask] = (
        "keep_documented_anomaly"
    )

    decision.loc[normal_exclude_mask] = (
        "exclude_duplicate_normal_copy"
    )

    decision.loc[quarantine_mask] = (
        "quarantine_cross_label_measurement_conflict"
    )

    modeling_eligible = decision.str.startswith(
        "keep_"
    )

    final_label = pd.Series(
        pd.NA,
        index=frame.index,
        dtype="string",
    )

    final_label.loc[modeling_eligible] = (
        event_label
    )

    split_assignment = pd.Series(
        pd.NA,
        index=frame.index,
        dtype="string",
    )

    manifest_piece = pd.DataFrame(
        {
            "farm": event.farm,
            "asset_id": event.asset_id,
            "asset_key": event.asset_key,
            "event_id": event.event_id,
            "event_key": event_key,
            "source_event_label": event_label,
            "source_row_index": np.arange(
                len(frame),
                dtype=np.int64,
            ),
            "raw_id": parsed_ids.to_numpy(),
            "timestamp_utc":
                parsed_timestamps.to_numpy(),
            "within_official_event_window":
                official_mask.to_numpy(),
            "manifest_decision":
                decision.to_numpy(),
            "modeling_eligible":
                modeling_eligible.to_numpy(),
            "final_label":
                final_label.to_numpy(),
            "split_assignment":
                split_assignment.to_numpy(),
        }
    )

    manifest_frames.append(manifest_piece)

    eligible_timestamps = pd.DatetimeIndex(
        manifest_piece.loc[
            manifest_piece["modeling_eligible"],
            "timestamp_utc",
        ]
    )

    ELIGIBLE_TIMESTAMP_CACHE[event_key] = (
        eligible_timestamps.sort_values()
    )

    summary_records.append(
        {
            "farm": event.farm,
            "asset_id": event.asset_id,
            "event_key": event_key,
            "source_event_label": event_label,
            "source_rows": len(frame),
            "official_window_rows":
                int(official_mask.sum()),
            "outside_window_excluded": int(
                decision.eq(
                    "exclude_outside_event_window"
                ).sum()
            ),
            "ordinary_source_label_kept": int(
                decision.eq(
                    "keep_source_label"
                ).sum()
            ),
            "documented_anomaly_copy_kept": int(
                decision.eq(
                    "keep_documented_anomaly"
                ).sum()
            ),
            "duplicate_normal_copy_excluded": int(
                decision.eq(
                    "exclude_duplicate_normal_copy"
                ).sum()
            ),
            "measurement_conflict_quarantined": int(
                decision.eq(
                    "quarantine_cross_label_"
                    "measurement_conflict"
                ).sum()
            ),
            "modeling_eligible_rows": int(
                modeling_eligible.sum()
            ),
        }
    )


ALL_EVENT_BOUNDARY_AUDIT = pd.DataFrame(
    boundary_records
)

ROW_LABEL_MANIFEST = pd.concat(
    manifest_frames,
    ignore_index=True,
)

ROW_LABEL_MANIFEST_SUMMARY = pd.DataFrame(
    summary_records
)


# ---------------------------------------------------------
# 5. Validate pair-level policy application
# ---------------------------------------------------------

pair_resolution_records = []

for pair in (
    OFFICIAL_WINDOW_CROSS_LABEL_AUDIT
    .itertuples(index=False)
):
    shared_timestamps = (
        ALL_OFFICIAL_EVENT_TIMESTAMP_CACHE[
            pair.event_a_key
        ]
        .intersection(
            ALL_OFFICIAL_EVENT_TIMESTAMP_CACHE[
                pair.event_b_key
            ]
        )
        .sort_values()
    )

    rows_a = ROW_LABEL_MANIFEST.loc[
        ROW_LABEL_MANIFEST["event_key"].eq(
            pair.event_a_key
        )
        & ROW_LABEL_MANIFEST[
            "timestamp_utc"
        ].isin(shared_timestamps)
    ]

    rows_b = ROW_LABEL_MANIFEST.loc[
        ROW_LABEL_MANIFEST["event_key"].eq(
            pair.event_b_key
        )
        & ROW_LABEL_MANIFEST[
            "timestamp_utc"
        ].isin(shared_timestamps)
    ]

    retained_anomaly_rows = 0
    excluded_normal_rows = 0
    quarantined_event_a_rows = 0
    quarantined_event_b_rows = 0

    action = pair.recommended_overlap_action

    if action == (
        "keep_anomaly_copy_exclude_normal_copy"
    ):
        anomaly_key = pair.anomaly_event_key
        normal_key = pair.normal_event_key

        anomaly_rows = (
            rows_a
            if pair.event_a_key == anomaly_key
            else rows_b
        )

        normal_rows = (
            rows_a
            if pair.event_a_key == normal_key
            else rows_b
        )

        retained_anomaly_rows = int(
            anomaly_rows[
                "manifest_decision"
            ].eq(
                "keep_documented_anomaly"
            ).sum()
        )

        excluded_normal_rows = int(
            normal_rows[
                "manifest_decision"
            ].eq(
                "exclude_duplicate_normal_copy"
            ).sum()
        )

    elif action == (
        "quarantine_both_overlap_slices"
    ):
        quarantine_decision = (
            "quarantine_cross_label_"
            "measurement_conflict"
        )

        quarantined_event_a_rows = int(
            rows_a[
                "manifest_decision"
            ].eq(
                quarantine_decision
            ).sum()
        )

        quarantined_event_b_rows = int(
            rows_b[
                "manifest_decision"
            ].eq(
                quarantine_decision
            ).sum()
        )

    remaining_shared_eligible = (
        ELIGIBLE_TIMESTAMP_CACHE[
            pair.event_a_key
        ]
        .intersection(
            ELIGIBLE_TIMESTAMP_CACHE[
                pair.event_b_key
            ]
        )
    )

    shared_count = len(shared_timestamps)

    if action == (
        "keep_anomaly_copy_exclude_normal_copy"
    ):
        policy_satisfied = all(
            [
                retained_anomaly_rows
                == shared_count,
                excluded_normal_rows
                == shared_count,
                len(remaining_shared_eligible) == 0,
            ]
        )

        excluded_source_rows = (
            excluded_normal_rows
        )

        retained_conflict_source_rows = (
            retained_anomaly_rows
        )

    else:
        policy_satisfied = all(
            [
                quarantined_event_a_rows
                == shared_count,
                quarantined_event_b_rows
                == shared_count,
                len(remaining_shared_eligible) == 0,
            ]
        )

        excluded_source_rows = (
            quarantined_event_a_rows
            + quarantined_event_b_rows
        )

        retained_conflict_source_rows = 0

    pair_resolution_records.append(
        {
            "farm": pair.farm,
            "asset_id": pair.asset_id,
            "event_a_key": pair.event_a_key,
            "event_b_key": pair.event_b_key,
            "policy_action": action,
            "shared_official_timestamps":
                shared_count,
            "conflicting_source_rows_before_policy":
                2 * shared_count,
            "retained_anomaly_copy_rows":
                retained_anomaly_rows,
            "excluded_normal_copy_rows":
                excluded_normal_rows,
            "quarantined_event_a_rows":
                quarantined_event_a_rows,
            "quarantined_event_b_rows":
                quarantined_event_b_rows,
            "excluded_source_rows_by_policy":
                excluded_source_rows,
            "retained_conflict_source_rows":
                retained_conflict_source_rows,
            "shared_timestamps_with_two_eligible_copies":
                len(remaining_shared_eligible),
            "policy_status": (
                "policy_satisfied"
                if policy_satisfied
                else "review_required"
            ),
        }
    )


CROSS_LABEL_MANIFEST_RESOLUTION_AUDIT = (
    pd.DataFrame(pair_resolution_records)
    .sort_values(
        ["farm", "asset_id", "event_a_key"]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 6. Global integrity checks
# ---------------------------------------------------------

if ROW_LABEL_MANIFEST[
    ["event_key", "source_row_index"]
].duplicated().any():
    raise ValueError(
        "Manifest contains duplicate source-row keys."
    )


eligible_mask = ROW_LABEL_MANIFEST[
    "modeling_eligible"
]

official_mask = ROW_LABEL_MANIFEST[
    "within_official_event_window"
]

if (
    eligible_mask
    & ~official_mask
).any():
    raise ValueError(
        "Rows outside official event windows were "
        "marked modeling-eligible."
    )


if ROW_LABEL_MANIFEST.loc[
    eligible_mask,
    "final_label",
].isna().any():
    raise ValueError(
        "At least one eligible row lacks a final label."
    )


if ROW_LABEL_MANIFEST.loc[
    ~eligible_mask,
    "final_label",
].notna().any():
    raise ValueError(
        "At least one excluded row retains a final label."
    )


if ROW_LABEL_MANIFEST[
    "split_assignment"
].notna().any():
    raise ValueError(
        "A split was assigned before asset-grouped "
        "split construction."
    )


if not CROSS_LABEL_MANIFEST_RESOLUTION_AUDIT[
    "policy_status"
].eq("policy_satisfied").all():
    raise ValueError(
        "At least one cross-label policy failed "
        "manifest validation."
    )


remaining_cross_label_duplicates = int(
    CROSS_LABEL_MANIFEST_RESOLUTION_AUDIT[
        "shared_timestamps_with_two_eligible_copies"
    ].sum()
)

if remaining_cross_label_duplicates != 0:
    raise ValueError(
        "Cross-label timestamp duplication remains "
        "among eligible rows."
    )


decision_counts = (
    ROW_LABEL_MANIFEST[
        "manifest_decision"
    ]
    .value_counts()
    .rename_axis("manifest_decision")
    .reset_index(name="rows")
)

LABEL_MANIFEST_DECISION_COUNTS = (
    decision_counts
)

FINAL_LABEL_COUNTS = (
    ROW_LABEL_MANIFEST.loc[
        eligible_mask
    ]
    .groupby(
        "final_label",
        dropna=False,
    )
    .size()
    .rename("eligible_rows")
    .reset_index()
    .sort_values("final_label")
    .reset_index(drop=True)
)


conflict_event_keys = sorted(
    set(
        OFFICIAL_WINDOW_CROSS_LABEL_AUDIT[
            "event_a_key"
        ]
    )
    | set(
        OFFICIAL_WINDOW_CROSS_LABEL_AUDIT[
            "event_b_key"
        ]
    )
)

AFFECTED_ROW_LABEL_MANIFEST_SUMMARY = (
    ROW_LABEL_MANIFEST_SUMMARY.loc[
        ROW_LABEL_MANIFEST_SUMMARY[
            "event_key"
        ].isin(conflict_event_keys)
    ]
    .sort_values(
        ["farm", "asset_id", "event_key"]
    )
    .reset_index(drop=True)
)


total_source_rows = len(
    ROW_LABEL_MANIFEST
)

total_official_rows = int(
    official_mask.sum()
)

outside_window_rows = int(
    ROW_LABEL_MANIFEST[
        "manifest_decision"
    ].eq(
        "exclude_outside_event_window"
    ).sum()
)

duplicate_normal_rows_excluded = int(
    ROW_LABEL_MANIFEST[
        "manifest_decision"
    ].eq(
        "exclude_duplicate_normal_copy"
    ).sum()
)

measurement_conflict_rows_quarantined = int(
    ROW_LABEL_MANIFEST[
        "manifest_decision"
    ].eq(
        "quarantine_cross_label_"
        "measurement_conflict"
    ).sum()
)

eligible_rows = int(
    eligible_mask.sum()
)

if total_source_rows != (
    outside_window_rows
    + duplicate_normal_rows_excluded
    + measurement_conflict_rows_quarantined
    + eligible_rows
):
    raise ValueError(
        "Manifest row-conservation check failed."
    )


# ---------------------------------------------------------
# 7. Display compact results
# ---------------------------------------------------------

print("Affected-event row-label manifest summary:")
print(
    AFFECTED_ROW_LABEL_MANIFEST_SUMMARY
    .to_string(index=False)
)

print("\nCross-label manifest resolution audit:")
print(
    CROSS_LABEL_MANIFEST_RESOLUTION_AUDIT
    .to_string(index=False)
)

print("\nManifest decision counts:")
print(
    LABEL_MANIFEST_DECISION_COUNTS
    .to_string(index=False)
)

print("\nEligible final-label counts:")
print(
    FINAL_LABEL_COUNTS.to_string(index=False)
)

print("\nCell 3I completed successfully.")
print(
    "Total source rows represented:",
    total_source_rows,
)
print(
    "Rows inside official event windows:",
    total_official_rows,
)
print(
    "Rows excluded outside official windows:",
    outside_window_rows,
)
print(
    "Nested normal duplicate rows excluded:",
    duplicate_normal_rows_excluded,
)
print(
    "Measurement-conflict rows quarantined:",
    measurement_conflict_rows_quarantined,
)
print(
    "Rows eligible before further deduplication:",
    eligible_rows,
)
print(
    "Cross-label timestamps retaining two "
    "eligible copies:",
    remaining_cross_label_duplicates,
)
print(
    "Assigned train/validation/test rows:",
    int(
        ROW_LABEL_MANIFEST[
            "split_assignment"
        ].notna().sum()
    ),
)

print(
    "\nThe inherited train_test field was not read. "
    "Source CSVs and source labels remain unchanged."
)

Affected-event row-label manifest summary:
farm asset_id       event_key source_event_label  source_rows  official_window_rows  outside_window_excluded  ordinary_source_label_kept  documented_anomaly_copy_kept  duplicate_normal_copy_excluded  measurement_conflict_quarantined  modeling_eligible_rows
   B        7 farm_b_event_27            anomaly        62268                  8785                    53483                        6480                          2305                               0                                 0                    8785
   B        7 farm_b_event_87             normal        55356                  2305                    53051                           0                             0                            2305                                 0                       0
   C       34  farm_c_event_4            anomaly        56449                  2737                    53712                        1186                             0                    

In [31]:
# Cell 3J — audit same-label eligible-row duplication.
#
# Row identity for source-duplication purposes is:
#   asset_key + timestamp_utc + final_label
#
# Measurement fingerprints distinguish:
#   1. exact same-label copies at the same timestamp,
#   2. same-label timestamp collisions with different values,
#   3. repeated measurement vectors at different timestamps.
#
# Repetition at different timestamps is reported separately
# and is not automatically treated as source duplication.
#
# No manifest decision, source label, source CSV, or split
# assignment is modified.

import hashlib

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 1. Validate prerequisites
# ---------------------------------------------------------

required_objects = [
    "EVENTS",
    "ROW_LABEL_MANIFEST",
    "ROW_LABEL_MANIFEST_SUMMARY",
    "EXCLUDED_NONFEATURE_COLUMNS",
    "inspect_3c_event_schema",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing required notebook objects: {missing_objects}"
    )


expected_eligible_rows = int(
    ROW_LABEL_MANIFEST["modeling_eligible"].sum()
)

if expected_eligible_rows != 214546:
    raise ValueError(
        "Unexpected Cell 3I eligible-row count: "
        f"{expected_eligible_rows:,}."
    )


if ROW_LABEL_MANIFEST[
    "split_assignment"
].notna().any():
    raise ValueError(
        "Cell 3J must run before split assignment."
    )


eligible_manifest = ROW_LABEL_MANIFEST.loc[
    ROW_LABEL_MANIFEST["modeling_eligible"]
].copy()


if eligible_manifest["final_label"].isna().any():
    raise ValueError(
        "At least one eligible row lacks a final label."
    )


eligible_manifest["final_label"] = (
    eligible_manifest["final_label"]
    .astype("string")
    .str.strip()
    .str.lower()
)

eligible_manifest["source_event_label"] = (
    eligible_manifest["source_event_label"]
    .astype("string")
    .str.strip()
    .str.lower()
)


if not eligible_manifest[
    "final_label"
].eq(
    eligible_manifest["source_event_label"]
).all():
    raise ValueError(
        "An eligible row appears to have been relabeled."
    )


eligible_manifest["timestamp_utc"] = pd.to_datetime(
    eligible_manifest["timestamp_utc"],
    errors="coerce",
    utc=True,
)

if eligible_manifest["timestamp_utc"].isna().any():
    raise ValueError(
        "At least one eligible manifest timestamp is invalid."
    )


numeric_source_indices = pd.to_numeric(
    eligible_manifest["source_row_index"],
    errors="coerce",
)

if numeric_source_indices.isna().any():
    raise ValueError(
        "At least one eligible source_row_index is invalid."
    )

if numeric_source_indices.ne(
    np.floor(numeric_source_indices)
).any():
    raise ValueError(
        "At least one eligible source_row_index "
        "is non-integral."
    )

eligible_manifest["source_row_index"] = (
    numeric_source_indices.astype("int64")
)


if eligible_manifest[
    ["event_key", "source_row_index"]
].duplicated().any():
    raise ValueError(
        "Eligible manifest source-row keys are not unique."
    )


# Each official event was previously verified to have unique
# timestamps. Reconfirm this after policy application.

if eligible_manifest[
    ["event_key", "timestamp_utc"]
].duplicated().any():
    raise ValueError(
        "An eligible event contains duplicate timestamps."
    )


# ---------------------------------------------------------
# 2. Measurement-schema helpers
# ---------------------------------------------------------

def normalize_3j_name(value):
    return str(value).strip().lower()


excluded_feature_names = {
    normalize_3j_name(name)
    for name in EXCLUDED_NONFEATURE_COLUMNS
}


# Explicit target and provenance safeguards. These columns
# must never participate in a measurement fingerprint.

prohibited_feature_names = {
    "id",
    "time_stamp",
    "timestamp",
    "train_test",
    "status_type_id",
    "status_label",
    "event_label",
    "source_event_label",
    "final_label",
    "label",
    "target",
}


# Apply both the notebook-wide exclusions and the explicit
# fingerprint leakage safeguards to every event file.

fingerprint_excluded_names = (
    excluded_feature_names
    | prohibited_feature_names
)


MISSING_3J_TOKENS = {
    "",
    "na",
    "nan",
    "n/a",
    "null",
    "none",
}


def fingerprint_3j_rows(numeric_values):
    """Return a SHA-256 fingerprint for every numeric row."""

    values = np.ascontiguousarray(
        numeric_values,
        dtype="<f8",
    ).copy()

    # Canonicalize representations that compare equal but
    # could otherwise have different binary encodings.

    values[values == 0.0] = 0.0
    values[np.isnan(values)] = np.nan

    return np.fromiter(
        (
            hashlib.sha256(
                row.tobytes(order="C")
            ).hexdigest()
            for row in values
        ),
        dtype="U64",
        count=len(values),
    )


# ---------------------------------------------------------
# 3. Fingerprint every eligible measurement row
# ---------------------------------------------------------

measurement_index_frames = []
schema_records = []

eligible_event_keys = sorted(
    eligible_manifest["event_key"].unique()
)


for event_number, event_key in enumerate(
    eligible_event_keys,
    start=1,
):
    event_rows = (
        eligible_manifest.loc[
            eligible_manifest["event_key"].eq(event_key)
        ]
        .sort_values("source_row_index")
        .reset_index(drop=True)
    )

    schema = inspect_3c_event_schema(event_key)

    normalized_to_raw = schema["normalized_to_raw"]
    timestamp_raw = schema["timestamp_raw"]

    normalized_column_map = {}

    for normalized_name, raw_name in (
        normalized_to_raw.items()
    ):
        normalized_name = normalize_3j_name(
            normalized_name
        )

        if normalized_name in normalized_column_map:
            raise ValueError(
                f"{event_key}: duplicate normalized column "
                f"{normalized_name!r}."
            )

        normalized_column_map[
            normalized_name
        ] = raw_name


    timestamp_candidates = [
        normalized_name
        for normalized_name, raw_name
        in normalized_column_map.items()
        if raw_name == timestamp_raw
    ]

    if len(timestamp_candidates) != 1:
        raise ValueError(
            f"{event_key}: timestamp column could not be "
            "mapped uniquely."
        )

    timestamp_normalized = timestamp_candidates[0]


    # Exclude all known nonfeature, target, label, and
    # provenance columns consistently across every event.

    feature_names = sorted(
        normalized_name
        for normalized_name in normalized_column_map
        if normalized_name
        not in fingerprint_excluded_names
        and normalized_name != timestamp_normalized
    )


    # Defense-in-depth: no explicitly prohibited field may
    # reach the fingerprint measurement matrix.

    leaked_columns = (
        set(feature_names)
        & prohibited_feature_names
    )

    if leaked_columns:
        raise ValueError(
            f"{event_key}: target/provenance columns remain "
            f"in the feature set: {sorted(leaked_columns)}"
        )


    if not feature_names:
        raise ValueError(
            f"{event_key}: no measurement features remain."
        )


    raw_feature_names = [
        normalized_column_map[name]
        for name in feature_names
    ]

    if len(set(raw_feature_names)) != len(
        raw_feature_names
    ):
        raise ValueError(
            f"{event_key}: multiple normalized features map "
            "to the same raw column."
        )


    source_features = pd.read_csv(
        schema["event_path"],
        sep=None,
        engine="python",
        quotechar="'",
        dtype=str,
        keep_default_na=False,
        skipinitialspace=True,
        usecols=raw_feature_names,
    ).rename(
        columns={
            raw_name: normalized_name
            for normalized_name, raw_name
            in normalized_column_map.items()
            if normalized_name in feature_names
        }
    )


    missing_read_columns = (
        set(feature_names)
        - set(source_features.columns)
    )

    if missing_read_columns:
        raise ValueError(
            f"{event_key}: measurement columns were not "
            f"read: {sorted(missing_read_columns)}"
        )

    source_features = source_features.loc[
        :,
        feature_names,
    ]


    source_indices = event_rows[
        "source_row_index"
    ].to_numpy(dtype=np.int64)

    if len(source_indices) == 0:
        continue

    if source_indices.min() < 0:
        raise ValueError(
            f"{event_key}: negative source-row index found."
        )

    if source_indices.max() >= len(source_features):
        raise ValueError(
            f"{event_key}: source-row index exceeds the "
            "measurement file length."
        )


    selected_features = (
        source_features.iloc[source_indices]
        .reset_index(drop=True)
    )

    parsed_feature_columns = []
    missing_feature_cells = 0
    infinite_feature_cells = 0


    for feature_name in feature_names:
        cleaned = (
            selected_features[feature_name]
            .astype("string")
            .str.strip()
            .str.strip("'\"")
            .str.strip()
        )

        missing_mask = (
            cleaned.isna()
            | cleaned.str.lower().isin(
                MISSING_3J_TOKENS
            )
        )

        numeric = pd.to_numeric(
            cleaned.mask(missing_mask),
            errors="coerce",
        )

        invalid_nonmissing = (
            ~missing_mask
            & numeric.isna()
        )

        if invalid_nonmissing.any():
            examples = (
                cleaned.loc[invalid_nonmissing]
                .drop_duplicates()
                .head(5)
                .tolist()
            )

            raise ValueError(
                f"{event_key}: nonnumeric values in "
                f"{feature_name!r}: {examples}"
            )

        numeric_values = numeric.to_numpy(
            dtype=np.float64
        )

        missing_feature_cells += int(
            np.isnan(numeric_values).sum()
        )

        infinite_feature_cells += int(
            np.isinf(numeric_values).sum()
        )

        parsed_feature_columns.append(
            numeric_values
        )


    measurement_values = np.column_stack(
        parsed_feature_columns
    )

    measurement_fingerprints = (
        fingerprint_3j_rows(measurement_values)
    )

    feature_schema_text = "\x1f".join(
        feature_names
    )

    measurement_schema_id = hashlib.sha256(
        feature_schema_text.encode("utf-8")
    ).hexdigest()


    measurement_piece = event_rows.loc[
        :,
        [
            "farm",
            "asset_id",
            "asset_key",
            "event_id",
            "event_key",
            "source_row_index",
            "raw_id",
            "timestamp_utc",
            "source_event_label",
            "final_label",
        ],
    ].copy()

    measurement_piece[
        "measurement_schema_id"
    ] = measurement_schema_id

    measurement_piece[
        "measurement_feature_count"
    ] = len(feature_names)

    measurement_piece[
        "measurement_fingerprint"
    ] = measurement_fingerprints

    measurement_index_frames.append(
        measurement_piece
    )


    schema_records.append(
        {
            "event_key": event_key,
            "farm": event_rows["farm"].iloc[0],
            "asset_id": event_rows[
                "asset_id"
            ].iloc[0],
            "asset_key": event_rows[
                "asset_key"
            ].iloc[0],
            "eligible_rows": len(event_rows),
            "measurement_feature_count":
                len(feature_names),
            "measurement_schema_id":
                measurement_schema_id,
            "missing_feature_cells":
                missing_feature_cells,
            "infinite_feature_cells":
                infinite_feature_cells,
        }
    )


    if (
        event_number % 10 == 0
        or event_number == len(eligible_event_keys)
    ):
        print(
            f"Fingerprint progress: {event_number}/"
            f"{len(eligible_event_keys)} event files"
        )


if not measurement_index_frames:
    raise ValueError(
        "No eligible measurement rows were fingerprinted."
    )


ELIGIBLE_MEASUREMENT_SCHEMA_AUDIT = (
    pd.DataFrame(schema_records)
)

ELIGIBLE_MEASUREMENT_INDEX = pd.concat(
    measurement_index_frames,
    ignore_index=True,
)


if len(ELIGIBLE_MEASUREMENT_INDEX) != (
    expected_eligible_rows
):
    raise ValueError(
        "Eligible measurement-index row count does not "
        "match Cell 3I."
    )


if ELIGIBLE_MEASUREMENT_INDEX[
    ["event_key", "source_row_index"]
].duplicated().any():
    raise ValueError(
        "Measurement index contains duplicate source keys."
    )


if ELIGIBLE_MEASUREMENT_INDEX[
    "measurement_fingerprint"
].isna().any():
    raise ValueError(
        "At least one eligible row lacks a fingerprint."
    )


# ---------------------------------------------------------
# 4. Verify comparable schemas within each physical asset
# ---------------------------------------------------------

ASSET_FEATURE_SCHEMA_AUDIT = (
    ELIGIBLE_MEASUREMENT_INDEX
    .groupby(
        [
            "farm",
            "asset_id",
            "asset_key",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        eligible_rows=(
            "event_key",
            "size",
        ),
        event_files=(
            "event_key",
            "nunique",
        ),
        measurement_schema_count=(
            "measurement_schema_id",
            "nunique",
        ),
        minimum_feature_count=(
            "measurement_feature_count",
            "min",
        ),
        maximum_feature_count=(
            "measurement_feature_count",
            "max",
        ),
    )
)


incompatible_asset_schemas = (
    ASSET_FEATURE_SCHEMA_AUDIT.loc[
        ASSET_FEATURE_SCHEMA_AUDIT[
            "measurement_schema_count"
        ].ne(1)
    ]
)


if len(incompatible_asset_schemas):
    print(
        "\nAssets with incompatible eligible schemas:"
    )
    print(
        incompatible_asset_schemas.to_string(
            index=False
        )
    )

    raise ValueError(
        "Same-asset duplicate comparison is incomplete "
        "because feature schemas differ."
    )


# ---------------------------------------------------------
# 5. Global post-policy cross-label timestamp check
# ---------------------------------------------------------

GLOBAL_ELIGIBLE_TIMESTAMP_AUDIT = (
    ELIGIBLE_MEASUREMENT_INDEX
    .groupby(
        [
            "farm",
            "asset_id",
            "asset_key",
            "timestamp_utc",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        eligible_source_rows=(
            "event_key",
            "size",
        ),
        eligible_event_files=(
            "event_key",
            "nunique",
        ),
        eligible_label_count=(
            "final_label",
            "nunique",
        ),
    )
)


CROSS_LABEL_ELIGIBLE_TIMESTAMP_GROUPS = (
    GLOBAL_ELIGIBLE_TIMESTAMP_AUDIT.loc[
        GLOBAL_ELIGIBLE_TIMESTAMP_AUDIT[
            "eligible_label_count"
        ].gt(1)
    ]
    .sort_values(
        [
            "farm",
            "asset_id",
            "timestamp_utc",
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 6. Classify same-label timestamp collisions
# ---------------------------------------------------------

same_label_identity_columns = [
    "farm",
    "asset_id",
    "asset_key",
    "final_label",
    "timestamp_utc",
]


SAME_LABEL_TIMESTAMP_COLLISION_GROUPS = (
    ELIGIBLE_MEASUREMENT_INDEX
    .groupby(
        same_label_identity_columns,
        as_index=False,
        dropna=False,
    )
    .agg(
        source_row_copies=(
            "event_key",
            "size",
        ),
        distinct_event_files=(
            "event_key",
            "nunique",
        ),
        event_keys=(
            "event_key",
            lambda values: "|".join(
                sorted(
                    set(
                        values.astype(str)
                    )
                )
            ),
        ),
        unique_measurement_vectors=(
            "measurement_fingerprint",
            "nunique",
        ),
    )
)


SAME_LABEL_TIMESTAMP_COLLISION_GROUPS = (
    SAME_LABEL_TIMESTAMP_COLLISION_GROUPS.loc[
        SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
            "source_row_copies"
        ].gt(1)
    ]
    .copy()
)


SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
    "collision_relationship"
] = np.select(
    [
        SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
            "unique_measurement_vectors"
        ].eq(1),
        SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
            "unique_measurement_vectors"
        ].eq(
            SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
                "source_row_copies"
            ]
        ),
    ],
    [
        "exact_same_label_source_copy",
        "same_label_timestamp_different_measurements",
    ],
    default=(
        "mixed_exact_and_different_measurements"
    ),
)


SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
    "redundant_exact_source_rows"
] = np.where(
    SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
        "collision_relationship"
    ].eq("exact_same_label_source_copy"),
    SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
        "source_row_copies"
    ] - 1,
    0,
)


SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
    "source_rows_requiring_review"
] = np.where(
    SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
        "collision_relationship"
    ].eq("exact_same_label_source_copy"),
    0,
    SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
        "source_row_copies"
    ],
)


SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
    "recommended_next_action"
] = np.where(
    SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
        "collision_relationship"
    ].eq("exact_same_label_source_copy"),
    "retain_one_canonical_copy_after_pair_review",
    "review_provenance_before_retention",
)


SAME_LABEL_TIMESTAMP_COLLISION_GROUPS = (
    SAME_LABEL_TIMESTAMP_COLLISION_GROUPS
    .sort_values(
        [
            "farm",
            "asset_id",
            "final_label",
            "timestamp_utc",
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 7. Produce event-pair-level results
# ---------------------------------------------------------

pair_candidates = (
    ELIGIBLE_MEASUREMENT_INDEX.loc[
        ELIGIBLE_MEASUREMENT_INDEX.duplicated(
            same_label_identity_columns,
            keep=False,
        )
    ]
    .copy()
)


pair_audit_columns = [
    "farm",
    "asset_id",
    "asset_key",
    "final_label",
    "event_a_key",
    "event_b_key",
    "shared_eligible_timestamps",
    "exact_measurement_timestamps",
    "differing_measurement_timestamps",
    "pair_relationship",
]


if len(pair_candidates):
    pair_rows = pair_candidates.merge(
        pair_candidates,
        on=same_label_identity_columns,
        how="inner",
        suffixes=("_a", "_b"),
        validate="many_to_many",
    )

    pair_rows = pair_rows.loc[
        pair_rows["event_key_a"].astype(str)
        < pair_rows["event_key_b"].astype(str)
    ].copy()

    pair_rows["exact_measurement_match"] = (
        pair_rows[
            "measurement_fingerprint_a"
        ].eq(
            pair_rows[
                "measurement_fingerprint_b"
            ]
        )
    )

    pair_rows[
        "differing_measurement_match"
    ] = ~pair_rows["exact_measurement_match"]


    SAME_LABEL_EVENT_PAIR_AUDIT = (
        pair_rows
        .groupby(
            [
                "farm",
                "asset_id",
                "asset_key",
                "final_label",
                "event_key_a",
                "event_key_b",
            ],
            as_index=False,
            dropna=False,
        )
        .agg(
            shared_eligible_timestamps=(
                "timestamp_utc",
                "nunique",
            ),
            exact_measurement_timestamps=(
                "exact_measurement_match",
                "sum",
            ),
            differing_measurement_timestamps=(
                "differing_measurement_match",
                "sum",
            ),
        )
        .rename(
            columns={
                "event_key_a": "event_a_key",
                "event_key_b": "event_b_key",
            }
        )
    )


    SAME_LABEL_EVENT_PAIR_AUDIT[
        "pair_relationship"
    ] = np.select(
        [
            SAME_LABEL_EVENT_PAIR_AUDIT[
                "differing_measurement_timestamps"
            ].eq(0),
            SAME_LABEL_EVENT_PAIR_AUDIT[
                "exact_measurement_timestamps"
            ].eq(0),
        ],
        [
            "all_shared_rows_exact",
            "all_shared_timestamps_differ",
        ],
        default="mixed_exact_and_different",
    )


    SAME_LABEL_EVENT_PAIR_AUDIT = (
        SAME_LABEL_EVENT_PAIR_AUDIT.loc[
            :,
            pair_audit_columns,
        ]
        .sort_values(
            [
                "farm",
                "asset_id",
                "final_label",
                "event_a_key",
                "event_b_key",
            ]
        )
        .reset_index(drop=True)
    )

else:
    SAME_LABEL_EVENT_PAIR_AUDIT = pd.DataFrame(
        columns=pair_audit_columns
    )


# ---------------------------------------------------------
# 8. Summarize collision relationships
# ---------------------------------------------------------

relationship_summary_columns = [
    "collision_relationship",
    "collision_timestamp_groups",
    "source_rows_in_groups",
    "redundant_exact_source_rows",
    "source_rows_requiring_review",
]


if len(SAME_LABEL_TIMESTAMP_COLLISION_GROUPS):
    SAME_LABEL_COLLISION_RELATIONSHIP_COUNTS = (
        SAME_LABEL_TIMESTAMP_COLLISION_GROUPS
        .groupby(
            "collision_relationship",
            as_index=False,
            dropna=False,
        )
        .agg(
            collision_timestamp_groups=(
                "timestamp_utc",
                "size",
            ),
            source_rows_in_groups=(
                "source_row_copies",
                "sum",
            ),
            redundant_exact_source_rows=(
                "redundant_exact_source_rows",
                "sum",
            ),
            source_rows_requiring_review=(
                "source_rows_requiring_review",
                "sum",
            ),
        )
        .loc[
            :,
            relationship_summary_columns,
        ]
        .sort_values("collision_relationship")
        .reset_index(drop=True)
    )

else:
    SAME_LABEL_COLLISION_RELATIONSHIP_COUNTS = (
        pd.DataFrame(
            columns=relationship_summary_columns
        )
    )


# ---------------------------------------------------------
# 9. Audit vector reuse at different timestamps
# ---------------------------------------------------------

# Collapse true same-timestamp source copies first. This
# section measures recurrence through time, not multiple
# source copies of the same timestamped observation.

distinct_timestamp_vectors = (
    ELIGIBLE_MEASUREMENT_INDEX
    .sort_values(
        [
            "asset_key",
            "final_label",
            "timestamp_utc",
            "event_key",
        ]
    )
    .drop_duplicates(
        [
            "asset_key",
            "final_label",
            "timestamp_utc",
            "measurement_schema_id",
            "measurement_fingerprint",
        ]
    )
)


MEASUREMENT_VECTOR_REUSE_GROUPS = (
    distinct_timestamp_vectors
    .groupby(
        [
            "farm",
            "asset_id",
            "asset_key",
            "final_label",
            "measurement_schema_id",
            "measurement_fingerprint",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        distinct_timestamp_occurrences=(
            "timestamp_utc",
            "nunique",
        ),
        contributing_event_files=(
            "event_key",
            "nunique",
        ),
        first_timestamp_utc=(
            "timestamp_utc",
            "min",
        ),
        last_timestamp_utc=(
            "timestamp_utc",
            "max",
        ),
    )
)


MEASUREMENT_VECTOR_REUSE_GROUPS = (
    MEASUREMENT_VECTOR_REUSE_GROUPS.loc[
        MEASUREMENT_VECTOR_REUSE_GROUPS[
            "distinct_timestamp_occurrences"
        ].gt(1)
    ]
    .copy()
)


MEASUREMENT_VECTOR_REUSE_GROUPS[
    "occurrences_beyond_first"
] = (
    MEASUREMENT_VECTOR_REUSE_GROUPS[
        "distinct_timestamp_occurrences"
    ] - 1
)


reuse_summary_columns = [
    "farm",
    "final_label",
    "reused_vector_groups",
    "distinct_timestamp_occurrences",
    "occurrences_beyond_first",
    "maximum_occurrences_of_one_vector",
]


if len(MEASUREMENT_VECTOR_REUSE_GROUPS):
    MEASUREMENT_VECTOR_REUSE_SUMMARY = (
        MEASUREMENT_VECTOR_REUSE_GROUPS
        .groupby(
            [
                "farm",
                "final_label",
            ],
            as_index=False,
            dropna=False,
        )
        .agg(
            reused_vector_groups=(
                "measurement_fingerprint",
                "size",
            ),
            distinct_timestamp_occurrences=(
                "distinct_timestamp_occurrences",
                "sum",
            ),
            occurrences_beyond_first=(
                "occurrences_beyond_first",
                "sum",
            ),
            maximum_occurrences_of_one_vector=(
                "distinct_timestamp_occurrences",
                "max",
            ),
        )
        .loc[
            :,
            reuse_summary_columns,
        ]
        .sort_values(
            [
                "farm",
                "final_label",
            ]
        )
        .reset_index(drop=True)
    )

else:
    MEASUREMENT_VECTOR_REUSE_SUMMARY = pd.DataFrame(
        columns=reuse_summary_columns
    )


# ---------------------------------------------------------
# 10. Final counts and non-destructive checks
# ---------------------------------------------------------

cross_label_eligible_timestamp_groups = len(
    CROSS_LABEL_ELIGIBLE_TIMESTAMP_GROUPS
)

same_label_collision_groups = len(
    SAME_LABEL_TIMESTAMP_COLLISION_GROUPS
)

exact_same_label_collision_groups = int(
    SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
        "collision_relationship"
    ].eq(
        "exact_same_label_source_copy"
    ).sum()
)

redundant_exact_same_label_source_rows = int(
    SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
        "redundant_exact_source_rows"
    ].sum()
)

ambiguous_same_label_collision_groups = int(
    (
        ~SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
            "collision_relationship"
        ].eq(
            "exact_same_label_source_copy"
        )
    ).sum()
)

ambiguous_same_label_source_rows = int(
    SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
        "source_rows_requiring_review"
    ].sum()
)

reused_measurement_vector_groups = len(
    MEASUREMENT_VECTOR_REUSE_GROUPS
)


# Row conservation: this audit must not change Cell 3I.

if int(
    ROW_LABEL_MANIFEST["modeling_eligible"].sum()
) != expected_eligible_rows:
    raise ValueError(
        "Cell 3J unexpectedly changed manifest eligibility."
    )

if ROW_LABEL_MANIFEST[
    "split_assignment"
].notna().any():
    raise ValueError(
        "Cell 3J unexpectedly assigned a split."
    )


# ---------------------------------------------------------
# 11. Display compact results
# ---------------------------------------------------------

DISPLAY_LIMIT_3J = 100


print("\nSame-label event-pair audit:")

if len(SAME_LABEL_EVENT_PAIR_AUDIT):
    print(
        SAME_LABEL_EVENT_PAIR_AUDIT.head(
            DISPLAY_LIMIT_3J
        ).to_string(index=False)
    )

    if len(SAME_LABEL_EVENT_PAIR_AUDIT) > (
        DISPLAY_LIMIT_3J
    ):
        print(
            f"... display truncated: "
            f"{len(SAME_LABEL_EVENT_PAIR_AUDIT):,} "
            "total event pairs."
        )
else:
    print("No same-label timestamp-sharing event pairs.")


print("\nSame-label collision relationship counts:")

if len(SAME_LABEL_COLLISION_RELATIONSHIP_COUNTS):
    print(
        SAME_LABEL_COLLISION_RELATIONSHIP_COUNTS
        .to_string(index=False)
    )
else:
    print("No same-label eligible timestamp collisions.")


print("\nMeasurement-vector reuse at different timestamps:")

if len(MEASUREMENT_VECTOR_REUSE_SUMMARY):
    print(
        MEASUREMENT_VECTOR_REUSE_SUMMARY.to_string(
            index=False
        )
    )
else:
    print(
        "No measurement vector recurs at different "
        "timestamps within the same asset and label."
    )


if cross_label_eligible_timestamp_groups:
    print(
        "\nUnexpected eligible cross-label timestamp "
        "groups:"
    )
    print(
        CROSS_LABEL_ELIGIBLE_TIMESTAMP_GROUPS.head(
            DISPLAY_LIMIT_3J
        ).to_string(index=False)
    )


print("\nCell 3J completed successfully.")

print(
    "Eligible rows fingerprinted:",
    len(ELIGIBLE_MEASUREMENT_INDEX),
)

print(
    "Assets with incompatible eligible schemas:",
    len(incompatible_asset_schemas),
)

print(
    "Eligible cross-label timestamp groups:",
    cross_label_eligible_timestamp_groups,
)

print(
    "Same-label timestamp collision groups:",
    same_label_collision_groups,
)

print(
    "Exact same-label source-copy groups:",
    exact_same_label_collision_groups,
)

print(
    "Redundant exact same-label source rows:",
    redundant_exact_same_label_source_rows,
)

print(
    "Ambiguous same-label collision groups:",
    ambiguous_same_label_collision_groups,
)

print(
    "Ambiguous same-label source rows:",
    ambiguous_same_label_source_rows,
)

print(
    "Measurement vectors reused at different timestamps:",
    reused_measurement_vector_groups,
)

print(
    "Assigned train/validation/test rows:",
    int(
        ROW_LABEL_MANIFEST[
            "split_assignment"
        ].notna().sum()
    ),
)

print(
    "\nNo source row, source label, manifest decision, "
    "or split assignment was modified."
)

Fingerprint progress: 10/94 event files
Fingerprint progress: 20/94 event files
Fingerprint progress: 30/94 event files
Fingerprint progress: 40/94 event files
Fingerprint progress: 50/94 event files
Fingerprint progress: 60/94 event files
Fingerprint progress: 70/94 event files
Fingerprint progress: 80/94 event files
Fingerprint progress: 90/94 event files
Fingerprint progress: 94/94 event files

Same-label event-pair audit:
farm asset_id        asset_key final_label     event_a_key     event_b_key  shared_eligible_timestamps  exact_measurement_timestamps  differing_measurement_timestamps     pair_relationship
   A       21 farm_a::asset_21     anomaly farm_a_event_51 farm_a_event_72                        1009                          1009                                 0 all_shared_rows_exact

Same-label collision relationship counts:
      collision_relationship  collision_timestamp_groups  source_rows_in_groups  redundant_exact_source_rows  source_rows_requiring_review
exact_same

In [32]:
# Cell 3K — deterministic canonical retention for the exact
# same-label source copies verified by Cell 3J.
#
# Canonical source:  farm_a_event_51
# Redundant source:  farm_a_event_72
# Rows excluded:     1,009
# Eligible rows:     214,546 -> 213,537
#
# This cell changes only manifest eligibility for the verified
# redundant rows. It does not delete rows, edit source files,
# change labels/timestamps, or assign data splits.

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 1. Validate prerequisites and the exact Cell 3J result
# ---------------------------------------------------------

required_objects_3k = [
    "ROW_LABEL_MANIFEST",
    "ELIGIBLE_MEASUREMENT_INDEX",
    "SAME_LABEL_TIMESTAMP_COLLISION_GROUPS",
    "SAME_LABEL_EVENT_PAIR_AUDIT",
    "CROSS_LABEL_ELIGIBLE_TIMESTAMP_GROUPS",
    "incompatible_asset_schemas",
]

missing_objects_3k = [
    name for name in required_objects_3k
    if name not in globals()
]

if missing_objects_3k:
    raise NameError(
        "Run Cell 3J successfully before Cell 3K. "
        f"Missing objects: {missing_objects_3k}"
    )


required_manifest_columns_3k = {
    "farm",
    "asset_id",
    "asset_key",
    "event_id",
    "event_key",
    "source_row_index",
    "raw_id",
    "timestamp_utc",
    "source_event_label",
    "final_label",
    "modeling_eligible",
    "split_assignment",
}

missing_manifest_columns_3k = (
    required_manifest_columns_3k
    - set(ROW_LABEL_MANIFEST.columns)
)

if missing_manifest_columns_3k:
    raise ValueError(
        "ROW_LABEL_MANIFEST is missing required columns: "
        f"{sorted(missing_manifest_columns_3k)}"
    )


required_measurement_columns_3k = {
    "farm",
    "asset_id",
    "asset_key",
    "event_id",
    "event_key",
    "source_row_index",
    "raw_id",
    "timestamp_utc",
    "final_label",
    "measurement_schema_id",
    "measurement_fingerprint",
}

missing_measurement_columns_3k = (
    required_measurement_columns_3k
    - set(ELIGIBLE_MEASUREMENT_INDEX.columns)
)

if missing_measurement_columns_3k:
    raise ValueError(
        "ELIGIBLE_MEASUREMENT_INDEX is missing columns: "
        f"{sorted(missing_measurement_columns_3k)}"
    )


EXPECTED_ELIGIBLE_BEFORE_3K = 214_546
EXPECTED_DUPLICATE_GROUPS_3K = 1_009
EXPECTED_REDUNDANT_ROWS_3K = 1_009
EXPECTED_ELIGIBLE_AFTER_3K = 213_537

CANONICAL_EVENT_KEY_3K = "farm_a_event_51"
REDUNDANT_EVENT_KEY_3K = "farm_a_event_72"

EXPECTED_EVENT_KEYS_3K = frozenset(
    {
        CANONICAL_EVENT_KEY_3K,
        REDUNDANT_EVENT_KEY_3K,
    }
)


eligible_before_3k = int(
    ROW_LABEL_MANIFEST["modeling_eligible"].sum()
)

if eligible_before_3k != EXPECTED_ELIGIBLE_BEFORE_3K:
    raise ValueError(
        "Expected 214,546 eligible rows before Cell 3K, "
        f"but found {eligible_before_3k:,}. Restart from "
        "Cell 3I, then rerun Cells 3J and 3K once."
    )

if ROW_LABEL_MANIFEST[
    "split_assignment"
].notna().any():
    raise ValueError(
        "Cell 3K must run before split assignment."
    )

if len(CROSS_LABEL_ELIGIBLE_TIMESTAMP_GROUPS):
    raise ValueError(
        "Cross-label eligible timestamp collisions remain."
    )

if len(incompatible_asset_schemas):
    raise ValueError(
        "Incompatible same-asset feature schemas remain."
    )

if len(SAME_LABEL_TIMESTAMP_COLLISION_GROUPS) != (
    EXPECTED_DUPLICATE_GROUPS_3K
):
    raise ValueError(
        "Expected 1,009 same-label collision groups, "
        f"but found "
        f"{len(SAME_LABEL_TIMESTAMP_COLLISION_GROUPS):,}."
    )

if not SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
    "collision_relationship"
].eq("exact_same_label_source_copy").all():
    raise ValueError(
        "At least one same-label collision is ambiguous."
    )

if not SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
    "source_row_copies"
].eq(2).all():
    raise ValueError(
        "At least one collision group does not contain "
        "exactly two source rows."
    )

if int(
    SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
        "redundant_exact_source_rows"
    ].sum()
) != EXPECTED_REDUNDANT_ROWS_3K:
    raise ValueError(
        "The audited redundant-row count is not 1,009."
    )


# Confirm the single event-pair result.

if len(SAME_LABEL_EVENT_PAIR_AUDIT) != 1:
    raise ValueError(
        "Expected exactly one audited same-label event pair."
    )

pair_record_3k = SAME_LABEL_EVENT_PAIR_AUDIT.iloc[0]

observed_pair_keys_3k = frozenset(
    {
        str(pair_record_3k["event_a_key"]),
        str(pair_record_3k["event_b_key"]),
    }
)

if observed_pair_keys_3k != EXPECTED_EVENT_KEYS_3K:
    raise ValueError(
        "The audited pair is not farm_a_event_51 and "
        "farm_a_event_72."
    )

if (
    int(pair_record_3k["shared_eligible_timestamps"])
    != EXPECTED_DUPLICATE_GROUPS_3K
    or int(pair_record_3k["exact_measurement_timestamps"])
    != EXPECTED_DUPLICATE_GROUPS_3K
    or int(pair_record_3k["differing_measurement_timestamps"])
    != 0
    or pair_record_3k["pair_relationship"]
    != "all_shared_rows_exact"
):
    raise ValueError(
        "The event-pair audit no longer matches the verified "
        "1,009 exact-copy result."
    )


# ---------------------------------------------------------
# 2. Reconstruct and revalidate every collision member
# ---------------------------------------------------------

collision_identity_columns_3k = [
    "farm",
    "asset_id",
    "asset_key",
    "final_label",
    "timestamp_utc",
]

exact_collision_keys_3k = (
    SAME_LABEL_TIMESTAMP_COLLISION_GROUPS.loc[
        :,
        collision_identity_columns_3k,
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

if len(exact_collision_keys_3k) != (
    EXPECTED_DUPLICATE_GROUPS_3K
):
    raise ValueError(
        "Exact collision identities are not unique."
    )

exact_collision_members_3k = (
    ELIGIBLE_MEASUREMENT_INDEX.merge(
        exact_collision_keys_3k,
        on=collision_identity_columns_3k,
        how="inner",
        validate="many_to_one",
    )
    .sort_values(
        collision_identity_columns_3k
        + ["event_key", "source_row_index"]
    )
    .reset_index(drop=True)
)

if len(exact_collision_members_3k) != (
    EXPECTED_DUPLICATE_GROUPS_3K * 2
):
    raise ValueError(
        "Expected 2,018 collision-member rows, but found "
        f"{len(exact_collision_members_3k):,}."
    )

collision_member_check_3k = (
    exact_collision_members_3k.groupby(
        collision_identity_columns_3k,
        as_index=False,
        dropna=False,
    )
    .agg(
        source_rows=("event_key", "size"),
        event_files=("event_key", "nunique"),
        schemas=("measurement_schema_id", "nunique"),
        vectors=("measurement_fingerprint", "nunique"),
        event_key_set=(
            "event_key",
            lambda values: frozenset(
                values.astype(str)
            ),
        ),
    )
)

valid_member_groups_3k = (
    collision_member_check_3k["source_rows"].eq(2)
    & collision_member_check_3k["event_files"].eq(2)
    & collision_member_check_3k["schemas"].eq(1)
    & collision_member_check_3k["vectors"].eq(1)
    & collision_member_check_3k["event_key_set"].map(
        lambda value: value == EXPECTED_EVENT_KEYS_3K
    )
)

if not valid_member_groups_3k.all():
    raise ValueError(
        "At least one collision group is not one exact row "
        "from each verified event."
    )


# ---------------------------------------------------------
# 3. Build the complete row-level retention audit
# ---------------------------------------------------------

canonical_members_3k = exact_collision_members_3k.loc[
    exact_collision_members_3k["event_key"].eq(
        CANONICAL_EVENT_KEY_3K
    )
].copy()

redundant_members_3k = exact_collision_members_3k.loc[
    exact_collision_members_3k["event_key"].eq(
        REDUNDANT_EVENT_KEY_3K
    )
].copy()

if len(canonical_members_3k) != EXPECTED_DUPLICATE_GROUPS_3K:
    raise ValueError(
        "The canonical event does not contribute 1,009 rows."
    )

if len(redundant_members_3k) != EXPECTED_REDUNDANT_ROWS_3K:
    raise ValueError(
        "The redundant event does not contribute 1,009 rows."
    )

retention_pairs_3k = redundant_members_3k.merge(
    canonical_members_3k,
    on=collision_identity_columns_3k,
    how="inner",
    suffixes=("_excluded", "_retained"),
    validate="one_to_one",
)

if len(retention_pairs_3k) != EXPECTED_REDUNDANT_ROWS_3K:
    raise ValueError(
        "Every redundant row could not be paired with exactly "
        "one canonical row."
    )

if not retention_pairs_3k[
    "measurement_schema_id_excluded"
].eq(
    retention_pairs_3k[
        "measurement_schema_id_retained"
    ]
).all():
    raise ValueError(
        "A proposed retention pair has different schemas."
    )

if not retention_pairs_3k[
    "measurement_fingerprint_excluded"
].eq(
    retention_pairs_3k[
        "measurement_fingerprint_retained"
    ]
).all():
    raise ValueError(
        "A proposed retention pair has different measurements."
    )

EXACT_SAME_LABEL_CANONICAL_RETENTION_AUDIT = pd.DataFrame(
    {
        "farm": retention_pairs_3k["farm"],
        "asset_id": retention_pairs_3k["asset_id"],
        "asset_key": retention_pairs_3k["asset_key"],
        "final_label": retention_pairs_3k["final_label"],
        "timestamp_utc": retention_pairs_3k[
            "timestamp_utc"
        ],
        "measurement_schema_id": retention_pairs_3k[
            "measurement_schema_id_retained"
        ],
        "measurement_fingerprint": retention_pairs_3k[
            "measurement_fingerprint_retained"
        ],
        "retained_event_id": retention_pairs_3k[
            "event_id_retained"
        ],
        "retained_event_key": retention_pairs_3k[
            "event_key_retained"
        ],
        "retained_source_row_index": retention_pairs_3k[
            "source_row_index_retained"
        ],
        "retained_raw_id": retention_pairs_3k[
            "raw_id_retained"
        ],
        "excluded_event_id": retention_pairs_3k[
            "event_id_excluded"
        ],
        "excluded_event_key": retention_pairs_3k[
            "event_key_excluded"
        ],
        "excluded_source_row_index": retention_pairs_3k[
            "source_row_index_excluded"
        ],
        "excluded_raw_id": retention_pairs_3k[
            "raw_id_excluded"
        ],
        "retention_rule": (
            "retain_farm_a_event_51_exclude_farm_a_event_72"
        ),
        "manifest_action": (
            "exclude_redundant_exact_same_label_source_copy"
        ),
    }
).sort_values(
    [
        "farm",
        "asset_id",
        "timestamp_utc",
        "excluded_source_row_index",
    ]
).reset_index(drop=True)

if len(EXACT_SAME_LABEL_CANONICAL_RETENTION_AUDIT) != (
    EXPECTED_REDUNDANT_ROWS_3K
):
    raise ValueError(
        "The retention audit does not contain 1,009 rows."
    )

if EXACT_SAME_LABEL_CANONICAL_RETENTION_AUDIT[
    ["excluded_event_key", "excluded_source_row_index"]
].duplicated().any():
    raise ValueError(
        "The retention audit has duplicate excluded keys."
    )


# ---------------------------------------------------------
# 4. Resolve retained/excluded source keys in the manifest
# ---------------------------------------------------------

if ROW_LABEL_MANIFEST[
    ["event_key", "source_row_index"]
].duplicated().any():
    raise ValueError(
        "ROW_LABEL_MANIFEST source keys are not unique."
    )

manifest_source_indices_3k = pd.to_numeric(
    ROW_LABEL_MANIFEST["source_row_index"],
    errors="coerce",
)

if (
    manifest_source_indices_3k.isna().any()
    or manifest_source_indices_3k.ne(
        np.floor(manifest_source_indices_3k)
    ).any()
):
    raise ValueError(
        "ROW_LABEL_MANIFEST contains an invalid "
        "source_row_index."
    )

manifest_key_frame_3k = pd.DataFrame(
    {
        "event_key": ROW_LABEL_MANIFEST[
            "event_key"
        ].astype(str),
        "source_row_index": (
            manifest_source_indices_3k.astype("int64")
        ),
    },
    index=ROW_LABEL_MANIFEST.index,
)

manifest_key_index_3k = pd.MultiIndex.from_frame(
    manifest_key_frame_3k
)


def normalized_3k_source_keys(
    audit_frame,
    event_column,
    row_column,
):
    key_frame = audit_frame.loc[
        :,
        [event_column, row_column],
    ].rename(
        columns={
            event_column: "event_key",
            row_column: "source_row_index",
        }
    )

    key_frame["event_key"] = (
        key_frame["event_key"].astype(str)
    )
    key_frame["source_row_index"] = pd.to_numeric(
        key_frame["source_row_index"],
        errors="raise",
    ).astype("int64")

    return key_frame


excluded_key_frame_3k = normalized_3k_source_keys(
    EXACT_SAME_LABEL_CANONICAL_RETENTION_AUDIT,
    "excluded_event_key",
    "excluded_source_row_index",
)

canonical_key_frame_3k = normalized_3k_source_keys(
    EXACT_SAME_LABEL_CANONICAL_RETENTION_AUDIT,
    "retained_event_key",
    "retained_source_row_index",
)

excluded_key_index_3k = pd.MultiIndex.from_frame(
    excluded_key_frame_3k
)

canonical_key_index_3k = pd.MultiIndex.from_frame(
    canonical_key_frame_3k
)

if not excluded_key_index_3k.is_unique:
    raise ValueError(
        "Excluded source keys are not unique."
    )

if not canonical_key_index_3k.is_unique:
    raise ValueError(
        "Canonical source keys are not unique."
    )

if len(
    excluded_key_index_3k.intersection(
        canonical_key_index_3k
    )
):
    raise ValueError(
        "Canonical and excluded source-key sets overlap."
    )

exclusion_mask_3k = manifest_key_index_3k.isin(
    excluded_key_index_3k
)

canonical_mask_3k = manifest_key_index_3k.isin(
    canonical_key_index_3k
)

if int(exclusion_mask_3k.sum()) != EXPECTED_REDUNDANT_ROWS_3K:
    raise ValueError(
        "Not all 1,009 excluded keys resolve uniquely in the "
        "manifest."
    )

if int(canonical_mask_3k.sum()) != EXPECTED_DUPLICATE_GROUPS_3K:
    raise ValueError(
        "Not all 1,009 canonical keys resolve uniquely in the "
        "manifest."
    )

if not ROW_LABEL_MANIFEST.loc[
    exclusion_mask_3k,
    "modeling_eligible",
].eq(True).all():
    raise ValueError(
        "At least one proposed exclusion is not currently "
        "modeling-eligible."
    )

if not ROW_LABEL_MANIFEST.loc[
    canonical_mask_3k,
    "modeling_eligible",
].eq(True).all():
    raise ValueError(
        "At least one canonical row is not currently eligible."
    )


# Reconfirm that each resolved exclusion has the same asset,
# normalized label, and UTC timestamp as its audit record.

manifest_exclusion_rows_3k = ROW_LABEL_MANIFEST.loc[
    exclusion_mask_3k,
    [
        "farm",
        "asset_id",
        "asset_key",
        "final_label",
        "timestamp_utc",
        "event_key",
        "source_row_index",
    ],
].copy()

manifest_exclusion_rows_3k["final_label"] = (
    manifest_exclusion_rows_3k["final_label"]
    .astype("string")
    .str.strip()
    .str.lower()
)

manifest_exclusion_rows_3k["timestamp_utc"] = pd.to_datetime(
    manifest_exclusion_rows_3k["timestamp_utc"],
    errors="coerce",
    utc=True,
)

manifest_exclusion_rows_3k["event_key"] = (
    manifest_exclusion_rows_3k["event_key"].astype(str)
)

manifest_exclusion_rows_3k["source_row_index"] = pd.to_numeric(
    manifest_exclusion_rows_3k["source_row_index"],
    errors="raise",
).astype("int64")

if manifest_exclusion_rows_3k[
    "timestamp_utc"
].isna().any():
    raise ValueError(
        "A proposed exclusion has an invalid timestamp."
    )

normalized_retention_audit_3k = (
    EXACT_SAME_LABEL_CANONICAL_RETENTION_AUDIT.copy()
)

normalized_retention_audit_3k[
    "excluded_event_key"
] = normalized_retention_audit_3k[
    "excluded_event_key"
].astype(str)

normalized_retention_audit_3k[
    "excluded_source_row_index"
] = pd.to_numeric(
    normalized_retention_audit_3k[
        "excluded_source_row_index"
    ],
    errors="raise",
).astype("int64")

manifest_exclusion_check_3k = manifest_exclusion_rows_3k.merge(
    normalized_retention_audit_3k.loc[
        :,
        [
            "farm",
            "asset_id",
            "asset_key",
            "final_label",
            "timestamp_utc",
            "excluded_event_key",
            "excluded_source_row_index",
        ],
    ],
    left_on=[
        "farm",
        "asset_id",
        "asset_key",
        "final_label",
        "timestamp_utc",
        "event_key",
        "source_row_index",
    ],
    right_on=[
        "farm",
        "asset_id",
        "asset_key",
        "final_label",
        "timestamp_utc",
        "excluded_event_key",
        "excluded_source_row_index",
    ],
    how="inner",
    validate="one_to_one",
)

if len(manifest_exclusion_check_3k) != (
    EXPECTED_REDUNDANT_ROWS_3K
):
    raise ValueError(
        "Manifest exclusion records do not fully agree with "
        "the row-level audit."
    )


# ---------------------------------------------------------
# 5. Apply the manifest-only eligibility decision
# ---------------------------------------------------------

audit_columns_3k = [
    "same_label_duplicate_action",
    "same_label_duplicate_canonical_event_key",
    "same_label_duplicate_reason",
]

existing_audit_columns_3k = [
    name for name in audit_columns_3k
    if name in ROW_LABEL_MANIFEST.columns
]

if existing_audit_columns_3k:
    raise ValueError(
        "Cell 3K audit columns already exist: "
        f"{existing_audit_columns_3k}. Restart from Cell 3I "
        "before rerunning Cell 3K."
    )

manifest_before_3k = ROW_LABEL_MANIFEST.copy(deep=True)

eligibility_before_series_3k = ROW_LABEL_MANIFEST[
    "modeling_eligible"
].copy(deep=True)

for column_name_3k in audit_columns_3k:
    ROW_LABEL_MANIFEST[column_name_3k] = pd.Series(
        pd.NA,
        index=ROW_LABEL_MANIFEST.index,
        dtype="string",
    )

ROW_LABEL_MANIFEST.loc[
    canonical_mask_3k,
    "same_label_duplicate_action",
] = "canonical_exact_copy_retained"

ROW_LABEL_MANIFEST.loc[
    exclusion_mask_3k,
    "same_label_duplicate_action",
] = "redundant_exact_copy_excluded"

ROW_LABEL_MANIFEST.loc[
    canonical_mask_3k | exclusion_mask_3k,
    "same_label_duplicate_canonical_event_key",
] = CANONICAL_EVENT_KEY_3K

ROW_LABEL_MANIFEST.loc[
    canonical_mask_3k | exclusion_mask_3k,
    "same_label_duplicate_reason",
] = (
    "verified_same_asset_same_label_same_timestamp_"
    "same_measurement_vector"
)


# This is the only change to a pre-existing manifest column.

ROW_LABEL_MANIFEST.loc[
    exclusion_mask_3k,
    "modeling_eligible",
] = False


# ---------------------------------------------------------
# 6. Verify mutation boundaries and row conservation
# ---------------------------------------------------------

if len(ROW_LABEL_MANIFEST) != len(manifest_before_3k):
    raise ValueError(
        "Cell 3K changed the manifest row count."
    )

if not ROW_LABEL_MANIFEST.index.equals(
    manifest_before_3k.index
):
    raise ValueError(
        "Cell 3K changed the manifest index."
    )

unchanged_columns_3k = [
    name for name in manifest_before_3k.columns
    if name != "modeling_eligible"
]

if not ROW_LABEL_MANIFEST.loc[
    :,
    unchanged_columns_3k,
].equals(
    manifest_before_3k.loc[
        :,
        unchanged_columns_3k,
    ]
):
    raise ValueError(
        "A pre-existing manifest column other than "
        "modeling_eligible changed unexpectedly."
    )

eligibility_change_mask_3k = (
    eligibility_before_series_3k.ne(
        ROW_LABEL_MANIFEST["modeling_eligible"]
    )
    .fillna(False)
    .to_numpy()
)

if not np.array_equal(
    eligibility_change_mask_3k,
    exclusion_mask_3k,
):
    raise ValueError(
        "Eligibility changed outside the verified redundant "
        "source-key set."
    )

if not ROW_LABEL_MANIFEST.loc[
    exclusion_mask_3k,
    "modeling_eligible",
].eq(False).all():
    raise ValueError(
        "At least one redundant row remains eligible."
    )

if not ROW_LABEL_MANIFEST.loc[
    canonical_mask_3k,
    "modeling_eligible",
].eq(True).all():
    raise ValueError(
        "At least one canonical row was excluded."
    )

eligible_after_3k = int(
    ROW_LABEL_MANIFEST["modeling_eligible"].sum()
)

if eligible_after_3k != EXPECTED_ELIGIBLE_AFTER_3K:
    raise ValueError(
        "Expected 213,537 eligible rows after retention, "
        f"but found {eligible_after_3k:,}."
    )

if eligible_after_3k != (
    eligible_before_3k - EXPECTED_REDUNDANT_ROWS_3K
):
    raise ValueError(
        "Eligible-row subtraction is not conserved."
    )

if ROW_LABEL_MANIFEST[
    "split_assignment"
].notna().any():
    raise ValueError(
        "Cell 3K unexpectedly assigned a split."
    )


# ---------------------------------------------------------
# 7. Produce post-retention summaries
# ---------------------------------------------------------

EXACT_SAME_LABEL_CANONICAL_RETENTION_SUMMARY = (
    EXACT_SAME_LABEL_CANONICAL_RETENTION_AUDIT.groupby(
        [
            "farm",
            "asset_id",
            "asset_key",
            "final_label",
            "retained_event_key",
            "excluded_event_key",
            "retention_rule",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        duplicate_timestamp_groups=(
            "timestamp_utc",
            "nunique",
        ),
        canonical_rows_retained=(
            "retained_source_row_index",
            "size",
        ),
        redundant_rows_excluded=(
            "excluded_source_row_index",
            "size",
        ),
        first_duplicate_timestamp_utc=(
            "timestamp_utc",
            "min",
        ),
        last_duplicate_timestamp_utc=(
            "timestamp_utc",
            "max",
        ),
    )
)

affected_event_mask_3k = ROW_LABEL_MANIFEST[
    "event_key"
].astype(str).isin(EXPECTED_EVENT_KEYS_3K)

POST_DEDUPLICATION_AFFECTED_EVENT_SUMMARY = (
    ROW_LABEL_MANIFEST.loc[
        affected_event_mask_3k
    ]
    .groupby(
        [
            "farm",
            "asset_id",
            "asset_key",
            "event_key",
            "final_label",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        manifest_rows=("source_row_index", "size"),
        modeling_eligible_rows=(
            "modeling_eligible",
            "sum",
        ),
        canonical_rows_retained=(
            "same_label_duplicate_action",
            lambda values: int(
                values.eq(
                    "canonical_exact_copy_retained"
                ).sum()
            ),
        ),
        redundant_rows_excluded=(
            "same_label_duplicate_action",
            lambda values: int(
                values.eq(
                    "redundant_exact_copy_excluded"
                ).sum()
            ),
        ),
    )
    .sort_values("event_key")
    .reset_index(drop=True)
)

POST_DEDUPLICATION_ELIGIBLE_LABEL_SUMMARY = (
    ROW_LABEL_MANIFEST.loc[
        ROW_LABEL_MANIFEST["modeling_eligible"]
    ]
    .groupby(
        ["farm", "final_label"],
        as_index=False,
        dropna=False,
    )
    .agg(
        eligible_rows=("event_key", "size"),
        eligible_event_files=("event_key", "nunique"),
        eligible_assets=("asset_key", "nunique"),
    )
    .sort_values(["farm", "final_label"])
    .reset_index(drop=True)
)

ROW_LABEL_MANIFEST_SUMMARY_3K = (
    ROW_LABEL_MANIFEST.groupby(
        [
            "farm",
            "final_label",
            "modeling_eligible",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        manifest_rows=("event_key", "size"),
        event_files=("event_key", "nunique"),
        assets=("asset_key", "nunique"),
    )
    .sort_values(
        [
            "farm",
            "final_label",
            "modeling_eligible",
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 8. Display compact results
# ---------------------------------------------------------

DISPLAY_LIMIT_3K = 100

print("\nCanonical-retention summary:")
print(
    EXACT_SAME_LABEL_CANONICAL_RETENTION_SUMMARY
    .to_string(index=False)
)

print("\nAffected-event manifest summary:")
print(
    POST_DEDUPLICATION_AFFECTED_EVENT_SUMMARY
    .to_string(index=False)
)

print("\nPost-deduplication eligible-label summary:")
print(
    POST_DEDUPLICATION_ELIGIBLE_LABEL_SUMMARY
    .to_string(index=False)
)

print("\nRow-level exclusion audit:")
print(
    EXACT_SAME_LABEL_CANONICAL_RETENTION_AUDIT.head(
        DISPLAY_LIMIT_3K
    ).to_string(index=False)
)

if len(
    EXACT_SAME_LABEL_CANONICAL_RETENTION_AUDIT
) > DISPLAY_LIMIT_3K:
    print(
        f"... display truncated: "
        f"{len(EXACT_SAME_LABEL_CANONICAL_RETENTION_AUDIT):,} "
        "total rows retained in "
        "EXACT_SAME_LABEL_CANONICAL_RETENTION_AUDIT."
    )

ambiguous_groups_after_audit_3k = int(
    (
        ~SAME_LABEL_TIMESTAMP_COLLISION_GROUPS[
            "collision_relationship"
        ].eq("exact_same_label_source_copy")
    ).sum()
)

print("\nCell 3K completed successfully.")

print(
    "Eligible rows before canonical retention:",
    eligible_before_3k,
)

print(
    "Verified exact duplicate timestamp groups:",
    len(EXACT_SAME_LABEL_CANONICAL_RETENTION_AUDIT),
)

print(
    "Canonical farm_a_event_51 rows retained:",
    int(canonical_mask_3k.sum()),
)

print(
    "Redundant farm_a_event_72 rows excluded:",
    int(exclusion_mask_3k.sum()),
)

print(
    "Eligible rows after canonical retention:",
    eligible_after_3k,
)

print(
    "Eligible cross-label timestamp groups:",
    len(CROSS_LABEL_ELIGIBLE_TIMESTAMP_GROUPS),
)

print(
    "Ambiguous same-label collision groups:",
    ambiguous_groups_after_audit_3k,
)

print(
    "Assigned train/validation/test rows:",
    int(
        ROW_LABEL_MANIFEST[
            "split_assignment"
        ].notna().sum()
    ),
)

print(
    "Manifest rows deleted:",
    len(manifest_before_3k) - len(ROW_LABEL_MANIFEST),
)

print(
    "\nOnly the 1,009 verified redundant rows were marked "
    "modeling-ineligible. No source row, label, timestamp, "
    "or split assignment was changed."
)


Canonical-retention summary:
farm asset_id        asset_key final_label retained_event_key excluded_event_key                                 retention_rule  duplicate_timestamp_groups  canonical_rows_retained  redundant_rows_excluded first_duplicate_timestamp_utc last_duplicate_timestamp_utc
   A       21 farm_a::asset_21     anomaly    farm_a_event_51    farm_a_event_72 retain_farm_a_event_51_exclude_farm_a_event_72                        1009                     1009                     1009     2023-10-10 08:40:00+00:00    2023-10-17 08:40:00+00:00

Affected-event manifest summary:
farm asset_id        asset_key       event_key final_label  manifest_rows  modeling_eligible_rows  canonical_rows_retained  redundant_rows_excluded
   A       21 farm_a::asset_21 farm_a_event_51     anomaly           1673                    1673                     1009                        0
   A       21 farm_a::asset_21 farm_a_event_51         NaN          52763                       0             

In [33]:
"""Cell 3L — publication visualization suite after deterministic retention.

Run inside the notebook namespace immediately after Cell 3K:

    %run -i cell_3l_paper_visualization_suite.py

The cell writes publication-ready PNG and PDF figures, CSV/LaTeX tables,
and a machine-readable figure registry. It uses only post-policy manifest rows
and never changes ROW_LABEL_MANIFEST or assigns a data split.
"""

from __future__ import annotations

import math
import re
from pathlib import Path

import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# 1. Validate Cell 3K and prepare an immutable plotting frame
# -----------------------------------------------------------------------------

if "ROW_LABEL_MANIFEST" not in globals():
    raise NameError("Run Cells 3I–3K before Cell 3L.")

REQUIRED_3L = {
    "farm",
    "asset_id",
    "asset_key",
    "event_key",
    "source_row_index",
    "timestamp_utc",
    "final_label",
    "modeling_eligible",
    "split_assignment",
}

missing_3l = REQUIRED_3L - set(ROW_LABEL_MANIFEST.columns)
if missing_3l:
    raise ValueError(
        "ROW_LABEL_MANIFEST is missing columns required by Cell 3L: "
        f"{sorted(missing_3l)}"
    )

EXPECTED_ELIGIBLE_3L = 213_537
eligible_count_3l = int(ROW_LABEL_MANIFEST["modeling_eligible"].sum())
if eligible_count_3l != EXPECTED_ELIGIBLE_3L:
    raise ValueError(
        "Cell 3L expects the verified post-3K eligible count of 213,537, "
        f"but found {eligible_count_3l:,}."
    )

if ROW_LABEL_MANIFEST["split_assignment"].notna().any():
    raise ValueError("Cell 3L must run before train/validation/test assignment.")

optional_3l = [
    c
    for c in [
        "event_id",
        "raw_id",
        "source_event_label",
        "same_label_duplicate_action",
        "same_label_duplicate_canonical_event_key",
        "same_label_duplicate_reason",
    ]
    if c in ROW_LABEL_MANIFEST.columns
]

manifest_columns_3l = sorted(REQUIRED_3L | set(optional_3l))
manifest_3l = ROW_LABEL_MANIFEST.loc[:, manifest_columns_3l].copy()

manifest_3l["farm"] = manifest_3l["farm"].astype("string").str.strip()
manifest_3l["asset_key"] = manifest_3l["asset_key"].astype("string")
manifest_3l["event_key"] = manifest_3l["event_key"].astype("string")
manifest_3l["final_label"] = (
    manifest_3l["final_label"].astype("string").str.strip().str.lower()
)
manifest_3l["timestamp_utc"] = pd.to_datetime(
    manifest_3l["timestamp_utc"], errors="coerce", utc=True
)
manifest_3l["modeling_eligible"] = (
    manifest_3l["modeling_eligible"].fillna(False).astype(bool)
)

eligible_3l = manifest_3l.loc[
    manifest_3l["modeling_eligible"]
    & manifest_3l["final_label"].isin(["normal", "anomaly"])
    & manifest_3l["timestamp_utc"].notna()
].copy()

if len(eligible_3l) != EXPECTED_ELIGIBLE_3L:
    raise ValueError(
        "All 213,537 post-retention eligible rows must have a valid timestamp "
        "and a normalized normal/anomaly label."
    )

if eligible_3l[["event_key", "source_row_index"]].duplicated().any():
    raise ValueError("Eligible manifest source keys are not unique.")

FARMS_3L = sorted(eligible_3l["farm"].dropna().astype(str).unique())
LABELS_3L = ["normal", "anomaly"]

if FARMS_3L != ["A", "B", "C"]:
    raise ValueError(f"Expected farms A, B, and C; found {FARMS_3L}.")

eligible_3l["date_utc"] = eligible_3l["timestamp_utc"].dt.floor("D")
eligible_3l["month_utc"] = (
    eligible_3l["timestamp_utc"]
    .dt.tz_convert(None)
    .dt.to_period("M")
    .dt.to_timestamp()
)
eligible_3l["hour_utc"] = eligible_3l["timestamp_utc"].dt.hour
eligible_3l["weekday_number"] = eligible_3l["timestamp_utc"].dt.dayofweek
eligible_3l["weekday"] = eligible_3l["timestamp_utc"].dt.day_name().str[:3]


# -----------------------------------------------------------------------------
# 2. Journal-oriented style and artifact registry
# -----------------------------------------------------------------------------

OUTPUT_ROOT_3L = Path("paper_visuals_3l")
PNG_DIR_3L = OUTPUT_ROOT_3L / "figures_png"
PDF_DIR_3L = OUTPUT_ROOT_3L / "figures_pdf"
TABLE_DIR_3L = OUTPUT_ROOT_3L / "tables"

for directory_3l in [PNG_DIR_3L, PDF_DIR_3L, TABLE_DIR_3L]:
    directory_3l.mkdir(parents=True, exist_ok=True)

COLORS_3L = {
    "normal": "#0072B2",
    "anomaly": "#D55E00",
    "eligible": "#009E73",
    "exact_copy_excluded": "#CC79A7",
    "other_ineligible": "#999999",
    "A": "#0072B2",
    "B": "#E69F00",
    "C": "#009E73",
}

mpl.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.family": "DejaVu Serif",
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

FIGURE_REGISTRY_3L: list[dict] = []
TABLE_REGISTRY_3L: list[dict] = []


def slug_3l(value: object) -> str:
    value = re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_").lower()
    return value or "item"


def finish_axes_3l(ax, xlabel=None, ylabel=None, legend=True):
    if xlabel is not None:
        ax.set_xlabel(xlabel)
    if ylabel is not None:
        ax.set_ylabel(ylabel)
    ax.grid(axis="y", color="#D9D9D9", linewidth=0.6, alpha=0.7)
    ax.set_axisbelow(True)
    if legend:
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            ax.legend(frameon=False)


def save_figure_3l(
    fig,
    figure_id,
    slug,
    title,
    tier,
    claim,
    source,
):
    filename = f"{figure_id}_{slug_3l(slug)}"
    png_path = PNG_DIR_3L / f"{filename}.png"
    pdf_path = PDF_DIR_3L / f"{filename}.pdf"
    fig.savefig(png_path, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    FIGURE_REGISTRY_3L.append(
        {
            "figure_id": figure_id,
            "title": title,
            "tier": tier,
            "claim_supported": claim,
            "source_table": source,
            "png_path": str(png_path),
            "pdf_path": str(pdf_path),
        }
    )


def save_table_3l(table_id, name, frame, purpose):
    clean = frame.copy()
    csv_path = TABLE_DIR_3L / f"{table_id}_{slug_3l(name)}.csv"
    tex_path = TABLE_DIR_3L / f"{table_id}_{slug_3l(name)}.tex"
    clean.to_csv(csv_path, index=False)

    # Use a dependency-free LaTeX writer so tables remain exportable even when
    # pandas' optional Jinja2 dependency is unavailable.
    def latex_escape_3l(value):
        text = str(value)
        replacements = [
            ("\\", r"\textbackslash{}"),
            ("&", r"\&"),
            ("%", r"\%"),
            ("$", r"\$"),
            ("#", r"\#"),
            ("_", r"\_"),
            ("{", r"\{"),
            ("}", r"\}"),
            ("~", r"\textasciitilde{}"),
            ("^", r"\textasciicircum{}"),
        ]
        for old, new in replacements:
            text = text.replace(old, new)
        return text

    def latex_value_3l(value):
        if pd.isna(value):
            return "--"
        if isinstance(value, (pd.Timestamp, np.datetime64)):
            return latex_escape_3l(pd.Timestamp(value).isoformat())
        if isinstance(value, (float, np.floating)):
            return f"{float(value):.6g}"
        return latex_escape_3l(value)

    alignment_3l = "".join(
        "r" if pd.api.types.is_numeric_dtype(clean[column]) else "l"
        for column in clean.columns
    )
    header_3l = " & ".join(
        latex_escape_3l(column.replace("_", " ")) for column in clean.columns
    )
    with tex_path.open("w", encoding="utf-8") as handle_3l:
        handle_3l.write(f"\\begin{{tabular}}{{{alignment_3l}}}\n")
        handle_3l.write("\\hline\n")
        handle_3l.write(header_3l + r" \\" + "\n")
        handle_3l.write("\\hline\n")
        for row_3l in clean.itertuples(index=False, name=None):
            handle_3l.write(
                " & ".join(latex_value_3l(value) for value in row_3l)
                + r" \\" + "\n"
            )
        handle_3l.write("\\hline\n")
        handle_3l.write("\\end{tabular}\n")
    TABLE_REGISTRY_3L.append(
        {
            "table_id": table_id,
            "name": name,
            "purpose": purpose,
            "rows": len(clean),
            "columns": len(clean.columns),
            "csv_path": str(csv_path),
            "latex_path": str(tex_path) if tex_path else pd.NA,
        }
    )


def add_bar_labels_3l(ax, fmt="{x:,.0f}", fontsize=7):
    for container in ax.containers:
        try:
            ax.bar_label(
                container,
                labels=[fmt.format(x=v) if np.isfinite(v) else "" for v in container.datavalues],
                padding=2,
                fontsize=fontsize,
            )
        except Exception:
            pass


def grouped_bar_3l(
    frame,
    index,
    columns,
    values,
    figure_id,
    slug,
    title,
    ylabel,
    tier,
    claim,
    source,
    normalize=False,
):
    pivot = frame.pivot_table(
        index=index,
        columns=columns,
        values=values,
        aggfunc="sum",
        fill_value=0,
        observed=False,
    )
    pivot = pivot.reindex(columns=[x for x in LABELS_3L if x in pivot.columns])
    if normalize:
        pivot = pivot.div(pivot.sum(axis=1).replace(0, np.nan), axis=0) * 100
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    pivot.plot(
        kind="bar",
        ax=ax,
        color=[COLORS_3L.get(str(c), "#777777") for c in pivot.columns],
        width=0.75,
    )
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=0)
    finish_axes_3l(ax, index.replace("_", " ").title(), ylabel)
    add_bar_labels_3l(ax, fmt="{x:.1f}" if normalize else "{x:,.0f}")
    fig.tight_layout()
    save_figure_3l(fig, figure_id, slug, title, tier, claim, source)


def horizontal_rank_3l(
    frame,
    label_column,
    value_column,
    figure_id,
    slug,
    title,
    xlabel,
    tier,
    claim,
    source,
    top_n=25,
    color="#0072B2",
):
    plot = frame.nlargest(top_n, value_column).sort_values(value_column)
    height = max(4.0, 0.24 * len(plot) + 1.2)
    fig, ax = plt.subplots(figsize=(7.4, height))
    ax.barh(plot[label_column].astype(str), plot[value_column], color=color)
    ax.set_title(title)
    finish_axes_3l(ax, xlabel, None, legend=False)
    ax.grid(axis="x", color="#D9D9D9", linewidth=0.6, alpha=0.7)
    ax.grid(axis="y", visible=False)
    add_bar_labels_3l(ax)
    fig.tight_layout()
    save_figure_3l(fig, figure_id, slug, title, tier, claim, source)


def matrix_figure_3l(
    matrix,
    figure_id,
    slug,
    title,
    xlabel,
    ylabel,
    tier,
    claim,
    source,
    cmap="viridis",
):
    matrix = matrix.copy()
    fig_width = max(6.0, 0.24 * len(matrix.columns) + 2.2)
    fig_height = max(3.8, 0.24 * len(matrix.index) + 1.8)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    image = ax.imshow(matrix.to_numpy(dtype=float), aspect="auto", cmap=cmap)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(len(matrix.columns)))
    ax.set_xticklabels(matrix.columns, rotation=90)
    ax.set_yticks(range(len(matrix.index)))
    ax.set_yticklabels(matrix.index)
    colorbar = fig.colorbar(image, ax=ax, pad=0.02)
    colorbar.ax.set_ylabel("Count", rotation=270, labelpad=12)
    fig.tight_layout()
    save_figure_3l(fig, figure_id, slug, title, tier, claim, source)


# -----------------------------------------------------------------------------
# 3. Reusable paper tables
# -----------------------------------------------------------------------------

eligible_label_summary_3l = (
    eligible_3l.groupby(["farm", "final_label"], as_index=False)
    .agg(
        eligible_rows=("event_key", "size"),
        eligible_event_files=("event_key", "nunique"),
        eligible_assets=("asset_key", "nunique"),
        first_timestamp_utc=("timestamp_utc", "min"),
        last_timestamp_utc=("timestamp_utc", "max"),
    )
    .sort_values(["farm", "final_label"])
    .reset_index(drop=True)
)

event_summary_3l = (
    eligible_3l.groupby(
        ["farm", "asset_id", "asset_key", "event_key", "final_label"],
        as_index=False,
        dropna=False,
    )
    .agg(
        eligible_rows=("source_row_index", "size"),
        unique_timestamps=("timestamp_utc", "nunique"),
        start_utc=("timestamp_utc", "min"),
        end_utc=("timestamp_utc", "max"),
    )
)
event_summary_3l["duration_hours"] = (
    event_summary_3l["end_utc"] - event_summary_3l["start_utc"]
).dt.total_seconds() / 3600.0

ordered_eligible_3l = eligible_3l.sort_values(["event_key", "timestamp_utc"])
intervals_3l = ordered_eligible_3l.loc[
    :,
    ["farm", "asset_key", "event_key", "final_label", "timestamp_utc"],
].copy()
intervals_3l["interval_minutes"] = (
    intervals_3l.groupby("event_key", sort=False)["timestamp_utc"]
    .diff()
    .dt.total_seconds()
    .div(60.0)
)
intervals_3l = intervals_3l.loc[
    intervals_3l["interval_minutes"].gt(0)
    & intervals_3l["interval_minutes"].le(24 * 60)
].copy()

event_cadence_3l = (
    intervals_3l.groupby("event_key", as_index=False)
    .agg(
        median_interval_minutes=("interval_minutes", "median"),
        p95_interval_minutes=("interval_minutes", lambda x: x.quantile(0.95)),
        maximum_interval_minutes=("interval_minutes", "max"),
    )
)
event_summary_3l = event_summary_3l.merge(
    event_cadence_3l, on="event_key", how="left", validate="one_to_one"
)

asset_summary_3l = (
    eligible_3l.groupby(["farm", "asset_id", "asset_key"], as_index=False)
    .agg(
        eligible_rows=("event_key", "size"),
        eligible_events=("event_key", "nunique"),
        labels=("final_label", "nunique"),
        first_timestamp_utc=("timestamp_utc", "min"),
        last_timestamp_utc=("timestamp_utc", "max"),
    )
)

asset_label_rows_3l = (
    eligible_3l.groupby(["farm", "asset_key", "final_label"], as_index=False)
    .size()
    .rename(columns={"size": "eligible_rows"})
)
asset_label_events_3l = (
    eligible_3l.groupby(["farm", "asset_key", "final_label"], as_index=False)
    .agg(eligible_events=("event_key", "nunique"))
)

row_pivot_3l = asset_label_rows_3l.pivot_table(
    index=["farm", "asset_key"],
    columns="final_label",
    values="eligible_rows",
    aggfunc="sum",
    fill_value=0,
).reset_index()
for label_3l in LABELS_3L:
    if label_3l not in row_pivot_3l:
        row_pivot_3l[label_3l] = 0
row_pivot_3l["anomaly_to_normal_row_ratio"] = (
    row_pivot_3l["anomaly"] / row_pivot_3l["normal"].replace(0, np.nan)
)

event_pivot_3l = asset_label_events_3l.pivot_table(
    index=["farm", "asset_key"],
    columns="final_label",
    values="eligible_events",
    aggfunc="sum",
    fill_value=0,
).reset_index()
for label_3l in LABELS_3L:
    if label_3l not in event_pivot_3l:
        event_pivot_3l[label_3l] = 0
event_pivot_3l["anomaly_to_normal_event_ratio"] = (
    event_pivot_3l["anomaly"] / event_pivot_3l["normal"].replace(0, np.nan)
)

manifest_status_3l = pd.Series(
    "other_ineligible", index=manifest_3l.index, dtype="string"
)
manifest_status_3l.loc[manifest_3l["modeling_eligible"]] = "eligible"
if "same_label_duplicate_action" in manifest_3l:
    manifest_status_3l.loc[
        manifest_3l["same_label_duplicate_action"]
        .astype("string")
        .eq("redundant_exact_copy_excluded")
    ] = "exact_copy_excluded"
manifest_3l["manifest_status"] = manifest_status_3l

status_summary_3l = (
    manifest_3l.groupby(["farm", "manifest_status"], as_index=False)
    .size()
    .rename(columns={"size": "manifest_rows"})
)

farm_temporal_summary_3l = (
    eligible_3l.groupby("farm", as_index=False)
    .agg(
        eligible_rows=("event_key", "size"),
        assets=("asset_key", "nunique"),
        events=("event_key", "nunique"),
        first_timestamp_utc=("timestamp_utc", "min"),
        last_timestamp_utc=("timestamp_utc", "max"),
        active_days=("date_utc", "nunique"),
    )
)
farm_temporal_summary_3l["calendar_span_days"] = (
    farm_temporal_summary_3l["last_timestamp_utc"]
    - farm_temporal_summary_3l["first_timestamp_utc"]
).dt.total_seconds() / 86400.0

tables_to_save_3l = [
    ("T001", "eligible_label_summary", eligible_label_summary_3l,
     "Post-deduplication class support by farm."),
    ("T002", "event_summary", event_summary_3l,
     "Event size, time coverage, duration, and sampling cadence."),
    ("T003", "asset_summary", asset_summary_3l,
     "Asset-level data support and temporal coverage."),
    ("T004", "asset_label_rows", asset_label_rows_3l,
     "Eligible normal/anomaly row counts by asset."),
    ("T005", "asset_label_events", asset_label_events_3l,
     "Eligible normal/anomaly event counts by asset."),
    ("T006", "manifest_status_summary", status_summary_3l,
     "Manifest disposition counts after canonical retention."),
    ("T007", "farm_temporal_summary", farm_temporal_summary_3l,
     "Temporal coverage and support by farm."),
]

for table_spec_3l in tables_to_save_3l:
    save_table_3l(*table_spec_3l)

optional_tables_3l = [
    ("T008", "canonical_retention_summary",
     "EXACT_SAME_LABEL_CANONICAL_RETENTION_SUMMARY",
     "Canonical source retention and redundant source exclusion."),
    ("T009", "affected_event_summary",
     "POST_DEDUPLICATION_AFFECTED_EVENT_SUMMARY",
     "Manifest effect for the two audited overlapping events."),
    ("T010", "collision_relationship_counts",
     "SAME_LABEL_COLLISION_RELATIONSHIP_COUNTS",
     "Exact versus ambiguous same-label collision relationships."),
    ("T011", "measurement_vector_reuse_summary",
     "MEASUREMENT_VECTOR_REUSE_SUMMARY",
     "Measurement vectors recurring at different timestamps."),
]

for table_id_3l, name_3l, object_name_3l, purpose_3l in optional_tables_3l:
    if object_name_3l in globals():
        save_table_3l(table_id_3l, name_3l, globals()[object_name_3l], purpose_3l)


# -----------------------------------------------------------------------------
# 4. Manuscript/overview candidates (C001–C035)
# -----------------------------------------------------------------------------

grouped_bar_3l(
    eligible_label_summary_3l,
    "farm", "final_label", "eligible_rows",
    "C001", "eligible_rows_by_farm_label",
    "Post-deduplication eligible observations",
    "Eligible observations", "core",
    "Shows class support after verified duplicate exclusion.",
    "eligible_label_summary_3l",
)

grouped_bar_3l(
    eligible_label_summary_3l,
    "farm", "final_label", "eligible_rows",
    "C002", "class_composition_by_farm",
    "Within-farm class composition",
    "Share of eligible observations (%)", "core",
    "Shows farm-specific class imbalance without conflating farm size.",
    "eligible_label_summary_3l", normalize=True,
)

grouped_bar_3l(
    eligible_label_summary_3l,
    "farm", "final_label", "eligible_event_files",
    "C003", "eligible_events_by_farm_label",
    "Eligible event files by farm and class",
    "Event files", "core",
    "Shows independent event support for each class.",
    "eligible_label_summary_3l",
)

grouped_bar_3l(
    eligible_label_summary_3l,
    "farm", "final_label", "eligible_assets",
    "C004", "eligible_assets_by_farm_label",
    "Eligible physical assets by farm and class",
    "Physical assets", "core",
    "Shows the asset-level support available for leakage-safe evaluation.",
    "eligible_label_summary_3l",
)

overall_label_3l = (
    eligible_3l.groupby("final_label", as_index=False)
    .size()
    .rename(columns={"size": "eligible_rows"})
)
fig, ax = plt.subplots(figsize=(5.4, 4.0))
ax.bar(
    overall_label_3l["final_label"],
    overall_label_3l["eligible_rows"],
    color=[COLORS_3L.get(x, "#777777") for x in overall_label_3l["final_label"]],
)
ax.set_title("Overall post-deduplication class support")
finish_axes_3l(ax, "Class", "Eligible observations", legend=False)
add_bar_labels_3l(ax)
fig.tight_layout()
save_figure_3l(
    fig, "C005", "overall_class_support", "Overall class support", "core",
    "Summarizes the final row-level class balance.", "eligible_3l"
)

event_groups_3l = []
event_group_labels_3l = []
event_group_colors_3l = []
for farm_3l in FARMS_3L:
    for label_3l in LABELS_3L:
        values_3l = event_summary_3l.loc[
            event_summary_3l["farm"].eq(farm_3l)
            & event_summary_3l["final_label"].eq(label_3l),
            "eligible_rows",
        ].to_numpy()
        if len(values_3l):
            event_groups_3l.append(values_3l)
            event_group_labels_3l.append(f"{farm_3l}\n{label_3l}")
            event_group_colors_3l.append(COLORS_3L[label_3l])
fig, ax = plt.subplots(figsize=(7.6, 4.5))
box_3l = ax.boxplot(event_groups_3l, tick_labels=event_group_labels_3l,
                    showfliers=False, patch_artist=True)
for patch_3l, color_3l in zip(box_3l["boxes"], event_group_colors_3l):
    patch_3l.set_facecolor(color_3l)
    patch_3l.set_alpha(0.75)
ax.set_title("Eligible observations per event")
finish_axes_3l(ax, "Farm and class", "Observations per event", legend=False)
fig.tight_layout()
save_figure_3l(
    fig, "C006", "event_size_distribution", "Event size distribution", "core",
    "Shows event-size heterogeneity across farms and classes.", "event_summary_3l"
)

horizontal_rank_3l(
    asset_summary_3l, "asset_key", "eligible_rows",
    "C007", "asset_row_support", "Eligible observations by asset",
    "Eligible observations", "core",
    "Identifies concentration of row support in particular assets.",
    "asset_summary_3l", top_n=35,
)

horizontal_rank_3l(
    asset_summary_3l, "asset_key", "eligible_events",
    "C008", "asset_event_support", "Eligible event files by asset",
    "Eligible event files", "core",
    "Identifies the number of independent events contributed by each asset.",
    "asset_summary_3l", top_n=35, color="#E69F00",
)

ratio_rows_3l = row_pivot_3l.dropna(subset=["anomaly_to_normal_row_ratio"]).copy()
horizontal_rank_3l(
    ratio_rows_3l, "asset_key", "anomaly_to_normal_row_ratio",
    "C009", "asset_row_class_ratio", "Anomaly-to-normal row ratio by asset",
    "Anomaly / normal observations", "core",
    "Shows asset-level class imbalance among assets with normal support.",
    "row_pivot_3l", top_n=35, color="#D55E00",
)

ratio_events_3l = event_pivot_3l.dropna(subset=["anomaly_to_normal_event_ratio"]).copy()
horizontal_rank_3l(
    ratio_events_3l, "asset_key", "anomaly_to_normal_event_ratio",
    "C010", "asset_event_class_ratio", "Anomaly-to-normal event ratio by asset",
    "Anomaly / normal events", "core",
    "Shows imbalance in independent event support rather than row volume.",
    "event_pivot_3l", top_n=35, color="#CC79A7",
)

duration_groups_3l = []
duration_labels_3l = []
duration_colors_3l = []
for farm_3l in FARMS_3L:
    for label_3l in LABELS_3L:
        values_3l = event_summary_3l.loc[
            event_summary_3l["farm"].eq(farm_3l)
            & event_summary_3l["final_label"].eq(label_3l),
            "duration_hours",
        ].dropna().to_numpy()
        if len(values_3l):
            duration_groups_3l.append(values_3l)
            duration_labels_3l.append(f"{farm_3l}\n{label_3l}")
            duration_colors_3l.append(COLORS_3L[label_3l])
fig, ax = plt.subplots(figsize=(7.6, 4.5))
box_3l = ax.boxplot(duration_groups_3l, tick_labels=duration_labels_3l,
                    showfliers=False, patch_artist=True)
for patch_3l, color_3l in zip(box_3l["boxes"], duration_colors_3l):
    patch_3l.set_facecolor(color_3l)
    patch_3l.set_alpha(0.75)
ax.set_title("Event duration by farm and class")
finish_axes_3l(ax, "Farm and class", "Duration (hours)", legend=False)
fig.tight_layout()
save_figure_3l(
    fig, "C011", "event_duration_distribution", "Event duration distribution", "core",
    "Shows whether event windows differ materially by source and class.", "event_summary_3l"
)

fig, ax = plt.subplots(figsize=(6.4, 4.5))
for label_3l in LABELS_3L:
    subset_3l = event_summary_3l.loc[event_summary_3l["final_label"].eq(label_3l)]
    ax.scatter(
        subset_3l["duration_hours"], subset_3l["eligible_rows"],
        s=34, alpha=0.72, color=COLORS_3L[label_3l], label=label_3l.title(),
    )
ax.set_title("Event duration and eligible row support")
finish_axes_3l(ax, "Duration (hours)", "Eligible observations")
fig.tight_layout()
save_figure_3l(
    fig, "C012", "event_duration_vs_rows", "Event duration versus rows", "core",
    "Checks whether event size is primarily explained by temporal duration.", "event_summary_3l"
)

cadence_summary_3l = (
    event_summary_3l.groupby(["farm", "final_label"], as_index=False)
    .agg(median_interval_minutes=("median_interval_minutes", "median"))
)
grouped_bar_3l(
    cadence_summary_3l,
    "farm", "final_label", "median_interval_minutes",
    "C013", "median_sampling_interval", "Median within-event sampling interval",
    "Median interval (minutes)", "core",
    "Verifies sampling cadence comparability across farms and classes.",
    "event_summary_3l",
)

cadence_groups_3l = []
cadence_labels_3l = []
cadence_colors_3l = []
for farm_3l in FARMS_3L:
    for label_3l in LABELS_3L:
        values_3l = event_summary_3l.loc[
            event_summary_3l["farm"].eq(farm_3l)
            & event_summary_3l["final_label"].eq(label_3l),
            "p95_interval_minutes",
        ].dropna().to_numpy()
        if len(values_3l):
            cadence_groups_3l.append(values_3l)
            cadence_labels_3l.append(f"{farm_3l}\n{label_3l}")
            cadence_colors_3l.append(COLORS_3L[label_3l])
fig, ax = plt.subplots(figsize=(7.6, 4.5))
box_3l = ax.boxplot(cadence_groups_3l, tick_labels=cadence_labels_3l,
                    showfliers=False, patch_artist=True)
for patch_3l, color_3l in zip(box_3l["boxes"], cadence_colors_3l):
    patch_3l.set_facecolor(color_3l)
    patch_3l.set_alpha(0.75)
ax.set_title("Event-level 95th-percentile sampling gaps")
finish_axes_3l(ax, "Farm and class", "Gap (minutes)", legend=False)
fig.tight_layout()
save_figure_3l(
    fig, "C014", "sampling_gap_distribution", "Sampling gap distribution", "core",
    "Shows long within-event sampling gaps that may affect window construction.",
    "event_summary_3l"
)

monthly_farm_3l = (
    eligible_3l.groupby(["month_utc", "farm"], as_index=False)
    .size().rename(columns={"size": "eligible_rows"})
)
fig, ax = plt.subplots(figsize=(8.2, 4.4))
for farm_3l in FARMS_3L:
    subset_3l = monthly_farm_3l.loc[monthly_farm_3l["farm"].eq(farm_3l)]
    ax.plot(subset_3l["month_utc"], subset_3l["eligible_rows"], marker="o",
            linewidth=1.5, markersize=3.5, color=COLORS_3L[farm_3l], label=f"Farm {farm_3l}")
ax.set_title("Monthly eligible observations by farm")
finish_axes_3l(ax, "Month (UTC)", "Eligible observations")
ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=10))
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
fig.tight_layout()
save_figure_3l(
    fig, "C015", "monthly_rows_by_farm", "Monthly rows by farm", "core",
    "Shows temporal coverage and source-specific acquisition periods.", "eligible_3l"
)

monthly_label_3l = (
    eligible_3l.groupby(["month_utc", "final_label"], as_index=False)
    .size().rename(columns={"size": "eligible_rows"})
)
fig, ax = plt.subplots(figsize=(8.2, 4.4))
for label_3l in LABELS_3L:
    subset_3l = monthly_label_3l.loc[monthly_label_3l["final_label"].eq(label_3l)]
    ax.plot(subset_3l["month_utc"], subset_3l["eligible_rows"], marker="o",
            linewidth=1.5, markersize=3.5, color=COLORS_3L[label_3l], label=label_3l.title())
ax.set_title("Monthly eligible observations by class")
finish_axes_3l(ax, "Month (UTC)", "Eligible observations")
ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=10))
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
fig.tight_layout()
save_figure_3l(
    fig, "C016", "monthly_rows_by_label", "Monthly rows by class", "core",
    "Shows whether class support is temporally separated.", "eligible_3l"
)

daily_label_3l = (
    eligible_3l.groupby(["date_utc", "final_label"], as_index=False)
    .size().rename(columns={"size": "eligible_rows"})
    .sort_values("date_utc")
)
daily_label_3l["cumulative_rows"] = (
    daily_label_3l.groupby("final_label")["eligible_rows"].cumsum()
)
fig, ax = plt.subplots(figsize=(8.2, 4.4))
for label_3l in LABELS_3L:
    subset_3l = daily_label_3l.loc[daily_label_3l["final_label"].eq(label_3l)]
    ax.plot(subset_3l["date_utc"], subset_3l["cumulative_rows"],
            linewidth=1.6, color=COLORS_3L[label_3l], label=label_3l.title())
ax.set_title("Cumulative eligible observations through time")
finish_axes_3l(ax, "Date (UTC)", "Cumulative observations")
ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=10))
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
fig.tight_layout()
save_figure_3l(
    fig, "C017", "cumulative_rows_by_label", "Cumulative class support", "core",
    "Shows how class evidence accumulates over the observation horizon.", "eligible_3l"
)

hour_profile_3l = (
    eligible_3l.groupby(["hour_utc", "final_label"], as_index=False)
    .size().rename(columns={"size": "eligible_rows"})
)
fig, ax = plt.subplots(figsize=(7.8, 4.2))
for label_3l in LABELS_3L:
    subset_3l = hour_profile_3l.loc[hour_profile_3l["final_label"].eq(label_3l)]
    ax.plot(subset_3l["hour_utc"], subset_3l["eligible_rows"], marker="o",
            color=COLORS_3L[label_3l], label=label_3l.title())
ax.set_xticks(range(0, 24, 2))
ax.set_title("UTC hour-of-day sampling profile")
finish_axes_3l(ax, "Hour (UTC)", "Eligible observations")
fig.tight_layout()
save_figure_3l(
    fig, "C018", "hourly_sampling_profile", "Hourly sampling profile", "core",
    "Checks for class-dependent hour-of-day acquisition bias.", "eligible_3l"
)

weekday_order_3l = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
weekday_profile_3l = (
    eligible_3l.groupby(["weekday", "final_label"], as_index=False)
    .size().rename(columns={"size": "eligible_rows"})
)
weekday_profile_3l["weekday"] = pd.Categorical(
    weekday_profile_3l["weekday"], categories=weekday_order_3l, ordered=True
)
weekday_profile_3l = weekday_profile_3l.sort_values("weekday")
grouped_bar_3l(
    weekday_profile_3l,
    "weekday", "final_label", "eligible_rows",
    "C019", "weekday_sampling_profile", "UTC weekday sampling profile",
    "Eligible observations", "core",
    "Checks for class-dependent day-of-week acquisition bias.",
    "eligible_3l",
)

for number_3l, label_3l in [(20, "anomaly"), (21, "normal")]:
    heat_3l = (
        eligible_3l.loc[eligible_3l["final_label"].eq(label_3l)]
        .groupby(["weekday_number", "hour_utc"]).size()
        .unstack(fill_value=0)
        .reindex(index=range(7), columns=range(24), fill_value=0)
    )
    heat_3l.index = weekday_order_3l
    matrix_figure_3l(
        heat_3l, f"C{number_3l:03d}", f"{label_3l}_weekday_hour_heatmap",
        f"{label_3l.title()} observations by weekday and UTC hour",
        "Hour (UTC)", "Weekday", "core",
        f"Shows the temporal acquisition footprint of {label_3l} observations.",
        "eligible_3l", cmap="magma" if label_3l == "anomaly" else "Blues",
    )

status_pivot_3l = status_summary_3l.pivot_table(
    index="farm", columns="manifest_status", values="manifest_rows",
    aggfunc="sum", fill_value=0,
)
status_order_3l = [
    x for x in ["eligible", "exact_copy_excluded", "other_ineligible"]
    if x in status_pivot_3l.columns
]
status_pivot_3l = status_pivot_3l.reindex(columns=status_order_3l)

fig, ax = plt.subplots(figsize=(7.2, 4.3))
status_pivot_3l.plot(
    kind="bar", stacked=True, ax=ax,
    color=[COLORS_3L.get(x, "#777777") for x in status_pivot_3l.columns],
)
ax.set_title("Manifest row disposition by farm")
ax.tick_params(axis="x", rotation=0)
finish_axes_3l(ax, "Farm", "Manifest rows")
fig.tight_layout()
save_figure_3l(
    fig, "C022", "manifest_status_counts", "Manifest disposition counts", "core",
    "Shows eligible, exact-copy-excluded, and other non-modeling rows.", "status_summary_3l"
)

status_share_3l = status_pivot_3l.div(status_pivot_3l.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(7.2, 4.3))
status_share_3l.plot(
    kind="bar", stacked=True, ax=ax,
    color=[COLORS_3L.get(x, "#777777") for x in status_share_3l.columns],
)
ax.set_title("Within-farm manifest disposition")
ax.tick_params(axis="x", rotation=0)
finish_axes_3l(ax, "Farm", "Share of manifest rows (%)")
fig.tight_layout()
save_figure_3l(
    fig, "C023", "manifest_status_shares", "Manifest disposition shares", "core",
    "Normalizes manifest decisions for comparison across differently sized farms.",
    "status_summary_3l"
)

exact_excluded_3l = int(
    manifest_3l["manifest_status"].eq("exact_copy_excluded").sum()
)
dedup_overall_3l = pd.DataFrame(
    {"stage": ["Before retention", "After retention"],
     "eligible_rows": [len(eligible_3l) + exact_excluded_3l, len(eligible_3l)]}
)
fig, ax = plt.subplots(figsize=(5.6, 4.1))
ax.bar(dedup_overall_3l["stage"], dedup_overall_3l["eligible_rows"],
       color=["#999999", COLORS_3L["eligible"]])
ax.set_title("Effect of verified canonical retention")
finish_axes_3l(ax, "Audit stage", "Eligible observations", legend=False)
add_bar_labels_3l(ax)
fig.tight_layout()
save_figure_3l(
    fig, "C024", "dedup_before_after", "Canonical-retention effect", "core",
    "Shows the exact 1,009-row eligibility reduction without deleting data.",
    "manifest_3l"
)

dedup_farm_3l = (
    eligible_3l.groupby("farm").size().rename("after_retention").to_frame()
)
excluded_by_farm_3l = (
    manifest_3l.loc[manifest_3l["manifest_status"].eq("exact_copy_excluded")]
    .groupby("farm").size().rename("excluded_exact_copies")
)
dedup_farm_3l = dedup_farm_3l.join(excluded_by_farm_3l, how="left").fillna(0)
dedup_farm_3l["before_retention"] = (
    dedup_farm_3l["after_retention"] + dedup_farm_3l["excluded_exact_copies"]
)
fig, ax = plt.subplots(figsize=(7.0, 4.2))
dedup_farm_3l[["before_retention", "after_retention"]].plot(
    kind="bar", ax=ax, color=["#999999", COLORS_3L["eligible"]]
)
ax.set_title("Canonical-retention effect by farm")
ax.tick_params(axis="x", rotation=0)
finish_axes_3l(ax, "Farm", "Eligible observations")
add_bar_labels_3l(ax)
fig.tight_layout()
save_figure_3l(
    fig, "C025", "dedup_by_farm", "Canonical retention by farm", "core",
    "Localizes the exact-copy exclusion to its source farm.", "manifest_3l"
)

excluded_event_3l = (
    manifest_3l.loc[manifest_3l["manifest_status"].eq("exact_copy_excluded")]
    .groupby("event_key", as_index=False).size()
    .rename(columns={"size": "excluded_rows"})
)
horizontal_rank_3l(
    excluded_event_3l, "event_key", "excluded_rows",
    "C026", "excluded_rows_by_event", "Verified exact-copy exclusions by event",
    "Excluded rows", "core",
    "Identifies the redundant source event affected by canonical retention.",
    "manifest_3l", top_n=20, color=COLORS_3L["exact_copy_excluded"],
)

if "EXACT_SAME_LABEL_CANONICAL_RETENTION_AUDIT" in globals():
    retention_3l = EXACT_SAME_LABEL_CANONICAL_RETENTION_AUDIT.copy()
    retention_3l["timestamp_utc"] = pd.to_datetime(
        retention_3l["timestamp_utc"], errors="coerce", utc=True
    )
    retention_3l["date_utc"] = retention_3l["timestamp_utc"].dt.floor("D")
    retention_daily_3l = (
        retention_3l.groupby("date_utc", as_index=False).size()
        .rename(columns={"size": "duplicate_pairs"})
    )
    fig, ax = plt.subplots(figsize=(8.0, 4.2))
    ax.plot(retention_daily_3l["date_utc"], retention_daily_3l["duplicate_pairs"],
            marker="o", color=COLORS_3L["exact_copy_excluded"])
    ax.set_title("Temporal extent of verified exact source copies")
    finish_axes_3l(ax, "Date (UTC)", "Duplicate timestamp pairs", legend=False)
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=9))
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
    fig.tight_layout()
    save_figure_3l(
        fig, "C027", "duplicate_overlap_timeline", "Duplicate overlap timeline", "core",
        "Shows the seven-day interval covered by verified exact copies.",
        "EXACT_SAME_LABEL_CANONICAL_RETENTION_AUDIT"
    )

    fig, ax = plt.subplots(figsize=(5.7, 5.0))
    ax.scatter(
        retention_3l["retained_source_row_index"],
        retention_3l["excluded_source_row_index"],
        s=10, alpha=0.55, color=COLORS_3L["exact_copy_excluded"],
    )
    ax.set_title("One-to-one retained and excluded source-row mapping")
    finish_axes_3l(ax, "Retained source-row index", "Excluded source-row index", legend=False)
    fig.tight_layout()
    save_figure_3l(
        fig, "C028", "retained_excluded_row_mapping", "Canonical row mapping", "core",
        "Demonstrates deterministic one-to-one source-row retention.",
        "EXACT_SAME_LABEL_CANONICAL_RETENTION_AUDIT"
    )

event_order_3l = event_summary_3l.sort_values(["farm", "start_utc"]).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9.0, max(6.0, len(event_order_3l) * 0.11 + 1.8)))
for y_3l, row_3l in event_order_3l.iterrows():
    ax.plot([row_3l["start_utc"], row_3l["end_utc"]], [y_3l, y_3l],
            linewidth=2.2, color=COLORS_3L[row_3l["final_label"]])
ax.set_yticks(range(len(event_order_3l)))
ax.set_yticklabels(event_order_3l["event_key"], fontsize=5.5)
ax.set_title("Eligible event temporal coverage")
ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Event file")
ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=12))
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
fig.tight_layout()
save_figure_3l(
    fig, "C029", "event_temporal_coverage", "Event temporal coverage", "core",
    "Shows separation and overlap among eligible event windows.", "event_summary_3l"
)

horizontal_rank_3l(
    event_summary_3l, "event_key", "duration_hours",
    "C030", "longest_events", "Longest eligible event windows",
    "Duration (hours)", "core",
    "Identifies events with the widest temporal support.", "event_summary_3l",
    top_n=20, color="#56B4E9",
)

horizontal_rank_3l(
    event_summary_3l, "event_key", "eligible_rows",
    "C031", "largest_events", "Largest eligible event files",
    "Eligible observations", "core",
    "Identifies events dominating the row-level dataset.", "event_summary_3l",
    top_n=25, color="#E69F00",
)

asset_row_matrix_3l = asset_label_rows_3l.pivot_table(
    index="asset_key", columns="final_label", values="eligible_rows",
    aggfunc="sum", fill_value=0,
).reindex(columns=LABELS_3L, fill_value=0)
matrix_figure_3l(
    asset_row_matrix_3l, "C032", "asset_label_row_heatmap",
    "Eligible observations by asset and class", "Class", "Asset",
    "core", "Shows asset-by-class row support and missing class combinations.",
    "asset_label_rows_3l", cmap="viridis",
)

asset_event_matrix_3l = asset_label_events_3l.pivot_table(
    index="asset_key", columns="final_label", values="eligible_events",
    aggfunc="sum", fill_value=0,
).reindex(columns=LABELS_3L, fill_value=0)
matrix_figure_3l(
    asset_event_matrix_3l, "C033", "asset_label_event_heatmap",
    "Eligible event files by asset and class", "Class", "Asset",
    "core", "Shows independent event support for every asset-class combination.",
    "asset_label_events_3l", cmap="cividis",
)

monthly_farm_matrix_3l = monthly_farm_3l.pivot_table(
    index="farm", columns="month_utc", values="eligible_rows",
    aggfunc="sum", fill_value=0,
)
monthly_farm_matrix_3l.columns = [x.strftime("%Y-%m") for x in monthly_farm_matrix_3l.columns]
matrix_figure_3l(
    monthly_farm_matrix_3l, "C034", "farm_month_activity_heatmap",
    "Monthly acquisition footprint by farm", "Month (UTC)", "Farm",
    "core", "Shows whether farms occupy distinct calendar periods.",
    "monthly_farm_3l", cmap="Blues",
)

fig, ax = plt.subplots(figsize=(8.0, 3.8))
for y_3l, row_3l in farm_temporal_summary_3l.sort_values("farm").reset_index(drop=True).iterrows():
    ax.plot([row_3l["first_timestamp_utc"], row_3l["last_timestamp_utc"]],
            [y_3l, y_3l], linewidth=8, solid_capstyle="round",
            color=COLORS_3L[str(row_3l["farm"])])
ax.set_yticks(range(len(farm_temporal_summary_3l)))
ax.set_yticklabels([f"Farm {x}" for x in sorted(farm_temporal_summary_3l["farm"])])
ax.set_title("Farm-level temporal coverage")
ax.set_xlabel("Time (UTC)")
ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=10))
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
fig.tight_layout()
save_figure_3l(
    fig, "C035", "farm_temporal_coverage", "Farm temporal coverage", "core",
    "Summarizes the observation horizon of each farm.", "farm_temporal_summary_3l"
)


# -----------------------------------------------------------------------------
# 5. Farm-specific supplementary figures (three per farm)
# -----------------------------------------------------------------------------

for farm_number_3l, farm_3l in enumerate(FARMS_3L, start=1):
    farm_events_3l = event_summary_3l.loc[
        event_summary_3l["farm"].eq(farm_3l)
    ].sort_values("start_utc").reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(8.5, max(4.5, len(farm_events_3l) * 0.18 + 1.5)))
    for y_3l, row_3l in farm_events_3l.iterrows():
        ax.plot([row_3l["start_utc"], row_3l["end_utc"]], [y_3l, y_3l],
                linewidth=3, color=COLORS_3L[row_3l["final_label"]])
    ax.set_yticks(range(len(farm_events_3l)))
    ax.set_yticklabels(farm_events_3l["event_key"], fontsize=6)
    ax.set_title(f"Farm {farm_3l}: eligible event timeline")
    ax.set_xlabel("Time (UTC)")
    ax.set_ylabel("Event file")
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=10))
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
    fig.tight_layout()
    save_figure_3l(
        fig, f"F{farm_number_3l:02d}01", f"farm_{farm_3l}_event_timeline",
        f"Farm {farm_3l} event timeline", "supplement",
        "Shows within-farm event ordering, overlap, and class identity.", "event_summary_3l"
    )

    farm_monthly_3l = (
        eligible_3l.loc[eligible_3l["farm"].eq(farm_3l)]
        .groupby(["month_utc", "final_label"], as_index=False).size()
        .rename(columns={"size": "eligible_rows"})
    )
    fig, ax = plt.subplots(figsize=(8.0, 4.2))
    for label_3l in LABELS_3L:
        subset_3l = farm_monthly_3l.loc[farm_monthly_3l["final_label"].eq(label_3l)]
        ax.plot(subset_3l["month_utc"], subset_3l["eligible_rows"], marker="o",
                color=COLORS_3L[label_3l], label=label_3l.title())
    ax.set_title(f"Farm {farm_3l}: monthly class support")
    finish_axes_3l(ax, "Month (UTC)", "Eligible observations")
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=9))
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
    fig.tight_layout()
    save_figure_3l(
        fig, f"F{farm_number_3l:02d}02", f"farm_{farm_3l}_monthly_labels",
        f"Farm {farm_3l} monthly class support", "supplement",
        "Shows temporal separation of normal and anomaly evidence within the farm.",
        "eligible_3l"
    )

    farm_assets_3l = asset_summary_3l.loc[
        asset_summary_3l["farm"].eq(farm_3l)
    ]
    horizontal_rank_3l(
        farm_assets_3l, "asset_key", "eligible_rows",
        f"F{farm_number_3l:02d}03", f"farm_{farm_3l}_asset_ranking",
        f"Farm {farm_3l}: eligible observations by asset",
        "Eligible observations", "supplement",
        "Shows within-farm concentration of evidence by asset.",
        "asset_summary_3l", top_n=50, color=COLORS_3L[farm_3l],
    )


# -----------------------------------------------------------------------------
# 6. One supplementary activity timeline per physical asset
# -----------------------------------------------------------------------------

asset_keys_3l = sorted(eligible_3l["asset_key"].astype(str).unique())

for asset_number_3l, asset_key_3l in enumerate(asset_keys_3l, start=1):
    asset_daily_3l = (
        eligible_3l.loc[eligible_3l["asset_key"].eq(asset_key_3l)]
        .groupby(["date_utc", "final_label"], as_index=False).size()
        .rename(columns={"size": "eligible_rows"})
    )
    farm_3l = str(
        eligible_3l.loc[eligible_3l["asset_key"].eq(asset_key_3l), "farm"].iloc[0]
    )
    fig, ax = plt.subplots(figsize=(8.0, 4.0))
    for label_3l in LABELS_3L:
        subset_3l = asset_daily_3l.loc[asset_daily_3l["final_label"].eq(label_3l)]
        if len(subset_3l):
            ax.plot(
                subset_3l["date_utc"], subset_3l["eligible_rows"],
                marker="o", markersize=3, linewidth=1.3,
                color=COLORS_3L[label_3l], label=label_3l.title(),
            )
    ax.set_title(f"{asset_key_3l}: daily eligible observations")
    finish_axes_3l(ax, "Date (UTC)", "Eligible observations")
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=9))
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
    fig.tight_layout()
    save_figure_3l(
        fig, f"A{asset_number_3l:03d}", f"{asset_key_3l}_daily_activity",
        f"{asset_key_3l} daily activity", "supplement",
        "Shows asset-specific temporal and class coverage used for grouped splitting.",
        "eligible_3l"
    )


# If a future reduced dataset has fewer assets, add event diagnostics until the
# suite still contains at least 55 logical figures.
event_fallback_number_3l = 0
for event_key_3l in event_summary_3l["event_key"].astype(str):
    if len(FIGURE_REGISTRY_3L) >= 55:
        break
    event_fallback_number_3l += 1
    event_rows_3l = eligible_3l.loc[eligible_3l["event_key"].eq(event_key_3l)]
    event_daily_3l = (
        event_rows_3l.groupby("date_utc", as_index=False).size()
        .rename(columns={"size": "eligible_rows"})
    )
    fig, ax = plt.subplots(figsize=(7.6, 3.8))
    ax.plot(event_daily_3l["date_utc"], event_daily_3l["eligible_rows"],
            marker="o", color="#56B4E9")
    ax.set_title(f"{event_key_3l}: daily eligible observations")
    finish_axes_3l(ax, "Date (UTC)", "Eligible observations", legend=False)
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=8))
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
    fig.tight_layout()
    save_figure_3l(
        fig, f"E{event_fallback_number_3l:03d}", f"{event_key_3l}_daily_activity",
        f"{event_key_3l} daily activity", "supplement",
        "Provides event-level temporal support when the asset count is small.",
        "eligible_3l"
    )


# -----------------------------------------------------------------------------
# 7. Registries, workbook, and non-destructive final checks
# -----------------------------------------------------------------------------

PAPER_FIGURE_REGISTRY_3L = pd.DataFrame(FIGURE_REGISTRY_3L)
PAPER_TABLE_REGISTRY_3L = pd.DataFrame(TABLE_REGISTRY_3L)

if PAPER_FIGURE_REGISTRY_3L["figure_id"].duplicated().any():
    duplicates_3l = PAPER_FIGURE_REGISTRY_3L.loc[
        PAPER_FIGURE_REGISTRY_3L["figure_id"].duplicated(keep=False),
        "figure_id",
    ].tolist()
    raise ValueError(f"Duplicate figure identifiers generated: {duplicates_3l}")

if len(PAPER_FIGURE_REGISTRY_3L) < 50:
    raise ValueError(
        f"Only {len(PAPER_FIGURE_REGISTRY_3L)} figures were generated; "
        "the Cell 3L minimum is 50."
    )

registry_csv_3l = OUTPUT_ROOT_3L / "paper_figure_registry.csv"
table_registry_csv_3l = OUTPUT_ROOT_3L / "paper_table_registry.csv"
PAPER_FIGURE_REGISTRY_3L.to_csv(registry_csv_3l, index=False)
PAPER_TABLE_REGISTRY_3L.to_csv(table_registry_csv_3l, index=False)

try:
    workbook_path_3l = OUTPUT_ROOT_3L / "paper_tables_3l.xlsx"

    def excel_safe_3l(frame):
        output_3l = frame.copy()
        for column_3l in output_3l.columns:
            if isinstance(output_3l[column_3l].dtype, pd.DatetimeTZDtype):
                output_3l[column_3l] = output_3l[column_3l].dt.tz_convert(None)
        return output_3l

    with pd.ExcelWriter(workbook_path_3l) as writer_3l:
        excel_safe_3l(eligible_label_summary_3l).to_excel(
            writer_3l, sheet_name="eligible_by_farm", index=False
        )
        excel_safe_3l(event_summary_3l).to_excel(
            writer_3l, sheet_name="event_summary", index=False
        )
        excel_safe_3l(asset_summary_3l).to_excel(
            writer_3l, sheet_name="asset_summary", index=False
        )
        excel_safe_3l(status_summary_3l).to_excel(
            writer_3l, sheet_name="manifest_status", index=False
        )
        excel_safe_3l(farm_temporal_summary_3l).to_excel(
            writer_3l, sheet_name="farm_temporal", index=False
        )
        PAPER_FIGURE_REGISTRY_3L.to_excel(
            writer_3l, sheet_name="figure_registry", index=False
        )
        PAPER_TABLE_REGISTRY_3L.to_excel(
            writer_3l, sheet_name="table_registry", index=False
        )
except Exception as exc:
    workbook_path_3l = None
    print(f"Excel workbook export skipped: {exc}")

if int(ROW_LABEL_MANIFEST["modeling_eligible"].sum()) != EXPECTED_ELIGIBLE_3L:
    raise ValueError("Cell 3L unexpectedly changed manifest eligibility.")
if ROW_LABEL_MANIFEST["split_assignment"].notna().any():
    raise ValueError("Cell 3L unexpectedly assigned a split.")

print("\nCell 3L completed successfully.")
print("Logical figures generated:", len(PAPER_FIGURE_REGISTRY_3L))
print("Core/overview figures:", int(PAPER_FIGURE_REGISTRY_3L["tier"].eq("core").sum()))
print("Supplementary figures:", int(PAPER_FIGURE_REGISTRY_3L["tier"].eq("supplement").sum()))
print("Physical assets with individual timelines:", len(asset_keys_3l))
print("Paper tables generated:", len(PAPER_TABLE_REGISTRY_3L))
print("Eligible rows visualized:", len(eligible_3l))
print("Figure registry:", registry_csv_3l)
print("Table registry:", table_registry_csv_3l)
if workbook_path_3l is not None:
    print("Combined table workbook:", workbook_path_3l)
print("Assigned train/validation/test rows:", int(ROW_LABEL_MANIFEST["split_assignment"].notna().sum()))
print("\nNo manifest row, label, eligibility decision, or split assignment was modified.")

print("\nRecommended manuscript candidates:")
print(
    PAPER_FIGURE_REGISTRY_3L.loc[
        PAPER_FIGURE_REGISTRY_3L["tier"].eq("core"),
        ["figure_id", "title", "claim_supported"],
    ].head(12).to_string(index=False)
)


Excel workbook export skipped: No module named 'openpyxl'

Cell 3L completed successfully.
Logical figures generated: 80
Core/overview figures: 35
Supplementary figures: 45
Physical assets with individual timelines: 36
Paper tables generated: 11
Eligible rows visualized: 213537
Figure registry: paper_visuals_3l\paper_figure_registry.csv
Table registry: paper_visuals_3l\paper_table_registry.csv
Assigned train/validation/test rows: 0

No manifest row, label, eligibility decision, or split assignment was modified.

Recommended manuscript candidates:
figure_id                                      title                                                        claim_supported
     C001   Post-deduplication eligible observations                Shows class support after verified duplicate exclusion.
     C002              Within-farm class composition      Shows farm-specific class imbalance without conflating farm size.
     C003     Eligible event files by farm and class                       

In [35]:
%pip install -q openpyxl

Note: you may need to restart the kernel to use updated packages.


In [36]:
from pathlib import Path
import re
import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter


def safe_sheet_name_3l(table_id, name, used_names):
    candidate = re.sub(
        r"[:\\/?*\[\]]",
        "_",
        f"{table_id}_{name}",
    )[:31]

    base = candidate
    suffix = 1

    while candidate in used_names:
        suffix += 1
        candidate = f"{base[:27]}_{suffix}"

    used_names.add(candidate)
    return candidate


workbook_path_3l = (
    Path(OUTPUT_ROOT_3L) / "paper_tables_3l.xlsx"
)

used_sheet_names_3l = set()

with pd.ExcelWriter(
    workbook_path_3l,
    engine="openpyxl",
) as writer_3l:

    # Export all 11 registered paper tables.
    for record_3l in PAPER_TABLE_REGISTRY_3L.itertuples(
        index=False
    ):
        table_frame_3l = pd.read_csv(record_3l.csv_path)

        sheet_name_3l = safe_sheet_name_3l(
            str(record_3l.table_id),
            str(record_3l.name),
            used_sheet_names_3l,
        )

        table_frame_3l.to_excel(
            writer_3l,
            sheet_name=sheet_name_3l,
            index=False,
        )

    # Add the figure and table registries.
    PAPER_FIGURE_REGISTRY_3L.to_excel(
        writer_3l,
        sheet_name="figure_registry",
        index=False,
    )

    PAPER_TABLE_REGISTRY_3L.to_excel(
        writer_3l,
        sheet_name="table_registry",
        index=False,
    )

    # Apply readable workbook formatting.
    for worksheet_3l in writer_3l.book.worksheets:
        worksheet_3l.freeze_panes = "A2"
        worksheet_3l.auto_filter.ref = (
            worksheet_3l.dimensions
        )

        for cell_3l in worksheet_3l[1]:
            cell_3l.font = Font(
                bold=True,
                color="FFFFFF",
            )
            cell_3l.fill = PatternFill(
                fill_type="solid",
                fgColor="1F4E78",
            )
            cell_3l.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True,
            )

        for column_number_3l, column_cells_3l in enumerate(
            worksheet_3l.columns,
            start=1,
        ):
            maximum_length_3l = max(
                len(str(cell_3l.value))
                if cell_3l.value is not None
                else 0
                for cell_3l in column_cells_3l
            )

            worksheet_3l.column_dimensions[
                get_column_letter(column_number_3l)
            ].width = min(
                max(maximum_length_3l + 2, 10),
                45,
            )


if not workbook_path_3l.exists():
    raise FileNotFoundError(
        "The Excel workbook was not created."
    )

print("Excel workbook created successfully:")
print(workbook_path_3l)
print("Registered paper tables exported:", 11)
print("Total workbook sheets:", 13)

Excel workbook created successfully:
paper_visuals_3l\paper_tables_3l.xlsx
Registered paper tables exported: 11
Total workbook sheets: 13


In [37]:
"""Cell 3M — deterministic, leakage-safe asset-grouped data split.

Run inside the notebook namespace immediately after Cell 3L:

    %run -i cell_3m_asset_grouped_split.py

The cell assigns every modeling-eligible row to train, validation, or test by
physical ``asset_key``. It balances farm, label, row, event, and asset support;
exports split diagnostics; and changes no pre-existing manifest column except
``split_assignment``.
"""

from __future__ import annotations

import math
import re
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# 1. Validate the audited post-3K/post-3L checkpoint
# -----------------------------------------------------------------------------

if "ROW_LABEL_MANIFEST" not in globals():
    raise NameError("Run Cells 3I–3L before Cell 3M.")

REQUIRED_COLUMNS_3M = {
    "farm",
    "asset_id",
    "asset_key",
    "event_key",
    "source_row_index",
    "timestamp_utc",
    "final_label",
    "modeling_eligible",
    "split_assignment",
}

missing_columns_3m = REQUIRED_COLUMNS_3M - set(ROW_LABEL_MANIFEST.columns)
if missing_columns_3m:
    raise ValueError(
        "ROW_LABEL_MANIFEST is missing columns required by Cell 3M: "
        f"{sorted(missing_columns_3m)}"
    )

EXPECTED_ELIGIBLE_ROWS_3M = 213_537
EXPECTED_PHYSICAL_ASSETS_3M = 36
SPLIT_ORDER_3M = ["train", "validation", "test"]
TARGET_FRACTIONS_3M = np.array([0.70, 0.15, 0.15], dtype=float)
RANDOM_SEED_3M = 20260810
RANDOM_CANDIDATES_3M = 30_000

modeling_eligible_mask_3m = (
    ROW_LABEL_MANIFEST["modeling_eligible"].fillna(False).astype(bool)
)
eligible_rows_before_3m = int(modeling_eligible_mask_3m.sum())

if eligible_rows_before_3m != EXPECTED_ELIGIBLE_ROWS_3M:
    raise ValueError(
        "Cell 3M expects the verified post-retention eligible count of "
        f"213,537, but found {eligible_rows_before_3m:,}."
    )

if ROW_LABEL_MANIFEST["split_assignment"].notna().any():
    raise ValueError(
        "A split assignment already exists. Restart from Cell 3K before "
        "running Cell 3M again."
    )

manifest_length_before_3m = len(ROW_LABEL_MANIFEST)
manifest_index_before_3m = ROW_LABEL_MANIFEST.index.copy()
unchanged_columns_3m = [
    column
    for column in ROW_LABEL_MANIFEST.columns
    if column != "split_assignment"
]
manifest_integrity_hash_before_3m = pd.util.hash_pandas_object(
    ROW_LABEL_MANIFEST.loc[:, unchanged_columns_3m],
    index=True,
).to_numpy(copy=True)


# -----------------------------------------------------------------------------
# 2. Normalize an immutable eligible-row working frame
# -----------------------------------------------------------------------------

eligible_3m = ROW_LABEL_MANIFEST.loc[
    modeling_eligible_mask_3m,
    [
        "farm",
        "asset_id",
        "asset_key",
        "event_key",
        "source_row_index",
        "timestamp_utc",
        "final_label",
    ],
].copy()

eligible_3m["farm"] = eligible_3m["farm"].astype("string").str.strip()
eligible_3m["asset_id"] = eligible_3m["asset_id"].astype("string").str.strip()
eligible_3m["asset_key"] = eligible_3m["asset_key"].astype("string").str.strip()
eligible_3m["event_key"] = eligible_3m["event_key"].astype("string").str.strip()
eligible_3m["final_label"] = (
    eligible_3m["final_label"].astype("string").str.strip().str.lower()
)
eligible_3m["timestamp_utc"] = pd.to_datetime(
    eligible_3m["timestamp_utc"],
    errors="coerce",
    utc=True,
)

if len(eligible_3m) != EXPECTED_ELIGIBLE_ROWS_3M:
    raise ValueError("Eligible-row extraction was not row-conserving.")

required_nonmissing_3m = [
    "farm",
    "asset_id",
    "asset_key",
    "event_key",
    "timestamp_utc",
    "final_label",
]
if eligible_3m[required_nonmissing_3m].isna().any().any():
    bad_columns_3m = eligible_3m[required_nonmissing_3m].columns[
        eligible_3m[required_nonmissing_3m].isna().any()
    ].tolist()
    raise ValueError(
        "Eligible rows have missing split-critical values in: "
        f"{bad_columns_3m}"
    )

for text_column_3m in ["farm", "asset_id", "asset_key", "event_key"]:
    if eligible_3m[text_column_3m].eq("").any():
        raise ValueError(
            f"Eligible rows contain a blank {text_column_3m}."
        )

FARMS_3M = sorted(eligible_3m["farm"].astype(str).unique().tolist())
LABELS_3M = ["normal", "anomaly"]

if FARMS_3M != ["A", "B", "C"]:
    raise ValueError(f"Expected farms A, B, and C; found {FARMS_3M}.")

observed_labels_3m = sorted(
    eligible_3m["final_label"].astype(str).unique().tolist()
)
if observed_labels_3m != sorted(LABELS_3M):
    raise ValueError(
        "Expected exactly the normalized labels normal and anomaly; found "
        f"{observed_labels_3m}."
    )

if eligible_3m[["event_key", "source_row_index"]].duplicated().any():
    raise ValueError("Eligible manifest source keys are not unique.")

asset_identity_check_3m = eligible_3m.groupby(
    "asset_key",
    dropna=False,
).agg(
    farms=("farm", "nunique"),
    asset_ids=("asset_id", "nunique"),
)
if not (
    asset_identity_check_3m["farms"].eq(1).all()
    and asset_identity_check_3m["asset_ids"].eq(1).all()
):
    raise ValueError("An asset_key maps to multiple farm/asset identities.")

event_identity_check_3m = eligible_3m.groupby(
    "event_key",
    dropna=False,
).agg(
    assets=("asset_key", "nunique"),
    labels=("final_label", "nunique"),
)
if not (
    event_identity_check_3m["assets"].eq(1).all()
    and event_identity_check_3m["labels"].eq(1).all()
):
    raise ValueError(
        "An eligible event spans multiple assets or normalized labels, so "
        "an asset-only split would not isolate event sources."
    )

physical_asset_count_3m = int(eligible_3m["asset_key"].nunique())
if physical_asset_count_3m != EXPECTED_PHYSICAL_ASSETS_3M:
    raise ValueError(
        "Cell 3L reported 36 eligible physical assets, but Cell 3M found "
        f"{physical_asset_count_3m}."
    )


# -----------------------------------------------------------------------------
# 3. Build one balance profile per indivisible physical asset
# -----------------------------------------------------------------------------

asset_profile_3m = (
    eligible_3m.groupby("asset_key", as_index=False, dropna=False)
    .agg(
        farm=("farm", "first"),
        asset_id=("asset_id", "first"),
        eligible_rows=("event_key", "size"),
        eligible_events=("event_key", "nunique"),
        first_timestamp_utc=("timestamp_utc", "min"),
        last_timestamp_utc=("timestamp_utc", "max"),
    )
    .sort_values("asset_key")
    .reset_index(drop=True)
)

balance_feature_values_3m: list[np.ndarray] = []
balance_feature_names_3m: list[str] = []
balance_feature_types_3m: list[str] = []
balance_feature_weights_3m: list[float] = []
coverage_feature_indices_3m: list[int] = []


def add_balance_feature_3m(name, feature_type, values, weight, coverage=False):
    values = np.asarray(values, dtype=float)
    if len(values) != len(asset_profile_3m):
        raise ValueError(f"Balance feature {name} has the wrong length.")
    balance_feature_names_3m.append(str(name))
    balance_feature_types_3m.append(str(feature_type))
    balance_feature_values_3m.append(values)
    balance_feature_weights_3m.append(float(weight))
    if coverage:
        coverage_feature_indices_3m.append(len(balance_feature_names_3m) - 1)


for farm_3m in FARMS_3M:
    for label_3m in LABELS_3M:
        subset_3m = eligible_3m.loc[
            eligible_3m["farm"].eq(farm_3m)
            & eligible_3m["final_label"].eq(label_3m)
        ]
        row_counts_3m = (
            subset_3m.groupby("asset_key").size().reindex(
                asset_profile_3m["asset_key"], fill_value=0
            )
        )
        event_counts_3m = (
            subset_3m.groupby("asset_key")["event_key"].nunique().reindex(
                asset_profile_3m["asset_key"], fill_value=0
            )
        )

        row_column_3m = f"{label_3m}_rows"
        event_column_3m = f"{label_3m}_events"
        asset_profile_3m.loc[
            asset_profile_3m["farm"].eq(farm_3m), row_column_3m
        ] = row_counts_3m.loc[
            asset_profile_3m.loc[
                asset_profile_3m["farm"].eq(farm_3m), "asset_key"
            ]
        ].to_numpy()
        asset_profile_3m.loc[
            asset_profile_3m["farm"].eq(farm_3m), event_column_3m
        ] = event_counts_3m.loc[
            asset_profile_3m.loc[
                asset_profile_3m["farm"].eq(farm_3m), "asset_key"
            ]
        ].to_numpy()

        add_balance_feature_3m(
            f"farm_label_rows::{farm_3m}::{label_3m}",
            "eligible_rows",
            row_counts_3m.to_numpy(),
            weight=5.0,
        )
        add_balance_feature_3m(
            f"farm_label_events::{farm_3m}::{label_3m}",
            "eligible_events",
            event_counts_3m.to_numpy(),
            weight=4.0,
        )
        support_values_3m = row_counts_3m.gt(0).astype(int).to_numpy()
        add_balance_feature_3m(
            f"farm_label_assets::{farm_3m}::{label_3m}",
            "supporting_assets",
            support_values_3m,
            weight=2.5,
            coverage=int(support_values_3m.sum()) >= len(SPLIT_ORDER_3M),
        )

for farm_3m in FARMS_3M:
    farm_support_3m = asset_profile_3m["farm"].eq(farm_3m).astype(int).to_numpy()
    add_balance_feature_3m(
        f"farm_assets::{farm_3m}",
        "farm_assets",
        farm_support_3m,
        weight=3.0,
        coverage=int(farm_support_3m.sum()) >= len(SPLIT_ORDER_3M),
    )

for label_3m in LABELS_3M:
    label_support_3m = np.zeros(len(asset_profile_3m), dtype=int)
    for farm_3m in FARMS_3M:
        subset_assets_3m = set(
            eligible_3m.loc[
                eligible_3m["farm"].eq(farm_3m)
                & eligible_3m["final_label"].eq(label_3m),
                "asset_key",
            ].astype(str)
        )
        label_support_3m += asset_profile_3m["asset_key"].astype(str).isin(
            subset_assets_3m
        ).astype(int).to_numpy()
    label_support_3m = (label_support_3m > 0).astype(int)
    add_balance_feature_3m(
        f"label_assets::{label_3m}",
        "label_assets",
        label_support_3m,
        weight=3.0,
        coverage=int(label_support_3m.sum()) >= len(SPLIT_ORDER_3M),
    )

balance_matrix_3m = np.column_stack(balance_feature_values_3m).astype(float)
balance_totals_3m = balance_matrix_3m.sum(axis=0)
balance_weights_3m = np.asarray(balance_feature_weights_3m, dtype=float)

if np.any(balance_totals_3m <= 0):
    empty_features_3m = [
        balance_feature_names_3m[index]
        for index in np.flatnonzero(balance_totals_3m <= 0)
    ]
    raise ValueError(f"Empty split-balance features: {empty_features_3m}")


# -----------------------------------------------------------------------------
# 4. Determine exact asset counts and optimize the deterministic assignment
# -----------------------------------------------------------------------------


def target_asset_counts_3m(number_of_assets):
    raw_counts_3m = TARGET_FRACTIONS_3M * int(number_of_assets)
    counts_3m = np.floor(raw_counts_3m).astype(int)
    remaining_3m = int(number_of_assets - counts_3m.sum())
    fractional_3m = raw_counts_3m - counts_3m
    # A tie is awarded to test before validation so the untouched holdout gets
    # the slightly larger of the two small asset allocations.
    tie_priority_3m = {0: 0, 1: 1, 2: 2}
    award_order_3m = sorted(
        range(len(SPLIT_ORDER_3M)),
        key=lambda index: (fractional_3m[index], tie_priority_3m[index]),
        reverse=True,
    )
    for index_3m in award_order_3m[:remaining_3m]:
        counts_3m[index_3m] += 1
    return counts_3m


target_asset_counts_array_3m = target_asset_counts_3m(physical_asset_count_3m)
expected_asset_counts_array_3m = np.array([25, 5, 6], dtype=int)
if not np.array_equal(
    target_asset_counts_array_3m,
    expected_asset_counts_array_3m,
):
    raise ValueError(
        "The 70/15/15 target should allocate 25/5/6 of the 36 assets, "
        f"but produced {target_asset_counts_array_3m.tolist()}."
    )

target_feature_matrix_3m = (
    TARGET_FRACTIONS_3M[:, None] * balance_totals_3m[None, :]
)


def evaluate_assignment_3m(assignment_codes_3m, return_actual=False):
    actual_3m = np.vstack(
        [
            balance_matrix_3m[assignment_codes_3m == split_code_3m].sum(axis=0)
            for split_code_3m in range(len(SPLIT_ORDER_3M))
        ]
    )
    normalized_error_3m = (
        actual_3m - target_feature_matrix_3m
    ) / np.maximum(balance_totals_3m[None, :], 1.0)
    weighted_mse_3m = float(
        np.sum(balance_weights_3m[None, :] * normalized_error_3m**2)
    )
    maximum_deviation_3m = float(np.max(np.abs(normalized_error_3m)))
    missing_required_coverage_3m = int(
        (actual_3m[:, coverage_feature_indices_3m] <= 0).sum()
    )
    score_3m = (
        weighted_mse_3m
        + 0.25 * maximum_deviation_3m**2
        + 1_000.0 * missing_required_coverage_3m
    )
    if return_actual:
        return score_3m, actual_3m, missing_required_coverage_3m
    return score_3m


assignment_template_3m = np.concatenate(
    [
        np.full(count_3m, split_code_3m, dtype=np.int8)
        for split_code_3m, count_3m in enumerate(target_asset_counts_array_3m)
    ]
)

rng_3m = np.random.default_rng(RANDOM_SEED_3M)
best_assignment_codes_3m = None
best_assignment_score_3m = math.inf

for _ in range(RANDOM_CANDIDATES_3M):
    candidate_codes_3m = rng_3m.permutation(assignment_template_3m)
    candidate_score_3m = evaluate_assignment_3m(candidate_codes_3m)
    if candidate_score_3m < best_assignment_score_3m:
        best_assignment_score_3m = candidate_score_3m
        best_assignment_codes_3m = candidate_codes_3m.copy()

if best_assignment_codes_3m is None:
    raise RuntimeError("The deterministic asset-assignment search failed.")

# Pairwise local improvement preserves the exact 25/5/6 asset counts.
local_search_iterations_3m = 0
while True:
    best_swap_3m = None
    best_swap_score_3m = best_assignment_score_3m
    for left_3m in range(len(best_assignment_codes_3m) - 1):
        for right_3m in range(left_3m + 1, len(best_assignment_codes_3m)):
            if (
                best_assignment_codes_3m[left_3m]
                == best_assignment_codes_3m[right_3m]
            ):
                continue
            trial_codes_3m = best_assignment_codes_3m.copy()
            trial_codes_3m[left_3m], trial_codes_3m[right_3m] = (
                trial_codes_3m[right_3m],
                trial_codes_3m[left_3m],
            )
            trial_score_3m = evaluate_assignment_3m(trial_codes_3m)
            if trial_score_3m < best_swap_score_3m - 1e-15:
                best_swap_score_3m = trial_score_3m
                best_swap_3m = (left_3m, right_3m)
    if best_swap_3m is None:
        break
    left_3m, right_3m = best_swap_3m
    best_assignment_codes_3m[left_3m], best_assignment_codes_3m[right_3m] = (
        best_assignment_codes_3m[right_3m],
        best_assignment_codes_3m[left_3m],
    )
    best_assignment_score_3m = best_swap_score_3m
    local_search_iterations_3m += 1
    if local_search_iterations_3m > 200:
        raise RuntimeError("The split local search did not converge.")

(
    best_assignment_score_3m,
    best_feature_actual_3m,
    missing_required_coverage_3m,
) = evaluate_assignment_3m(best_assignment_codes_3m, return_actual=True)

if missing_required_coverage_3m:
    raise ValueError(
        "No feasible split satisfying required farm/label coverage was found. "
        "ROW_LABEL_MANIFEST was not modified."
    )

observed_asset_counts_3m = np.bincount(
    best_assignment_codes_3m,
    minlength=len(SPLIT_ORDER_3M),
)
if not np.array_equal(
    observed_asset_counts_3m,
    target_asset_counts_array_3m,
):
    raise ValueError("Optimized split does not preserve target asset counts.")

asset_profile_3m["split_assignment"] = [
    SPLIT_ORDER_3M[split_code_3m]
    for split_code_3m in best_assignment_codes_3m
]
asset_to_split_3m = asset_profile_3m.set_index("asset_key")[
    "split_assignment"
].to_dict()

eligible_3m["split_assignment"] = eligible_3m["asset_key"].map(
    asset_to_split_3m
)
if eligible_3m["split_assignment"].isna().any():
    raise ValueError("An eligible asset was not assigned to a split.")


# -----------------------------------------------------------------------------
# 5. Pre-mutation leakage and coverage gates
# -----------------------------------------------------------------------------

asset_split_counts_pre_3m = eligible_3m.groupby("asset_key")[
    "split_assignment"
].nunique()
event_split_counts_pre_3m = eligible_3m.groupby("event_key")[
    "split_assignment"
].nunique()

if asset_split_counts_pre_3m.gt(1).any():
    raise ValueError("At least one physical asset crosses proposed splits.")
if event_split_counts_pre_3m.gt(1).any():
    raise ValueError("At least one event source crosses proposed splits.")

pre_split_label_coverage_3m = eligible_3m.groupby("split_assignment")[
    "final_label"
].nunique().reindex(SPLIT_ORDER_3M, fill_value=0)
if not pre_split_label_coverage_3m.eq(len(LABELS_3M)).all():
    raise ValueError("Every split must contain both normal and anomaly rows.")

pre_split_farm_coverage_3m = eligible_3m.groupby("split_assignment")[
    "farm"
].nunique().reindex(SPLIT_ORDER_3M, fill_value=0)
if not pre_split_farm_coverage_3m.eq(len(FARMS_3M)).all():
    raise ValueError("Every split must contain all three farms.")

for farm_3m in FARMS_3M:
    for label_3m in LABELS_3M:
        category_frame_3m = eligible_3m.loc[
            eligible_3m["farm"].eq(farm_3m)
            & eligible_3m["final_label"].eq(label_3m)
        ]
        supporting_assets_3m = int(category_frame_3m["asset_key"].nunique())
        if supporting_assets_3m >= len(SPLIT_ORDER_3M):
            covered_splits_3m = int(
                category_frame_3m["split_assignment"].nunique()
            )
            if covered_splits_3m != len(SPLIT_ORDER_3M):
                raise ValueError(
                    f"Farm {farm_3m}, label {label_3m} has enough asset "
                    "support for all splits but is not represented in all."
                )


# -----------------------------------------------------------------------------
# 6. Apply the split_assignment mutation and verify its exact boundary
# -----------------------------------------------------------------------------

manifest_asset_keys_3m = (
    ROW_LABEL_MANIFEST["asset_key"].astype("string").str.strip()
)
manifest_split_values_3m = manifest_asset_keys_3m.map(asset_to_split_3m)

if manifest_split_values_3m.loc[modeling_eligible_mask_3m].isna().any():
    raise ValueError("Not all eligible manifest rows resolve to an asset split.")

# Convert the previously empty split column to a nullable string column, then
# write assignments only on modeling-eligible rows.
ROW_LABEL_MANIFEST["split_assignment"] = ROW_LABEL_MANIFEST[
    "split_assignment"
].astype("string")
ROW_LABEL_MANIFEST.loc[
    modeling_eligible_mask_3m,
    "split_assignment",
] = manifest_split_values_3m.loc[modeling_eligible_mask_3m].astype("string")

if len(ROW_LABEL_MANIFEST) != manifest_length_before_3m:
    raise ValueError("Cell 3M changed the manifest row count.")
if not ROW_LABEL_MANIFEST.index.equals(manifest_index_before_3m):
    raise ValueError("Cell 3M changed the manifest index.")

manifest_integrity_hash_after_3m = pd.util.hash_pandas_object(
    ROW_LABEL_MANIFEST.loc[:, unchanged_columns_3m],
    index=True,
).to_numpy()
if not np.array_equal(
    manifest_integrity_hash_before_3m,
    manifest_integrity_hash_after_3m,
):
    raise ValueError(
        "A pre-existing manifest value outside split_assignment changed."
    )

if int(ROW_LABEL_MANIFEST["modeling_eligible"].sum()) != (
    EXPECTED_ELIGIBLE_ROWS_3M
):
    raise ValueError("Cell 3M changed manifest eligibility.")

eligible_assignments_3m = ROW_LABEL_MANIFEST.loc[
    modeling_eligible_mask_3m,
    "split_assignment",
]
ineligible_assignments_3m = ROW_LABEL_MANIFEST.loc[
    ~modeling_eligible_mask_3m,
    "split_assignment",
]

if eligible_assignments_3m.isna().any():
    raise ValueError("At least one modeling-eligible row is unassigned.")
if not eligible_assignments_3m.isin(SPLIT_ORDER_3M).all():
    raise ValueError("An eligible row has an invalid split name.")
if ineligible_assignments_3m.notna().any():
    raise ValueError("Cell 3M assigned a modeling-ineligible row.")

assigned_manifest_3m = ROW_LABEL_MANIFEST.loc[
    modeling_eligible_mask_3m,
    ["asset_key", "event_key", "split_assignment"],
].copy()
assets_crossing_splits_3m = int(
    assigned_manifest_3m.groupby("asset_key")["split_assignment"]
    .nunique()
    .gt(1)
    .sum()
)
events_crossing_splits_3m = int(
    assigned_manifest_3m.groupby("event_key")["split_assignment"]
    .nunique()
    .gt(1)
    .sum()
)
if assets_crossing_splits_3m or events_crossing_splits_3m:
    raise ValueError("Post-mutation asset/event leakage was detected.")


# -----------------------------------------------------------------------------
# 7. Create paper-ready split tables and audit objects
# -----------------------------------------------------------------------------

split_category_3m = pd.CategoricalDtype(SPLIT_ORDER_3M, ordered=True)
eligible_3m["split_assignment"] = eligible_3m["split_assignment"].astype(
    split_category_3m
)
asset_profile_3m["split_assignment"] = asset_profile_3m[
    "split_assignment"
].astype(split_category_3m)

for label_3m in LABELS_3M:
    row_column_3m = f"{label_3m}_rows"
    event_column_3m = f"{label_3m}_events"
    if row_column_3m not in asset_profile_3m:
        asset_profile_3m[row_column_3m] = 0
    if event_column_3m not in asset_profile_3m:
        asset_profile_3m[event_column_3m] = 0
    asset_profile_3m[row_column_3m] = (
        asset_profile_3m[row_column_3m].fillna(0).astype(int)
    )
    asset_profile_3m[event_column_3m] = (
        asset_profile_3m[event_column_3m].fillna(0).astype(int)
    )

ASSET_SPLIT_ASSIGNMENT_3M = asset_profile_3m.loc[
    :,
    [
        "farm",
        "asset_id",
        "asset_key",
        "split_assignment",
        "eligible_rows",
        "eligible_events",
        "normal_rows",
        "anomaly_rows",
        "normal_events",
        "anomaly_events",
        "first_timestamp_utc",
        "last_timestamp_utc",
    ],
].sort_values(["split_assignment", "farm", "asset_key"]).reset_index(drop=True)
ASSET_SPLIT_ASSIGNMENT_3M["split_assignment"] = (
    ASSET_SPLIT_ASSIGNMENT_3M["split_assignment"].astype("string")
)

split_summary_records_3m = []
for split_name_3m, target_fraction_3m in zip(
    SPLIT_ORDER_3M,
    TARGET_FRACTIONS_3M,
):
    subset_3m = eligible_3m.loc[
        eligible_3m["split_assignment"].eq(split_name_3m)
    ]
    split_summary_records_3m.append(
        {
            "split_assignment": split_name_3m,
            "target_fraction": target_fraction_3m,
            "eligible_rows": len(subset_3m),
            "eligible_row_fraction": len(subset_3m) / len(eligible_3m),
            "eligible_events": subset_3m["event_key"].nunique(),
            "event_fraction": (
                subset_3m["event_key"].nunique()
                / eligible_3m["event_key"].nunique()
            ),
            "physical_assets": subset_3m["asset_key"].nunique(),
            "asset_fraction": (
                subset_3m["asset_key"].nunique()
                / eligible_3m["asset_key"].nunique()
            ),
            "farms": subset_3m["farm"].nunique(),
            "normal_rows": int(subset_3m["final_label"].eq("normal").sum()),
            "anomaly_rows": int(subset_3m["final_label"].eq("anomaly").sum()),
        }
    )

SPLIT_SUMMARY_3M = pd.DataFrame(split_summary_records_3m)

farm_label_index_3m = pd.MultiIndex.from_product(
    [SPLIT_ORDER_3M, FARMS_3M, LABELS_3M],
    names=["split_assignment", "farm", "final_label"],
)
SPLIT_FARM_LABEL_SUMMARY_3M = (
    eligible_3m.groupby(
        ["split_assignment", "farm", "final_label"],
        observed=True,
    )
    .agg(
        eligible_rows=("event_key", "size"),
        eligible_events=("event_key", "nunique"),
        physical_assets=("asset_key", "nunique"),
    )
    .reindex(farm_label_index_3m, fill_value=0)
    .reset_index()
)
farm_label_totals_3m = SPLIT_FARM_LABEL_SUMMARY_3M.groupby(
    ["farm", "final_label"]
)[["eligible_rows", "eligible_events", "physical_assets"]].transform("sum")
for metric_3m in ["eligible_rows", "eligible_events", "physical_assets"]:
    SPLIT_FARM_LABEL_SUMMARY_3M[f"{metric_3m}_fraction"] = np.divide(
        SPLIT_FARM_LABEL_SUMMARY_3M[metric_3m],
        farm_label_totals_3m[metric_3m],
        out=np.zeros(len(SPLIT_FARM_LABEL_SUMMARY_3M), dtype=float),
        where=farm_label_totals_3m[metric_3m].to_numpy() > 0,
    )

class_index_3m = pd.MultiIndex.from_product(
    [SPLIT_ORDER_3M, LABELS_3M],
    names=["split_assignment", "final_label"],
)
SPLIT_CLASS_BALANCE_3M = (
    eligible_3m.groupby(
        ["split_assignment", "final_label"],
        observed=True,
    )
    .agg(
        eligible_rows=("event_key", "size"),
        eligible_events=("event_key", "nunique"),
        physical_assets=("asset_key", "nunique"),
    )
    .reindex(class_index_3m, fill_value=0)
    .reset_index()
)
split_row_totals_3m = SPLIT_CLASS_BALANCE_3M.groupby("split_assignment")[
    "eligible_rows"
].transform("sum")
SPLIT_CLASS_BALANCE_3M["within_split_row_fraction"] = np.divide(
    SPLIT_CLASS_BALANCE_3M["eligible_rows"],
    split_row_totals_3m,
    out=np.zeros(len(SPLIT_CLASS_BALANCE_3M), dtype=float),
    where=split_row_totals_3m.to_numpy() > 0,
)

optimization_records_3m = []
for split_code_3m, split_name_3m in enumerate(SPLIT_ORDER_3M):
    for feature_index_3m, feature_name_3m in enumerate(
        balance_feature_names_3m
    ):
        total_3m = balance_totals_3m[feature_index_3m]
        actual_3m = best_feature_actual_3m[split_code_3m, feature_index_3m]
        actual_fraction_3m = actual_3m / total_3m
        optimization_records_3m.append(
            {
                "split_assignment": split_name_3m,
                "balance_feature": feature_name_3m,
                "feature_type": balance_feature_types_3m[feature_index_3m],
                "feature_weight": balance_weights_3m[feature_index_3m],
                "target_fraction": TARGET_FRACTIONS_3M[split_code_3m],
                "actual_count": actual_3m,
                "total_count": total_3m,
                "actual_fraction": actual_fraction_3m,
                "absolute_fraction_error": abs(
                    actual_fraction_3m - TARGET_FRACTIONS_3M[split_code_3m]
                ),
                "coverage_required": (
                    feature_index_3m in coverage_feature_indices_3m
                ),
            }
        )

SPLIT_OPTIMIZATION_DIAGNOSTICS_3M = pd.DataFrame(
    optimization_records_3m
)

leakage_audit_records_3m = [
    {
        "check": "eligible_row_conservation",
        "observed": int(eligible_assignments_3m.notna().sum()),
        "required": EXPECTED_ELIGIBLE_ROWS_3M,
        "passed": int(eligible_assignments_3m.notna().sum())
        == EXPECTED_ELIGIBLE_ROWS_3M,
    },
    {
        "check": "unassigned_eligible_rows",
        "observed": int(eligible_assignments_3m.isna().sum()),
        "required": 0,
        "passed": int(eligible_assignments_3m.isna().sum()) == 0,
    },
    {
        "check": "assigned_ineligible_rows",
        "observed": int(ineligible_assignments_3m.notna().sum()),
        "required": 0,
        "passed": int(ineligible_assignments_3m.notna().sum()) == 0,
    },
    {
        "check": "assets_crossing_splits",
        "observed": assets_crossing_splits_3m,
        "required": 0,
        "passed": assets_crossing_splits_3m == 0,
    },
    {
        "check": "events_crossing_splits",
        "observed": events_crossing_splits_3m,
        "required": 0,
        "passed": events_crossing_splits_3m == 0,
    },
    {
        "check": "eligible_labels_per_split_minimum",
        "observed": int(pre_split_label_coverage_3m.min()),
        "required": len(LABELS_3M),
        "passed": int(pre_split_label_coverage_3m.min()) == len(LABELS_3M),
    },
    {
        "check": "farms_per_split_minimum",
        "observed": int(pre_split_farm_coverage_3m.min()),
        "required": len(FARMS_3M),
        "passed": int(pre_split_farm_coverage_3m.min()) == len(FARMS_3M),
    },
    {
        "check": "manifest_rows_deleted",
        "observed": manifest_length_before_3m - len(ROW_LABEL_MANIFEST),
        "required": 0,
        "passed": manifest_length_before_3m == len(ROW_LABEL_MANIFEST),
    },
    {
        "check": "non_split_manifest_values_changed",
        "observed": int(
            np.count_nonzero(
                manifest_integrity_hash_before_3m
                != manifest_integrity_hash_after_3m
            )
        ),
        "required": 0,
        "passed": np.array_equal(
            manifest_integrity_hash_before_3m,
            manifest_integrity_hash_after_3m,
        ),
    },
]
SPLIT_LEAKAGE_AUDIT_3M = pd.DataFrame(leakage_audit_records_3m)

if not SPLIT_LEAKAGE_AUDIT_3M["passed"].all():
    failed_checks_3m = SPLIT_LEAKAGE_AUDIT_3M.loc[
        ~SPLIT_LEAKAGE_AUDIT_3M["passed"], "check"
    ].tolist()
    raise ValueError(f"Split audit failed: {failed_checks_3m}")


# -----------------------------------------------------------------------------
# 8. Export tables, workbook, and publication-quality split figures
# -----------------------------------------------------------------------------

OUTPUT_ROOT_3M = Path("paper_visuals_3l") / "split_diagnostics_3m"
TABLE_DIR_3M = OUTPUT_ROOT_3M / "tables"
PNG_DIR_3M = OUTPUT_ROOT_3M / "figures_png"
PDF_DIR_3M = OUTPUT_ROOT_3M / "figures_pdf"
for directory_3m in [TABLE_DIR_3M, PNG_DIR_3M, PDF_DIR_3M]:
    directory_3m.mkdir(parents=True, exist_ok=True)


def slug_3m(value):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_").lower()


table_exports_3m = [
    (
        "T3M01",
        "asset_split_assignment",
        ASSET_SPLIT_ASSIGNMENT_3M,
        "Canonical asset-to-split assignment and asset-level support.",
    ),
    (
        "T3M02",
        "split_summary",
        SPLIT_SUMMARY_3M,
        "Overall row, event, and asset support by split.",
    ),
    (
        "T3M03",
        "split_farm_label_summary",
        SPLIT_FARM_LABEL_SUMMARY_3M,
        "Farm- and label-stratified split support.",
    ),
    (
        "T3M04",
        "split_class_balance",
        SPLIT_CLASS_BALANCE_3M,
        "Within-split class composition and independent support.",
    ),
    (
        "T3M05",
        "split_optimization_diagnostics",
        SPLIT_OPTIMIZATION_DIAGNOSTICS_3M,
        "Target-versus-actual balance for every optimization feature.",
    ),
    (
        "T3M06",
        "split_leakage_audit",
        SPLIT_LEAKAGE_AUDIT_3M,
        "Hard row-conservation and leakage checks.",
    ),
]

table_registry_records_3m = []
for table_id_3m, table_name_3m, table_frame_3m, purpose_3m in table_exports_3m:
    csv_path_3m = TABLE_DIR_3M / (
        f"{table_id_3m}_{slug_3m(table_name_3m)}.csv"
    )
    tex_path_3m = TABLE_DIR_3M / (
        f"{table_id_3m}_{slug_3m(table_name_3m)}.tex"
    )
    table_frame_3m.to_csv(csv_path_3m, index=False)
    try:
        table_frame_3m.to_latex(tex_path_3m, index=False, escape=True)
        latex_path_value_3m = str(tex_path_3m)
    except Exception as latex_error_3m:
        latex_path_value_3m = pd.NA
        print(
            f"LaTeX export skipped for {table_id_3m}: "
            f"{latex_error_3m}"
        )
    table_registry_records_3m.append(
        {
            "table_id": table_id_3m,
            "name": table_name_3m,
            "purpose": purpose_3m,
            "rows": len(table_frame_3m),
            "csv_path": str(csv_path_3m),
            "latex_path": latex_path_value_3m,
        }
    )

SPLIT_TABLE_REGISTRY_3M = pd.DataFrame(table_registry_records_3m)
split_table_registry_path_3m = OUTPUT_ROOT_3M / "split_table_registry_3m.csv"
SPLIT_TABLE_REGISTRY_3M.to_csv(split_table_registry_path_3m, index=False)

workbook_path_3m = OUTPUT_ROOT_3M / "split_diagnostics_3m.xlsx"


def excel_safe_frame_3m(frame):
    """Return an Excel-compatible copy without changing source datetimes."""
    clean_3m = frame.copy()
    for column_3m in clean_3m.columns:
        if isinstance(clean_3m[column_3m].dtype, pd.DatetimeTZDtype):
            clean_3m[column_3m] = (
                clean_3m[column_3m]
                .dt.tz_convert("UTC")
                .dt.tz_localize(None)
            )
    return clean_3m


try:
    with pd.ExcelWriter(workbook_path_3m, engine="openpyxl") as writer_3m:
        for table_id_3m, table_name_3m, table_frame_3m, _ in table_exports_3m:
            sheet_name_3m = f"{table_id_3m}_{slug_3m(table_name_3m)}"[:31]
            excel_safe_frame_3m(table_frame_3m).to_excel(
                writer_3m,
                sheet_name=sheet_name_3m,
                index=False,
            )
        SPLIT_TABLE_REGISTRY_3M.to_excel(
            writer_3m,
            sheet_name="table_registry",
            index=False,
        )

        from openpyxl.styles import Alignment, Font, PatternFill
        from openpyxl.utils import get_column_letter

        for worksheet_3m in writer_3m.book.worksheets:
            worksheet_3m.freeze_panes = "A2"
            worksheet_3m.auto_filter.ref = worksheet_3m.dimensions
            for header_cell_3m in worksheet_3m[1]:
                header_cell_3m.font = Font(bold=True, color="FFFFFF")
                header_cell_3m.fill = PatternFill(
                    fill_type="solid",
                    fgColor="1F4E78",
                )
                header_cell_3m.alignment = Alignment(
                    horizontal="center",
                    vertical="center",
                    wrap_text=True,
                )
            for column_number_3m, column_cells_3m in enumerate(
                worksheet_3m.columns,
                start=1,
            ):
                maximum_length_3m = max(
                    len(str(cell_3m.value)) if cell_3m.value is not None else 0
                    for cell_3m in column_cells_3m
                )
                worksheet_3m.column_dimensions[
                    get_column_letter(column_number_3m)
                ].width = min(max(maximum_length_3m + 2, 10), 45)
except ImportError as excel_error_3m:
    workbook_path_3m = None
    print(f"Excel workbook export skipped: {excel_error_3m}")

mpl.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.family": "DejaVu Serif",
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

SPLIT_COLORS_3M = {
    "train": "#0072B2",
    "validation": "#E69F00",
    "test": "#009E73",
}
LABEL_COLORS_3M = {
    "normal": "#0072B2",
    "anomaly": "#D55E00",
}
figure_registry_records_3m = []


def save_split_figure_3m(figure, figure_id, name, title, claim, source_table):
    base_name_3m = f"{figure_id}_{slug_3m(name)}"
    png_path_3m = PNG_DIR_3M / f"{base_name_3m}.png"
    pdf_path_3m = PDF_DIR_3M / f"{base_name_3m}.pdf"
    figure.savefig(png_path_3m, bbox_inches="tight", facecolor="white")
    figure.savefig(pdf_path_3m, bbox_inches="tight", facecolor="white")
    plt.close(figure)
    figure_registry_records_3m.append(
        {
            "figure_id": figure_id,
            "title": title,
            "claim_supported": claim,
            "source_table": source_table,
            "png_path": str(png_path_3m),
            "pdf_path": str(pdf_path_3m),
        }
    )


# D3M01 — absolute eligible row support by class.
row_pivot_3m = SPLIT_CLASS_BALANCE_3M.pivot(
    index="split_assignment",
    columns="final_label",
    values="eligible_rows",
).reindex(index=SPLIT_ORDER_3M, columns=LABELS_3M, fill_value=0)
fig_3m, ax_3m = plt.subplots(figsize=(6.6, 4.0))
bottom_3m = np.zeros(len(SPLIT_ORDER_3M))
for label_3m in LABELS_3M:
    values_3m = row_pivot_3m[label_3m].to_numpy()
    ax_3m.bar(
        SPLIT_ORDER_3M,
        values_3m,
        bottom=bottom_3m,
        color=LABEL_COLORS_3M[label_3m],
        label=label_3m.capitalize(),
    )
    bottom_3m += values_3m
ax_3m.set_title("Eligible observations in the asset-grouped split")
ax_3m.set_ylabel("Eligible observations")
ax_3m.grid(axis="y", color="#D9D9D9", linewidth=0.6, alpha=0.7)
ax_3m.legend(frameon=False)
save_split_figure_3m(
    fig_3m,
    "D3M01",
    "eligible_rows_by_split_and_class",
    "Eligible observations by split and class",
    "Shows final class support after physical-asset isolation.",
    "SPLIT_CLASS_BALANCE_3M",
)

# D3M02 — normalized class composition.
composition_pivot_3m = SPLIT_CLASS_BALANCE_3M.pivot(
    index="split_assignment",
    columns="final_label",
    values="within_split_row_fraction",
).reindex(index=SPLIT_ORDER_3M, columns=LABELS_3M, fill_value=0)
fig_3m, ax_3m = plt.subplots(figsize=(6.6, 4.0))
bottom_3m = np.zeros(len(SPLIT_ORDER_3M))
for label_3m in LABELS_3M:
    values_3m = 100.0 * composition_pivot_3m[label_3m].to_numpy()
    ax_3m.bar(
        SPLIT_ORDER_3M,
        values_3m,
        bottom=bottom_3m,
        color=LABEL_COLORS_3M[label_3m],
        label=label_3m.capitalize(),
    )
    bottom_3m += values_3m
ax_3m.set_title("Within-split class composition")
ax_3m.set_ylabel("Eligible observations (%)")
ax_3m.set_ylim(0, 100)
ax_3m.grid(axis="y", color="#D9D9D9", linewidth=0.6, alpha=0.7)
ax_3m.legend(frameon=False)
save_split_figure_3m(
    fig_3m,
    "D3M02",
    "within_split_class_composition",
    "Within-split class composition",
    "Checks that class balance remains comparable across grouped splits.",
    "SPLIT_CLASS_BALANCE_3M",
)

# D3M03 — independent support units.
fig_3m, axes_3m = plt.subplots(1, 2, figsize=(9.0, 3.8))
for ax_3m, metric_3m, title_3m in zip(
    axes_3m,
    ["physical_assets", "eligible_events"],
    ["Physical assets", "Independent event files"],
):
    values_3m = SPLIT_SUMMARY_3M.set_index("split_assignment").reindex(
        SPLIT_ORDER_3M
    )[metric_3m]
    ax_3m.bar(
        SPLIT_ORDER_3M,
        values_3m,
        color=[SPLIT_COLORS_3M[name] for name in SPLIT_ORDER_3M],
    )
    ax_3m.set_title(title_3m)
    ax_3m.set_ylabel("Count")
    ax_3m.grid(axis="y", color="#D9D9D9", linewidth=0.6, alpha=0.7)
fig_3m.suptitle("Independent support retained in each split")
fig_3m.tight_layout()
save_split_figure_3m(
    fig_3m,
    "D3M03",
    "independent_asset_and_event_support",
    "Independent asset and event support by split",
    "Reports evaluation support in independent assets and events, not rows alone.",
    "SPLIT_SUMMARY_3M",
)

# D3M04 — farm/label support within each split.
farm_label_plot_3m = SPLIT_FARM_LABEL_SUMMARY_3M.copy()
farm_label_plot_3m["farm_split"] = (
    farm_label_plot_3m["farm"].astype(str)
    + "–"
    + farm_label_plot_3m["split_assignment"].astype(str).str[:3]
)
farm_split_order_3m = [
    f"{farm_3m}–{split_name_3m[:3]}"
    for farm_3m in FARMS_3M
    for split_name_3m in SPLIT_ORDER_3M
]
farm_label_pivot_3m = farm_label_plot_3m.pivot(
    index="farm_split",
    columns="final_label",
    values="eligible_rows",
).reindex(index=farm_split_order_3m, columns=LABELS_3M, fill_value=0)
fig_3m, ax_3m = plt.subplots(figsize=(9.0, 4.2))
bottom_3m = np.zeros(len(farm_split_order_3m))
for label_3m in LABELS_3M:
    values_3m = farm_label_pivot_3m[label_3m].to_numpy()
    ax_3m.bar(
        farm_split_order_3m,
        values_3m,
        bottom=bottom_3m,
        color=LABEL_COLORS_3M[label_3m],
        label=label_3m.capitalize(),
    )
    bottom_3m += values_3m
ax_3m.set_title("Farm- and class-specific support after asset grouping")
ax_3m.set_ylabel("Eligible observations")
ax_3m.tick_params(axis="x", rotation=40)
ax_3m.grid(axis="y", color="#D9D9D9", linewidth=0.6, alpha=0.7)
ax_3m.legend(frameon=False)
save_split_figure_3m(
    fig_3m,
    "D3M04",
    "farm_class_support_by_split",
    "Farm and class support by split",
    "Verifies that every farm and label remains represented in every split.",
    "SPLIT_FARM_LABEL_SUMMARY_3M",
)

# D3M05 — actual versus target shares for the main support units.
share_plot_3m = SPLIT_SUMMARY_3M.loc[
    :,
    [
        "split_assignment",
        "target_fraction",
        "eligible_row_fraction",
        "event_fraction",
        "asset_fraction",
    ],
].copy()
x_3m = np.arange(len(SPLIT_ORDER_3M), dtype=float)
width_3m = 0.20
fig_3m, ax_3m = plt.subplots(figsize=(7.4, 4.2))
for offset_3m, metric_3m, label_3m in [
    (-width_3m, "eligible_row_fraction", "Rows"),
    (0.0, "event_fraction", "Events"),
    (width_3m, "asset_fraction", "Assets"),
]:
    ax_3m.bar(
        x_3m + offset_3m,
        100.0 * share_plot_3m[metric_3m].to_numpy(),
        width=width_3m,
        label=label_3m,
    )
ax_3m.scatter(
    x_3m,
    100.0 * share_plot_3m["target_fraction"].to_numpy(),
    marker="D",
    s=35,
    color="black",
    label="Target",
    zorder=5,
)
ax_3m.set_xticks(x_3m, SPLIT_ORDER_3M)
ax_3m.set_ylabel("Share of complete eligible dataset (%)")
ax_3m.set_title("Actual split support versus the 70/15/15 target")
ax_3m.grid(axis="y", color="#D9D9D9", linewidth=0.6, alpha=0.7)
ax_3m.legend(frameon=False, ncol=4)
save_split_figure_3m(
    fig_3m,
    "D3M05",
    "actual_vs_target_split_share",
    "Actual versus target split shares",
    "Quantifies unavoidable deviations caused by indivisible physical assets.",
    "SPLIT_SUMMARY_3M",
)

SPLIT_FIGURE_REGISTRY_3M = pd.DataFrame(figure_registry_records_3m)
split_figure_registry_path_3m = OUTPUT_ROOT_3M / "split_figure_registry_3m.csv"
SPLIT_FIGURE_REGISTRY_3M.to_csv(split_figure_registry_path_3m, index=False)


# -----------------------------------------------------------------------------
# 9. Compact notebook report
# -----------------------------------------------------------------------------

print("\nAsset-grouped split assignment:")
print(ASSET_SPLIT_ASSIGNMENT_3M.to_string(index=False))

print("\nOverall split summary:")
print(SPLIT_SUMMARY_3M.to_string(index=False))

print("\nFarm- and label-specific split summary:")
print(SPLIT_FARM_LABEL_SUMMARY_3M.to_string(index=False))

print("\nLeakage and conservation audit:")
print(SPLIT_LEAKAGE_AUDIT_3M.to_string(index=False))

print("\nCell 3M completed successfully.")
print("Optimization seed:", RANDOM_SEED_3M)
print("Random candidate assignments evaluated:", RANDOM_CANDIDATES_3M)
print("Local-search improving swaps:", local_search_iterations_3m)
print("Final weighted balance score:", f"{best_assignment_score_3m:.8f}")
print("Eligible rows assigned:", int(eligible_assignments_3m.notna().sum()))
print("Physical assets assigned:", physical_asset_count_3m)
print("Eligible event files assigned:", eligible_3m["event_key"].nunique())
print(
    "Train/validation/test physical assets:",
    "/".join(str(value) for value in observed_asset_counts_3m),
)
print("Assets crossing splits:", assets_crossing_splits_3m)
print("Events crossing splits:", events_crossing_splits_3m)
print("Unassigned eligible rows:", int(eligible_assignments_3m.isna().sum()))
print("Assigned modeling-ineligible rows:", int(ineligible_assignments_3m.notna().sum()))
print("Split tables generated:", len(SPLIT_TABLE_REGISTRY_3M))
print("Split figures generated:", len(SPLIT_FIGURE_REGISTRY_3M))
print("Split table registry:", split_table_registry_path_3m)
print("Split figure registry:", split_figure_registry_path_3m)
if workbook_path_3m is not None:
    print("Split diagnostics workbook:", workbook_path_3m)
print(
    "\nOnly modeling-eligible rows received a split_assignment. No source row, "
    "label, timestamp, eligibility decision, or audit value was changed."
)
print(
    "Keep the test split untouched until model selection, threshold selection, "
    "and all validation-driven decisions are complete."
)


LaTeX export skipped for T3M01: Missing optional dependency 'Jinja2'. DataFrame.style requires jinja2. Use pip or conda to install Jinja2.
LaTeX export skipped for T3M02: Missing optional dependency 'Jinja2'. DataFrame.style requires jinja2. Use pip or conda to install Jinja2.
LaTeX export skipped for T3M03: Missing optional dependency 'Jinja2'. DataFrame.style requires jinja2. Use pip or conda to install Jinja2.
LaTeX export skipped for T3M04: Missing optional dependency 'Jinja2'. DataFrame.style requires jinja2. Use pip or conda to install Jinja2.
LaTeX export skipped for T3M05: Missing optional dependency 'Jinja2'. DataFrame.style requires jinja2. Use pip or conda to install Jinja2.
LaTeX export skipped for T3M06: Missing optional dependency 'Jinja2'. DataFrame.style requires jinja2. Use pip or conda to install Jinja2.

Asset-grouped split assignment:
farm asset_id        asset_key split_assignment  eligible_rows  eligible_events  normal_rows  anomaly_rows  normal_events  anomaly_ev

In [39]:
%pip install -q Jinja2

Note: you may need to restart the kernel to use updated packages.


In [40]:
# Repair the six skipped Cell 3M LaTeX exports.
# This does not recalculate or modify the split.

for (
    table_id_3m,
    table_name_3m,
    table_frame_3m,
    purpose_3m,
) in table_exports_3m:

    tex_path_3m = TABLE_DIR_3M / (
        f"{table_id_3m}_{slug_3m(table_name_3m)}.tex"
    )

    table_frame_3m.to_latex(
        tex_path_3m,
        index=False,
        escape=True,
    )

    SPLIT_TABLE_REGISTRY_3M.loc[
        SPLIT_TABLE_REGISTRY_3M["table_id"].eq(table_id_3m),
        "latex_path",
    ] = str(tex_path_3m)


SPLIT_TABLE_REGISTRY_3M.to_csv(
    split_table_registry_path_3m,
    index=False,
)

# Replace only the workbook registry sheet.
with pd.ExcelWriter(
    workbook_path_3m,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace",
) as writer_3m:

    SPLIT_TABLE_REGISTRY_3M.to_excel(
        writer_3m,
        sheet_name="table_registry",
        index=False,
    )


missing_latex_3m = int(
    SPLIT_TABLE_REGISTRY_3M["latex_path"].isna().sum()
)

if missing_latex_3m:
    raise ValueError(
        f"{missing_latex_3m} LaTeX exports remain missing."
    )

print("Cell 3M LaTeX exports repaired successfully.")
print("LaTeX tables exported:", len(SPLIT_TABLE_REGISTRY_3M))
print("Split assignment unchanged.")

Cell 3M LaTeX exports repaired successfully.
LaTeX tables exported: 6
Split assignment unchanged.


In [41]:
%run -i cell_3n_train_only_preprocessing.py


Measurement-source resolution:
resolution_mode selected_source_name selected_source_type  source_rows  eligible_keys_required  eligible_keys_matched  unmatched_eligible_rows          feature_column_mode  candidate_features  retained_features  excluded_features
      automatic          eligible_3l      keyed_dataframe       213537                  213537                 213537                        0 automatic_metadata_exclusion                   4                  3                  1

Training-only feature decisions:
decision                                 decision_reason  features
  retain              passed_training_only_quality_gates         3
 exclude insufficient_numeric_parse_fraction_in_training         1

Frozen split-transform audit:
split_assignment   rows  features raw_missing_cells raw_missing_fraction  post_transform_missing_cells  post_transform_nonfinite_cells transformed_mean_absolute_mean transformed_mean_feature_std parameters_fitted_on distribution_reporting
   

F:\Umar-Wisal-Work\Wisal-Bearings-Work\Code\cell_3n_train_only_preprocessing.py:1721: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell_3n.font = cell_3n.font.copy(bold=True, color="FFFFFF")
F:\Umar-Wisal-Work\Wisal-Bearings-Work\Code\cell_3n_train_only_preprocessing.py:1722: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell_3n.fill = cell_3n.fill.copy(


In [42]:
"""Cell 3N — split-aware feature audit and train-only preprocessing.

Run once in the notebook namespace after the accepted Cell 3M split:

    %run -i cell_3n_train_only_preprocessing.py

The cell resolves raw sensor measurements by the immutable source key
(``event_key``, ``source_row_index``), fits every quality decision and every
preprocessing statistic on training assets only, and applies the frozen
transformation to validation and test rows. Validation is used only for
non-destructive shift diagnostics. Test distributions are not summarized,
ranked, plotted, or used in any decision.

Preferred explicit input (optional):

    FEATURE_SOURCE_3N = <DataFrame with event_key, source_row_index, sensors>

or:

    FEATURE_SOURCE_3N = {event_key: raw_event_dataframe, ...}

If ``FEATURE_SOURCE_3N`` is absent, the cell safely searches existing notebook
DataFrames and event-frame mappings. It refuses to continue unless all 213,537
eligible manifest keys resolve exactly once.
"""

from __future__ import annotations

import hashlib
import json
import math
import re
from collections import Counter
from collections.abc import Mapping
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# 1. Lock and validate the accepted Cell 3M checkpoint
# -----------------------------------------------------------------------------

if "ROW_LABEL_MANIFEST" not in globals():
    raise NameError("Run Cells 3I–3M before Cell 3N.")

REQUIRED_MANIFEST_COLUMNS_3N = {
    "farm",
    "asset_id",
    "asset_key",
    "event_key",
    "source_row_index",
    "timestamp_utc",
    "final_label",
    "modeling_eligible",
    "split_assignment",
}

missing_manifest_columns_3n = (
    REQUIRED_MANIFEST_COLUMNS_3N - set(ROW_LABEL_MANIFEST.columns)
)
if missing_manifest_columns_3n:
    raise ValueError(
        "ROW_LABEL_MANIFEST is missing Cell 3N columns: "
        f"{sorted(missing_manifest_columns_3n)}"
    )

EXPECTED_ELIGIBLE_ROWS_3N = 213_537
EXPECTED_SPLIT_ROWS_3N = {
    "train": 152_236,
    "validation": 30_261,
    "test": 31_040,
}
EXPECTED_SPLIT_ASSETS_3N = {
    "train": 25,
    "validation": 5,
    "test": 6,
}
EXPECTED_SPLIT_EVENTS_3N = {
    "train": 64,
    "validation": 14,
    "test": 15,
}
SPLIT_ORDER_3N = ["train", "validation", "test"]
KEY_COLUMNS_3N = ["event_key", "source_row_index"]
RANDOM_SEED_3N = 20260810

# Training-only quality policy. Extreme values are audited but deliberately not
# clipped: genuine fault excursions may carry useful anomaly information.
MIN_TRAIN_NUMERIC_PARSE_FRACTION_3N = 0.98
MAX_TRAIN_MISSING_FRACTION_3N = 0.30
MAX_TRAIN_FARM_MISSING_FRACTION_3N = 0.50
MIN_TRAIN_FINITE_VALUES_3N = 100
NUMERICAL_EPSILON_3N = 1e-12
CORRELATION_ALERT_3N = 0.98
CORRELATION_SAMPLE_ROWS_3N = 50_000
PSI_EPSILON_3N = 1e-6

eligible_mask_3n = (
    ROW_LABEL_MANIFEST["modeling_eligible"].fillna(False).astype(bool)
)

if int(eligible_mask_3n.sum()) != EXPECTED_ELIGIBLE_ROWS_3N:
    raise ValueError(
        "Cell 3N expects 213,537 post-deduplication eligible rows, but found "
        f"{int(eligible_mask_3n.sum()):,}."
    )

if ROW_LABEL_MANIFEST.loc[
    eligible_mask_3n, "split_assignment"
].isna().any():
    raise ValueError("At least one eligible row lacks a Cell 3M split.")

if ROW_LABEL_MANIFEST.loc[
    ~eligible_mask_3n, "split_assignment"
].notna().any():
    raise ValueError("A modeling-ineligible row has a split assignment.")

manifest_length_before_3n = len(ROW_LABEL_MANIFEST)
manifest_index_before_3n = ROW_LABEL_MANIFEST.index.copy()
manifest_hash_before_3n = pd.util.hash_pandas_object(
    ROW_LABEL_MANIFEST,
    index=True,
).to_numpy(copy=True)

eligible_manifest_3n = ROW_LABEL_MANIFEST.loc[
    eligible_mask_3n,
    [
        "farm",
        "asset_id",
        "asset_key",
        "event_key",
        "source_row_index",
        "timestamp_utc",
        "final_label",
        "split_assignment",
    ],
].copy()

for column_3n in [
    "farm",
    "asset_id",
    "asset_key",
    "event_key",
    "final_label",
    "split_assignment",
]:
    eligible_manifest_3n[column_3n] = (
        eligible_manifest_3n[column_3n].astype("string").str.strip()
    )

eligible_manifest_3n["final_label"] = (
    eligible_manifest_3n["final_label"].str.lower()
)
eligible_manifest_3n["timestamp_utc"] = pd.to_datetime(
    eligible_manifest_3n["timestamp_utc"],
    errors="coerce",
    utc=True,
)
eligible_manifest_3n["source_row_index"] = pd.to_numeric(
    eligible_manifest_3n["source_row_index"],
    errors="coerce",
)

if eligible_manifest_3n[
    [
        "farm",
        "asset_id",
        "asset_key",
        "event_key",
        "source_row_index",
        "timestamp_utc",
        "final_label",
        "split_assignment",
    ]
].isna().any().any():
    raise ValueError("Eligible manifest rows contain missing critical values.")

if not np.isclose(
    eligible_manifest_3n["source_row_index"],
    np.floor(eligible_manifest_3n["source_row_index"]),
).all():
    raise ValueError("Manifest source_row_index contains non-integer values.")

eligible_manifest_3n["source_row_index"] = eligible_manifest_3n[
    "source_row_index"
].astype("int64")

if eligible_manifest_3n[KEY_COLUMNS_3N].duplicated().any():
    raise ValueError("Eligible manifest source keys are not unique.")

if sorted(eligible_manifest_3n["farm"].astype(str).unique()) != [
    "A",
    "B",
    "C",
]:
    raise ValueError("Cell 3N expects farms A, B, and C.")

if sorted(eligible_manifest_3n["final_label"].astype(str).unique()) != [
    "anomaly",
    "normal",
]:
    raise ValueError("Cell 3N expects labels normal and anomaly.")

observed_split_rows_3n = (
    eligible_manifest_3n["split_assignment"]
    .value_counts()
    .reindex(SPLIT_ORDER_3N, fill_value=0)
    .astype(int)
    .to_dict()
)
observed_split_assets_3n = (
    eligible_manifest_3n.groupby("split_assignment")["asset_key"]
    .nunique()
    .reindex(SPLIT_ORDER_3N, fill_value=0)
    .astype(int)
    .to_dict()
)
observed_split_events_3n = (
    eligible_manifest_3n.groupby("split_assignment")["event_key"]
    .nunique()
    .reindex(SPLIT_ORDER_3N, fill_value=0)
    .astype(int)
    .to_dict()
)

if observed_split_rows_3n != EXPECTED_SPLIT_ROWS_3N:
    raise ValueError(
        "Cell 3M row assignment differs from the accepted checkpoint: "
        f"{observed_split_rows_3n}."
    )
if observed_split_assets_3n != EXPECTED_SPLIT_ASSETS_3N:
    raise ValueError(
        "Cell 3M asset assignment differs from the accepted checkpoint: "
        f"{observed_split_assets_3n}."
    )
if observed_split_events_3n != EXPECTED_SPLIT_EVENTS_3N:
    raise ValueError(
        "Cell 3M event assignment differs from the accepted checkpoint: "
        f"{observed_split_events_3n}."
    )

assets_crossing_splits_3n = int(
    (
        eligible_manifest_3n.groupby("asset_key")["split_assignment"]
        .nunique()
        .gt(1)
    ).sum()
)
events_crossing_splits_3n = int(
    (
        eligible_manifest_3n.groupby("event_key")["split_assignment"]
        .nunique()
        .gt(1)
    ).sum()
)
if assets_crossing_splits_3n or events_crossing_splits_3n:
    raise ValueError("The accepted asset/event isolation no longer holds.")

eligible_key_index_3n = pd.MultiIndex.from_frame(
    eligible_manifest_3n[KEY_COLUMNS_3N]
)
eligible_event_keys_3n = frozenset(
    eligible_manifest_3n["event_key"].astype(str).unique()
)


# -----------------------------------------------------------------------------
# 2. Resolve a raw measurement source without guessing row identity
# -----------------------------------------------------------------------------

RESERVED_COLUMN_EXACT_3N = {
    "farm",
    "farm_id",
    "asset",
    "asset_id",
    "asset_key",
    "turbine",
    "turbine_id",
    "wtg",
    "event",
    "event_id",
    "event_key",
    "source_row_index",
    "raw_id",
    "row_id",
    "index",
    "timestamp",
    "timestamp_utc",
    "datetime",
    "date",
    "time",
    "label",
    "class",
    "target",
    "state",
    "status_label",
    "source_event_label",
    "final_label",
    "modeling_eligible",
    "split",
    "split_assignment",
    "measurement_schema_id",
    "measurement_fingerprint",
    "file",
    "filename",
    "file_path",
    "path",
}

RESERVED_COLUMN_PATTERN_3N = re.compile(
    r"(^|_)(label|class|target|split|eligible|event|asset|turbine|farm|"
    r"timestamp|datetime|date|time|index|row|raw|schema|fingerprint|"
    r"filename|filepath|path|audit|reason|action|fault_code|anomaly_flag)"
    r"($|_)",
    flags=re.IGNORECASE,
)


def slug_3n(value):
    value = re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_").lower()
    return value or "item"


def normalized_source_index_3n(series, source_name):
    numeric_3n = pd.to_numeric(series, errors="coerce")
    if numeric_3n.isna().any() or not np.isclose(
        numeric_3n, np.floor(numeric_3n)
    ).all():
        raise ValueError(
            f"{source_name} contains invalid source-row indices."
        )
    return numeric_3n.astype("int64")


def is_reserved_feature_name_3n(column):
    normalized_3n = slug_3n(column)
    return (
        normalized_3n in RESERVED_COLUMN_EXACT_3N
        or RESERVED_COLUMN_PATTERN_3N.search(normalized_3n) is not None
    )


def candidate_feature_columns_3n(frame):
    return [
        column_3n
        for column_3n in frame.columns
        if column_3n not in KEY_COLUMNS_3N
        and not is_reserved_feature_name_3n(column_3n)
    ]


def normalize_keyed_frame_3n(frame, source_name):
    if not isinstance(frame, pd.DataFrame):
        raise TypeError(f"{source_name} is not a pandas DataFrame.")
    if frame.columns.duplicated().any():
        duplicated_3n = frame.columns[frame.columns.duplicated()].tolist()
        raise ValueError(
            f"{source_name} has duplicate columns: {duplicated_3n}."
        )
    missing_keys_3n = set(KEY_COLUMNS_3N) - set(frame.columns)
    if missing_keys_3n:
        raise ValueError(
            f"{source_name} lacks source keys {sorted(missing_keys_3n)}."
        )
    output_3n = frame.copy(deep=False)
    output_3n["event_key"] = output_3n["event_key"].astype("string").str.strip()
    output_3n["source_row_index"] = normalized_source_index_3n(
        output_3n["source_row_index"], source_name
    )
    if output_3n["event_key"].isna().any() or output_3n[
        "event_key"
    ].eq("").any():
        raise ValueError(f"{source_name} contains blank event keys.")
    return output_3n


def infer_leaf_event_key_3n(path_parts, frame):
    if "event_key" in frame.columns:
        values_3n = (
            frame["event_key"].dropna().astype(str).str.strip().unique().tolist()
        )
        if len(values_3n) == 1 and values_3n[0] in eligible_event_keys_3n:
            return values_3n[0]

    for part_3n in reversed(path_parts):
        part_text_3n = str(part_3n).strip()
        if part_text_3n in eligible_event_keys_3n:
            return part_text_3n

    if {"farm", "event_id"}.issubset(frame.columns):
        farm_values_3n = frame["farm"].dropna().astype(str).str.strip().unique()
        event_values_3n = frame["event_id"].dropna().astype(str).str.strip().unique()
        if len(farm_values_3n) == 1 and len(event_values_3n) == 1:
            farm_token_3n = farm_values_3n[0].lower().replace("farm_", "")
            event_token_3n = re.sub(r"\.0$", "", event_values_3n[0])
            candidate_3n = f"farm_{farm_token_3n}_event_{event_token_3n}"
            if candidate_3n in eligible_event_keys_3n:
                return candidate_3n

    path_text_3n = "::".join(str(part_3n) for part_3n in path_parts)
    farm_match_3n = re.search(r"(?:farm[_\s-]*)?([abc])", path_text_3n, re.I)
    event_match_3n = re.search(r"(?:event[_\s-]*)?(\d+)(?!.*\d)", path_text_3n, re.I)
    if farm_match_3n and event_match_3n:
        candidate_3n = (
            f"farm_{farm_match_3n.group(1).lower()}_event_{event_match_3n.group(1)}"
        )
        if candidate_3n in eligible_event_keys_3n:
            return candidate_3n
    return None


def iter_dataframe_leaves_3n(value, path=(), depth=0, seen=None):
    if seen is None:
        seen = set()
    object_id_3n = id(value)
    if object_id_3n in seen:
        return
    seen.add(object_id_3n)

    if isinstance(value, pd.DataFrame):
        yield path, value
        return
    if isinstance(value, Mapping) and depth < 4:
        for key_3n, child_3n in value.items():
            yield from iter_dataframe_leaves_3n(
                child_3n,
                path=path + (key_3n,),
                depth=depth + 1,
                seen=seen,
            )


def materialize_event_mapping_3n(mapping, source_name):
    pieces_3n = []
    resolved_events_3n = []
    unresolved_leaf_count_3n = 0

    for path_3n, frame_3n in iter_dataframe_leaves_3n(mapping):
        event_key_3n = infer_leaf_event_key_3n(path_3n, frame_3n)
        if event_key_3n is None:
            unresolved_leaf_count_3n += 1
            continue
        if event_key_3n not in eligible_event_keys_3n:
            continue

        piece_3n = frame_3n.copy(deep=False)
        if "event_key" not in piece_3n.columns:
            piece_3n = piece_3n.assign(event_key=event_key_3n)
        else:
            piece_3n = piece_3n.assign(event_key=event_key_3n)

        if "source_row_index" not in piece_3n.columns:
            if "raw_id" in piece_3n.columns:
                source_indices_3n = piece_3n["raw_id"]
            else:
                source_indices_3n = pd.Series(
                    piece_3n.index,
                    index=piece_3n.index,
                )
            piece_3n = piece_3n.assign(
                source_row_index=normalized_source_index_3n(
                    source_indices_3n,
                    f"{source_name}:{event_key_3n}",
                ).to_numpy()
            )

        pieces_3n.append(piece_3n)
        resolved_events_3n.append(event_key_3n)

    if not pieces_3n:
        raise ValueError(
            f"{source_name} contains no event DataFrames that can be mapped "
            "to eligible event keys."
        )

    output_3n = pd.concat(
        pieces_3n,
        axis=0,
        ignore_index=True,
        sort=False,
    )
    output_3n.attrs["resolved_event_count_3n"] = len(set(resolved_events_3n))
    output_3n.attrs["unresolved_leaf_count_3n"] = unresolved_leaf_count_3n
    return normalize_keyed_frame_3n(output_3n, source_name)


def inspect_source_candidate_3n(name, value, explicit=False):
    try:
        if isinstance(value, pd.DataFrame):
            keyed_3n = normalize_keyed_frame_3n(value, name)
            source_type_3n = "keyed_dataframe"
        elif isinstance(value, Mapping):
            keyed_3n = materialize_event_mapping_3n(value, name)
            source_type_3n = "event_dataframe_mapping"
        else:
            return None, None

        feature_candidates_3n = candidate_feature_columns_3n(keyed_3n)
        if not feature_candidates_3n:
            raise ValueError("no non-metadata feature columns were found")

        duplicated_keys_3n = int(keyed_3n[KEY_COLUMNS_3N].duplicated().sum())
        source_keys_3n = pd.MultiIndex.from_frame(keyed_3n[KEY_COLUMNS_3N])
        matched_keys_3n = int(source_keys_3n.isin(eligible_key_index_3n).sum())
        eligible_keys_matched_3n = int(eligible_key_index_3n.isin(source_keys_3n).sum())
        exact_coverage_3n = (
            eligible_keys_matched_3n == EXPECTED_ELIGIBLE_ROWS_3N
            and duplicated_keys_3n == 0
        )

        record_3n = {
            "source_name": str(name),
            "source_type": source_type_3n,
            "explicit": bool(explicit),
            "source_rows": int(len(keyed_3n)),
            "candidate_feature_columns": int(len(feature_candidates_3n)),
            "eligible_keys_matched": eligible_keys_matched_3n,
            "source_rows_matching_eligible": matched_keys_3n,
            "duplicated_source_keys": duplicated_keys_3n,
            "exact_eligible_coverage": bool(exact_coverage_3n),
            "inspection_error": pd.NA,
        }
        return record_3n, keyed_3n
    except Exception as error_3n:
        if explicit:
            raise
        record_3n = {
            "source_name": str(name),
            "source_type": type(value).__name__,
            "explicit": False,
            "source_rows": int(len(value)) if hasattr(value, "__len__") else pd.NA,
            "candidate_feature_columns": pd.NA,
            "eligible_keys_matched": 0,
            "source_rows_matching_eligible": 0,
            "duplicated_source_keys": pd.NA,
            "exact_eligible_coverage": False,
            "inspection_error": f"{type(error_3n).__name__}: {error_3n}",
        }
        return record_3n, None


source_candidate_records_3n = []
source_candidate_frames_3n = {}
source_resolution_mode_3n = "automatic"

if "FEATURE_SOURCE_3N" in globals():
    source_resolution_mode_3n = "explicit"
    source_record_3n, source_frame_3n = inspect_source_candidate_3n(
        "FEATURE_SOURCE_3N",
        globals()["FEATURE_SOURCE_3N"],
        explicit=True,
    )
    source_candidate_records_3n.append(source_record_3n)
    source_candidate_frames_3n["FEATURE_SOURCE_3N"] = source_frame_3n
else:
    globals_snapshot_3n = list(globals().items())
    excluded_source_names_3n = {
        "ROW_LABEL_MANIFEST",
        "eligible_manifest_3n",
        "ELIGIBLE_MEASUREMENT_INDEX",
        "EXACT_SAME_LABEL_CANONICAL_RETENTION_AUDIT",
    }

    for name_3n, value_3n in globals_snapshot_3n:
        if name_3n in excluded_source_names_3n or name_3n.startswith("_"):
            continue
        if isinstance(value_3n, pd.DataFrame) and set(KEY_COLUMNS_3N).issubset(
            value_3n.columns
        ):
            record_3n, frame_3n = inspect_source_candidate_3n(
                name_3n, value_3n
            )
            if record_3n is not None:
                source_candidate_records_3n.append(record_3n)
                if frame_3n is not None:
                    source_candidate_frames_3n[name_3n] = frame_3n

    mapping_name_pattern_3n = re.compile(
        r"event|data|frame|raw|measurement|source|farm", re.IGNORECASE
    )
    for name_3n, value_3n in globals_snapshot_3n:
        if (
            name_3n.startswith("_")
            or not isinstance(value_3n, Mapping)
            or not mapping_name_pattern_3n.search(name_3n)
        ):
            continue
        record_3n, frame_3n = inspect_source_candidate_3n(name_3n, value_3n)
        if record_3n is not None:
            source_candidate_records_3n.append(record_3n)
            if frame_3n is not None:
                source_candidate_frames_3n[name_3n] = frame_3n


FEATURE_SOURCE_CANDIDATES_3N = pd.DataFrame(source_candidate_records_3n)
if FEATURE_SOURCE_CANDIDATES_3N.empty:
    raise NameError(
        "Cell 3N could not find raw sensor measurements. Define "
        "FEATURE_SOURCE_3N as either a DataFrame containing event_key, "
        "source_row_index, and sensor columns, or a mapping from event_key "
        "to raw event DataFrames; then rerun Cell 3N."
    )

valid_sources_3n = FEATURE_SOURCE_CANDIDATES_3N.loc[
    FEATURE_SOURCE_CANDIDATES_3N["exact_eligible_coverage"].fillna(False)
].copy()
if valid_sources_3n.empty:
    compact_candidates_3n = FEATURE_SOURCE_CANDIDATES_3N.loc[
        :,
        [
            "source_name",
            "source_type",
            "source_rows",
            "candidate_feature_columns",
            "eligible_keys_matched",
            "duplicated_source_keys",
            "inspection_error",
        ],
    ]
    raise ValueError(
        "No measurement source covers every eligible manifest key exactly "
        "once. Inspect FEATURE_SOURCE_CANDIDATES_3N. Preferred repair:\n"
        "FEATURE_SOURCE_3N = <keyed measurement DataFrame or event mapping>\n\n"
        + compact_candidates_3n.to_string(index=False)
    )

valid_sources_3n = valid_sources_3n.sort_values(
    ["explicit", "candidate_feature_columns", "source_rows", "source_name"],
    ascending=[False, False, True, True],
).reset_index(drop=True)

selected_source_name_3n = str(valid_sources_3n.iloc[0]["source_name"])
selected_source_type_3n = str(valid_sources_3n.iloc[0]["source_type"])
MEASUREMENT_SOURCE_3N = source_candidate_frames_3n[selected_source_name_3n]

if MEASUREMENT_SOURCE_3N[KEY_COLUMNS_3N].duplicated().any():
    raise ValueError("The selected measurement source has duplicate source keys.")

if "FEATURE_COLUMNS_3N" in globals() and globals()["FEATURE_COLUMNS_3N"] is not None:
    requested_feature_columns_3n = list(globals()["FEATURE_COLUMNS_3N"])
    if len(requested_feature_columns_3n) != len(set(requested_feature_columns_3n)):
        raise ValueError("FEATURE_COLUMNS_3N contains duplicates.")
    missing_requested_features_3n = set(requested_feature_columns_3n) - set(
        MEASUREMENT_SOURCE_3N.columns
    )
    if missing_requested_features_3n:
        raise ValueError(
            "FEATURE_COLUMNS_3N is missing from the measurement source: "
            f"{sorted(missing_requested_features_3n)}"
        )
    leaking_requested_features_3n = [
        column_3n
        for column_3n in requested_feature_columns_3n
        if is_reserved_feature_name_3n(column_3n)
    ]
    if leaking_requested_features_3n:
        raise ValueError(
            "FEATURE_COLUMNS_3N includes identity/label/split-like columns: "
            f"{leaking_requested_features_3n}"
        )
    candidate_features_3n = requested_feature_columns_3n
    feature_column_mode_3n = "explicit"
else:
    candidate_features_3n = candidate_feature_columns_3n(MEASUREMENT_SOURCE_3N)
    feature_column_mode_3n = "automatic_metadata_exclusion"

if not candidate_features_3n:
    raise ValueError("No candidate sensor columns remain after metadata exclusion.")

source_subset_3n = MEASUREMENT_SOURCE_3N.loc[
    :,
    KEY_COLUMNS_3N + candidate_features_3n,
].copy()

RAW_ELIGIBLE_FEATURES_3N = eligible_manifest_3n.merge(
    source_subset_3n,
    on=KEY_COLUMNS_3N,
    how="left",
    validate="one_to_one",
    indicator="_measurement_match_3n",
)

unmatched_eligible_rows_3n = int(
    RAW_ELIGIBLE_FEATURES_3N["_measurement_match_3n"].ne("both").sum()
)
if unmatched_eligible_rows_3n:
    raise ValueError(
        f"{unmatched_eligible_rows_3n:,} eligible rows lack raw measurements."
    )
RAW_ELIGIBLE_FEATURES_3N = RAW_ELIGIBLE_FEATURES_3N.drop(
    columns="_measurement_match_3n"
)

if len(RAW_ELIGIBLE_FEATURES_3N) != EXPECTED_ELIGIBLE_ROWS_3N:
    raise ValueError("Measurement join did not conserve eligible rows.")


# -----------------------------------------------------------------------------
# 3. Fit feature-quality decisions using training assets only
# -----------------------------------------------------------------------------

train_mask_3n = RAW_ELIGIBLE_FEATURES_3N["split_assignment"].eq("train")
validation_mask_3n = RAW_ELIGIBLE_FEATURES_3N[
    "split_assignment"
].eq("validation")
test_mask_3n = RAW_ELIGIBLE_FEATURES_3N["split_assignment"].eq("test")

if int(train_mask_3n.sum()) != EXPECTED_SPLIT_ROWS_3N["train"]:
    raise ValueError("Training-row count changed during the measurement join.")

numeric_feature_data_3n = pd.DataFrame(
    index=RAW_ELIGIBLE_FEATURES_3N.index
)
feature_quality_records_3n = []

train_farms_3n = sorted(
    RAW_ELIGIBLE_FEATURES_3N.loc[train_mask_3n, "farm"].astype(str).unique()
)
if train_farms_3n != ["A", "B", "C"]:
    raise ValueError("Training assets do not cover all three farms.")

for feature_3n in candidate_features_3n:
    original_3n = RAW_ELIGIBLE_FEATURES_3N[feature_3n]
    numeric_3n = pd.to_numeric(original_3n, errors="coerce").astype("float64")
    numeric_3n = numeric_3n.mask(~np.isfinite(numeric_3n), np.nan)
    numeric_feature_data_3n[feature_3n] = numeric_3n

    train_original_3n = original_3n.loc[train_mask_3n]
    train_values_3n = numeric_3n.loc[train_mask_3n]
    original_nonmissing_3n = train_original_3n.notna()
    finite_train_3n = train_values_3n.dropna()
    parse_denominator_3n = int(original_nonmissing_3n.sum())
    parse_numerator_3n = int(
        (original_nonmissing_3n & train_values_3n.notna()).sum()
    )
    parse_fraction_3n = (
        parse_numerator_3n / parse_denominator_3n
        if parse_denominator_3n
        else 0.0
    )
    train_missing_fraction_3n = float(train_values_3n.isna().mean())

    farm_missing_fractions_3n = {}
    for farm_3n in train_farms_3n:
        farm_mask_3n = train_mask_3n & RAW_ELIGIBLE_FEATURES_3N[
            "farm"
        ].eq(farm_3n)
        farm_missing_fractions_3n[farm_3n] = float(
            numeric_3n.loc[farm_mask_3n].isna().mean()
        )

    finite_count_3n = int(len(finite_train_3n))
    unique_count_3n = int(finite_train_3n.nunique(dropna=True))

    if finite_count_3n:
        q1_3n = float(finite_train_3n.quantile(0.25))
        median_3n = float(finite_train_3n.median())
        q3_3n = float(finite_train_3n.quantile(0.75))
        iqr_3n = float(q3_3n - q1_3n)
        mean_3n = float(finite_train_3n.mean())
        std_3n = float(finite_train_3n.std(ddof=0))
        min_3n = float(finite_train_3n.min())
        max_3n = float(finite_train_3n.max())
        mad_3n = float((finite_train_3n - median_3n).abs().median())
    else:
        q1_3n = median_3n = q3_3n = iqr_3n = np.nan
        mean_3n = std_3n = min_3n = max_3n = mad_3n = np.nan

    decision_3n = "retain"
    reason_3n = "passed_training_only_quality_gates"
    scale_source_3n = "training_iqr"
    scale_3n = iqr_3n

    if parse_fraction_3n < MIN_TRAIN_NUMERIC_PARSE_FRACTION_3N:
        decision_3n = "exclude"
        reason_3n = "insufficient_numeric_parse_fraction_in_training"
    elif finite_count_3n < MIN_TRAIN_FINITE_VALUES_3N:
        decision_3n = "exclude"
        reason_3n = "insufficient_finite_training_values"
    elif train_missing_fraction_3n > MAX_TRAIN_MISSING_FRACTION_3N:
        decision_3n = "exclude"
        reason_3n = "excessive_training_missingness"
    elif max(farm_missing_fractions_3n.values()) > (
        MAX_TRAIN_FARM_MISSING_FRACTION_3N
    ):
        decision_3n = "exclude"
        reason_3n = "excessive_missingness_in_a_training_farm"
    elif unique_count_3n <= 1:
        decision_3n = "exclude"
        reason_3n = "constant_in_training"
    elif not np.isfinite(scale_3n) or abs(scale_3n) <= NUMERICAL_EPSILON_3N:
        scale_3n = std_3n
        scale_source_3n = "training_standard_deviation_fallback"
        if not np.isfinite(scale_3n) or abs(scale_3n) <= NUMERICAL_EPSILON_3N:
            decision_3n = "exclude"
            reason_3n = "near_constant_in_training"

    if decision_3n == "exclude":
        scale_source_3n = pd.NA
        scale_3n = np.nan

    feature_quality_records_3n.append(
        {
            "feature": str(feature_3n),
            "decision": decision_3n,
            "decision_reason": reason_3n,
            "fit_split": "train",
            "train_rows": int(train_mask_3n.sum()),
            "train_original_nonmissing": parse_denominator_3n,
            "train_numeric_parse_fraction": parse_fraction_3n,
            "train_finite_values": finite_count_3n,
            "train_missing_fraction": train_missing_fraction_3n,
            "train_farm_A_missing_fraction": farm_missing_fractions_3n["A"],
            "train_farm_B_missing_fraction": farm_missing_fractions_3n["B"],
            "train_farm_C_missing_fraction": farm_missing_fractions_3n["C"],
            "train_unique_values": unique_count_3n,
            "train_min": min_3n,
            "train_q1": q1_3n,
            "train_median": median_3n,
            "train_q3": q3_3n,
            "train_max": max_3n,
            "train_mean": mean_3n,
            "train_std": std_3n,
            "train_mad": mad_3n,
            "imputation_method": (
                "training_median" if decision_3n == "retain" else pd.NA
            ),
            "imputation_value": (
                median_3n if decision_3n == "retain" else np.nan
            ),
            "scaling_method": (
                "robust_center_and_scale" if decision_3n == "retain" else pd.NA
            ),
            "center_value": (
                median_3n if decision_3n == "retain" else np.nan
            ),
            "scale_value": scale_3n,
            "scale_source": scale_source_3n,
            "outlier_clipping": "none_fault_excursions_preserved",
        }
    )


FEATURE_QUALITY_AUDIT_3N = pd.DataFrame(feature_quality_records_3n).sort_values(
    ["decision", "feature"],
    ascending=[False, True],
).reset_index(drop=True)

FEATURE_NAMES_3N = FEATURE_QUALITY_AUDIT_3N.loc[
    FEATURE_QUALITY_AUDIT_3N["decision"].eq("retain"), "feature"
].astype(str).tolist()
EXCLUDED_FEATURES_3N = FEATURE_QUALITY_AUDIT_3N.loc[
    FEATURE_QUALITY_AUDIT_3N["decision"].eq("exclude"), "feature"
].astype(str).tolist()

if not FEATURE_NAMES_3N:
    raise ValueError(
        "No sensor feature passed the training-only quality gates. Inspect "
        "FEATURE_QUALITY_AUDIT_3N; do not relax gates using validation/test."
    )

PREPROCESSING_PARAMETERS_3N = FEATURE_QUALITY_AUDIT_3N.loc[
    FEATURE_QUALITY_AUDIT_3N["decision"].eq("retain"),
    [
        "feature",
        "fit_split",
        "imputation_method",
        "imputation_value",
        "scaling_method",
        "center_value",
        "scale_value",
        "scale_source",
        "outlier_clipping",
    ],
].reset_index(drop=True)

if not PREPROCESSING_PARAMETERS_3N["fit_split"].eq("train").all():
    raise ValueError("A preprocessing parameter was not fitted on training data.")
if not np.isfinite(
    PREPROCESSING_PARAMETERS_3N[
        ["imputation_value", "center_value", "scale_value"]
    ].to_numpy(dtype=float)
).all():
    raise ValueError("A retained feature has a non-finite fitted parameter.")
if PREPROCESSING_PARAMETERS_3N["scale_value"].abs().le(
    NUMERICAL_EPSILON_3N
).any():
    raise ValueError("A retained feature has a zero preprocessing scale.")


# -----------------------------------------------------------------------------
# 4. Freeze the training transform and apply without refitting
# -----------------------------------------------------------------------------

parameter_lookup_3n = PREPROCESSING_PARAMETERS_3N.set_index("feature")


def transform_features_3n(frame):
    """Apply the already-fitted Cell 3N transform; never refits parameters."""
    missing_features_3n = set(FEATURE_NAMES_3N) - set(frame.columns)
    if missing_features_3n:
        raise ValueError(
            "Transform input is missing retained features: "
            f"{sorted(missing_features_3n)}"
        )
    transformed_3n = pd.DataFrame(index=frame.index)
    for feature_3n in FEATURE_NAMES_3N:
        values_3n = pd.to_numeric(frame[feature_3n], errors="coerce").astype(
            "float64"
        )
        values_3n = values_3n.mask(~np.isfinite(values_3n), np.nan)
        params_3n = parameter_lookup_3n.loc[feature_3n]
        values_3n = values_3n.fillna(float(params_3n["imputation_value"]))
        values_3n = (
            values_3n - float(params_3n["center_value"])
        ) / float(params_3n["scale_value"])
        transformed_3n[feature_3n] = values_3n.astype("float32")
    return transformed_3n


transformed_features_3n = transform_features_3n(
    RAW_ELIGIBLE_FEATURES_3N[FEATURE_NAMES_3N]
)

if transformed_features_3n.isna().any().any():
    raise ValueError("Missing values remain after the frozen transform.")
if not np.isfinite(transformed_features_3n.to_numpy(dtype="float32")).all():
    raise ValueError("Non-finite values remain after the frozen transform.")

metadata_columns_3n = [
    "farm",
    "asset_id",
    "asset_key",
    "event_key",
    "source_row_index",
    "timestamp_utc",
    "final_label",
    "split_assignment",
]

PREPROCESSED_FEATURE_MATRIX_3N = pd.concat(
    [
        RAW_ELIGIBLE_FEATURES_3N[metadata_columns_3n].reset_index(drop=True),
        transformed_features_3n.reset_index(drop=True),
    ],
    axis=1,
)

label_map_3n = {"normal": 0, "anomaly": 1}


def split_outputs_3n(split_name):
    split_mask_3n = PREPROCESSED_FEATURE_MATRIX_3N[
        "split_assignment"
    ].eq(split_name)
    x_3n = PREPROCESSED_FEATURE_MATRIX_3N.loc[
        split_mask_3n, FEATURE_NAMES_3N
    ].reset_index(drop=True)
    y_3n = (
        PREPROCESSED_FEATURE_MATRIX_3N.loc[
            split_mask_3n, "final_label"
        ]
        .map(label_map_3n)
        .astype("int8")
        .reset_index(drop=True)
    )
    meta_3n = PREPROCESSED_FEATURE_MATRIX_3N.loc[
        split_mask_3n, metadata_columns_3n
    ].reset_index(drop=True)
    return x_3n, y_3n, meta_3n


X_TRAIN_3N, Y_TRAIN_3N, META_TRAIN_3N = split_outputs_3n("train")
X_VALIDATION_3N, Y_VALIDATION_3N, META_VALIDATION_3N = split_outputs_3n(
    "validation"
)
X_TEST_3N, Y_TEST_3N, META_TEST_3N = split_outputs_3n("test")

if [len(X_TRAIN_3N), len(X_VALIDATION_3N), len(X_TEST_3N)] != [
    EXPECTED_SPLIT_ROWS_3N[name_3n] for name_3n in SPLIT_ORDER_3N
]:
    raise ValueError("Preprocessed split outputs do not conserve rows.")

PREPROCESSOR_STATE_3N = {
    "cell": "3N",
    "version": 1,
    "fit_split": "train",
    "random_seed": RANDOM_SEED_3N,
    "source_name": selected_source_name_3n,
    "source_type": selected_source_type_3n,
    "feature_column_mode": feature_column_mode_3n,
    "candidate_features": [str(value_3n) for value_3n in candidate_features_3n],
    "retained_features": FEATURE_NAMES_3N,
    "excluded_features": EXCLUDED_FEATURES_3N,
    "label_map": label_map_3n,
    "imputation": "training median",
    "scaling": "training median and IQR; training std fallback",
    "outlier_clipping": "none",
    "test_policy": (
        "frozen transform only; no test feature selection, drift statistics, "
        "ranking, plotting, model selection, or threshold selection"
    ),
    "parameters": PREPROCESSING_PARAMETERS_3N.to_dict(orient="records"),
}


# -----------------------------------------------------------------------------
# 5. Train/validation diagnostics; keep test distributions sealed
# -----------------------------------------------------------------------------

SOURCE_RESOLUTION_AUDIT_3N = pd.DataFrame(
    [
        {
            "resolution_mode": source_resolution_mode_3n,
            "selected_source_name": selected_source_name_3n,
            "selected_source_type": selected_source_type_3n,
            "source_rows": int(len(MEASUREMENT_SOURCE_3N)),
            "eligible_keys_required": EXPECTED_ELIGIBLE_ROWS_3N,
            "eligible_keys_matched": EXPECTED_ELIGIBLE_ROWS_3N,
            "unmatched_eligible_rows": unmatched_eligible_rows_3n,
            "feature_column_mode": feature_column_mode_3n,
            "candidate_features": len(candidate_features_3n),
            "retained_features": len(FEATURE_NAMES_3N),
            "excluded_features": len(EXCLUDED_FEATURES_3N),
        }
    ]
)

transform_audit_records_3n = []
for split_3n in SPLIT_ORDER_3N:
    mask_3n = RAW_ELIGIBLE_FEATURES_3N["split_assignment"].eq(split_3n)
    transformed_split_3n = transformed_features_3n.loc[mask_3n]
    if split_3n == "test":
        raw_missing_cells_3n = pd.NA
        raw_missing_fraction_3n = pd.NA
        transformed_mean_abs_3n = pd.NA
        transformed_std_mean_3n = pd.NA
        distribution_reporting_3n = "sealed"
    else:
        raw_selected_3n = numeric_feature_data_3n.loc[
            mask_3n, FEATURE_NAMES_3N
        ]
        raw_missing_cells_3n = int(raw_selected_3n.isna().sum().sum())
        raw_missing_fraction_3n = float(raw_selected_3n.isna().to_numpy().mean())
        transformed_mean_abs_3n = float(
            transformed_split_3n.mean().abs().mean()
        )
        transformed_std_mean_3n = float(
            transformed_split_3n.std(ddof=0).mean()
        )
        distribution_reporting_3n = "reported"

    transform_audit_records_3n.append(
        {
            "split_assignment": split_3n,
            "rows": int(mask_3n.sum()),
            "features": len(FEATURE_NAMES_3N),
            "raw_missing_cells": raw_missing_cells_3n,
            "raw_missing_fraction": raw_missing_fraction_3n,
            "post_transform_missing_cells": int(
                transformed_split_3n.isna().sum().sum()
            ),
            "post_transform_nonfinite_cells": int(
                (~np.isfinite(transformed_split_3n.to_numpy())).sum()
            ),
            "transformed_mean_absolute_mean": transformed_mean_abs_3n,
            "transformed_mean_feature_std": transformed_std_mean_3n,
            "parameters_fitted_on": "train",
            "distribution_reporting": distribution_reporting_3n,
        }
    )

SPLIT_TRANSFORM_AUDIT_3N = pd.DataFrame(transform_audit_records_3n)

TRAIN_TRANSFORMED_FEATURE_SUMMARY_3N = pd.DataFrame(
    {
        "feature": FEATURE_NAMES_3N,
        "mean": X_TRAIN_3N.mean().reindex(FEATURE_NAMES_3N).to_numpy(),
        "std": X_TRAIN_3N.std(ddof=0).reindex(FEATURE_NAMES_3N).to_numpy(),
        "min": X_TRAIN_3N.min().reindex(FEATURE_NAMES_3N).to_numpy(),
        "q1": X_TRAIN_3N.quantile(0.25).reindex(FEATURE_NAMES_3N).to_numpy(),
        "median": X_TRAIN_3N.median().reindex(FEATURE_NAMES_3N).to_numpy(),
        "q3": X_TRAIN_3N.quantile(0.75).reindex(FEATURE_NAMES_3N).to_numpy(),
        "max": X_TRAIN_3N.max().reindex(FEATURE_NAMES_3N).to_numpy(),
    }
)


def population_stability_index_3n(train_values, comparison_values, bins=10):
    train_values_3n = np.asarray(train_values, dtype=float)
    comparison_values_3n = np.asarray(comparison_values, dtype=float)
    train_values_3n = train_values_3n[np.isfinite(train_values_3n)]
    comparison_values_3n = comparison_values_3n[
        np.isfinite(comparison_values_3n)
    ]
    if not len(train_values_3n) or not len(comparison_values_3n):
        return np.nan, 0

    edges_3n = np.unique(
        np.quantile(train_values_3n, np.linspace(0.0, 1.0, bins + 1))
    )
    if len(edges_3n) < 3:
        return 0.0, max(len(edges_3n) - 1, 1)
    edges_3n[0] = -np.inf
    edges_3n[-1] = np.inf
    train_counts_3n, _ = np.histogram(train_values_3n, bins=edges_3n)
    comparison_counts_3n, _ = np.histogram(
        comparison_values_3n, bins=edges_3n
    )
    train_props_3n = train_counts_3n / train_counts_3n.sum()
    comparison_props_3n = comparison_counts_3n / comparison_counts_3n.sum()
    train_props_3n = np.clip(train_props_3n, PSI_EPSILON_3N, None)
    comparison_props_3n = np.clip(
        comparison_props_3n, PSI_EPSILON_3N, None
    )
    psi_3n = float(
        np.sum(
            (comparison_props_3n - train_props_3n)
            * np.log(comparison_props_3n / train_props_3n)
        )
    )
    return psi_3n, len(edges_3n) - 1


validation_shift_records_3n = []
for feature_3n in FEATURE_NAMES_3N:
    psi_3n, bins_3n = population_stability_index_3n(
        X_TRAIN_3N[feature_3n],
        X_VALIDATION_3N[feature_3n],
    )
    if not np.isfinite(psi_3n):
        shift_band_3n = "not_estimable"
    elif psi_3n < 0.10:
        shift_band_3n = "small"
    elif psi_3n < 0.25:
        shift_band_3n = "moderate"
    else:
        shift_band_3n = "large"
    validation_shift_records_3n.append(
        {
            "feature": feature_3n,
            "comparison": "validation_vs_train",
            "train_defined_bins": bins_3n,
            "population_stability_index": psi_3n,
            "shift_band": shift_band_3n,
            "used_for_feature_exclusion": False,
            "test_distribution_accessed": False,
        }
    )

VALIDATION_SHIFT_AUDIT_3N = pd.DataFrame(
    validation_shift_records_3n
).sort_values(
    "population_stability_index",
    ascending=False,
    na_position="last",
).reset_index(drop=True)

correlation_sample_n_3n = min(CORRELATION_SAMPLE_ROWS_3N, len(X_TRAIN_3N))
if correlation_sample_n_3n < len(X_TRAIN_3N):
    correlation_sample_3n = X_TRAIN_3N.sample(
        n=correlation_sample_n_3n,
        random_state=RANDOM_SEED_3N,
        replace=False,
    )
else:
    correlation_sample_3n = X_TRAIN_3N

TRAIN_SPEARMAN_CORRELATION_3N = correlation_sample_3n.corr(method="spearman")
correlation_pair_records_3n = []
for index_a_3n, feature_a_3n in enumerate(FEATURE_NAMES_3N):
    for feature_b_3n in FEATURE_NAMES_3N[index_a_3n + 1 :]:
        correlation_3n = float(
            TRAIN_SPEARMAN_CORRELATION_3N.loc[feature_a_3n, feature_b_3n]
        )
        correlation_pair_records_3n.append(
            {
                "feature_a": feature_a_3n,
                "feature_b": feature_b_3n,
                "train_spearman_correlation": correlation_3n,
                "absolute_correlation": abs(correlation_3n),
                "high_correlation_alert": abs(correlation_3n)
                >= CORRELATION_ALERT_3N,
                "used_for_feature_exclusion": False,
                "sample_rows": correlation_sample_n_3n,
            }
        )

TRAIN_CORRELATION_PAIR_AUDIT_3N = pd.DataFrame(correlation_pair_records_3n)
if not TRAIN_CORRELATION_PAIR_AUDIT_3N.empty:
    TRAIN_CORRELATION_PAIR_AUDIT_3N = (
        TRAIN_CORRELATION_PAIR_AUDIT_3N.sort_values(
            "absolute_correlation", ascending=False
        ).reset_index(drop=True)
    )
else:
    TRAIN_CORRELATION_PAIR_AUDIT_3N = pd.DataFrame(
        columns=[
            "feature_a",
            "feature_b",
            "train_spearman_correlation",
            "absolute_correlation",
            "high_correlation_alert",
            "used_for_feature_exclusion",
            "sample_rows",
        ]
    )


# -----------------------------------------------------------------------------
# 6. Mutation boundary, leakage discipline, and conservation audit
# -----------------------------------------------------------------------------

manifest_hash_after_3n = pd.util.hash_pandas_object(
    ROW_LABEL_MANIFEST,
    index=True,
).to_numpy(copy=True)
manifest_values_changed_3n = int(
    np.count_nonzero(manifest_hash_before_3n != manifest_hash_after_3n)
)

if len(ROW_LABEL_MANIFEST) != manifest_length_before_3n:
    raise ValueError("Cell 3N changed the manifest row count.")
if not ROW_LABEL_MANIFEST.index.equals(manifest_index_before_3n):
    raise ValueError("Cell 3N changed the manifest index.")
if manifest_values_changed_3n:
    raise ValueError("Cell 3N changed ROW_LABEL_MANIFEST values.")

test_used_for_fitting_3n = int(
    PREPROCESSING_PARAMETERS_3N["fit_split"].ne("train").sum()
)
test_distribution_statistics_exported_3n = int(
    SPLIT_TRANSFORM_AUDIT_3N.loc[
        SPLIT_TRANSFORM_AUDIT_3N["split_assignment"].eq("test"),
        [
            "raw_missing_cells",
            "raw_missing_fraction",
            "transformed_mean_absolute_mean",
            "transformed_mean_feature_std",
        ],
    ].notna().sum().sum()
)

PREPROCESSING_LEAKAGE_AUDIT_3N = pd.DataFrame(
    [
        {
            "check": "eligible_row_conservation",
            "observed": len(PREPROCESSED_FEATURE_MATRIX_3N),
            "required": EXPECTED_ELIGIBLE_ROWS_3N,
        },
        {
            "check": "unmatched_eligible_measurement_keys",
            "observed": unmatched_eligible_rows_3n,
            "required": 0,
        },
        {
            "check": "post_transform_missing_cells",
            "observed": int(transformed_features_3n.isna().sum().sum()),
            "required": 0,
        },
        {
            "check": "post_transform_nonfinite_cells",
            "observed": int(
                (~np.isfinite(transformed_features_3n.to_numpy())).sum()
            ),
            "required": 0,
        },
        {
            "check": "parameters_not_fitted_on_train",
            "observed": test_used_for_fitting_3n,
            "required": 0,
        },
        {
            "check": "test_distribution_statistics_exported",
            "observed": test_distribution_statistics_exported_3n,
            "required": 0,
        },
        {
            "check": "assets_crossing_splits",
            "observed": assets_crossing_splits_3n,
            "required": 0,
        },
        {
            "check": "events_crossing_splits",
            "observed": events_crossing_splits_3n,
            "required": 0,
        },
        {
            "check": "manifest_rows_deleted",
            "observed": manifest_length_before_3n - len(ROW_LABEL_MANIFEST),
            "required": 0,
        },
        {
            "check": "manifest_values_changed",
            "observed": manifest_values_changed_3n,
            "required": 0,
        },
    ]
)
PREPROCESSING_LEAKAGE_AUDIT_3N["passed"] = (
    PREPROCESSING_LEAKAGE_AUDIT_3N["observed"]
    == PREPROCESSING_LEAKAGE_AUDIT_3N["required"]
)

if not PREPROCESSING_LEAKAGE_AUDIT_3N["passed"].all():
    failed_checks_3n = PREPROCESSING_LEAKAGE_AUDIT_3N.loc[
        ~PREPROCESSING_LEAKAGE_AUDIT_3N["passed"]
    ]
    raise ValueError(
        "Cell 3N leakage/conservation checks failed:\n"
        + failed_checks_3n.to_string(index=False)
    )


# -----------------------------------------------------------------------------
# 7. Export paper-ready tables, figures, registries, and frozen state
# -----------------------------------------------------------------------------

OUTPUT_ROOT_3N = Path(
    globals().get("OUTPUT_ROOT_3L", Path("paper_visuals_3l"))
) / "preprocessing_3n"
TABLE_DIR_3N = OUTPUT_ROOT_3N / "tables"
FIGURE_DIR_3N = OUTPUT_ROOT_3N / "figures"
for directory_3n in [OUTPUT_ROOT_3N, TABLE_DIR_3N, FIGURE_DIR_3N]:
    directory_3n.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.size": 9,
        "axes.titlesize": 11,
        "axes.labelsize": 9,
        "legend.fontsize": 8,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

SPLIT_COLORS_3N = {"train": "#2E6F9E", "validation": "#E69F00"}
DECISION_COLORS_3N = {"retain": "#2A9D8F", "exclude": "#C94C4C"}
figure_registry_records_3n = []


def save_figure_3n(figure, figure_id, title, claim_supported, source_objects):
    stem_3n = f"{figure_id}_{slug_3n(title)}"
    png_path_3n = FIGURE_DIR_3N / f"{stem_3n}.png"
    pdf_path_3n = FIGURE_DIR_3N / f"{stem_3n}.pdf"
    figure.savefig(png_path_3n, bbox_inches="tight", dpi=300)
    figure.savefig(pdf_path_3n, bbox_inches="tight")
    plt.close(figure)
    figure_registry_records_3n.append(
        {
            "figure_id": figure_id,
            "title": title,
            "claim_supported": claim_supported,
            "source_objects": source_objects,
            "fit_or_comparison_scope": "training and validation only; test sealed",
            "png_path": str(png_path_3n),
            "pdf_path": str(pdf_path_3n),
        }
    )


# F3N01 — feature decision counts.
decision_counts_3n = (
    FEATURE_QUALITY_AUDIT_3N["decision"]
    .value_counts()
    .reindex(["retain", "exclude"], fill_value=0)
)
figure_3n, axis_3n = plt.subplots(figsize=(5.8, 4.0))
bars_3n = axis_3n.bar(
    decision_counts_3n.index,
    decision_counts_3n.values,
    color=[DECISION_COLORS_3N[value_3n] for value_3n in decision_counts_3n.index],
)
axis_3n.bar_label(bars_3n, padding=3)
axis_3n.set_ylabel("Candidate sensor features")
axis_3n.set_title("Training-only feature-quality decisions")
axis_3n.grid(axis="y", alpha=0.2)
save_figure_3n(
    figure_3n,
    "F3N01",
    "Training-only feature quality decisions",
    "Documents how many candidate sensors passed predeclared training-only gates.",
    "FEATURE_QUALITY_AUDIT_3N",
)


# F3N02 — training missingness.
missing_plot_3n = FEATURE_QUALITY_AUDIT_3N.sort_values(
    "train_missing_fraction", ascending=True
).copy()
figure_height_3n = max(4.5, min(16.0, 0.28 * len(missing_plot_3n) + 1.8))
figure_3n, axis_3n = plt.subplots(figsize=(8.2, figure_height_3n))
axis_3n.barh(
    np.arange(len(missing_plot_3n)),
    100.0 * missing_plot_3n["train_missing_fraction"],
    color=[DECISION_COLORS_3N[value_3n] for value_3n in missing_plot_3n["decision"]],
)
axis_3n.axvline(
    100.0 * MAX_TRAIN_MISSING_FRACTION_3N,
    color="#333333",
    linestyle="--",
    linewidth=1.0,
    label="Global exclusion threshold",
)
axis_3n.set_yticks(np.arange(len(missing_plot_3n)))
axis_3n.set_yticklabels(missing_plot_3n["feature"])
axis_3n.set_xlabel("Missing training values (%)")
axis_3n.set_title("Training missingness by candidate sensor")
axis_3n.legend(loc="lower right")
axis_3n.grid(axis="x", alpha=0.2)
save_figure_3n(
    figure_3n,
    "F3N02",
    "Training missingness by candidate sensor",
    "Shows the missingness evidence used by the train-only feature gate.",
    "FEATURE_QUALITY_AUDIT_3N",
)


# F3N03 — farm-specific training missingness.
farm_missing_columns_3n = [
    "train_farm_A_missing_fraction",
    "train_farm_B_missing_fraction",
    "train_farm_C_missing_fraction",
]
farm_missing_matrix_3n = FEATURE_QUALITY_AUDIT_3N.set_index("feature").loc[
    :, farm_missing_columns_3n
]
figure_height_3n = max(4.5, min(16.0, 0.25 * len(farm_missing_matrix_3n) + 2.0))
figure_3n, axis_3n = plt.subplots(figsize=(6.8, figure_height_3n))
image_3n = axis_3n.imshow(
    100.0 * farm_missing_matrix_3n.to_numpy(dtype=float),
    aspect="auto",
    cmap="YlOrRd",
    vmin=0,
    vmax=max(
        1.0,
        min(100.0, 100.0 * float(farm_missing_matrix_3n.max().max())),
    ),
)
axis_3n.set_xticks([0, 1, 2])
axis_3n.set_xticklabels(["Farm A", "Farm B", "Farm C"])
axis_3n.set_yticks(np.arange(len(farm_missing_matrix_3n)))
axis_3n.set_yticklabels(farm_missing_matrix_3n.index)
axis_3n.set_title("Farm-specific training missingness")
colorbar_3n = figure_3n.colorbar(image_3n, ax=axis_3n, pad=0.02)
colorbar_3n.set_label("Missing values (%)")
save_figure_3n(
    figure_3n,
    "F3N03",
    "Farm-specific training missingness",
    "Verifies that retained features have usable training support in every farm.",
    "FEATURE_QUALITY_AUDIT_3N",
)


# F3N04 — validation PSI, never test PSI.
psi_plot_3n = VALIDATION_SHIFT_AUDIT_3N.sort_values(
    "population_stability_index", ascending=True
)
figure_height_3n = max(4.5, min(16.0, 0.28 * len(psi_plot_3n) + 1.8))
figure_3n, axis_3n = plt.subplots(figsize=(8.2, figure_height_3n))
axis_3n.barh(
    np.arange(len(psi_plot_3n)),
    psi_plot_3n["population_stability_index"],
    color="#6C5B7B",
)
axis_3n.axvline(0.10, color="#E69F00", linestyle="--", label="Moderate shift")
axis_3n.axvline(0.25, color="#C94C4C", linestyle="--", label="Large shift")
axis_3n.set_yticks(np.arange(len(psi_plot_3n)))
axis_3n.set_yticklabels(psi_plot_3n["feature"])
axis_3n.set_xlabel("Population stability index")
axis_3n.set_title("Validation-to-training feature shift")
axis_3n.legend(loc="lower right")
axis_3n.grid(axis="x", alpha=0.2)
save_figure_3n(
    figure_3n,
    "F3N04",
    "Validation to training feature shift",
    "Quantifies unseen-asset validation shift without consulting test distributions.",
    "VALIDATION_SHIFT_AUDIT_3N",
)


# F3N05 — training-only Spearman correlations.
max_heatmap_features_3n = min(30, len(FEATURE_NAMES_3N))
heatmap_features_3n = (
    TRAIN_TRANSFORMED_FEATURE_SUMMARY_3N.sort_values("std", ascending=False)
    .head(max_heatmap_features_3n)["feature"]
    .tolist()
)
heatmap_matrix_3n = TRAIN_SPEARMAN_CORRELATION_3N.loc[
    heatmap_features_3n, heatmap_features_3n
]
heatmap_size_3n = max(6.0, min(13.0, 0.42 * len(heatmap_features_3n) + 2.5))
figure_3n, axis_3n = plt.subplots(figsize=(heatmap_size_3n, heatmap_size_3n))
image_3n = axis_3n.imshow(
    heatmap_matrix_3n.to_numpy(dtype=float),
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
)
axis_3n.set_xticks(np.arange(len(heatmap_features_3n)))
axis_3n.set_xticklabels(heatmap_features_3n, rotation=90)
axis_3n.set_yticks(np.arange(len(heatmap_features_3n)))
axis_3n.set_yticklabels(heatmap_features_3n)
axis_3n.set_title("Training-only Spearman feature correlation")
colorbar_3n = figure_3n.colorbar(image_3n, ax=axis_3n, pad=0.02)
colorbar_3n.set_label("Spearman correlation")
save_figure_3n(
    figure_3n,
    "F3N05",
    "Training only Spearman feature correlation",
    "Documents multicollinearity without automatically deleting correlated sensors.",
    "TRAIN_SPEARMAN_CORRELATION_3N",
)


# F3N06 — train and validation transformed location/scale; test remains sealed.
comparison_summary_3n = pd.DataFrame(
    {
        "feature": FEATURE_NAMES_3N,
        "train_mean": X_TRAIN_3N.mean().reindex(FEATURE_NAMES_3N).to_numpy(),
        "validation_mean": X_VALIDATION_3N.mean().reindex(FEATURE_NAMES_3N).to_numpy(),
        "train_std": X_TRAIN_3N.std(ddof=0).reindex(FEATURE_NAMES_3N).to_numpy(),
        "validation_std": X_VALIDATION_3N.std(ddof=0).reindex(FEATURE_NAMES_3N).to_numpy(),
    }
)
plot_order_3n = comparison_summary_3n.assign(
    validation_mean_abs=lambda frame_3n: frame_3n["validation_mean"].abs()
).sort_values("validation_mean_abs", ascending=True)
figure_height_3n = max(5.0, min(16.0, 0.30 * len(plot_order_3n) + 2.2))
figure_3n, axes_3n = plt.subplots(
    1, 2, figsize=(11.0, figure_height_3n), sharey=True
)
y_positions_3n = np.arange(len(plot_order_3n))
for axis_3n, statistic_3n, title_3n in [
    (axes_3n[0], "mean", "Transformed mean"),
    (axes_3n[1], "std", "Transformed standard deviation"),
]:
    axis_3n.scatter(
        plot_order_3n[f"train_{statistic_3n}"],
        y_positions_3n,
        color=SPLIT_COLORS_3N["train"],
        marker="o",
        label="Train",
        s=25,
    )
    axis_3n.scatter(
        plot_order_3n[f"validation_{statistic_3n}"],
        y_positions_3n,
        color=SPLIT_COLORS_3N["validation"],
        marker="^",
        label="Validation",
        s=28,
    )
    axis_3n.set_xlabel(title_3n)
    axis_3n.grid(axis="x", alpha=0.2)
axes_3n[0].set_yticks(y_positions_3n)
axes_3n[0].set_yticklabels(plot_order_3n["feature"])
axes_3n[0].axvline(0, color="#555555", linewidth=0.8)
axes_3n[1].axvline(1, color="#555555", linewidth=0.8)
axes_3n[1].legend(loc="lower right")
figure_3n.suptitle("Frozen preprocessing across train and validation")
save_figure_3n(
    figure_3n,
    "F3N06",
    "Frozen preprocessing across train and validation",
    "Shows validation behavior under training-fitted preprocessing; test remains sealed.",
    "X_TRAIN_3N; X_VALIDATION_3N",
)


PREPROCESSING_FIGURE_REGISTRY_3N = pd.DataFrame(figure_registry_records_3n)

table_exports_3n = [
    (
        "T3N01",
        "source_resolution_audit",
        SOURCE_RESOLUTION_AUDIT_3N,
        "Documents exact source-key resolution before feature processing.",
    ),
    (
        "T3N02",
        "feature_quality_audit",
        FEATURE_QUALITY_AUDIT_3N,
        "Reports training-only feature decisions and fitted statistics.",
    ),
    (
        "T3N03",
        "preprocessing_parameters",
        PREPROCESSING_PARAMETERS_3N,
        "Provides the frozen median-imputation and robust-scaling parameters.",
    ),
    (
        "T3N04",
        "split_transform_audit",
        SPLIT_TRANSFORM_AUDIT_3N,
        "Verifies finite, row-conserving transformation while sealing test statistics.",
    ),
    (
        "T3N05",
        "validation_shift_audit",
        VALIDATION_SHIFT_AUDIT_3N,
        "Quantifies validation-to-training drift using training-defined bins.",
    ),
    (
        "T3N06",
        "training_correlation_pairs",
        TRAIN_CORRELATION_PAIR_AUDIT_3N,
        "Reports training-only correlated sensor pairs without automatic removal.",
    ),
    (
        "T3N07",
        "training_transformed_feature_summary",
        TRAIN_TRANSFORMED_FEATURE_SUMMARY_3N,
        "Summarizes the transformed training feature space.",
    ),
    (
        "T3N08",
        "preprocessing_leakage_audit",
        PREPROCESSING_LEAKAGE_AUDIT_3N,
        "Confirms train-only fitting, sealed test diagnostics, and manifest conservation.",
    ),
]

table_registry_records_3n = []
for table_id_3n, table_name_3n, table_frame_3n, purpose_3n in table_exports_3n:
    stem_3n = f"{table_id_3n}_{slug_3n(table_name_3n)}"
    csv_path_3n = TABLE_DIR_3N / f"{stem_3n}.csv"
    tex_path_3n = TABLE_DIR_3N / f"{stem_3n}.tex"
    table_frame_3n.to_csv(csv_path_3n, index=False)
    latex_status_3n = "exported"
    try:
        table_frame_3n.to_latex(tex_path_3n, index=False, escape=True)
        latex_path_value_3n = str(tex_path_3n)
    except Exception as error_3n:
        latex_status_3n = f"skipped: {type(error_3n).__name__}: {error_3n}"
        latex_path_value_3n = pd.NA
        print(f"LaTeX export skipped for {table_id_3n}: {error_3n}")

    table_registry_records_3n.append(
        {
            "table_id": table_id_3n,
            "name": table_name_3n,
            "purpose": purpose_3n,
            "rows": len(table_frame_3n),
            "columns": len(table_frame_3n.columns),
            "csv_path": str(csv_path_3n),
            "latex_path": latex_path_value_3n,
            "latex_status": latex_status_3n,
        }
    )

PREPROCESSING_TABLE_REGISTRY_3N = pd.DataFrame(table_registry_records_3n)

figure_registry_path_3n = OUTPUT_ROOT_3N / "preprocessing_figure_registry_3n.csv"
table_registry_path_3n = OUTPUT_ROOT_3N / "preprocessing_table_registry_3n.csv"
PREPROCESSING_FIGURE_REGISTRY_3N.to_csv(figure_registry_path_3n, index=False)
PREPROCESSING_TABLE_REGISTRY_3N.to_csv(table_registry_path_3n, index=False)

state_json_path_3n = OUTPUT_ROOT_3N / "preprocessor_state_3n.json"


def json_safe_3n(value):
    if value is pd.NA or value is None:
        return None
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


with state_json_path_3n.open("w", encoding="utf-8") as state_file_3n:
    json.dump(
        PREPROCESSOR_STATE_3N,
        state_file_3n,
        indent=2,
        ensure_ascii=False,
        default=json_safe_3n,
    )

workbook_path_3n = OUTPUT_ROOT_3N / "preprocessing_diagnostics_3n.xlsx"
workbook_exported_3n = False
try:
    with pd.ExcelWriter(workbook_path_3n, engine="openpyxl") as writer_3n:
        for table_id_3n, table_name_3n, table_frame_3n, _ in table_exports_3n:
            sheet_name_3n = f"{table_id_3n}_{slug_3n(table_name_3n)}"[:31]
            workbook_frame_3n = table_frame_3n.copy()
            for column_3n in workbook_frame_3n.columns:
                if isinstance(workbook_frame_3n[column_3n].dtype, pd.DatetimeTZDtype):
                    workbook_frame_3n[column_3n] = workbook_frame_3n[
                        column_3n
                    ].dt.tz_convert("UTC").dt.tz_localize(None)
            workbook_frame_3n.to_excel(
                writer_3n,
                sheet_name=sheet_name_3n,
                index=False,
            )
        PREPROCESSING_FIGURE_REGISTRY_3N.to_excel(
            writer_3n, sheet_name="figure_registry", index=False
        )
        PREPROCESSING_TABLE_REGISTRY_3N.to_excel(
            writer_3n, sheet_name="table_registry", index=False
        )

        for worksheet_3n in writer_3n.book.worksheets:
            worksheet_3n.freeze_panes = "A2"
            worksheet_3n.auto_filter.ref = worksheet_3n.dimensions
            for cell_3n in worksheet_3n[1]:
                cell_3n.font = cell_3n.font.copy(bold=True, color="FFFFFF")
                cell_3n.fill = cell_3n.fill.copy(
                    fill_type="solid", fgColor="1F4E78"
                )
            for column_cells_3n in worksheet_3n.columns:
                maximum_length_3n = max(
                    len(str(cell_3n.value)) if cell_3n.value is not None else 0
                    for cell_3n in column_cells_3n
                )
                column_letter_3n = column_cells_3n[0].column_letter
                worksheet_3n.column_dimensions[column_letter_3n].width = min(
                    max(maximum_length_3n + 2, 10), 45
                )
    workbook_exported_3n = True
except Exception as error_3n:
    print(f"Excel workbook export skipped: {error_3n}")


# -----------------------------------------------------------------------------
# 8. Compact notebook report
# -----------------------------------------------------------------------------

decision_reason_summary_3n = (
    FEATURE_QUALITY_AUDIT_3N.groupby(
        ["decision", "decision_reason"], as_index=False
    )
    .agg(features=("feature", "size"))
    .sort_values(["decision", "features"], ascending=[False, False])
)

print("\nMeasurement-source resolution:")
print(SOURCE_RESOLUTION_AUDIT_3N.to_string(index=False))

print("\nTraining-only feature decisions:")
print(decision_reason_summary_3n.to_string(index=False))

print("\nFrozen split-transform audit:")
print(SPLIT_TRANSFORM_AUDIT_3N.to_string(index=False))

print("\nValidation-shift summary (test remains sealed):")
print(
    VALIDATION_SHIFT_AUDIT_3N.head(min(20, len(VALIDATION_SHIFT_AUDIT_3N)))
    .to_string(index=False)
)

print("\nPreprocessing leakage and conservation audit:")
print(PREPROCESSING_LEAKAGE_AUDIT_3N.to_string(index=False))

print("\nCell 3N completed successfully.")
print("Measurement source:", selected_source_name_3n)
print("Eligible rows resolved:", len(PREPROCESSED_FEATURE_MATRIX_3N))
print("Candidate sensor features:", len(candidate_features_3n))
print("Retained sensor features:", len(FEATURE_NAMES_3N))
print("Excluded sensor features:", len(EXCLUDED_FEATURES_3N))
print(
    "Train/validation/test rows:",
    f"{len(X_TRAIN_3N)}/{len(X_VALIDATION_3N)}/{len(X_TEST_3N)}",
)
print("Post-transform missing cells:", int(transformed_features_3n.isna().sum().sum()))
print(
    "Post-transform non-finite cells:",
    int((~np.isfinite(transformed_features_3n.to_numpy())).sum()),
)
print("Tables generated:", len(PREPROCESSING_TABLE_REGISTRY_3N))
print("Figures generated:", len(PREPROCESSING_FIGURE_REGISTRY_3N))
print("Figure registry:", figure_registry_path_3n)
print("Table registry:", table_registry_path_3n)
print("Frozen preprocessor state:", state_json_path_3n)
print("Excel workbook exported:", workbook_exported_3n)
print("Manifest values changed:", manifest_values_changed_3n)
print(
    "\nAll feature inclusion, imputation, and scaling parameters were fitted "
    "on training assets only. Validation was used only for shift diagnostics. "
    "Test received the frozen transform but its distributions remain sealed. "
    "No manifest row, label, eligibility decision, or split assignment changed."
)


C:\Users\Muhammad Umar\AppData\Local\Temp\ipykernel_12804\3472816551.py:480: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  output_3n = pd.concat(



Measurement-source resolution:
resolution_mode  selected_source_name selected_source_type  source_rows  eligible_keys_required  eligible_keys_matched  unmatched_eligible_rows          feature_column_mode  candidate_features  retained_features  excluded_features
      automatic MEASUREMENT_SOURCE_3N      keyed_dataframe       213537                  213537                 213537                        0 automatic_metadata_exclusion                   4                  3                  1

Training-only feature decisions:
decision                                 decision_reason  features
  retain              passed_training_only_quality_gates         3
 exclude insufficient_numeric_parse_fraction_in_training         1

Frozen split-transform audit:
split_assignment   rows  features raw_missing_cells raw_missing_fraction  post_transform_missing_cells  post_transform_nonfinite_cells transformed_mean_absolute_mean transformed_mean_feature_std parameters_fitted_on distribution_reporting
 

C:\Users\Muhammad Umar\AppData\Local\Temp\ipykernel_12804\3472816551.py:1721: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell_3n.font = cell_3n.font.copy(bold=True, color="FFFFFF")
C:\Users\Muhammad Umar\AppData\Local\Temp\ipykernel_12804\3472816551.py:1722: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell_3n.fill = cell_3n.fill.copy(


In [43]:
candidate_columns_3n = [
    "source_name",
    "source_type",
    "source_rows",
    "candidate_feature_columns",
    "eligible_keys_matched",
    "duplicated_source_keys",
    "exact_eligible_coverage",
    "inspection_error",
]

credible_sources_3n = (
    FEATURE_SOURCE_CANDIDATES_3N.loc[
        FEATURE_SOURCE_CANDIDATES_3N[
            "exact_eligible_coverage"
        ].fillna(False),
        candidate_columns_3n,
    ]
    .sort_values(
        "candidate_feature_columns",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(credible_sources_3n)

print("\nColumns in each complete-coverage source:")

for source_name_3n in credible_sources_3n["source_name"]:
    source_frame_3n = source_candidate_frames_3n.get(
        source_name_3n
    )

    if source_frame_3n is None:
        continue

    non_key_columns_3n = [
        str(column_3n)
        for column_3n in source_frame_3n.columns
        if column_3n not in {
            "event_key",
            "source_row_index",
        }
    ]

    print(f"\n{source_name_3n}:")
    print(non_key_columns_3n)

,source_name,source_type,source_rows,candidate_feature_columns,eligible_keys_matched,duplicated_source_keys,exact_eligible_coverage,inspection_error
0,eligible_3l,keyed_dataframe,213537,4,213537,0,True,<NA>
1,ordered_eligible_3l,keyed_dataframe,213537,4,213537,0,True,<NA>
2,MEASUREMENT_SOURCE_3N,keyed_dataframe,213537,4,213537,0,True,<NA>
3,source_subset_3n,keyed_dataframe,213537,4,213537,0,True,<NA>
4,RAW_ELIGIBLE_FEATURES_3N,keyed_dataframe,213537,4,213537,0,True,<NA>
5,PREPROCESSED_FEATURE_MATRIX_3N,keyed_dataframe,213537,3,213537,0,True,<NA>
6,eligible_manifest,keyed_dataframe,214546,1,213537,0,True,<NA>
7,distinct_timestamp_vectors,keyed_dataframe,213537,1,213537,0,True,<NA>
8,manifest_before_3k,keyed_dataframe,5242948,1,213537,0,True,<NA>
9,manifest_3l,keyed_dataframe,5242948,1,213537,0,True,<NA>



Columns in each complete-coverage source:

eligible_3l:
['asset_id', 'asset_key', 'event_id', 'farm', 'final_label', 'modeling_eligible', 'raw_id', 'same_label_duplicate_action', 'same_label_duplicate_canonical_event_key', 'same_label_duplicate_reason', 'source_event_label', 'split_assignment', 'timestamp_utc', 'date_utc', 'month_utc', 'hour_utc', 'weekday_number', 'weekday']

ordered_eligible_3l:
['asset_id', 'asset_key', 'event_id', 'farm', 'final_label', 'modeling_eligible', 'raw_id', 'same_label_duplicate_action', 'same_label_duplicate_canonical_event_key', 'same_label_duplicate_reason', 'source_event_label', 'split_assignment', 'timestamp_utc', 'date_utc', 'month_utc', 'hour_utc', 'weekday_number', 'weekday']

MEASUREMENT_SOURCE_3N:
['asset_id', 'asset_key', 'event_id', 'farm', 'final_label', 'modeling_eligible', 'raw_id', 'same_label_duplicate_action', 'same_label_duplicate_canonical_event_key', 'same_label_duplicate_reason', 'source_event_label', 'split_assignment', 'timestamp_

In [44]:
"""Cell 3N — split-aware feature audit and train-only preprocessing.

Run once in the notebook namespace after the accepted Cell 3M split:

    %run -i cell_3n_train_only_preprocessing.py

The cell resolves raw sensor measurements by the immutable source key
(``event_key``, ``source_row_index``), fits every quality decision and every
preprocessing statistic on training assets only, and applies the frozen
transformation to validation and test rows. Validation is used only for
non-destructive shift diagnostics. Test distributions are not summarized,
ranked, plotted, or used in any decision.

Required explicit input:

    FEATURE_SOURCE_3N = <DataFrame with event_key, source_row_index, sensors>

or:

    FEATURE_SOURCE_3N = {event_key: raw_event_dataframe, ...}

If ``FEATURE_SOURCE_3N`` is absent, the cell performs a structural source scan
for diagnosis but refuses to choose automatically. This prevents a derived
audit/visualization frame from being mistaken for raw sensor measurements. The
selected source must resolve all 213,537 eligible manifest keys exactly once.
"""

from __future__ import annotations

import hashlib
import json
import math
import re
from collections import Counter
from collections.abc import Mapping
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# 1. Lock and validate the accepted Cell 3M checkpoint
# -----------------------------------------------------------------------------

if "ROW_LABEL_MANIFEST" not in globals():
    raise NameError("Run Cells 3I–3M before Cell 3N.")

REQUIRED_MANIFEST_COLUMNS_3N = {
    "farm",
    "asset_id",
    "asset_key",
    "event_key",
    "source_row_index",
    "timestamp_utc",
    "final_label",
    "modeling_eligible",
    "split_assignment",
}

missing_manifest_columns_3n = (
    REQUIRED_MANIFEST_COLUMNS_3N - set(ROW_LABEL_MANIFEST.columns)
)
if missing_manifest_columns_3n:
    raise ValueError(
        "ROW_LABEL_MANIFEST is missing Cell 3N columns: "
        f"{sorted(missing_manifest_columns_3n)}"
    )

EXPECTED_ELIGIBLE_ROWS_3N = 213_537
EXPECTED_SPLIT_ROWS_3N = {
    "train": 152_236,
    "validation": 30_261,
    "test": 31_040,
}
EXPECTED_SPLIT_ASSETS_3N = {
    "train": 25,
    "validation": 5,
    "test": 6,
}
EXPECTED_SPLIT_EVENTS_3N = {
    "train": 64,
    "validation": 14,
    "test": 15,
}
SPLIT_ORDER_3N = ["train", "validation", "test"]
KEY_COLUMNS_3N = ["event_key", "source_row_index"]
RANDOM_SEED_3N = 20260810

# Training-only quality policy. Extreme values are audited but deliberately not
# clipped: genuine fault excursions may carry useful anomaly information.
MIN_TRAIN_NUMERIC_PARSE_FRACTION_3N = 0.98
MAX_TRAIN_MISSING_FRACTION_3N = 0.30
MAX_TRAIN_FARM_MISSING_FRACTION_3N = 0.50
MIN_TRAIN_FINITE_VALUES_3N = 100
NUMERICAL_EPSILON_3N = 1e-12
CORRELATION_ALERT_3N = 0.98
CORRELATION_SAMPLE_ROWS_3N = 50_000
PSI_EPSILON_3N = 1e-6

eligible_mask_3n = (
    ROW_LABEL_MANIFEST["modeling_eligible"].fillna(False).astype(bool)
)

if int(eligible_mask_3n.sum()) != EXPECTED_ELIGIBLE_ROWS_3N:
    raise ValueError(
        "Cell 3N expects 213,537 post-deduplication eligible rows, but found "
        f"{int(eligible_mask_3n.sum()):,}."
    )

if ROW_LABEL_MANIFEST.loc[
    eligible_mask_3n, "split_assignment"
].isna().any():
    raise ValueError("At least one eligible row lacks a Cell 3M split.")

if ROW_LABEL_MANIFEST.loc[
    ~eligible_mask_3n, "split_assignment"
].notna().any():
    raise ValueError("A modeling-ineligible row has a split assignment.")

manifest_length_before_3n = len(ROW_LABEL_MANIFEST)
manifest_index_before_3n = ROW_LABEL_MANIFEST.index.copy()
manifest_hash_before_3n = pd.util.hash_pandas_object(
    ROW_LABEL_MANIFEST,
    index=True,
).to_numpy(copy=True)

eligible_manifest_3n = ROW_LABEL_MANIFEST.loc[
    eligible_mask_3n,
    [
        "farm",
        "asset_id",
        "asset_key",
        "event_key",
        "source_row_index",
        "timestamp_utc",
        "final_label",
        "split_assignment",
    ],
].copy()

for column_3n in [
    "farm",
    "asset_id",
    "asset_key",
    "event_key",
    "final_label",
    "split_assignment",
]:
    eligible_manifest_3n[column_3n] = (
        eligible_manifest_3n[column_3n].astype("string").str.strip()
    )

eligible_manifest_3n["final_label"] = (
    eligible_manifest_3n["final_label"].str.lower()
)
eligible_manifest_3n["timestamp_utc"] = pd.to_datetime(
    eligible_manifest_3n["timestamp_utc"],
    errors="coerce",
    utc=True,
)
eligible_manifest_3n["source_row_index"] = pd.to_numeric(
    eligible_manifest_3n["source_row_index"],
    errors="coerce",
)

if eligible_manifest_3n[
    [
        "farm",
        "asset_id",
        "asset_key",
        "event_key",
        "source_row_index",
        "timestamp_utc",
        "final_label",
        "split_assignment",
    ]
].isna().any().any():
    raise ValueError("Eligible manifest rows contain missing critical values.")

if not np.isclose(
    eligible_manifest_3n["source_row_index"],
    np.floor(eligible_manifest_3n["source_row_index"]),
).all():
    raise ValueError("Manifest source_row_index contains non-integer values.")

eligible_manifest_3n["source_row_index"] = eligible_manifest_3n[
    "source_row_index"
].astype("int64")

if eligible_manifest_3n[KEY_COLUMNS_3N].duplicated().any():
    raise ValueError("Eligible manifest source keys are not unique.")

if sorted(eligible_manifest_3n["farm"].astype(str).unique()) != [
    "A",
    "B",
    "C",
]:
    raise ValueError("Cell 3N expects farms A, B, and C.")

if sorted(eligible_manifest_3n["final_label"].astype(str).unique()) != [
    "anomaly",
    "normal",
]:
    raise ValueError("Cell 3N expects labels normal and anomaly.")

observed_split_rows_3n = (
    eligible_manifest_3n["split_assignment"]
    .value_counts()
    .reindex(SPLIT_ORDER_3N, fill_value=0)
    .astype(int)
    .to_dict()
)
observed_split_assets_3n = (
    eligible_manifest_3n.groupby("split_assignment")["asset_key"]
    .nunique()
    .reindex(SPLIT_ORDER_3N, fill_value=0)
    .astype(int)
    .to_dict()
)
observed_split_events_3n = (
    eligible_manifest_3n.groupby("split_assignment")["event_key"]
    .nunique()
    .reindex(SPLIT_ORDER_3N, fill_value=0)
    .astype(int)
    .to_dict()
)

if observed_split_rows_3n != EXPECTED_SPLIT_ROWS_3N:
    raise ValueError(
        "Cell 3M row assignment differs from the accepted checkpoint: "
        f"{observed_split_rows_3n}."
    )
if observed_split_assets_3n != EXPECTED_SPLIT_ASSETS_3N:
    raise ValueError(
        "Cell 3M asset assignment differs from the accepted checkpoint: "
        f"{observed_split_assets_3n}."
    )
if observed_split_events_3n != EXPECTED_SPLIT_EVENTS_3N:
    raise ValueError(
        "Cell 3M event assignment differs from the accepted checkpoint: "
        f"{observed_split_events_3n}."
    )

assets_crossing_splits_3n = int(
    (
        eligible_manifest_3n.groupby("asset_key")["split_assignment"]
        .nunique()
        .gt(1)
    ).sum()
)
events_crossing_splits_3n = int(
    (
        eligible_manifest_3n.groupby("event_key")["split_assignment"]
        .nunique()
        .gt(1)
    ).sum()
)
if assets_crossing_splits_3n or events_crossing_splits_3n:
    raise ValueError("The accepted asset/event isolation no longer holds.")

eligible_key_index_3n = pd.MultiIndex.from_frame(
    eligible_manifest_3n[KEY_COLUMNS_3N]
)
eligible_event_keys_3n = frozenset(
    eligible_manifest_3n["event_key"].astype(str).unique()
)


# -----------------------------------------------------------------------------
# 2. Resolve a raw measurement source without guessing row identity
# -----------------------------------------------------------------------------

RESERVED_COLUMN_EXACT_3N = {
    "farm",
    "farm_id",
    "asset",
    "asset_id",
    "asset_key",
    "turbine",
    "turbine_id",
    "wtg",
    "event",
    "event_id",
    "event_key",
    "source_row_index",
    "raw_id",
    "row_id",
    "index",
    "timestamp",
    "timestamp_utc",
    "datetime",
    "date",
    "date_utc",
    "time",
    "year",
    "year_utc",
    "quarter",
    "quarter_utc",
    "month",
    "month_utc",
    "week",
    "week_utc",
    "weekday",
    "weekday_number",
    "day",
    "day_utc",
    "day_of_week",
    "dayofweek",
    "hour",
    "hour_utc",
    "minute",
    "minute_utc",
    "second",
    "second_utc",
    "label",
    "class",
    "target",
    "state",
    "status_label",
    "source_event_label",
    "final_label",
    "modeling_eligible",
    "split",
    "split_assignment",
    "measurement_schema_id",
    "measurement_fingerprint",
    "file",
    "filename",
    "file_path",
    "path",
}

RESERVED_COLUMN_PATTERN_3N = re.compile(
    r"(^|_)(label|class|target|split|eligible|event|asset|turbine|farm|"
    r"timestamp|datetime|date|time|calendar|year|quarter|month|week|weekday|"
    r"dayofweek|day|hour|minute|second|index|row|raw|schema|fingerprint|"
    r"filename|filepath|path|audit|reason|action|fault_code|anomaly_flag)"
    r"($|_)",
    flags=re.IGNORECASE,
)


def slug_3n(value):
    value = re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_").lower()
    return value or "item"


def normalized_source_index_3n(series, source_name):
    numeric_3n = pd.to_numeric(series, errors="coerce")
    if numeric_3n.isna().any() or not np.isclose(
        numeric_3n, np.floor(numeric_3n)
    ).all():
        raise ValueError(
            f"{source_name} contains invalid source-row indices."
        )
    return numeric_3n.astype("int64")


def is_reserved_feature_name_3n(column):
    normalized_3n = slug_3n(column)
    return (
        normalized_3n in RESERVED_COLUMN_EXACT_3N
        or RESERVED_COLUMN_PATTERN_3N.search(normalized_3n) is not None
    )


def candidate_feature_columns_3n(frame):
    return [
        column_3n
        for column_3n in frame.columns
        if column_3n not in KEY_COLUMNS_3N
        and not is_reserved_feature_name_3n(column_3n)
    ]


def normalize_keyed_frame_3n(frame, source_name):
    if not isinstance(frame, pd.DataFrame):
        raise TypeError(f"{source_name} is not a pandas DataFrame.")
    if frame.columns.duplicated().any():
        duplicated_3n = frame.columns[frame.columns.duplicated()].tolist()
        raise ValueError(
            f"{source_name} has duplicate columns: {duplicated_3n}."
        )
    missing_keys_3n = set(KEY_COLUMNS_3N) - set(frame.columns)
    if missing_keys_3n:
        raise ValueError(
            f"{source_name} lacks source keys {sorted(missing_keys_3n)}."
        )
    output_3n = frame.copy(deep=False)
    output_3n["event_key"] = output_3n["event_key"].astype("string").str.strip()
    output_3n["source_row_index"] = normalized_source_index_3n(
        output_3n["source_row_index"], source_name
    )
    if output_3n["event_key"].isna().any() or output_3n[
        "event_key"
    ].eq("").any():
        raise ValueError(f"{source_name} contains blank event keys.")
    return output_3n


def infer_leaf_event_key_3n(path_parts, frame):
    if "event_key" in frame.columns:
        values_3n = (
            frame["event_key"].dropna().astype(str).str.strip().unique().tolist()
        )
        if len(values_3n) == 1 and values_3n[0] in eligible_event_keys_3n:
            return values_3n[0]

    for part_3n in reversed(path_parts):
        part_text_3n = str(part_3n).strip()
        if part_text_3n in eligible_event_keys_3n:
            return part_text_3n

    if {"farm", "event_id"}.issubset(frame.columns):
        farm_values_3n = frame["farm"].dropna().astype(str).str.strip().unique()
        event_values_3n = frame["event_id"].dropna().astype(str).str.strip().unique()
        if len(farm_values_3n) == 1 and len(event_values_3n) == 1:
            farm_token_3n = farm_values_3n[0].lower().replace("farm_", "")
            event_token_3n = re.sub(r"\.0$", "", event_values_3n[0])
            candidate_3n = f"farm_{farm_token_3n}_event_{event_token_3n}"
            if candidate_3n in eligible_event_keys_3n:
                return candidate_3n

    path_text_3n = "::".join(str(part_3n) for part_3n in path_parts)
    farm_match_3n = re.search(r"(?:farm[_\s-]*)?([abc])", path_text_3n, re.I)
    event_match_3n = re.search(r"(?:event[_\s-]*)?(\d+)(?!.*\d)", path_text_3n, re.I)
    if farm_match_3n and event_match_3n:
        candidate_3n = (
            f"farm_{farm_match_3n.group(1).lower()}_event_{event_match_3n.group(1)}"
        )
        if candidate_3n in eligible_event_keys_3n:
            return candidate_3n
    return None


def iter_dataframe_leaves_3n(value, path=(), depth=0, seen=None):
    if seen is None:
        seen = set()
    object_id_3n = id(value)
    if object_id_3n in seen:
        return
    seen.add(object_id_3n)

    if isinstance(value, pd.DataFrame):
        yield path, value
        return
    if isinstance(value, Mapping) and depth < 4:
        for key_3n, child_3n in value.items():
            yield from iter_dataframe_leaves_3n(
                child_3n,
                path=path + (key_3n,),
                depth=depth + 1,
                seen=seen,
            )


def materialize_event_mapping_3n(mapping, source_name):
    pieces_3n = []
    resolved_events_3n = []
    unresolved_leaf_count_3n = 0

    for path_3n, frame_3n in iter_dataframe_leaves_3n(mapping):
        event_key_3n = infer_leaf_event_key_3n(path_3n, frame_3n)
        if event_key_3n is None:
            unresolved_leaf_count_3n += 1
            continue
        if event_key_3n not in eligible_event_keys_3n:
            continue

        piece_3n = frame_3n.copy(deep=False)
        if "event_key" not in piece_3n.columns:
            piece_3n = piece_3n.assign(event_key=event_key_3n)
        else:
            piece_3n = piece_3n.assign(event_key=event_key_3n)

        if "source_row_index" not in piece_3n.columns:
            if "raw_id" in piece_3n.columns:
                source_indices_3n = piece_3n["raw_id"]
            else:
                source_indices_3n = pd.Series(
                    piece_3n.index,
                    index=piece_3n.index,
                )
            piece_3n = piece_3n.assign(
                source_row_index=normalized_source_index_3n(
                    source_indices_3n,
                    f"{source_name}:{event_key_3n}",
                ).to_numpy()
            )

        pieces_3n.append(piece_3n)
        resolved_events_3n.append(event_key_3n)

    if not pieces_3n:
        raise ValueError(
            f"{source_name} contains no event DataFrames that can be mapped "
            "to eligible event keys."
        )

    # Pandas is deprecating dtype inference from empty/all-NA concat inputs.
    # Dropping only per-piece all-NA columns retains the established behavior:
    # a column is still kept whenever any event frame contains real values.
    concat_pieces_3n = [
        piece_3n.dropna(axis=1, how="all")
        for piece_3n in pieces_3n
        if not piece_3n.empty
    ]
    if not concat_pieces_3n:
        raise ValueError(f"{source_name} contains only empty event DataFrames.")

    output_3n = pd.concat(
        concat_pieces_3n,
        axis=0,
        ignore_index=True,
        sort=False,
    )
    output_3n.attrs["resolved_event_count_3n"] = len(set(resolved_events_3n))
    output_3n.attrs["unresolved_leaf_count_3n"] = unresolved_leaf_count_3n
    return normalize_keyed_frame_3n(output_3n, source_name)


def inspect_source_candidate_3n(name, value, explicit=False):
    try:
        if isinstance(value, pd.DataFrame):
            keyed_3n = normalize_keyed_frame_3n(value, name)
            source_type_3n = "keyed_dataframe"
        elif isinstance(value, Mapping):
            keyed_3n = materialize_event_mapping_3n(value, name)
            source_type_3n = "event_dataframe_mapping"
        else:
            return None, None

        feature_candidates_3n = candidate_feature_columns_3n(keyed_3n)
        if not feature_candidates_3n:
            raise ValueError("no non-metadata feature columns were found")

        duplicated_keys_3n = int(keyed_3n[KEY_COLUMNS_3N].duplicated().sum())
        source_keys_3n = pd.MultiIndex.from_frame(keyed_3n[KEY_COLUMNS_3N])
        matched_keys_3n = int(source_keys_3n.isin(eligible_key_index_3n).sum())
        eligible_keys_matched_3n = int(eligible_key_index_3n.isin(source_keys_3n).sum())
        exact_coverage_3n = (
            eligible_keys_matched_3n == EXPECTED_ELIGIBLE_ROWS_3N
            and duplicated_keys_3n == 0
        )

        record_3n = {
            "source_name": str(name),
            "source_type": source_type_3n,
            "explicit": bool(explicit),
            "source_rows": int(len(keyed_3n)),
            "candidate_feature_columns": int(len(feature_candidates_3n)),
            "eligible_keys_matched": eligible_keys_matched_3n,
            "source_rows_matching_eligible": matched_keys_3n,
            "duplicated_source_keys": duplicated_keys_3n,
            "exact_eligible_coverage": bool(exact_coverage_3n),
            "inspection_error": pd.NA,
        }
        return record_3n, keyed_3n
    except Exception as error_3n:
        if explicit:
            raise
        record_3n = {
            "source_name": str(name),
            "source_type": type(value).__name__,
            "explicit": False,
            "source_rows": int(len(value)) if hasattr(value, "__len__") else pd.NA,
            "candidate_feature_columns": pd.NA,
            "eligible_keys_matched": 0,
            "source_rows_matching_eligible": 0,
            "duplicated_source_keys": pd.NA,
            "exact_eligible_coverage": False,
            "inspection_error": f"{type(error_3n).__name__}: {error_3n}",
        }
        return record_3n, None


source_candidate_records_3n = []
source_candidate_frames_3n = {}
source_resolution_mode_3n = "automatic"
allow_automatic_source_3n = bool(
    globals().get("ALLOW_AUTOMATIC_FEATURE_SOURCE_3N", False)
)

if "FEATURE_SOURCE_3N" in globals():
    source_resolution_mode_3n = "explicit"
    source_record_3n, source_frame_3n = inspect_source_candidate_3n(
        "FEATURE_SOURCE_3N",
        globals()["FEATURE_SOURCE_3N"],
        explicit=True,
    )
    source_candidate_records_3n.append(source_record_3n)
    source_candidate_frames_3n["FEATURE_SOURCE_3N"] = source_frame_3n
else:
    globals_snapshot_3n = list(globals().items())
    excluded_source_names_3n = {
        "ROW_LABEL_MANIFEST",
        "eligible_manifest_3n",
        "ELIGIBLE_MEASUREMENT_INDEX",
        "EXACT_SAME_LABEL_CANONICAL_RETENTION_AUDIT",
        "MEASUREMENT_SOURCE_3N",
        "RAW_ELIGIBLE_FEATURES_3N",
        "PREPROCESSED_FEATURE_MATRIX_3N",
        "X_TRAIN_3N",
        "X_VALIDATION_3N",
        "X_TEST_3N",
    }

    for name_3n, value_3n in globals_snapshot_3n:
        if (
            name_3n in excluded_source_names_3n
            or name_3n.startswith("_")
            or name_3n.lower().endswith(("_3l", "_3m", "_3n"))
        ):
            continue
        if isinstance(value_3n, pd.DataFrame) and set(KEY_COLUMNS_3N).issubset(
            value_3n.columns
        ):
            record_3n, frame_3n = inspect_source_candidate_3n(
                name_3n, value_3n
            )
            if record_3n is not None:
                source_candidate_records_3n.append(record_3n)
                if frame_3n is not None:
                    source_candidate_frames_3n[name_3n] = frame_3n

    mapping_name_pattern_3n = re.compile(
        r"event|data|frame|raw|measurement|source|farm", re.IGNORECASE
    )
    for name_3n, value_3n in globals_snapshot_3n:
        if (
            name_3n.startswith("_")
            or name_3n.lower().endswith(("_3l", "_3m", "_3n"))
            or not isinstance(value_3n, Mapping)
            or not mapping_name_pattern_3n.search(name_3n)
        ):
            continue
        record_3n, frame_3n = inspect_source_candidate_3n(name_3n, value_3n)
        if record_3n is not None:
            source_candidate_records_3n.append(record_3n)
            if frame_3n is not None:
                source_candidate_frames_3n[name_3n] = frame_3n


FEATURE_SOURCE_CANDIDATES_3N = pd.DataFrame(source_candidate_records_3n)
if FEATURE_SOURCE_CANDIDATES_3N.empty:
    raise NameError(
        "Cell 3N could not find raw sensor measurements. Define "
        "FEATURE_SOURCE_3N as either a DataFrame containing event_key, "
        "source_row_index, and sensor columns, or a mapping from event_key "
        "to raw event DataFrames; then rerun Cell 3N."
    )

if "FEATURE_SOURCE_3N" not in globals() and not allow_automatic_source_3n:
    compact_candidates_3n = FEATURE_SOURCE_CANDIDATES_3N.loc[
        :,
        [
            "source_name",
            "source_type",
            "source_rows",
            "candidate_feature_columns",
            "eligible_keys_matched",
            "duplicated_source_keys",
            "exact_eligible_coverage",
            "inspection_error",
        ],
    ].sort_values(
        ["exact_eligible_coverage", "candidate_feature_columns"],
        ascending=[False, False],
        na_position="last",
    )
    raise NameError(
        "Cell 3N found possible sources but will not guess which object contains "
        "raw sensors. Review FEATURE_SOURCE_CANDIDATES_3N, then define:\n"
        "FEATURE_SOURCE_3N = <raw keyed DataFrame or raw event-frame mapping>\n"
        "Optionally define FEATURE_COLUMNS_3N as the physical sensor columns, "
        "then rerun Cell 3N.\n\n"
        + compact_candidates_3n.to_string(index=False)
    )

valid_sources_3n = FEATURE_SOURCE_CANDIDATES_3N.loc[
    FEATURE_SOURCE_CANDIDATES_3N["exact_eligible_coverage"].fillna(False)
].copy()
if valid_sources_3n.empty:
    compact_candidates_3n = FEATURE_SOURCE_CANDIDATES_3N.loc[
        :,
        [
            "source_name",
            "source_type",
            "source_rows",
            "candidate_feature_columns",
            "eligible_keys_matched",
            "duplicated_source_keys",
            "inspection_error",
        ],
    ]
    raise ValueError(
        "No measurement source covers every eligible manifest key exactly "
        "once. Inspect FEATURE_SOURCE_CANDIDATES_3N. Preferred repair:\n"
        "FEATURE_SOURCE_3N = <keyed measurement DataFrame or event mapping>\n\n"
        + compact_candidates_3n.to_string(index=False)
    )

valid_sources_3n = valid_sources_3n.sort_values(
    ["explicit", "candidate_feature_columns", "source_rows", "source_name"],
    ascending=[False, False, True, True],
).reset_index(drop=True)

selected_source_name_3n = str(valid_sources_3n.iloc[0]["source_name"])
selected_source_type_3n = str(valid_sources_3n.iloc[0]["source_type"])
MEASUREMENT_SOURCE_3N = source_candidate_frames_3n[selected_source_name_3n]

if MEASUREMENT_SOURCE_3N[KEY_COLUMNS_3N].duplicated().any():
    raise ValueError("The selected measurement source has duplicate source keys.")

if "FEATURE_COLUMNS_3N" in globals() and globals()["FEATURE_COLUMNS_3N"] is not None:
    requested_feature_columns_3n = list(globals()["FEATURE_COLUMNS_3N"])
    if len(requested_feature_columns_3n) != len(set(requested_feature_columns_3n)):
        raise ValueError("FEATURE_COLUMNS_3N contains duplicates.")
    missing_requested_features_3n = set(requested_feature_columns_3n) - set(
        MEASUREMENT_SOURCE_3N.columns
    )
    if missing_requested_features_3n:
        raise ValueError(
            "FEATURE_COLUMNS_3N is missing from the measurement source: "
            f"{sorted(missing_requested_features_3n)}"
        )
    leaking_requested_features_3n = [
        column_3n
        for column_3n in requested_feature_columns_3n
        if is_reserved_feature_name_3n(column_3n)
    ]
    if leaking_requested_features_3n:
        raise ValueError(
            "FEATURE_COLUMNS_3N includes identity/label/split-like columns: "
            f"{leaking_requested_features_3n}"
        )
    candidate_features_3n = requested_feature_columns_3n
    feature_column_mode_3n = "explicit"
else:
    candidate_features_3n = candidate_feature_columns_3n(MEASUREMENT_SOURCE_3N)
    feature_column_mode_3n = "automatic_metadata_exclusion"

if not candidate_features_3n:
    raise ValueError(
        "No physical sensor candidates remain after identity, label, split, "
        "timestamp, and calendar metadata exclusion. FEATURE_SOURCE_3N is "
        "probably a derived audit/visualization frame rather than raw measurements."
    )

source_subset_3n = MEASUREMENT_SOURCE_3N.loc[
    :,
    KEY_COLUMNS_3N + candidate_features_3n,
].copy()

RAW_ELIGIBLE_FEATURES_3N = eligible_manifest_3n.merge(
    source_subset_3n,
    on=KEY_COLUMNS_3N,
    how="left",
    validate="one_to_one",
    indicator="_measurement_match_3n",
)

unmatched_eligible_rows_3n = int(
    RAW_ELIGIBLE_FEATURES_3N["_measurement_match_3n"].ne("both").sum()
)
if unmatched_eligible_rows_3n:
    raise ValueError(
        f"{unmatched_eligible_rows_3n:,} eligible rows lack raw measurements."
    )
RAW_ELIGIBLE_FEATURES_3N = RAW_ELIGIBLE_FEATURES_3N.drop(
    columns="_measurement_match_3n"
)

if len(RAW_ELIGIBLE_FEATURES_3N) != EXPECTED_ELIGIBLE_ROWS_3N:
    raise ValueError("Measurement join did not conserve eligible rows.")


# -----------------------------------------------------------------------------
# 3. Fit feature-quality decisions using training assets only
# -----------------------------------------------------------------------------

train_mask_3n = RAW_ELIGIBLE_FEATURES_3N["split_assignment"].eq("train")
validation_mask_3n = RAW_ELIGIBLE_FEATURES_3N[
    "split_assignment"
].eq("validation")
test_mask_3n = RAW_ELIGIBLE_FEATURES_3N["split_assignment"].eq("test")

if int(train_mask_3n.sum()) != EXPECTED_SPLIT_ROWS_3N["train"]:
    raise ValueError("Training-row count changed during the measurement join.")

numeric_feature_data_3n = pd.DataFrame(
    index=RAW_ELIGIBLE_FEATURES_3N.index
)
feature_quality_records_3n = []

train_farms_3n = sorted(
    RAW_ELIGIBLE_FEATURES_3N.loc[train_mask_3n, "farm"].astype(str).unique()
)
if train_farms_3n != ["A", "B", "C"]:
    raise ValueError("Training assets do not cover all three farms.")

for feature_3n in candidate_features_3n:
    original_3n = RAW_ELIGIBLE_FEATURES_3N[feature_3n]
    numeric_3n = pd.to_numeric(original_3n, errors="coerce").astype("float64")
    numeric_3n = numeric_3n.mask(~np.isfinite(numeric_3n), np.nan)
    numeric_feature_data_3n[feature_3n] = numeric_3n

    train_original_3n = original_3n.loc[train_mask_3n]
    train_values_3n = numeric_3n.loc[train_mask_3n]
    original_nonmissing_3n = train_original_3n.notna()
    finite_train_3n = train_values_3n.dropna()
    parse_denominator_3n = int(original_nonmissing_3n.sum())
    parse_numerator_3n = int(
        (original_nonmissing_3n & train_values_3n.notna()).sum()
    )
    parse_fraction_3n = (
        parse_numerator_3n / parse_denominator_3n
        if parse_denominator_3n
        else 0.0
    )
    train_missing_fraction_3n = float(train_values_3n.isna().mean())

    farm_missing_fractions_3n = {}
    for farm_3n in train_farms_3n:
        farm_mask_3n = train_mask_3n & RAW_ELIGIBLE_FEATURES_3N[
            "farm"
        ].eq(farm_3n)
        farm_missing_fractions_3n[farm_3n] = float(
            numeric_3n.loc[farm_mask_3n].isna().mean()
        )

    finite_count_3n = int(len(finite_train_3n))
    unique_count_3n = int(finite_train_3n.nunique(dropna=True))

    if finite_count_3n:
        q1_3n = float(finite_train_3n.quantile(0.25))
        median_3n = float(finite_train_3n.median())
        q3_3n = float(finite_train_3n.quantile(0.75))
        iqr_3n = float(q3_3n - q1_3n)
        mean_3n = float(finite_train_3n.mean())
        std_3n = float(finite_train_3n.std(ddof=0))
        min_3n = float(finite_train_3n.min())
        max_3n = float(finite_train_3n.max())
        mad_3n = float((finite_train_3n - median_3n).abs().median())
    else:
        q1_3n = median_3n = q3_3n = iqr_3n = np.nan
        mean_3n = std_3n = min_3n = max_3n = mad_3n = np.nan

    decision_3n = "retain"
    reason_3n = "passed_training_only_quality_gates"
    scale_source_3n = "training_iqr"
    scale_3n = iqr_3n

    if parse_fraction_3n < MIN_TRAIN_NUMERIC_PARSE_FRACTION_3N:
        decision_3n = "exclude"
        reason_3n = "insufficient_numeric_parse_fraction_in_training"
    elif finite_count_3n < MIN_TRAIN_FINITE_VALUES_3N:
        decision_3n = "exclude"
        reason_3n = "insufficient_finite_training_values"
    elif train_missing_fraction_3n > MAX_TRAIN_MISSING_FRACTION_3N:
        decision_3n = "exclude"
        reason_3n = "excessive_training_missingness"
    elif max(farm_missing_fractions_3n.values()) > (
        MAX_TRAIN_FARM_MISSING_FRACTION_3N
    ):
        decision_3n = "exclude"
        reason_3n = "excessive_missingness_in_a_training_farm"
    elif unique_count_3n <= 1:
        decision_3n = "exclude"
        reason_3n = "constant_in_training"
    elif not np.isfinite(scale_3n) or abs(scale_3n) <= NUMERICAL_EPSILON_3N:
        scale_3n = std_3n
        scale_source_3n = "training_standard_deviation_fallback"
        if not np.isfinite(scale_3n) or abs(scale_3n) <= NUMERICAL_EPSILON_3N:
            decision_3n = "exclude"
            reason_3n = "near_constant_in_training"

    if decision_3n == "exclude":
        scale_source_3n = pd.NA
        scale_3n = np.nan

    feature_quality_records_3n.append(
        {
            "feature": str(feature_3n),
            "decision": decision_3n,
            "decision_reason": reason_3n,
            "fit_split": "train",
            "train_rows": int(train_mask_3n.sum()),
            "train_original_nonmissing": parse_denominator_3n,
            "train_numeric_parse_fraction": parse_fraction_3n,
            "train_finite_values": finite_count_3n,
            "train_missing_fraction": train_missing_fraction_3n,
            "train_farm_A_missing_fraction": farm_missing_fractions_3n["A"],
            "train_farm_B_missing_fraction": farm_missing_fractions_3n["B"],
            "train_farm_C_missing_fraction": farm_missing_fractions_3n["C"],
            "train_unique_values": unique_count_3n,
            "train_min": min_3n,
            "train_q1": q1_3n,
            "train_median": median_3n,
            "train_q3": q3_3n,
            "train_max": max_3n,
            "train_mean": mean_3n,
            "train_std": std_3n,
            "train_mad": mad_3n,
            "imputation_method": (
                "training_median" if decision_3n == "retain" else pd.NA
            ),
            "imputation_value": (
                median_3n if decision_3n == "retain" else np.nan
            ),
            "scaling_method": (
                "robust_center_and_scale" if decision_3n == "retain" else pd.NA
            ),
            "center_value": (
                median_3n if decision_3n == "retain" else np.nan
            ),
            "scale_value": scale_3n,
            "scale_source": scale_source_3n,
            "outlier_clipping": "none_fault_excursions_preserved",
        }
    )


FEATURE_QUALITY_AUDIT_3N = pd.DataFrame(feature_quality_records_3n).sort_values(
    ["decision", "feature"],
    ascending=[False, True],
).reset_index(drop=True)

FEATURE_NAMES_3N = FEATURE_QUALITY_AUDIT_3N.loc[
    FEATURE_QUALITY_AUDIT_3N["decision"].eq("retain"), "feature"
].astype(str).tolist()
EXCLUDED_FEATURES_3N = FEATURE_QUALITY_AUDIT_3N.loc[
    FEATURE_QUALITY_AUDIT_3N["decision"].eq("exclude"), "feature"
].astype(str).tolist()

if not FEATURE_NAMES_3N:
    raise ValueError(
        "No sensor feature passed the training-only quality gates. Inspect "
        "FEATURE_QUALITY_AUDIT_3N; do not relax gates using validation/test."
    )

PREPROCESSING_PARAMETERS_3N = FEATURE_QUALITY_AUDIT_3N.loc[
    FEATURE_QUALITY_AUDIT_3N["decision"].eq("retain"),
    [
        "feature",
        "fit_split",
        "imputation_method",
        "imputation_value",
        "scaling_method",
        "center_value",
        "scale_value",
        "scale_source",
        "outlier_clipping",
    ],
].reset_index(drop=True)

if not PREPROCESSING_PARAMETERS_3N["fit_split"].eq("train").all():
    raise ValueError("A preprocessing parameter was not fitted on training data.")
if not np.isfinite(
    PREPROCESSING_PARAMETERS_3N[
        ["imputation_value", "center_value", "scale_value"]
    ].to_numpy(dtype=float)
).all():
    raise ValueError("A retained feature has a non-finite fitted parameter.")
if PREPROCESSING_PARAMETERS_3N["scale_value"].abs().le(
    NUMERICAL_EPSILON_3N
).any():
    raise ValueError("A retained feature has a zero preprocessing scale.")


# -----------------------------------------------------------------------------
# 4. Freeze the training transform and apply without refitting
# -----------------------------------------------------------------------------

parameter_lookup_3n = PREPROCESSING_PARAMETERS_3N.set_index("feature")


def transform_features_3n(frame):
    """Apply the already-fitted Cell 3N transform; never refits parameters."""
    missing_features_3n = set(FEATURE_NAMES_3N) - set(frame.columns)
    if missing_features_3n:
        raise ValueError(
            "Transform input is missing retained features: "
            f"{sorted(missing_features_3n)}"
        )
    transformed_3n = pd.DataFrame(index=frame.index)
    for feature_3n in FEATURE_NAMES_3N:
        values_3n = pd.to_numeric(frame[feature_3n], errors="coerce").astype(
            "float64"
        )
        values_3n = values_3n.mask(~np.isfinite(values_3n), np.nan)
        params_3n = parameter_lookup_3n.loc[feature_3n]
        values_3n = values_3n.fillna(float(params_3n["imputation_value"]))
        values_3n = (
            values_3n - float(params_3n["center_value"])
        ) / float(params_3n["scale_value"])
        transformed_3n[feature_3n] = values_3n.astype("float32")
    return transformed_3n


transformed_features_3n = transform_features_3n(
    RAW_ELIGIBLE_FEATURES_3N[FEATURE_NAMES_3N]
)

if transformed_features_3n.isna().any().any():
    raise ValueError("Missing values remain after the frozen transform.")
if not np.isfinite(transformed_features_3n.to_numpy(dtype="float32")).all():
    raise ValueError("Non-finite values remain after the frozen transform.")

metadata_columns_3n = [
    "farm",
    "asset_id",
    "asset_key",
    "event_key",
    "source_row_index",
    "timestamp_utc",
    "final_label",
    "split_assignment",
]

PREPROCESSED_FEATURE_MATRIX_3N = pd.concat(
    [
        RAW_ELIGIBLE_FEATURES_3N[metadata_columns_3n].reset_index(drop=True),
        transformed_features_3n.reset_index(drop=True),
    ],
    axis=1,
)

label_map_3n = {"normal": 0, "anomaly": 1}


def split_outputs_3n(split_name):
    split_mask_3n = PREPROCESSED_FEATURE_MATRIX_3N[
        "split_assignment"
    ].eq(split_name)
    x_3n = PREPROCESSED_FEATURE_MATRIX_3N.loc[
        split_mask_3n, FEATURE_NAMES_3N
    ].reset_index(drop=True)
    y_3n = (
        PREPROCESSED_FEATURE_MATRIX_3N.loc[
            split_mask_3n, "final_label"
        ]
        .map(label_map_3n)
        .astype("int8")
        .reset_index(drop=True)
    )
    meta_3n = PREPROCESSED_FEATURE_MATRIX_3N.loc[
        split_mask_3n, metadata_columns_3n
    ].reset_index(drop=True)
    return x_3n, y_3n, meta_3n


X_TRAIN_3N, Y_TRAIN_3N, META_TRAIN_3N = split_outputs_3n("train")
X_VALIDATION_3N, Y_VALIDATION_3N, META_VALIDATION_3N = split_outputs_3n(
    "validation"
)
X_TEST_3N, Y_TEST_3N, META_TEST_3N = split_outputs_3n("test")

if [len(X_TRAIN_3N), len(X_VALIDATION_3N), len(X_TEST_3N)] != [
    EXPECTED_SPLIT_ROWS_3N[name_3n] for name_3n in SPLIT_ORDER_3N
]:
    raise ValueError("Preprocessed split outputs do not conserve rows.")

PREPROCESSOR_STATE_3N = {
    "cell": "3N",
    "version": 2,
    "fit_split": "train",
    "random_seed": RANDOM_SEED_3N,
    "source_name": selected_source_name_3n,
    "source_type": selected_source_type_3n,
    "feature_column_mode": feature_column_mode_3n,
    "candidate_features": [str(value_3n) for value_3n in candidate_features_3n],
    "retained_features": FEATURE_NAMES_3N,
    "excluded_features": EXCLUDED_FEATURES_3N,
    "label_map": label_map_3n,
    "imputation": "training median",
    "scaling": "training median and IQR; training std fallback",
    "outlier_clipping": "none",
    "test_policy": (
        "frozen transform only; no test feature selection, drift statistics, "
        "ranking, plotting, model selection, or threshold selection"
    ),
    "parameters": PREPROCESSING_PARAMETERS_3N.to_dict(orient="records"),
}


# -----------------------------------------------------------------------------
# 5. Train/validation diagnostics; keep test distributions sealed
# -----------------------------------------------------------------------------

SOURCE_RESOLUTION_AUDIT_3N = pd.DataFrame(
    [
        {
            "resolution_mode": source_resolution_mode_3n,
            "selected_source_name": selected_source_name_3n,
            "selected_source_type": selected_source_type_3n,
            "source_rows": int(len(MEASUREMENT_SOURCE_3N)),
            "eligible_keys_required": EXPECTED_ELIGIBLE_ROWS_3N,
            "eligible_keys_matched": EXPECTED_ELIGIBLE_ROWS_3N,
            "unmatched_eligible_rows": unmatched_eligible_rows_3n,
            "feature_column_mode": feature_column_mode_3n,
            "candidate_features": len(candidate_features_3n),
            "retained_features": len(FEATURE_NAMES_3N),
            "excluded_features": len(EXCLUDED_FEATURES_3N),
        }
    ]
)

transform_audit_records_3n = []
for split_3n in SPLIT_ORDER_3N:
    mask_3n = RAW_ELIGIBLE_FEATURES_3N["split_assignment"].eq(split_3n)
    transformed_split_3n = transformed_features_3n.loc[mask_3n]
    if split_3n == "test":
        raw_missing_cells_3n = pd.NA
        raw_missing_fraction_3n = pd.NA
        transformed_mean_abs_3n = pd.NA
        transformed_std_mean_3n = pd.NA
        distribution_reporting_3n = "sealed"
    else:
        raw_selected_3n = numeric_feature_data_3n.loc[
            mask_3n, FEATURE_NAMES_3N
        ]
        raw_missing_cells_3n = int(raw_selected_3n.isna().sum().sum())
        raw_missing_fraction_3n = float(raw_selected_3n.isna().to_numpy().mean())
        transformed_mean_abs_3n = float(
            transformed_split_3n.mean().abs().mean()
        )
        transformed_std_mean_3n = float(
            transformed_split_3n.std(ddof=0).mean()
        )
        distribution_reporting_3n = "reported"

    transform_audit_records_3n.append(
        {
            "split_assignment": split_3n,
            "rows": int(mask_3n.sum()),
            "features": len(FEATURE_NAMES_3N),
            "raw_missing_cells": raw_missing_cells_3n,
            "raw_missing_fraction": raw_missing_fraction_3n,
            "post_transform_missing_cells": int(
                transformed_split_3n.isna().sum().sum()
            ),
            "post_transform_nonfinite_cells": int(
                (~np.isfinite(transformed_split_3n.to_numpy())).sum()
            ),
            "transformed_mean_absolute_mean": transformed_mean_abs_3n,
            "transformed_mean_feature_std": transformed_std_mean_3n,
            "parameters_fitted_on": "train",
            "distribution_reporting": distribution_reporting_3n,
        }
    )

SPLIT_TRANSFORM_AUDIT_3N = pd.DataFrame(transform_audit_records_3n)

TRAIN_TRANSFORMED_FEATURE_SUMMARY_3N = pd.DataFrame(
    {
        "feature": FEATURE_NAMES_3N,
        "mean": X_TRAIN_3N.mean().reindex(FEATURE_NAMES_3N).to_numpy(),
        "std": X_TRAIN_3N.std(ddof=0).reindex(FEATURE_NAMES_3N).to_numpy(),
        "min": X_TRAIN_3N.min().reindex(FEATURE_NAMES_3N).to_numpy(),
        "q1": X_TRAIN_3N.quantile(0.25).reindex(FEATURE_NAMES_3N).to_numpy(),
        "median": X_TRAIN_3N.median().reindex(FEATURE_NAMES_3N).to_numpy(),
        "q3": X_TRAIN_3N.quantile(0.75).reindex(FEATURE_NAMES_3N).to_numpy(),
        "max": X_TRAIN_3N.max().reindex(FEATURE_NAMES_3N).to_numpy(),
    }
)


def population_stability_index_3n(train_values, comparison_values, bins=10):
    train_values_3n = np.asarray(train_values, dtype=float)
    comparison_values_3n = np.asarray(comparison_values, dtype=float)
    train_values_3n = train_values_3n[np.isfinite(train_values_3n)]
    comparison_values_3n = comparison_values_3n[
        np.isfinite(comparison_values_3n)
    ]
    if not len(train_values_3n) or not len(comparison_values_3n):
        return np.nan, 0

    edges_3n = np.unique(
        np.quantile(train_values_3n, np.linspace(0.0, 1.0, bins + 1))
    )
    if len(edges_3n) < 3:
        return 0.0, max(len(edges_3n) - 1, 1)
    edges_3n[0] = -np.inf
    edges_3n[-1] = np.inf
    train_counts_3n, _ = np.histogram(train_values_3n, bins=edges_3n)
    comparison_counts_3n, _ = np.histogram(
        comparison_values_3n, bins=edges_3n
    )
    train_props_3n = train_counts_3n / train_counts_3n.sum()
    comparison_props_3n = comparison_counts_3n / comparison_counts_3n.sum()
    train_props_3n = np.clip(train_props_3n, PSI_EPSILON_3N, None)
    comparison_props_3n = np.clip(
        comparison_props_3n, PSI_EPSILON_3N, None
    )
    psi_3n = float(
        np.sum(
            (comparison_props_3n - train_props_3n)
            * np.log(comparison_props_3n / train_props_3n)
        )
    )
    return psi_3n, len(edges_3n) - 1


validation_shift_records_3n = []
for feature_3n in FEATURE_NAMES_3N:
    psi_3n, bins_3n = population_stability_index_3n(
        X_TRAIN_3N[feature_3n],
        X_VALIDATION_3N[feature_3n],
    )
    if not np.isfinite(psi_3n):
        shift_band_3n = "not_estimable"
    elif psi_3n < 0.10:
        shift_band_3n = "small"
    elif psi_3n < 0.25:
        shift_band_3n = "moderate"
    else:
        shift_band_3n = "large"
    validation_shift_records_3n.append(
        {
            "feature": feature_3n,
            "comparison": "validation_vs_train",
            "train_defined_bins": bins_3n,
            "population_stability_index": psi_3n,
            "shift_band": shift_band_3n,
            "used_for_feature_exclusion": False,
            "test_distribution_accessed": False,
        }
    )

VALIDATION_SHIFT_AUDIT_3N = pd.DataFrame(
    validation_shift_records_3n
).sort_values(
    "population_stability_index",
    ascending=False,
    na_position="last",
).reset_index(drop=True)

correlation_sample_n_3n = min(CORRELATION_SAMPLE_ROWS_3N, len(X_TRAIN_3N))
if correlation_sample_n_3n < len(X_TRAIN_3N):
    correlation_sample_3n = X_TRAIN_3N.sample(
        n=correlation_sample_n_3n,
        random_state=RANDOM_SEED_3N,
        replace=False,
    )
else:
    correlation_sample_3n = X_TRAIN_3N

TRAIN_SPEARMAN_CORRELATION_3N = correlation_sample_3n.corr(method="spearman")
correlation_pair_records_3n = []
for index_a_3n, feature_a_3n in enumerate(FEATURE_NAMES_3N):
    for feature_b_3n in FEATURE_NAMES_3N[index_a_3n + 1 :]:
        correlation_3n = float(
            TRAIN_SPEARMAN_CORRELATION_3N.loc[feature_a_3n, feature_b_3n]
        )
        correlation_pair_records_3n.append(
            {
                "feature_a": feature_a_3n,
                "feature_b": feature_b_3n,
                "train_spearman_correlation": correlation_3n,
                "absolute_correlation": abs(correlation_3n),
                "high_correlation_alert": abs(correlation_3n)
                >= CORRELATION_ALERT_3N,
                "used_for_feature_exclusion": False,
                "sample_rows": correlation_sample_n_3n,
            }
        )

TRAIN_CORRELATION_PAIR_AUDIT_3N = pd.DataFrame(correlation_pair_records_3n)
if not TRAIN_CORRELATION_PAIR_AUDIT_3N.empty:
    TRAIN_CORRELATION_PAIR_AUDIT_3N = (
        TRAIN_CORRELATION_PAIR_AUDIT_3N.sort_values(
            "absolute_correlation", ascending=False
        ).reset_index(drop=True)
    )
else:
    TRAIN_CORRELATION_PAIR_AUDIT_3N = pd.DataFrame(
        columns=[
            "feature_a",
            "feature_b",
            "train_spearman_correlation",
            "absolute_correlation",
            "high_correlation_alert",
            "used_for_feature_exclusion",
            "sample_rows",
        ]
    )


# -----------------------------------------------------------------------------
# 6. Mutation boundary, leakage discipline, and conservation audit
# -----------------------------------------------------------------------------

manifest_hash_after_3n = pd.util.hash_pandas_object(
    ROW_LABEL_MANIFEST,
    index=True,
).to_numpy(copy=True)
manifest_values_changed_3n = int(
    np.count_nonzero(manifest_hash_before_3n != manifest_hash_after_3n)
)

if len(ROW_LABEL_MANIFEST) != manifest_length_before_3n:
    raise ValueError("Cell 3N changed the manifest row count.")
if not ROW_LABEL_MANIFEST.index.equals(manifest_index_before_3n):
    raise ValueError("Cell 3N changed the manifest index.")
if manifest_values_changed_3n:
    raise ValueError("Cell 3N changed ROW_LABEL_MANIFEST values.")

test_used_for_fitting_3n = int(
    PREPROCESSING_PARAMETERS_3N["fit_split"].ne("train").sum()
)
test_distribution_statistics_exported_3n = int(
    SPLIT_TRANSFORM_AUDIT_3N.loc[
        SPLIT_TRANSFORM_AUDIT_3N["split_assignment"].eq("test"),
        [
            "raw_missing_cells",
            "raw_missing_fraction",
            "transformed_mean_absolute_mean",
            "transformed_mean_feature_std",
        ],
    ].notna().sum().sum()
)

PREPROCESSING_LEAKAGE_AUDIT_3N = pd.DataFrame(
    [
        {
            "check": "eligible_row_conservation",
            "observed": len(PREPROCESSED_FEATURE_MATRIX_3N),
            "required": EXPECTED_ELIGIBLE_ROWS_3N,
        },
        {
            "check": "unmatched_eligible_measurement_keys",
            "observed": unmatched_eligible_rows_3n,
            "required": 0,
        },
        {
            "check": "post_transform_missing_cells",
            "observed": int(transformed_features_3n.isna().sum().sum()),
            "required": 0,
        },
        {
            "check": "post_transform_nonfinite_cells",
            "observed": int(
                (~np.isfinite(transformed_features_3n.to_numpy())).sum()
            ),
            "required": 0,
        },
        {
            "check": "parameters_not_fitted_on_train",
            "observed": test_used_for_fitting_3n,
            "required": 0,
        },
        {
            "check": "test_distribution_statistics_exported",
            "observed": test_distribution_statistics_exported_3n,
            "required": 0,
        },
        {
            "check": "assets_crossing_splits",
            "observed": assets_crossing_splits_3n,
            "required": 0,
        },
        {
            "check": "events_crossing_splits",
            "observed": events_crossing_splits_3n,
            "required": 0,
        },
        {
            "check": "manifest_rows_deleted",
            "observed": manifest_length_before_3n - len(ROW_LABEL_MANIFEST),
            "required": 0,
        },
        {
            "check": "manifest_values_changed",
            "observed": manifest_values_changed_3n,
            "required": 0,
        },
    ]
)
PREPROCESSING_LEAKAGE_AUDIT_3N["passed"] = (
    PREPROCESSING_LEAKAGE_AUDIT_3N["observed"]
    == PREPROCESSING_LEAKAGE_AUDIT_3N["required"]
)

if not PREPROCESSING_LEAKAGE_AUDIT_3N["passed"].all():
    failed_checks_3n = PREPROCESSING_LEAKAGE_AUDIT_3N.loc[
        ~PREPROCESSING_LEAKAGE_AUDIT_3N["passed"]
    ]
    raise ValueError(
        "Cell 3N leakage/conservation checks failed:\n"
        + failed_checks_3n.to_string(index=False)
    )


# -----------------------------------------------------------------------------
# 7. Export paper-ready tables, figures, registries, and frozen state
# -----------------------------------------------------------------------------

OUTPUT_ROOT_3N = Path(
    globals().get("OUTPUT_ROOT_3L", Path("paper_visuals_3l"))
) / "preprocessing_3n"
TABLE_DIR_3N = OUTPUT_ROOT_3N / "tables"
FIGURE_DIR_3N = OUTPUT_ROOT_3N / "figures"
for directory_3n in [OUTPUT_ROOT_3N, TABLE_DIR_3N, FIGURE_DIR_3N]:
    directory_3n.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.size": 9,
        "axes.titlesize": 11,
        "axes.labelsize": 9,
        "legend.fontsize": 8,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

SPLIT_COLORS_3N = {"train": "#2E6F9E", "validation": "#E69F00"}
DECISION_COLORS_3N = {"retain": "#2A9D8F", "exclude": "#C94C4C"}
figure_registry_records_3n = []


def save_figure_3n(figure, figure_id, title, claim_supported, source_objects):
    stem_3n = f"{figure_id}_{slug_3n(title)}"
    png_path_3n = FIGURE_DIR_3N / f"{stem_3n}.png"
    pdf_path_3n = FIGURE_DIR_3N / f"{stem_3n}.pdf"
    figure.savefig(png_path_3n, bbox_inches="tight", dpi=300)
    figure.savefig(pdf_path_3n, bbox_inches="tight")
    plt.close(figure)
    figure_registry_records_3n.append(
        {
            "figure_id": figure_id,
            "title": title,
            "claim_supported": claim_supported,
            "source_objects": source_objects,
            "fit_or_comparison_scope": "training and validation only; test sealed",
            "png_path": str(png_path_3n),
            "pdf_path": str(pdf_path_3n),
        }
    )


# F3N01 — feature decision counts.
decision_counts_3n = (
    FEATURE_QUALITY_AUDIT_3N["decision"]
    .value_counts()
    .reindex(["retain", "exclude"], fill_value=0)
)
figure_3n, axis_3n = plt.subplots(figsize=(5.8, 4.0))
bars_3n = axis_3n.bar(
    decision_counts_3n.index,
    decision_counts_3n.values,
    color=[DECISION_COLORS_3N[value_3n] for value_3n in decision_counts_3n.index],
)
axis_3n.bar_label(bars_3n, padding=3)
axis_3n.set_ylabel("Candidate sensor features")
axis_3n.set_title("Training-only feature-quality decisions")
axis_3n.grid(axis="y", alpha=0.2)
save_figure_3n(
    figure_3n,
    "F3N01",
    "Training-only feature quality decisions",
    "Documents how many candidate sensors passed predeclared training-only gates.",
    "FEATURE_QUALITY_AUDIT_3N",
)


# F3N02 — training missingness.
missing_plot_3n = FEATURE_QUALITY_AUDIT_3N.sort_values(
    "train_missing_fraction", ascending=True
).copy()
figure_height_3n = max(4.5, min(16.0, 0.28 * len(missing_plot_3n) + 1.8))
figure_3n, axis_3n = plt.subplots(figsize=(8.2, figure_height_3n))
axis_3n.barh(
    np.arange(len(missing_plot_3n)),
    100.0 * missing_plot_3n["train_missing_fraction"],
    color=[DECISION_COLORS_3N[value_3n] for value_3n in missing_plot_3n["decision"]],
)
axis_3n.axvline(
    100.0 * MAX_TRAIN_MISSING_FRACTION_3N,
    color="#333333",
    linestyle="--",
    linewidth=1.0,
    label="Global exclusion threshold",
)
axis_3n.set_yticks(np.arange(len(missing_plot_3n)))
axis_3n.set_yticklabels(missing_plot_3n["feature"])
axis_3n.set_xlabel("Missing training values (%)")
axis_3n.set_title("Training missingness by candidate sensor")
axis_3n.legend(loc="lower right")
axis_3n.grid(axis="x", alpha=0.2)
save_figure_3n(
    figure_3n,
    "F3N02",
    "Training missingness by candidate sensor",
    "Shows the missingness evidence used by the train-only feature gate.",
    "FEATURE_QUALITY_AUDIT_3N",
)


# F3N03 — farm-specific training missingness.
farm_missing_columns_3n = [
    "train_farm_A_missing_fraction",
    "train_farm_B_missing_fraction",
    "train_farm_C_missing_fraction",
]
farm_missing_matrix_3n = FEATURE_QUALITY_AUDIT_3N.set_index("feature").loc[
    :, farm_missing_columns_3n
]
figure_height_3n = max(4.5, min(16.0, 0.25 * len(farm_missing_matrix_3n) + 2.0))
figure_3n, axis_3n = plt.subplots(figsize=(6.8, figure_height_3n))
image_3n = axis_3n.imshow(
    100.0 * farm_missing_matrix_3n.to_numpy(dtype=float),
    aspect="auto",
    cmap="YlOrRd",
    vmin=0,
    vmax=max(
        1.0,
        min(100.0, 100.0 * float(farm_missing_matrix_3n.max().max())),
    ),
)
axis_3n.set_xticks([0, 1, 2])
axis_3n.set_xticklabels(["Farm A", "Farm B", "Farm C"])
axis_3n.set_yticks(np.arange(len(farm_missing_matrix_3n)))
axis_3n.set_yticklabels(farm_missing_matrix_3n.index)
axis_3n.set_title("Farm-specific training missingness")
colorbar_3n = figure_3n.colorbar(image_3n, ax=axis_3n, pad=0.02)
colorbar_3n.set_label("Missing values (%)")
save_figure_3n(
    figure_3n,
    "F3N03",
    "Farm-specific training missingness",
    "Verifies that retained features have usable training support in every farm.",
    "FEATURE_QUALITY_AUDIT_3N",
)


# F3N04 — validation PSI, never test PSI.
psi_plot_3n = VALIDATION_SHIFT_AUDIT_3N.sort_values(
    "population_stability_index", ascending=True
)
figure_height_3n = max(4.5, min(16.0, 0.28 * len(psi_plot_3n) + 1.8))
figure_3n, axis_3n = plt.subplots(figsize=(8.2, figure_height_3n))
axis_3n.barh(
    np.arange(len(psi_plot_3n)),
    psi_plot_3n["population_stability_index"],
    color="#6C5B7B",
)
axis_3n.axvline(0.10, color="#E69F00", linestyle="--", label="Moderate shift")
axis_3n.axvline(0.25, color="#C94C4C", linestyle="--", label="Large shift")
axis_3n.set_yticks(np.arange(len(psi_plot_3n)))
axis_3n.set_yticklabels(psi_plot_3n["feature"])
axis_3n.set_xlabel("Population stability index")
axis_3n.set_title("Validation-to-training feature shift")
axis_3n.legend(loc="lower right")
axis_3n.grid(axis="x", alpha=0.2)
save_figure_3n(
    figure_3n,
    "F3N04",
    "Validation to training feature shift",
    "Quantifies unseen-asset validation shift without consulting test distributions.",
    "VALIDATION_SHIFT_AUDIT_3N",
)


# F3N05 — training-only Spearman correlations.
max_heatmap_features_3n = min(30, len(FEATURE_NAMES_3N))
heatmap_features_3n = (
    TRAIN_TRANSFORMED_FEATURE_SUMMARY_3N.sort_values("std", ascending=False)
    .head(max_heatmap_features_3n)["feature"]
    .tolist()
)
heatmap_matrix_3n = TRAIN_SPEARMAN_CORRELATION_3N.loc[
    heatmap_features_3n, heatmap_features_3n
]
heatmap_size_3n = max(6.0, min(13.0, 0.42 * len(heatmap_features_3n) + 2.5))
figure_3n, axis_3n = plt.subplots(figsize=(heatmap_size_3n, heatmap_size_3n))
image_3n = axis_3n.imshow(
    heatmap_matrix_3n.to_numpy(dtype=float),
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
)
axis_3n.set_xticks(np.arange(len(heatmap_features_3n)))
axis_3n.set_xticklabels(heatmap_features_3n, rotation=90)
axis_3n.set_yticks(np.arange(len(heatmap_features_3n)))
axis_3n.set_yticklabels(heatmap_features_3n)
axis_3n.set_title("Training-only Spearman feature correlation")
colorbar_3n = figure_3n.colorbar(image_3n, ax=axis_3n, pad=0.02)
colorbar_3n.set_label("Spearman correlation")
save_figure_3n(
    figure_3n,
    "F3N05",
    "Training only Spearman feature correlation",
    "Documents multicollinearity without automatically deleting correlated sensors.",
    "TRAIN_SPEARMAN_CORRELATION_3N",
)


# F3N06 — train and validation transformed location/scale; test remains sealed.
comparison_summary_3n = pd.DataFrame(
    {
        "feature": FEATURE_NAMES_3N,
        "train_mean": X_TRAIN_3N.mean().reindex(FEATURE_NAMES_3N).to_numpy(),
        "validation_mean": X_VALIDATION_3N.mean().reindex(FEATURE_NAMES_3N).to_numpy(),
        "train_std": X_TRAIN_3N.std(ddof=0).reindex(FEATURE_NAMES_3N).to_numpy(),
        "validation_std": X_VALIDATION_3N.std(ddof=0).reindex(FEATURE_NAMES_3N).to_numpy(),
    }
)
plot_order_3n = comparison_summary_3n.assign(
    validation_mean_abs=lambda frame_3n: frame_3n["validation_mean"].abs()
).sort_values("validation_mean_abs", ascending=True)
figure_height_3n = max(5.0, min(16.0, 0.30 * len(plot_order_3n) + 2.2))
figure_3n, axes_3n = plt.subplots(
    1, 2, figsize=(11.0, figure_height_3n), sharey=True
)
y_positions_3n = np.arange(len(plot_order_3n))
for axis_3n, statistic_3n, title_3n in [
    (axes_3n[0], "mean", "Transformed mean"),
    (axes_3n[1], "std", "Transformed standard deviation"),
]:
    axis_3n.scatter(
        plot_order_3n[f"train_{statistic_3n}"],
        y_positions_3n,
        color=SPLIT_COLORS_3N["train"],
        marker="o",
        label="Train",
        s=25,
    )
    axis_3n.scatter(
        plot_order_3n[f"validation_{statistic_3n}"],
        y_positions_3n,
        color=SPLIT_COLORS_3N["validation"],
        marker="^",
        label="Validation",
        s=28,
    )
    axis_3n.set_xlabel(title_3n)
    axis_3n.grid(axis="x", alpha=0.2)
axes_3n[0].set_yticks(y_positions_3n)
axes_3n[0].set_yticklabels(plot_order_3n["feature"])
axes_3n[0].axvline(0, color="#555555", linewidth=0.8)
axes_3n[1].axvline(1, color="#555555", linewidth=0.8)
axes_3n[1].legend(loc="lower right")
figure_3n.suptitle("Frozen preprocessing across train and validation")
save_figure_3n(
    figure_3n,
    "F3N06",
    "Frozen preprocessing across train and validation",
    "Shows validation behavior under training-fitted preprocessing; test remains sealed.",
    "X_TRAIN_3N; X_VALIDATION_3N",
)


PREPROCESSING_FIGURE_REGISTRY_3N = pd.DataFrame(figure_registry_records_3n)

table_exports_3n = [
    (
        "T3N01",
        "source_resolution_audit",
        SOURCE_RESOLUTION_AUDIT_3N,
        "Documents exact source-key resolution before feature processing.",
    ),
    (
        "T3N02",
        "feature_quality_audit",
        FEATURE_QUALITY_AUDIT_3N,
        "Reports training-only feature decisions and fitted statistics.",
    ),
    (
        "T3N03",
        "preprocessing_parameters",
        PREPROCESSING_PARAMETERS_3N,
        "Provides the frozen median-imputation and robust-scaling parameters.",
    ),
    (
        "T3N04",
        "split_transform_audit",
        SPLIT_TRANSFORM_AUDIT_3N,
        "Verifies finite, row-conserving transformation while sealing test statistics.",
    ),
    (
        "T3N05",
        "validation_shift_audit",
        VALIDATION_SHIFT_AUDIT_3N,
        "Quantifies validation-to-training drift using training-defined bins.",
    ),
    (
        "T3N06",
        "training_correlation_pairs",
        TRAIN_CORRELATION_PAIR_AUDIT_3N,
        "Reports training-only correlated sensor pairs without automatic removal.",
    ),
    (
        "T3N07",
        "training_transformed_feature_summary",
        TRAIN_TRANSFORMED_FEATURE_SUMMARY_3N,
        "Summarizes the transformed training feature space.",
    ),
    (
        "T3N08",
        "preprocessing_leakage_audit",
        PREPROCESSING_LEAKAGE_AUDIT_3N,
        "Confirms train-only fitting, sealed test diagnostics, and manifest conservation.",
    ),
]

table_registry_records_3n = []
for table_id_3n, table_name_3n, table_frame_3n, purpose_3n in table_exports_3n:
    stem_3n = f"{table_id_3n}_{slug_3n(table_name_3n)}"
    csv_path_3n = TABLE_DIR_3N / f"{stem_3n}.csv"
    tex_path_3n = TABLE_DIR_3N / f"{stem_3n}.tex"
    table_frame_3n.to_csv(csv_path_3n, index=False)
    latex_status_3n = "exported"
    try:
        table_frame_3n.to_latex(tex_path_3n, index=False, escape=True)
        latex_path_value_3n = str(tex_path_3n)
    except Exception as error_3n:
        latex_status_3n = f"skipped: {type(error_3n).__name__}: {error_3n}"
        latex_path_value_3n = pd.NA
        print(f"LaTeX export skipped for {table_id_3n}: {error_3n}")

    table_registry_records_3n.append(
        {
            "table_id": table_id_3n,
            "name": table_name_3n,
            "purpose": purpose_3n,
            "rows": len(table_frame_3n),
            "columns": len(table_frame_3n.columns),
            "csv_path": str(csv_path_3n),
            "latex_path": latex_path_value_3n,
            "latex_status": latex_status_3n,
        }
    )

PREPROCESSING_TABLE_REGISTRY_3N = pd.DataFrame(table_registry_records_3n)

figure_registry_path_3n = OUTPUT_ROOT_3N / "preprocessing_figure_registry_3n.csv"
table_registry_path_3n = OUTPUT_ROOT_3N / "preprocessing_table_registry_3n.csv"
PREPROCESSING_FIGURE_REGISTRY_3N.to_csv(figure_registry_path_3n, index=False)
PREPROCESSING_TABLE_REGISTRY_3N.to_csv(table_registry_path_3n, index=False)

state_json_path_3n = OUTPUT_ROOT_3N / "preprocessor_state_3n.json"


def json_safe_3n(value):
    if value is pd.NA or value is None:
        return None
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


with state_json_path_3n.open("w", encoding="utf-8") as state_file_3n:
    json.dump(
        PREPROCESSOR_STATE_3N,
        state_file_3n,
        indent=2,
        ensure_ascii=False,
        default=json_safe_3n,
    )

workbook_path_3n = OUTPUT_ROOT_3N / "preprocessing_diagnostics_3n.xlsx"
workbook_exported_3n = False
try:
    from openpyxl.styles import Font, PatternFill

    with pd.ExcelWriter(workbook_path_3n, engine="openpyxl") as writer_3n:
        for table_id_3n, table_name_3n, table_frame_3n, _ in table_exports_3n:
            sheet_name_3n = f"{table_id_3n}_{slug_3n(table_name_3n)}"[:31]
            workbook_frame_3n = table_frame_3n.copy()
            for column_3n in workbook_frame_3n.columns:
                if isinstance(workbook_frame_3n[column_3n].dtype, pd.DatetimeTZDtype):
                    workbook_frame_3n[column_3n] = workbook_frame_3n[
                        column_3n
                    ].dt.tz_convert("UTC").dt.tz_localize(None)
            workbook_frame_3n.to_excel(
                writer_3n,
                sheet_name=sheet_name_3n,
                index=False,
            )
        PREPROCESSING_FIGURE_REGISTRY_3N.to_excel(
            writer_3n, sheet_name="figure_registry", index=False
        )
        PREPROCESSING_TABLE_REGISTRY_3N.to_excel(
            writer_3n, sheet_name="table_registry", index=False
        )

        for worksheet_3n in writer_3n.book.worksheets:
            worksheet_3n.freeze_panes = "A2"
            worksheet_3n.auto_filter.ref = worksheet_3n.dimensions
            for cell_3n in worksheet_3n[1]:
                cell_3n.font = Font(bold=True, color="FFFFFF")
                cell_3n.fill = PatternFill(
                    fill_type="solid",
                    fgColor="1F4E78",
                )
            for column_cells_3n in worksheet_3n.columns:
                maximum_length_3n = max(
                    len(str(cell_3n.value)) if cell_3n.value is not None else 0
                    for cell_3n in column_cells_3n
                )
                column_letter_3n = column_cells_3n[0].column_letter
                worksheet_3n.column_dimensions[column_letter_3n].width = min(
                    max(maximum_length_3n + 2, 10), 45
                )
    workbook_exported_3n = True
except Exception as error_3n:
    print(f"Excel workbook export skipped: {error_3n}")


# -----------------------------------------------------------------------------
# 8. Compact notebook report
# -----------------------------------------------------------------------------

decision_reason_summary_3n = (
    FEATURE_QUALITY_AUDIT_3N.groupby(
        ["decision", "decision_reason"], as_index=False
    )
    .agg(features=("feature", "size"))
    .sort_values(["decision", "features"], ascending=[False, False])
)

print("\nMeasurement-source resolution:")
print(SOURCE_RESOLUTION_AUDIT_3N.to_string(index=False))

print("\nTraining-only feature decisions:")
print(decision_reason_summary_3n.to_string(index=False))

print("\nFrozen split-transform audit:")
print(SPLIT_TRANSFORM_AUDIT_3N.to_string(index=False))

print("\nValidation-shift summary (test remains sealed):")
print(
    VALIDATION_SHIFT_AUDIT_3N.head(min(20, len(VALIDATION_SHIFT_AUDIT_3N)))
    .to_string(index=False)
)

print("\nPreprocessing leakage and conservation audit:")
print(PREPROCESSING_LEAKAGE_AUDIT_3N.to_string(index=False))

print("\nCell 3N completed successfully.")
print("Measurement source:", selected_source_name_3n)
print("Eligible rows resolved:", len(PREPROCESSED_FEATURE_MATRIX_3N))
print("Candidate sensor features:", len(candidate_features_3n))
print("Retained sensor features:", len(FEATURE_NAMES_3N))
print("Excluded sensor features:", len(EXCLUDED_FEATURES_3N))
print(
    "Train/validation/test rows:",
    f"{len(X_TRAIN_3N)}/{len(X_VALIDATION_3N)}/{len(X_TEST_3N)}",
)
print("Post-transform missing cells:", int(transformed_features_3n.isna().sum().sum()))
print(
    "Post-transform non-finite cells:",
    int((~np.isfinite(transformed_features_3n.to_numpy())).sum()),
)
print("Tables generated:", len(PREPROCESSING_TABLE_REGISTRY_3N))
print("Figures generated:", len(PREPROCESSING_FIGURE_REGISTRY_3N))
print("Figure registry:", figure_registry_path_3n)
print("Table registry:", table_registry_path_3n)
print("Frozen preprocessor state:", state_json_path_3n)
print("Excel workbook exported:", workbook_exported_3n)
print("Manifest values changed:", manifest_values_changed_3n)
print(
    "\nAll feature inclusion, imputation, and scaling parameters were fitted "
    "on training assets only. Validation was used only for shift diagnostics. "
    "Test received the frozen transform but its distributions remain sealed. "
    "No manifest row, label, eligibility decision, or split assignment changed."
)


NameError: Cell 3N found possible sources but will not guess which object contains raw sensors. Review FEATURE_SOURCE_CANDIDATES_3N, then define:
FEATURE_SOURCE_3N = <raw keyed DataFrame or raw event-frame mapping>
Optionally define FEATURE_COLUMNS_3N as the physical sensor columns, then rerun Cell 3N.

                       source_name     source_type  source_rows candidate_feature_columns  eligible_keys_matched duplicated_source_keys  exact_eligible_coverage                                                                                                       inspection_error
                 eligible_manifest keyed_dataframe       214546                         1                 213537                      0                     True                                                                                                                   <NA>
        distinct_timestamp_vectors keyed_dataframe       213537                         1                 213537                      0

In [45]:
import pandas as pd
import numpy as np


METADATA_NAMES_3N = {
    "farm",
    "asset_id",
    "asset_key",
    "event_id",
    "event_key",
    "raw_id",
    "source_row_index",
    "timestamp",
    "timestamp_utc",
    "date",
    "date_utc",
    "label",
    "final_label",
    "source_event_label",
    "split_assignment",
    "modeling_eligible",
    "within_official_event_window",
    "manifest_decision",
    "measurement_schema_id",
    "measurement_feature_count",
    "measurement_fingerprint",
}


def summarize_dataframe_3n(path_3n, frame_3n):
    columns_3n = [str(column) for column in frame_3n.columns]

    numeric_candidates_3n = []

    for column_3n in frame_3n.columns:
        normalized_name_3n = str(column_3n).strip().lower()

        if normalized_name_3n in METADATA_NAMES_3N:
            continue

        values_3n = pd.to_numeric(
            frame_3n[column_3n],
            errors="coerce",
        )

        finite_count_3n = int(
            np.isfinite(values_3n.to_numpy(dtype=float)).sum()
        )

        if finite_count_3n:
            numeric_candidates_3n.append(
                (
                    str(column_3n),
                    finite_count_3n,
                )
            )

    print(f"\nDATAFRAME: {path_3n}")
    print("Shape:", frame_3n.shape)
    print("Columns:", columns_3n)
    print(
        "Possible numeric sensor columns:",
        numeric_candidates_3n,
    )


def inspect_container_3n(
    value_3n,
    path_3n,
    depth_3n=0,
    maximum_depth_3n=5,
    maximum_children_3n=8,
    visited_3n=None,
):
    if visited_3n is None:
        visited_3n = set()

    object_id_3n = id(value_3n)

    if object_id_3n in visited_3n:
        return

    visited_3n.add(object_id_3n)

    if isinstance(value_3n, pd.DataFrame):
        summarize_dataframe_3n(path_3n, value_3n)
        return

    if depth_3n >= maximum_depth_3n:
        return

    if isinstance(value_3n, dict):
        print(
            f"\nDICT: {path_3n}; "
            f"items={len(value_3n):,}; "
            f"sample_keys={list(value_3n.keys())[:maximum_children_3n]}"
        )

        for key_3n, child_3n in list(
            value_3n.items()
        )[:maximum_children_3n]:
            inspect_container_3n(
                child_3n,
                f"{path_3n}[{key_3n!r}]",
                depth_3n + 1,
                maximum_depth_3n,
                maximum_children_3n,
                visited_3n,
            )

        return

    if isinstance(value_3n, (list, tuple)):
        print(
            f"\n{type(value_3n).__name__.upper()}: "
            f"{path_3n}; items={len(value_3n):,}"
        )

        for index_3n, child_3n in enumerate(
            value_3n[:maximum_children_3n]
        ):
            inspect_container_3n(
                child_3n,
                f"{path_3n}[{index_3n}]",
                depth_3n + 1,
                maximum_depth_3n,
                maximum_children_3n,
                visited_3n,
            )

        return

    if hasattr(value_3n, "__dict__"):
        public_attributes_3n = {
            name_3n: child_3n
            for name_3n, child_3n
            in vars(value_3n).items()
            if not name_3n.startswith("_")
        }

        if public_attributes_3n:
            print(
                f"\nOBJECT: {path_3n}; "
                f"type={type(value_3n).__name__}; "
                f"attributes="
                f"{list(public_attributes_3n)[:maximum_children_3n]}"
            )

            for name_3n, child_3n in list(
                public_attributes_3n.items()
            )[:maximum_children_3n]:
                inspect_container_3n(
                    child_3n,
                    f"{path_3n}.{name_3n}",
                    depth_3n + 1,
                    maximum_depth_3n,
                    maximum_children_3n,
                    visited_3n,
                )


likely_source_names_3n = [
    "FARMS",
    "EVENTS_BY_FARM",
    "event_a",
    "event_b",
    "SOURCE_EVENT_PROVENANCE_TABLES",
    "OFFICIAL_EVENT_TIMESTAMP_CACHE",
    "ALL_OFFICIAL_EVENT_TIMESTAMP_CACHE",
]

for source_name_3n in likely_source_names_3n:
    if source_name_3n not in globals():
        continue

    print("\n" + "=" * 80)
    print("ROOT SOURCE:", source_name_3n)

    inspect_container_3n(
        globals()[source_name_3n],
        source_name_3n,
    )


ROOT SOURCE: FARMS

DICT: FARMS; items=3; sample_keys=['A', 'B', 'C']

ROOT SOURCE: EVENTS_BY_FARM

DICT: EVENTS_BY_FARM; items=3; sample_keys=['A', 'B', 'C']

DATAFRAME: EVENTS_BY_FARM['A']
Shape: (22, 11)
Columns: ['farm', 'asset_id', 'event_id', 'event_label', 'event_start', 'event_start_id', 'event_end', 'event_end_id', 'event_description', 'event_key', 'event_duration_days']
Possible numeric sensor columns: [('event_start', 22), ('event_start_id', 22), ('event_end', 22), ('event_end_id', 22), ('event_duration_days', 22)]

DATAFRAME: EVENTS_BY_FARM['B']
Shape: (15, 11)
Columns: ['farm', 'asset_id', 'event_id', 'event_label', 'event_start', 'event_start_id', 'event_end', 'event_end_id', 'event_description', 'event_key', 'event_duration_days']
Possible numeric sensor columns: [('event_start', 15), ('event_start_id', 15), ('event_end', 15), ('event_end_id', 15), ('event_duration_days', 15)]

DATAFRAME: EVENTS_BY_FARM['C']
Shape: (58, 11)
Columns: ['farm', 'asset_id', 'event_id', 'eve

In [46]:
from pathlib import Path
import pandas as pd


def compact_value_3n(value_3n, maximum_length_3n=300):
    representation_3n = repr(value_3n)

    if len(representation_3n) > maximum_length_3n:
        representation_3n = (
            representation_3n[:maximum_length_3n]
            + "..."
        )

    return representation_3n


# ---------------------------------------------------------
# 1. Reveal the actual contents of FARMS
# ---------------------------------------------------------

print("FARMS structure:")

for farm_name_3n, farm_value_3n in FARMS.items():
    print(
        f"\nFarm {farm_name_3n!r}: "
        f"type={type(farm_value_3n).__name__}"
    )
    print(compact_value_3n(farm_value_3n))


# ---------------------------------------------------------
# 2. Find notebook variables that look like paths
# ---------------------------------------------------------

path_candidates_3n = []

for variable_name_3n, variable_value_3n in list(
    globals().items()
):
    if variable_name_3n.startswith("_"):
        continue

    if isinstance(variable_value_3n, (str, Path)):
        value_text_3n = str(variable_value_3n)

        path_signal_3n = any(
            token_3n in value_text_3n.lower()
            for token_3n in [
                ".csv",
                ".parquet",
                ".xlsx",
                ".xls",
                ".pkl",
                ".pickle",
                "\\",
                "/",
            ]
        )

        if path_signal_3n:
            path_object_3n = Path(value_text_3n)

            path_candidates_3n.append(
                {
                    "variable_name": variable_name_3n,
                    "value": value_text_3n,
                    "exists": path_object_3n.exists(),
                    "is_file": path_object_3n.is_file(),
                    "is_directory": path_object_3n.is_dir(),
                }
            )

PATH_CANDIDATES_3N = pd.DataFrame(path_candidates_3n)

print("\nNotebook path candidates:")

if PATH_CANDIDATES_3N.empty:
    print("No path-like scalar variables found.")
else:
    display(
        PATH_CANDIDATES_3N.sort_values(
            ["exists", "variable_name"],
            ascending=[False, True],
        ).reset_index(drop=True)
    )


# ---------------------------------------------------------
# 3. Find path/file columns in existing DataFrames
# ---------------------------------------------------------

path_column_records_3n = []

for variable_name_3n, variable_value_3n in list(
    globals().items()
):
    if not isinstance(variable_value_3n, pd.DataFrame):
        continue

    for column_3n in variable_value_3n.columns:
        normalized_column_3n = str(column_3n).lower()

        if not any(
            token_3n in normalized_column_3n
            for token_3n in [
                "path",
                "file",
                "directory",
                "folder",
            ]
        ):
            continue

        nonmissing_values_3n = (
            variable_value_3n[column_3n]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .head(10)
            .tolist()
        )

        path_column_records_3n.append(
            {
                "dataframe_name": variable_name_3n,
                "column": str(column_3n),
                "rows": len(variable_value_3n),
                "sample_values": nonmissing_values_3n,
            }
        )

PATH_COLUMNS_3N = pd.DataFrame(path_column_records_3n)

print("\nPath/file columns in notebook DataFrames:")

if PATH_COLUMNS_3N.empty:
    print("No path/file columns found.")
else:
    display(PATH_COLUMNS_3N)


# ---------------------------------------------------------
# 4. Inspect paths recorded in event audit tables
# ---------------------------------------------------------

event_audit_3n = SOURCE_EVENT_PROVENANCE_TABLES.get(
    "EVENT_TIMESTAMP_FILE_AUDIT"
)

if (
    isinstance(event_audit_3n, pd.DataFrame)
    and "event_path" in event_audit_3n.columns
):
    EVENT_PATH_AUDIT_3N = event_audit_3n[
        ["event_key", "event_path"]
    ].copy()

    EVENT_PATH_AUDIT_3N["path_exists"] = (
        EVENT_PATH_AUDIT_3N["event_path"]
        .astype(str)
        .map(lambda value_3n: Path(value_3n).exists())
    )

    print("\nRecorded event paths:")
    display(EVENT_PATH_AUDIT_3N)

FARMS structure:

Farm 'A': type=WindowsPath
WindowsPath('F:/Umar-Wisal-Work/Wisal-Bearings-Work/CARE_To_Compare/CARE_To_Compare/Wind Farm A')

Farm 'B': type=WindowsPath
WindowsPath('F:/Umar-Wisal-Work/Wisal-Bearings-Work/CARE_To_Compare/CARE_To_Compare/Wind Farm B')

Farm 'C': type=WindowsPath
WindowsPath('F:/Umar-Wisal-Work/Wisal-Bearings-Work/CARE_To_Compare/CARE_To_Compare/Wind Farm C')

Notebook path candidates:


,variable_name,value,exists,is_file,is_directory
0,ACTIVE_PYTHON,f:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To...,True,True,False
1,DATASET_ROOT,F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To...,True,False,True
2,FIGURE_DIR_3N,paper_visuals_3l\preprocessing_3n\figures,True,False,True
3,OUTPUT_ROOT_3M,paper_visuals_3l\split_diagnostics_3m,True,False,True
4,OUTPUT_ROOT_3N,paper_visuals_3l\preprocessing_3n,True,False,True
5,PDF_DIR_3L,paper_visuals_3l\figures_pdf,True,False,True
6,PDF_DIR_3M,paper_visuals_3l\split_diagnostics_3m\figures_pdf,True,False,True
7,PNG_DIR_3L,paper_visuals_3l\figures_png,True,False,True
8,PNG_DIR_3M,paper_visuals_3l\split_diagnostics_3m\figures_png,True,False,True
9,PROJECT_ROOT,F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To...,True,False,True



Path/file columns in notebook DataFrames:


,dataframe_name,column,rows,sample_values
0,EVENT_TIMESTAMP_FILE_AUDIT,event_path,6,[F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_T...
1,RAW_TIMESTAMP_PAIR_AUDIT,shared_timestamps_full_files,3,"[53983, 55356, 53391]"
2,RAW_TIMESTAMP_PAIR_AUDIT,full_file_shared_percent_a,3,"[99.16783, 88.899595, 99.953198]"
3,RAW_TIMESTAMP_PAIR_AUDIT,full_file_shared_percent_b,3,"[99.816945, 100.0, 94.582721]"
4,cross_label_audit,shared_timestamps_full_files,2,"[55356, 53391]"
5,cross_label_audit,full_file_shared_percent_a,2,"[88.899595, 99.953198]"
6,cross_label_audit,full_file_shared_percent_b,2,"[100.0, 94.582721]"
7,FULL_FILE_TIMESTAMP_SCOPE_AUDIT,shared_timestamps_full_files,2,"[55356, 53391]"
8,FULL_FILE_TIMESTAMP_SCOPE_AUDIT,full_file_shared_percent_a,2,"[88.899595, 99.953198]"
9,FULL_FILE_TIMESTAMP_SCOPE_AUDIT,full_file_shared_percent_b,2,"[100.0, 94.582721]"



Recorded event paths:


,event_key,event_path,path_exists
0,farm_b_event_27,F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To...,True
1,farm_b_event_87,F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To...,True
2,farm_c_event_4,F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To...,True
3,farm_c_event_56,F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To...,True


In [47]:
existing_event_paths_3n = (
    EVENT_PATH_AUDIT_3N.loc[
        EVENT_PATH_AUDIT_3N["path_exists"],
        "event_path",
    ]
    .astype(str)
    .tolist()
)

for event_path_3n in existing_event_paths_3n[:5]:
    event_path_object_3n = Path(event_path_3n)

    print("\nFILE:", event_path_object_3n)

    if event_path_object_3n.suffix.lower() == ".csv":
        preview_3n = pd.read_csv(
            event_path_object_3n,
            nrows=5,
        )
        print("Columns:", preview_3n.columns.tolist())
        display(preview_3n)


FILE: F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To_Compare\CARE_To_Compare\Wind Farm B\datasets\27.csv
Columns: ['time_stamp;asset_id;id;train_test;status_type_id;sensor_0_avg;sensor_0_max;sensor_0_min;sensor_0_std;sensor_1_avg;sensor_1_max;sensor_1_min;sensor_1_std;sensor_2_avg;sensor_2_max;sensor_2_min;sensor_2_std;sensor_3_avg;sensor_3_max;sensor_3_min;sensor_3_std;sensor_4_avg;sensor_4_max;sensor_4_min;sensor_4_std;sensor_5_avg;sensor_5_max;sensor_5_min;sensor_5_std;sensor_6_avg;sensor_6_max;sensor_6_min;sensor_6_std;sensor_7_avg;sensor_7_max;sensor_7_min;sensor_7_std;sensor_8_avg;sensor_8_max;sensor_8_min;sensor_8_std;sensor_9_avg;sensor_9_max;sensor_9_min;sensor_9_std;sensor_10_avg;sensor_10_max;sensor_10_min;sensor_10_std;reactive_power_11_avg;reactive_power_11_max;reactive_power_11_min;reactive_power_11_std;sensor_12_avg;sensor_12_max;sensor_12_min;sensor_12_std;sensor_13_avg;sensor_13_max;sensor_13_min;sensor_13_std;sensor_14_avg;sensor_14_max;sensor_14_min;sensor_14_std;se

,time_stamp;asset_id;id;train_test;status_type_id;sensor_0_avg;sensor_0_max;sensor_0_min;sensor_0_std;sensor_1_avg;sensor_1_max;sensor_1_min;sensor_1_std;sensor_2_avg;sensor_2_max;sensor_2_min;sensor_2_std;sensor_3_avg;sensor_3_max;sensor_3_min;sensor_3_std;sensor_4_avg;sensor_4_max;sensor_4_min;sensor_4_std;sensor_5_avg;sensor_5_max;sensor_5_min;sensor_5_std;sensor_6_avg;sensor_6_max;sensor_6_min;sensor_6_std;sensor_7_avg;sensor_7_max;sensor_7_min;sensor_7_std;sensor_8_avg;sensor_8_max;sensor_8_min;sensor_8_std;sensor_9_avg;sensor_9_max;sensor_9_min;sensor_9_std;sensor_10_avg;sensor_10_max;sensor_10_min;sensor_10_std;reactive_power_11_avg;reactive_power_11_max;reactive_power_11_min;reactive_power_11_std;sensor_12_avg;sensor_12_max;sensor_12_min;sensor_12_std;sensor_13_avg;sensor_13_max;sensor_13_min;sensor_13_std;sensor_14_avg;sensor_14_max;sensor_14_min;sensor_14_std;sensor_15_avg;sensor_15_max;sensor_15_min;sensor_15_std;sensor_16_avg;sensor_16_max;sensor_16_min;sensor_16_std;sensor_17_avg;sensor_17_max;sensor_17_min;sensor_17_std;sensor_18_avg;sensor_18_max;sensor_18_min;sensor_18_std;sensor_19_avg;sensor_19_max;sensor_19_min;sensor_19_std;sensor_20_avg;sensor_20_max;sensor_20_min;sensor_20_std;sensor_21_avg;sensor_21_max;sensor_21_min;sensor_21_std;sensor_22_avg;sensor_22_max;sensor_22_min;sensor_22_std;sensor_23_avg;sensor_23_max;sensor_23_min;sensor_23_std;sensor_24_avg;sensor_24_max;sensor_24_min;sensor_24_std;sensor_25_avg;sensor_25_max;sensor_25_min;sensor_25_std;sensor_26_avg;sensor_26_max;sensor_26_min;sensor_26_std;sensor_27_avg;sensor_27_max;sensor_27_min;sensor_27_std;sensor_28_avg;sensor_28_max;sensor_28_min;sensor_28_std;sensor_29_avg;sensor_29_max;sensor_29_min;sensor_29_std;sensor_30_avg;sensor_30_max;sensor_30_min;sensor_30_std;sensor_31_avg;sensor_31_max;sensor_31_min;sensor_31_std;sensor_32_avg;sensor_32_max;sensor_32_min;sensor_32_std;sensor_33_avg;sensor_33_max;sensor_33_min;sensor_33_std;sensor_34_avg;sensor_34_max;sensor_34_min;sensor_34_std;sensor_35_avg;sensor_35_max;sensor_35_min;sensor_35_std;sensor_36_avg;sensor_36_max;sensor_36_min;sensor_36_std;sensor_37_avg;sensor_37_max;sensor_37_min;sensor_37_std;sensor_38_avg;sensor_38_max;sensor_38_min;sensor_38_std;sensor_39_avg;sensor_39_max;sensor_39_min;sensor_39_std;sensor_40_avg;sensor_40_max;sensor_40_min;sensor_40_std;sensor_41_avg;sensor_41_max;sensor_41_min;sensor_41_std;sensor_42_avg;sensor_42_max;sensor_42_min;sensor_42_std;sensor_43_avg;sensor_43_max;sensor_43_min;sensor_43_std;sensor_44_avg;sensor_44_max;sensor_44_min;sensor_44_std;sensor_45_avg;sensor_45_max;sensor_45_min;sensor_45_std;sensor_46_avg;sensor_46_max;sensor_46_min;sensor_46_std;sensor_47_avg;sensor_47_max;sensor_47_min;sensor_47_std;sensor_48_avg;sensor_48_max;sensor_48_min;sensor_48_std;sensor_49_avg;sensor_49_max;sensor_49_min;sensor_49_std;sensor_50_avg;sensor_50_max;sensor_50_min;sensor_50_std;sensor_51_avg;sensor_51_max;sensor_51_min;sensor_51_std;sensor_52_avg;sensor_52_max;sensor_52_min;sensor_52_std;sensor_53_avg;sensor_53_max;sensor_53_min;sensor_53_std;sensor_54_avg;sensor_54_max;sensor_54_min;sensor_54_std;sensor_55_avg;sensor_55_max;sensor_55_min;sensor_55_std;sensor_56_avg;sensor_56_max;sensor_56_min;sensor_56_std;sensor_57_avg;sensor_57_max;sensor_57_min;sensor_57_std;power_58_avg;power_58_max;power_58_min;power_58_std;wind_speed_59_avg;wind_speed_59_max;wind_speed_59_min;wind_speed_59_std;wind_speed_60_avg;wind_speed_60_max;wind_speed_60_min;wind_speed_60_std;wind_speed_61_avg;wind_speed_61_max;wind_speed_61_min;wind_speed_61_std;power_62_avg;power_62_max;power_62_min;power_62_std
0,2022-08-30 00:00:00;7;0;train;0;0.0;0.0;0.0;0....
1,2022-08-30 00:10:00;7;1;train;0;1.0;0.0;0.0;0....
2,2022-08-30 00:20:00;7;2;train;0;1.0;0.0;0.0;0....
3,2022-08-30 00:30:00;7;3;train;0;4.0;0.0;0.0;0....
4,2022-08-30 00:40:00;7;4;train;0;4.0;0.0;0.0;0....



FILE: F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To_Compare\CARE_To_Compare\Wind Farm B\datasets\87.csv
Columns: ['time_stamp;asset_id;id;train_test;status_type_id;sensor_0_avg;sensor_0_max;sensor_0_min;sensor_0_std;sensor_1_avg;sensor_1_max;sensor_1_min;sensor_1_std;sensor_2_avg;sensor_2_max;sensor_2_min;sensor_2_std;sensor_3_avg;sensor_3_max;sensor_3_min;sensor_3_std;sensor_4_avg;sensor_4_max;sensor_4_min;sensor_4_std;sensor_5_avg;sensor_5_max;sensor_5_min;sensor_5_std;sensor_6_avg;sensor_6_max;sensor_6_min;sensor_6_std;sensor_7_avg;sensor_7_max;sensor_7_min;sensor_7_std;sensor_8_avg;sensor_8_max;sensor_8_min;sensor_8_std;sensor_9_avg;sensor_9_max;sensor_9_min;sensor_9_std;sensor_10_avg;sensor_10_max;sensor_10_min;sensor_10_std;reactive_power_11_avg;reactive_power_11_max;reactive_power_11_min;reactive_power_11_std;sensor_12_avg;sensor_12_max;sensor_12_min;sensor_12_std;sensor_13_avg;sensor_13_max;sensor_13_min;sensor_13_std;sensor_14_avg;sensor_14_max;sensor_14_min;sensor_14_std;se

,time_stamp;asset_id;id;train_test;status_type_id;sensor_0_avg;sensor_0_max;sensor_0_min;sensor_0_std;sensor_1_avg;sensor_1_max;sensor_1_min;sensor_1_std;sensor_2_avg;sensor_2_max;sensor_2_min;sensor_2_std;sensor_3_avg;sensor_3_max;sensor_3_min;sensor_3_std;sensor_4_avg;sensor_4_max;sensor_4_min;sensor_4_std;sensor_5_avg;sensor_5_max;sensor_5_min;sensor_5_std;sensor_6_avg;sensor_6_max;sensor_6_min;sensor_6_std;sensor_7_avg;sensor_7_max;sensor_7_min;sensor_7_std;sensor_8_avg;sensor_8_max;sensor_8_min;sensor_8_std;sensor_9_avg;sensor_9_max;sensor_9_min;sensor_9_std;sensor_10_avg;sensor_10_max;sensor_10_min;sensor_10_std;reactive_power_11_avg;reactive_power_11_max;reactive_power_11_min;reactive_power_11_std;sensor_12_avg;sensor_12_max;sensor_12_min;sensor_12_std;sensor_13_avg;sensor_13_max;sensor_13_min;sensor_13_std;sensor_14_avg;sensor_14_max;sensor_14_min;sensor_14_std;sensor_15_avg;sensor_15_max;sensor_15_min;sensor_15_std;sensor_16_avg;sensor_16_max;sensor_16_min;sensor_16_std;sensor_17_avg;sensor_17_max;sensor_17_min;sensor_17_std;sensor_18_avg;sensor_18_max;sensor_18_min;sensor_18_std;sensor_19_avg;sensor_19_max;sensor_19_min;sensor_19_std;sensor_20_avg;sensor_20_max;sensor_20_min;sensor_20_std;sensor_21_avg;sensor_21_max;sensor_21_min;sensor_21_std;sensor_22_avg;sensor_22_max;sensor_22_min;sensor_22_std;sensor_23_avg;sensor_23_max;sensor_23_min;sensor_23_std;sensor_24_avg;sensor_24_max;sensor_24_min;sensor_24_std;sensor_25_avg;sensor_25_max;sensor_25_min;sensor_25_std;sensor_26_avg;sensor_26_max;sensor_26_min;sensor_26_std;sensor_27_avg;sensor_27_max;sensor_27_min;sensor_27_std;sensor_28_avg;sensor_28_max;sensor_28_min;sensor_28_std;sensor_29_avg;sensor_29_max;sensor_29_min;sensor_29_std;sensor_30_avg;sensor_30_max;sensor_30_min;sensor_30_std;sensor_31_avg;sensor_31_max;sensor_31_min;sensor_31_std;sensor_32_avg;sensor_32_max;sensor_32_min;sensor_32_std;sensor_33_avg;sensor_33_max;sensor_33_min;sensor_33_std;sensor_34_avg;sensor_34_max;sensor_34_min;sensor_34_std;sensor_35_avg;sensor_35_max;sensor_35_min;sensor_35_std;sensor_36_avg;sensor_36_max;sensor_36_min;sensor_36_std;sensor_37_avg;sensor_37_max;sensor_37_min;sensor_37_std;sensor_38_avg;sensor_38_max;sensor_38_min;sensor_38_std;sensor_39_avg;sensor_39_max;sensor_39_min;sensor_39_std;sensor_40_avg;sensor_40_max;sensor_40_min;sensor_40_std;sensor_41_avg;sensor_41_max;sensor_41_min;sensor_41_std;sensor_42_avg;sensor_42_max;sensor_42_min;sensor_42_std;sensor_43_avg;sensor_43_max;sensor_43_min;sensor_43_std;sensor_44_avg;sensor_44_max;sensor_44_min;sensor_44_std;sensor_45_avg;sensor_45_max;sensor_45_min;sensor_45_std;sensor_46_avg;sensor_46_max;sensor_46_min;sensor_46_std;sensor_47_avg;sensor_47_max;sensor_47_min;sensor_47_std;sensor_48_avg;sensor_48_max;sensor_48_min;sensor_48_std;sensor_49_avg;sensor_49_max;sensor_49_min;sensor_49_std;sensor_50_avg;sensor_50_max;sensor_50_min;sensor_50_std;sensor_51_avg;sensor_51_max;sensor_51_min;sensor_51_std;sensor_52_avg;sensor_52_max;sensor_52_min;sensor_52_std;sensor_53_avg;sensor_53_max;sensor_53_min;sensor_53_std;sensor_54_avg;sensor_54_max;sensor_54_min;sensor_54_std;sensor_55_avg;sensor_55_max;sensor_55_min;sensor_55_std;sensor_56_avg;sensor_56_max;sensor_56_min;sensor_56_std;sensor_57_avg;sensor_57_max;sensor_57_min;sensor_57_std;power_58_avg;power_58_max;power_58_min;power_58_std;wind_speed_59_avg;wind_speed_59_max;wind_speed_59_min;wind_speed_59_std;wind_speed_60_avg;wind_speed_60_max;wind_speed_60_min;wind_speed_60_std;wind_speed_61_avg;wind_speed_61_max;wind_speed_61_min;wind_speed_61_std;power_62_avg;power_62_max;power_62_min;power_62_std
0,2022-09-13 23:00:00;7;0;train;0;0.0;0.0;0.0;0....
1,2022-09-13 23:10:00;7;1;train;0;0.0;0.0;0.0;0....
2,2022-09-13 23:20:00;7;2;train;0;0.0;0.0;0.0;0....
3,2022-09-13 23:30:00;7;3;train;0;0.0;0.0;0.0;0....
4,2022-09-13 23:40:00;7;4;train;0;0.0;0.0;0.0;0....



FILE: F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To_Compare\CARE_To_Compare\Wind Farm C\datasets\4.csv
Columns: ['time_stamp;asset_id;id;train_test;status_type_id;sensor_0_avg;sensor_0_max;sensor_0_min;sensor_0_std;sensor_1_avg;sensor_1_max;sensor_1_min;sensor_1_std;power_2_avg;power_2_max;power_2_min;power_2_std;sensor_3_avg;sensor_3_max;sensor_3_min;sensor_3_std;sensor_4_avg;sensor_4_max;sensor_4_min;sensor_4_std;power_5_avg;power_5_max;power_5_min;power_5_std;power_6_avg;power_6_max;power_6_min;power_6_std;sensor_7_avg;sensor_7_max;sensor_7_min;sensor_7_std;sensor_8_avg;sensor_8_max;sensor_8_min;sensor_8_std;sensor_9_avg;sensor_9_max;sensor_9_min;sensor_9_std;sensor_10_avg;sensor_10_max;sensor_10_min;sensor_10_std;sensor_11_avg;sensor_11_max;sensor_11_min;sensor_11_std;sensor_12_avg;sensor_12_max;sensor_12_min;sensor_12_std;sensor_13_avg;sensor_13_max;sensor_13_min;sensor_13_std;sensor_14_avg;sensor_14_max;sensor_14_min;sensor_14_std;sensor_15_avg;sensor_15_max;sensor_15_min;senso

time_stamp;asset_id;id;train_test;status_type_id;sensor_0_avg;sensor_0_max;sensor_0_min;sensor_0_std;sensor_1_avg;sensor_1_max;sensor_1_min;sensor_1_std;power_2_avg;power_2_max;power_2_min;power_2_std;sensor_3_avg;sensor_3_max;sensor_3_min;sensor_3_std;sensor_4_avg;sensor_4_max;sensor_4_min;sensor_4_std;power_5_avg;power_5_max;power_5_min;power_5_std;power_6_avg;power_6_max;power_6_min;power_6_std;sensor_7_avg;sensor_7_max;sensor_7_min;sensor_7_std;sensor_8_avg;sensor_8_max;sensor_8_min;sensor_8_std;sensor_9_avg;sensor_9_max;sensor_9_min;sensor_9_std;sensor_10_avg;sensor_10_max;sensor_10_min;sensor_10_std;sensor_11_avg;sensor_11_max;sensor_11_min;sensor_11_std;sensor_12_avg;sensor_12_max;sensor_12_min;sensor_12_std;sensor_13_avg;sensor_13_max;sensor_13_min;sensor_13_std;sensor_14_avg;sensor_14_max;sensor_14_min;sensor_14_std;sensor_15_avg;sensor_15_max;sensor_15_min;sensor_15_std;sensor_16_avg;sensor_16_max;sensor_16_min;sensor_16_std;power_17_avg;power_17_max;power_17_min;power_17_std;sensor_18_avg;sensor_18_max;sensor_18_min;sensor_18_std;sensor_19_avg;sensor_19_max;sensor_19_min;sensor_19_std;sensor_20_avg;sensor_20_max;sensor_20_min;sensor_20_std;sensor_21_avg;sensor_21_max;sensor_21_min;sensor_21_std;sensor_22_avg;sensor_22_max;sensor_22_min;sensor_22_std;sensor_23_avg;sensor_23_max;sensor_23_min;sensor_23_std;sensor_24_avg;sensor_24_max;sensor_24_min;sensor_24_std;sensor_25_avg;sensor_25_max;sensor_25_min;sensor_25_std;sensor_26_avg;sensor_26_max;sensor_26_min;sensor_26_std;sensor_27_avg;sensor_27_max;sensor_27_min;sensor_27_std;sensor_28_avg;sensor_28_max;sensor_28_min;sensor_28_std;sensor_29_avg;sensor_29_max;sensor_29_min;sensor_29_std;sensor_30_avg;sensor_30_max;sensor_30_min;sensor_30_std;sensor_31_avg;sensor_31_max;sensor_31_min;sensor_31_std;sensor_32_avg;sensor_32_max;sensor_32_min;sensor_32_std;sensor_33_avg;sensor_33_max;sensor_33_min;sensor_33_std;sensor_34_avg;sensor_34_max;sensor_34_min;sensor_34_std;sensor_35_avg;sensor_35_max;sensor_35_min;sensor_35_std;sensor_36_avg;sensor_36_max;sensor_36_min;sensor_36_std;sensor_37_avg;sensor_37_max;sensor_37_min;sensor_37_std;sensor_38_avg;sensor_38_max;sensor_38_min;sensor_38_std;sensor_39_avg;sensor_39_max;sensor_39_min;sensor_39_std;sensor_40_avg;sensor_40_max;sensor_40_min;sensor_40_std;sensor_41_avg;sensor_41_max;sensor_41_min;sensor_41_std;sensor_42_avg;sensor_42_max;sensor_42_min;sensor_42_std;sensor_43_avg;sensor_43_max;sensor_43_min;sensor_43_std;sensor_44_avg;sensor_44_max;sensor_44_min;sensor_44_std;sensor_45_avg;sensor_45_max;sensor_45_min;sensor_45_std;sensor_46_avg;sensor_46_max;sensor_46_min;sensor_46_std;sensor_47_avg;sensor_47_max;sensor_47_min;sensor_47_std;sensor_48_avg;sensor_48_max;sensor_48_min;sensor_48_std;sensor_49_avg;sensor_49_max;sensor_49_min;sensor_49_std;sensor_50_avg;sensor_50_max;sensor_50_min;sensor_50_std;sensor_51_avg;sensor_51_max;sensor_51_min;sensor_51_std;sensor_52_avg;sensor_52_max;sensor_52_min;sensor_52_std;sensor_53_avg;sensor_53_max;sensor_53_min;sensor_53_std;sensor_54_avg;sensor_54_max;sensor_54_min;sensor_54_std;sensor_55_avg;sensor_55_max;sensor_55_min;sensor_55_std;sensor_56_avg;sensor_56_max;sensor_56_min;sensor_56_std;sensor_57_avg;sensor_57_max;sensor_57_min;sensor_57_std;sensor_58_avg;sensor_58_max;sensor_58_min;sensor_58_std;sensor_59_avg;sensor_59_max;sensor_59_min;sensor_59_std;sensor_60_avg;sensor_60_max;sensor_60_min;sensor_60_std;sensor_61_avg;sensor_61_max;sensor_61_min;sensor_61_std;sensor_62_avg;sensor_62_max;sensor_62_min;sensor_62_std;sensor_63_avg;sensor_63_max;sensor_63_min;sensor_63_std;sensor_64_avg;sensor_64_max;sensor_64_min;sensor_64_std;sensor_65_avg;sensor_65_max;sensor_65_min;sensor_65_std;sensor_66_avg;sensor_66_max;sensor_66_min;sensor_66_std;sensor_67_avg;sensor_67_max;sensor_67_min;sensor_67_std;sensor_68_avg;sensor_68_max;sensor_68_min;sensor_68_std;sensor_69_avg;sensor_69_max;sensor_69_min;sensor_69_std;sensor_70_avg;sensor_70_max;sensor_70_min;sensor_70_std;sensor_71_avg;sensor_71_max;sen


FILE: F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To_Compare\CARE_To_Compare\Wind Farm C\datasets\56.csv
Columns: ['time_stamp;asset_id;id;train_test;status_type_id;sensor_0_avg;sensor_0_max;sensor_0_min;sensor_0_std;sensor_1_avg;sensor_1_max;sensor_1_min;sensor_1_std;power_2_avg;power_2_max;power_2_min;power_2_std;sensor_3_avg;sensor_3_max;sensor_3_min;sensor_3_std;sensor_4_avg;sensor_4_max;sensor_4_min;sensor_4_std;power_5_avg;power_5_max;power_5_min;power_5_std;power_6_avg;power_6_max;power_6_min;power_6_std;sensor_7_avg;sensor_7_max;sensor_7_min;sensor_7_std;sensor_8_avg;sensor_8_max;sensor_8_min;sensor_8_std;sensor_9_avg;sensor_9_max;sensor_9_min;sensor_9_std;sensor_10_avg;sensor_10_max;sensor_10_min;sensor_10_std;sensor_11_avg;sensor_11_max;sensor_11_min;sensor_11_std;sensor_12_avg;sensor_12_max;sensor_12_min;sensor_12_std;sensor_13_avg;sensor_13_max;sensor_13_min;sensor_13_std;sensor_14_avg;sensor_14_max;sensor_14_min;sensor_14_std;sensor_15_avg;sensor_15_max;sensor_15_min;sens

time_stamp;asset_id;id;train_test;status_type_id;sensor_0_avg;sensor_0_max;sensor_0_min;sensor_0_std;sensor_1_avg;sensor_1_max;sensor_1_min;sensor_1_std;power_2_avg;power_2_max;power_2_min;power_2_std;sensor_3_avg;sensor_3_max;sensor_3_min;sensor_3_std;sensor_4_avg;sensor_4_max;sensor_4_min;sensor_4_std;power_5_avg;power_5_max;power_5_min;power_5_std;power_6_avg;power_6_max;power_6_min;power_6_std;sensor_7_avg;sensor_7_max;sensor_7_min;sensor_7_std;sensor_8_avg;sensor_8_max;sensor_8_min;sensor_8_std;sensor_9_avg;sensor_9_max;sensor_9_min;sensor_9_std;sensor_10_avg;sensor_10_max;sensor_10_min;sensor_10_std;sensor_11_avg;sensor_11_max;sensor_11_min;sensor_11_std;sensor_12_avg;sensor_12_max;sensor_12_min;sensor_12_std;sensor_13_avg;sensor_13_max;sensor_13_min;sensor_13_std;sensor_14_avg;sensor_14_max;sensor_14_min;sensor_14_std;sensor_15_avg;sensor_15_max;sensor_15_min;sensor_15_std;sensor_16_avg;sensor_16_max;sensor_16_min;sensor_16_std;power_17_avg;power_17_max;power_17_min;power_17_std;sensor_18_avg;sensor_18_max;sensor_18_min;sensor_18_std;sensor_19_avg;sensor_19_max;sensor_19_min;sensor_19_std;sensor_20_avg;sensor_20_max;sensor_20_min;sensor_20_std;sensor_21_avg;sensor_21_max;sensor_21_min;sensor_21_std;sensor_22_avg;sensor_22_max;sensor_22_min;sensor_22_std;sensor_23_avg;sensor_23_max;sensor_23_min;sensor_23_std;sensor_24_avg;sensor_24_max;sensor_24_min;sensor_24_std;sensor_25_avg;sensor_25_max;sensor_25_min;sensor_25_std;sensor_26_avg;sensor_26_max;sensor_26_min;sensor_26_std;sensor_27_avg;sensor_27_max;sensor_27_min;sensor_27_std;sensor_28_avg;sensor_28_max;sensor_28_min;sensor_28_std;sensor_29_avg;sensor_29_max;sensor_29_min;sensor_29_std;sensor_30_avg;sensor_30_max;sensor_30_min;sensor_30_std;sensor_31_avg;sensor_31_max;sensor_31_min;sensor_31_std;sensor_32_avg;sensor_32_max;sensor_32_min;sensor_32_std;sensor_33_avg;sensor_33_max;sensor_33_min;sensor_33_std;sensor_34_avg;sensor_34_max;sensor_34_min;sensor_34_std;sensor_35_avg;sensor_35_max;sensor_35_min;sensor_35_std;sensor_36_avg;sensor_36_max;sensor_36_min;sensor_36_std;sensor_37_avg;sensor_37_max;sensor_37_min;sensor_37_std;sensor_38_avg;sensor_38_max;sensor_38_min;sensor_38_std;sensor_39_avg;sensor_39_max;sensor_39_min;sensor_39_std;sensor_40_avg;sensor_40_max;sensor_40_min;sensor_40_std;sensor_41_avg;sensor_41_max;sensor_41_min;sensor_41_std;sensor_42_avg;sensor_42_max;sensor_42_min;sensor_42_std;sensor_43_avg;sensor_43_max;sensor_43_min;sensor_43_std;sensor_44_avg;sensor_44_max;sensor_44_min;sensor_44_std;sensor_45_avg;sensor_45_max;sensor_45_min;sensor_45_std;sensor_46_avg;sensor_46_max;sensor_46_min;sensor_46_std;sensor_47_avg;sensor_47_max;sensor_47_min;sensor_47_std;sensor_48_avg;sensor_48_max;sensor_48_min;sensor_48_std;sensor_49_avg;sensor_49_max;sensor_49_min;sensor_49_std;sensor_50_avg;sensor_50_max;sensor_50_min;sensor_50_std;sensor_51_avg;sensor_51_max;sensor_51_min;sensor_51_std;sensor_52_avg;sensor_52_max;sensor_52_min;sensor_52_std;sensor_53_avg;sensor_53_max;sensor_53_min;sensor_53_std;sensor_54_avg;sensor_54_max;sensor_54_min;sensor_54_std;sensor_55_avg;sensor_55_max;sensor_55_min;sensor_55_std;sensor_56_avg;sensor_56_max;sensor_56_min;sensor_56_std;sensor_57_avg;sensor_57_max;sensor_57_min;sensor_57_std;sensor_58_avg;sensor_58_max;sensor_58_min;sensor_58_std;sensor_59_avg;sensor_59_max;sensor_59_min;sensor_59_std;sensor_60_avg;sensor_60_max;sensor_60_min;sensor_60_std;sensor_61_avg;sensor_61_max;sensor_61_min;sensor_61_std;sensor_62_avg;sensor_62_max;sensor_62_min;sensor_62_std;sensor_63_avg;sensor_63_max;sensor_63_min;sensor_63_std;sensor_64_avg;sensor_64_max;sensor_64_min;sensor_64_std;sensor_65_avg;sensor_65_max;sensor_65_min;sensor_65_std;sensor_66_avg;sensor_66_max;sensor_66_min;sensor_66_std;sensor_67_avg;sensor_67_max;sensor_67_min;sensor_67_std;sensor_68_avg;sensor_68_max;sensor_68_min;sensor_68_std;sensor_69_avg;sensor_69_max;sensor_69_min;sensor_69_std;sensor_70_avg;sensor_70_max;sensor_70_min;sensor_70_std;sensor_71_avg;sensor_71_max;sen

In [49]:
%run -i cell_3n_raw_file_adapter.py

ValueError: Invalid timestamps in F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To_Compare\CARE_To_Compare\Wind Farm A\datasets\10.csv.

In [51]:
"""Cell 3N adapter — rebuild the explicit raw sensor source from event CSVs.

Run after Cell 3M and immediately before cell_3n_train_only_preprocessing.py.
The adapter reads only events represented by modeling-eligible manifest rows,
detects the immutable source-row convention per event, and verifies timestamps.
"""

from pathlib import Path
import re

import numpy as np
import pandas as pd


if "ROW_LABEL_MANIFEST" not in globals() or "FARMS" not in globals():
    raise NameError("Run Cells 3I–3M before the Cell 3N raw-file adapter.")

required_manifest_columns_3n_adapter = {
    "farm", "event_key", "source_row_index", "timestamp_utc",
    "modeling_eligible",
}
missing_manifest_columns_3n_adapter = (
    required_manifest_columns_3n_adapter - set(ROW_LABEL_MANIFEST.columns)
)
if missing_manifest_columns_3n_adapter:
    raise ValueError(
        "ROW_LABEL_MANIFEST lacks adapter columns: "
        f"{sorted(missing_manifest_columns_3n_adapter)}"
    )

eligible_keys_3n_adapter = ROW_LABEL_MANIFEST.loc[
    ROW_LABEL_MANIFEST["modeling_eligible"].fillna(False).astype(bool),
    ["farm", "event_key", "source_row_index", "timestamp_utc"],
].copy()
eligible_keys_3n_adapter["farm"] = (
    eligible_keys_3n_adapter["farm"].astype("string").str.strip().str.upper()
)
eligible_keys_3n_adapter["event_key"] = (
    eligible_keys_3n_adapter["event_key"].astype("string").str.strip()
)
eligible_keys_3n_adapter["source_row_index"] = pd.to_numeric(
    eligible_keys_3n_adapter["source_row_index"], errors="raise"
).astype("int64")
eligible_keys_3n_adapter["timestamp_utc"] = pd.to_datetime(
    eligible_keys_3n_adapter["timestamp_utc"], errors="raise", utc=True
)

if eligible_keys_3n_adapter[["event_key", "source_row_index"]].duplicated().any():
    raise ValueError("Eligible manifest source keys are not unique.")


def event_id_from_key_3n_adapter(event_key_3n_adapter):
    match_3n_adapter = re.fullmatch(
        r"farm_([abc])_event_(\d+)", str(event_key_3n_adapter), flags=re.I
    )
    if match_3n_adapter is None:
        raise ValueError(f"Unsupported event key: {event_key_3n_adapter!r}")
    return match_3n_adapter.group(1).upper(), int(match_3n_adapter.group(2))


def locate_event_file_3n_adapter(farm_3n_adapter, event_id_3n_adapter):
    farm_root_3n_adapter = Path(FARMS[farm_3n_adapter])
    direct_candidates_3n_adapter = [
        farm_root_3n_adapter / "datasets" / f"{event_id_3n_adapter}.csv",
        farm_root_3n_adapter / "dataset" / f"{event_id_3n_adapter}.csv",
        farm_root_3n_adapter / f"{event_id_3n_adapter}.csv",
    ]
    existing_3n_adapter = [
        path_3n_adapter
        for path_3n_adapter in direct_candidates_3n_adapter
        if path_3n_adapter.is_file()
    ]
    if len(existing_3n_adapter) != 1:
        raise FileNotFoundError(
            f"Expected exactly one CSV for farm {farm_3n_adapter}, event "
            f"{event_id_3n_adapter}; found {existing_3n_adapter}."
        )
    return existing_3n_adapter[0]


def timestamp_match_count_3n_adapter(
    raw_frame_3n_adapter,
    manifest_event_3n_adapter,
    source_indices_3n_adapter,
):
    candidate_3n_adapter = pd.DataFrame({
        "source_row_index": pd.Series(source_indices_3n_adapter, dtype="int64"),
        "raw_timestamp_utc": raw_frame_3n_adapter["_timestamp_utc_3n"].to_numpy(),
    })
    if candidate_3n_adapter["source_row_index"].duplicated().any():
        return -1
    check_3n_adapter = manifest_event_3n_adapter.merge(
        candidate_3n_adapter,
        on="source_row_index",
        how="left",
        validate="one_to_one",
    )
    if check_3n_adapter["raw_timestamp_utc"].isna().any():
        return -1
    return int(
        check_3n_adapter["timestamp_utc"].eq(
            check_3n_adapter["raw_timestamp_utc"]
        ).sum()
    )


def parse_raw_timestamps_3n_adapter(values_3n_adapter):
    """Parse heterogeneous CSV timestamps without weakening exact matching.

    pandas 2.x may infer one format from the first value and coerce otherwise
    valid rows that use another representation.  ``format='mixed'`` performs
    element-wise inference.  The compatibility fallback preserves support for
    older pandas versions that do not implement that option.
    """
    cleaned_3n_adapter = values_3n_adapter.astype("string").str.strip()
    cleaned_3n_adapter = cleaned_3n_adapter.mask(
        cleaned_3n_adapter.eq("") | cleaned_3n_adapter.str.lower().isin({
            "nan", "nat", "none", "null",
        })
    )
    try:
        parsed_3n_adapter = pd.to_datetime(
            cleaned_3n_adapter,
            errors="coerce",
            utc=True,
            format="mixed",
        )
    except (TypeError, ValueError):
        parsed_3n_adapter = pd.to_datetime(
            cleaned_3n_adapter,
            errors="coerce",
            utc=True,
        )
    return parsed_3n_adapter


pieces_3n_adapter = []
audit_records_3n_adapter = []
sensor_schema_3n_adapter = None

for event_key_3n_adapter, manifest_event_3n_adapter in (
    eligible_keys_3n_adapter.groupby("event_key", sort=True, observed=True)
):
    farm_from_key_3n_adapter, event_id_3n_adapter = (
        event_id_from_key_3n_adapter(event_key_3n_adapter)
    )
    manifest_farms_3n_adapter = manifest_event_3n_adapter["farm"].unique().tolist()
    if manifest_farms_3n_adapter != [farm_from_key_3n_adapter]:
        raise ValueError(
            f"Farm mismatch for {event_key_3n_adapter}: "
            f"{manifest_farms_3n_adapter}."
        )

    event_path_3n_adapter = locate_event_file_3n_adapter(
        farm_from_key_3n_adapter, event_id_3n_adapter
    )
    raw_event_3n_adapter = pd.read_csv(
        event_path_3n_adapter,
        sep=";",
        low_memory=False,
    )
    if raw_event_3n_adapter.shape[1] <= 1:
        raise ValueError(
            f"Delimiter parsing failed for {event_path_3n_adapter}."
        )
    if "time_stamp" not in raw_event_3n_adapter.columns:
        raise ValueError(
            f"{event_path_3n_adapter} lacks the time_stamp column."
        )
    raw_event_3n_adapter["_timestamp_utc_3n"] = (
        parse_raw_timestamps_3n_adapter(raw_event_3n_adapter["time_stamp"])
    )
    invalid_file_timestamps_3n_adapter = int(
        raw_event_3n_adapter["_timestamp_utc_3n"].isna().sum()
    )

    sensor_columns_3n_adapter = [
        column_3n_adapter
        for column_3n_adapter in raw_event_3n_adapter.columns
        if re.search(r"_(avg|max|min|std)$", str(column_3n_adapter), flags=re.I)
    ]
    if not sensor_columns_3n_adapter:
        raise ValueError(f"No measurement columns in {event_path_3n_adapter}.")
    if sensor_schema_3n_adapter is None:
        sensor_schema_3n_adapter = sensor_columns_3n_adapter
    elif sensor_columns_3n_adapter != sensor_schema_3n_adapter:
        raise ValueError(
            f"Measurement schema mismatch in {event_path_3n_adapter}."
        )

    index_candidates_3n_adapter = {
        "zero_based_file_row": np.arange(len(raw_event_3n_adapter), dtype=np.int64),
        "one_based_file_row": np.arange(1, len(raw_event_3n_adapter) + 1, dtype=np.int64),
    }
    if "id" in raw_event_3n_adapter.columns:
        id_values_3n_adapter = pd.to_numeric(
            raw_event_3n_adapter["id"], errors="coerce"
        )
        if (
            id_values_3n_adapter.notna().all()
            and np.isclose(id_values_3n_adapter, np.floor(id_values_3n_adapter)).all()
        ):
            index_candidates_3n_adapter["id_column"] = (
                id_values_3n_adapter.astype("int64").to_numpy()
            )

    match_counts_3n_adapter = {
        name_3n_adapter: timestamp_match_count_3n_adapter(
            raw_event_3n_adapter,
            manifest_event_3n_adapter,
            values_3n_adapter,
        )
        for name_3n_adapter, values_3n_adapter in index_candidates_3n_adapter.items()
    }
    required_matches_3n_adapter = len(manifest_event_3n_adapter)
    exact_modes_3n_adapter = [
        name_3n_adapter
        for name_3n_adapter, count_3n_adapter in match_counts_3n_adapter.items()
        if count_3n_adapter == required_matches_3n_adapter
    ]
    if not exact_modes_3n_adapter:
        raise ValueError(
            f"No row-identity convention exactly matches timestamps for "
            f"{event_key_3n_adapter}: {match_counts_3n_adapter}."
        )

    # Equivalent conventions (commonly id == zero-based row) are harmless.
    chosen_mode_3n_adapter = exact_modes_3n_adapter[0]
    chosen_indices_3n_adapter = index_candidates_3n_adapter[
        chosen_mode_3n_adapter
    ]
    for other_mode_3n_adapter in exact_modes_3n_adapter[1:]:
        if not np.array_equal(
            chosen_indices_3n_adapter,
            index_candidates_3n_adapter[other_mode_3n_adapter],
        ):
            raise ValueError(
                f"Ambiguous non-equivalent row identities for "
                f"{event_key_3n_adapter}: {exact_modes_3n_adapter}."
            )

    keyed_event_3n_adapter = raw_event_3n_adapter.loc[
        :, sensor_schema_3n_adapter
    ].copy()
    keyed_event_3n_adapter.insert(0, "source_row_index", chosen_indices_3n_adapter)
    keyed_event_3n_adapter.insert(0, "event_key", str(event_key_3n_adapter))
    required_indices_3n_adapter = set(
        manifest_event_3n_adapter["source_row_index"].astype(int)
    )
    keyed_event_3n_adapter = keyed_event_3n_adapter.loc[
        keyed_event_3n_adapter["source_row_index"].isin(required_indices_3n_adapter)
    ].copy()
    if len(keyed_event_3n_adapter) != required_matches_3n_adapter:
        raise ValueError(f"Row conservation failed for {event_key_3n_adapter}.")
    pieces_3n_adapter.append(keyed_event_3n_adapter)
    audit_records_3n_adapter.append({
        "event_key": str(event_key_3n_adapter),
        "event_path": str(event_path_3n_adapter),
        "file_rows": int(len(raw_event_3n_adapter)),
        "eligible_rows": int(required_matches_3n_adapter),
        "row_identity_mode": chosen_mode_3n_adapter,
        "timestamp_matches": int(required_matches_3n_adapter),
        "invalid_file_timestamp_rows": invalid_file_timestamps_3n_adapter,
        "measurement_columns": int(len(sensor_schema_3n_adapter)),
    })

FEATURE_SOURCE_3N = pd.concat(
    pieces_3n_adapter,
    axis=0,
    ignore_index=True,
    sort=False,
)
FEATURE_COLUMNS_3N = list(sensor_schema_3n_adapter)
RAW_FILE_ADAPTER_AUDIT_3N = pd.DataFrame(audit_records_3n_adapter)

if len(FEATURE_SOURCE_3N) != len(eligible_keys_3n_adapter):
    raise ValueError("The adapter did not conserve all eligible rows.")
if FEATURE_SOURCE_3N[["event_key", "source_row_index"]].duplicated().any():
    raise ValueError("The adapter produced duplicate immutable source keys.")
if not pd.MultiIndex.from_frame(eligible_keys_3n_adapter[[
    "event_key", "source_row_index"
]]).isin(pd.MultiIndex.from_frame(FEATURE_SOURCE_3N[[
    "event_key", "source_row_index"
]])).all():
    raise ValueError("The adapter failed to cover every eligible source key.")

print("Cell 3N raw-file adapter completed successfully.")
print("Events loaded:", RAW_FILE_ADAPTER_AUDIT_3N["event_key"].nunique())
print("Eligible rows reconstructed:", len(FEATURE_SOURCE_3N))
print("Physical measurement columns:", len(FEATURE_COLUMNS_3N))
print("Delimiter: semicolon")
print("All manifest-to-file timestamps matched exactly.")


ValueError: Measurement schema mismatch in F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To_Compare\CARE_To_Compare\Wind Farm B\datasets\19.csv.